In [ ]:
# ==============================================================================
# DL-Docking Studio v12.0
# Pipeline de docking molecular com rescoring 3D-CNN (GNINA 1.3.2) | Google Colab (Python 3.12) | GPU NVIDIA A100
#
# Etapas do pipeline:
#   1. Alvo: RCSB PDB (ID), upload de PDB local ou AlphaFold DB (UniProt; model_v4 com contingência para model_v3). Todas as
#      origens passam pelo mesmo mapeamento de cadeias, ligantes e cofatores (_scan_pdb_text). Em modelos AlphaFold, o pLDDT
#      (fator B) do bolsão (resíduos a < 10 Å do centro da grid box) é reportado com classe de confiabilidade.
#   2. Receptor: PDBFixer/OpenMM em pH 7,4, com metais catalíticos preservados. Cofatores orgânicos (HEM, FAD, NAD...) e
#      complexos metálicos (SF4, CLA, B12...) não passam pelo PDBFixer: ficam retidos e são colados ao receptor protonado;
#      apenas o metal do cofator entra como "íon substituto" (para tratar corretamente a His/Cys axial) e é removido depois.
#   3. Ligante: isolamento do fragmento principal e protonação inicial (OpenBabel), ajuste iônico por subestrutura (RDKit),
#      ETKDGv3, minimização MMFF94, cargas de Gasteiger e contagem de torções (TORSDOF). Modo 1: triagem prospectiva;
#      modo 2: redocking cristalográfico (preserva o 3D nativo).
#   4. Docking: GNINA com parâmetros centralizados em GNINA_DOCK_PARAMS (fonte única do comando e do relatório), semente
#      fixa (RANDOM_SEED), grid box automática (ligante nativo + AUTOBOX_ADD por lado) ou manual, refino --minimize e
#      rescoring 3D-CNN (CNNscore, CNNaffinity, CNNvariance). A CNNvariance é lida da tag do SDF ou do log de refino,
#      casando por valores (tolerância 1e-4, Vina como desempate) e nunca por posição; pose ambígua vira "n/d".
#   5. Validação: PB_Valid usa SOMENTE os testes oficiais do modo "dock" do PoseBusters. As colunas diagnósticas
#      most_extreme_clash_* e not_too_far_away_* são exportadas à parte e não entram no veredito. O RMSD simétrico (spyrmsd,
#      sem superposição prévia) só é calculado quando a composição elementar do ligante coincide com a do nativo.
#   6. Interações: ProLIF restrito ao bolsão de 12 Å do ligante (contingência: esfera de 16 Å na grid box), com metais como
#      átomos isolados (MetalAcceptor) em ramos de construção do receptor com fallback. Se o ProLIF falhar, contatos
#      geométricos (H 3,5 Å; π 4,5 Å; hidrofóbico 4,0 Å; metal 2,8 Å) preenchem tabelas, figuras e relatório. A menor
#      distância O/N/S(ligante)-metal por pose é calculada independentemente do ProLIF.
#   7. Benchmark contra o ligante nativo (controle positivo): ΔVina, ΔpK, ΔLE e veredito multiparamétrico (PoseBusters,
#      consenso Vina/CNN e eficiência de ligante). Camada somente-leitura: uma falha nunca afeta as saídas já geradas.
#   8. Fila multi-ligante (PubChem, upload múltiplo, SDF multi-molécula), LEADERBOARD em Excel, gráficos de publicação
#      (PNG 600 DPI, SVG e PDF, em PT e EN), parecer científico automático e relatório acadêmico (ABNT NBR 10520/6023;
#      LaTeX abntex2 + biblatex-abnt) em .docx, .pdf, .md, .tex, .bib e .txt. Falhas documentais nunca interrompem o docking.
#   9. Dinâmica molecular: script turnkey run_md_openmm.py (OpenMM) da pose líder, em 08_dinamica_molecular/.
#
# Robustez: PDBFixer/OpenMM, PoseBusters e ProLIF rodam em processo filho vigiado (run_isolated: spawn + cloudpickle,
# faulthandler, teto de RAM); _free_memory() após cada ligante; travas de tamanho de caixa manual, de cadeia e de átomos
# ausentes; fila do Gradio com concorrência 1 (o motor guarda estado global; uma cópia do notebook por usuário).
# ==============================================================================
import os
import re
import sys
import glob
import shutil
import hashlib
import subprocess
import urllib.request
import urllib.error
import urllib.parse
import json
import time
import zipfile
import html as _html
import warnings
import importlib
import importlib.util
import importlib.metadata as importlib_metadata
from collections import Counter, OrderedDict
from pathlib import Path
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------
# Constantes globais
# ------------------------------------------------------------------------------
PIPELINE_VERSION = "12.0"
PLATFORM_NAME = f"DL-Docking Studio v{PIPELINE_VERSION}"
VERSION_TAG = f"v{PIPELINE_VERSION}"          # sufixo de versão nos nomes padrão dos arquivos exportados (planilha, relatório, .zip)
print(f"🧬 {PLATFORM_NAME}")
MAX_MANUAL_BOX_EDGE = 60.0        # Å: acima disso a amostragem Monte Carlo deixa de ser significativa
MAX_CHAIN_RESIDUES = 3000         # trava de sanidade espacial antes do PDBFixer/GNINA
WARN_CHAIN_RESIDUES = 1200
PDBFIXER_MAX_MISSING_ATOMS = 500   # trava de RAM: acima disso a reconstrução de átomos pesados é ignorada
SOLVENTS_AND_IONS = {"HOH", "DOD", "WAT", "NA", "CL", "K", "SO4", "PO4", "GOL", "EDO", "DMS", "ACT", "HOL"}
# Cofatores metálicos catalíticos reconhecidos (nome de RESÍDUO, colunas 18-20 do PDB).
CATALYTIC_METALS = {"CU", "CU1", "CU2", "ZN", "MG", "MN", "FE", "FE2", "CA", "CO", "NI"}
METAL_SYMBOL = {"CU": "Cu", "CU1": "Cu", "CU2": "Cu", "ZN": "Zn", "MG": "Mg", "MN": "Mn",
                "FE": "Fe", "FE2": "Fe", "CA": "Ca", "CO": "Co", "NI": "Ni"}
METAL_SYMBOLS_SET = set(METAL_SYMBOL.values())
# Cofatores ORGÂNICOS: sem templates no OpenMM/PDBFixer. São retidos à parte e colados ao receptor já protonado.
ORGANIC_COFACTORS = {"HEM", "FAD", "FMN", "NAD", "NDP", "NAP", "SAM", "PLP"}
# Elementos metálicos reconhecidos DENTRO de cofatores (ex.: FE do HEM, MG da clorofila, CO da B12).
METAL_ELEMENTS = {"FE", "CU", "ZN", "MG", "MN", "CO", "NI", "CA"}
# Sanidade do ligante (heurísticas conservadoras)
ORGANIC_LIGAND_ELEMENTS = {"H", "C", "N", "O", "F", "P", "S", "CL", "BR", "I"}
MAX_LIGAND_HEAVY_ATOMS = 150      # acima: recusa (tempo/RAM)
WARN_LIGAND_HEAVY_ATOMS = 70
WARN_LIGAND_TORSIONS = 20
METAL_COORD_CUTOFF = 2.8          # Å: N/O/S dentro deste raio de um metal => contato de coordenação
HIS_NAMES = {"HIS", "HID", "HIE", "HIP"}
RANDOM_SEED = 42
AUTOBOX_ADD = 4.0                 # Å por lado (regra do autobox do GNINA, McNutt et al. 2021)
GNINA_DOCK_PARAMS = {"num_modes": 10, "min_rmsd_filter": 0.5, "num_mc_saved": 50, "cnn_scoring": "rescore",
                     "pose_sort_order": "CNNscore"}   # única fonte: comando do docking e relatório acadêmico
LIG_N_CONFORMERS = 20             # conformeros ETKDGv3 do modo 1 (usado no preparo e no relatório)

BOX_AUTO = "Automática: caixa do ligante nativo (+4 Å por lado)"
BOX_MANUAL = "Manual: centro e tamanho em Å"

INPUT_RCSB = "Baixar do RCSB (PDB ID)"
INPUT_LOCAL = "Upload de PDB Local"
INPUT_AFDB = "Baixar do AlphaFold DB (UniProt ID)"
LOCAL_ID_PREFIX = "LOCAL_"
MAX_LOCAL_PDB_MB = 60                                   # trava de sanidade do upload
LOCAL_TITLE_PLACEHOLDER = "Modelo PDB Local customizado"
LOCAL_ORG_PLACEHOLDER = "Organismo não especificado"

PREP_MODES = [
    "1. Triagem prospectiva ➔ pH 7.4 + ETKDGv3 + MMFF94 + Gasteiger + torções",
    "2. Redocking cristalográfico ➔ preserva o 3D exato + Gasteiger + torções",
]


def _gnina_dock_timeout(exhaustiveness) -> int:
    return int(max(900, 900 + (int(exhaustiveness) - 32) * 15))

# ==============================================================================
# 1. INICIALIZAÇÃO DO AMBIENTE (COLAB: PYTHON 3.12 + GPU A100)
# ==============================================================================
def _sh(cmd, **kw):
    return subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, **kw)


# Requisitos pip: (distribuição, prefixo de versão exigido | None, especificação para o pip)
PIP_REQUIREMENTS = [
    ("rdkit", None, "rdkit"), ("spyrmsd", None, "spyrmsd"), ("posebusters", None, "posebusters"),
    ("prolif", None, "prolif"), ("MDAnalysis", None, "MDAnalysis"), ("biopython", None, "biopython"),
    ("gradio", "5.", "gradio>=5.0,<6.0"), ("openpyxl", None, "openpyxl"), ("openmm", None, "openmm"), ("matplotlib", None, "matplotlib"), ("seaborn", None, "seaborn"), ("adjustText", None, "adjustText"),
    ("nvidia-nvtx-cu11", None, "nvidia-nvtx-cu11"), ("nvidia-nvtx-cu12", None, "nvidia-nvtx-cu12"),
    ("docx", None, "python-docx"), ("weasyprint", None, "weasyprint"),            # relatório acadêmico em .docx e .pdf
]
OPTIONAL_PIP = {"nvidia-nvtx-cu11", "nvidia-nvtx-cu12"}   # tentados no máximo uma vez por VM
SOFT_PIP = {"docx", "weasyprint"}                  # relatório .docx/.pdf: um pip por pacote; falha não impede o docking e é retentada


_DIST_ALIAS = {"docx": "python-docx"}   # módulo `docx` => distribuição `python-docx` (a distribuição "docx" do PyPI é outro pacote, obsoleto)


def _dist_ok(dist: str, prefix=None) -> bool:
    try:
        v = importlib_metadata.version(_DIST_ALIAS.get(dist, dist))
    except Exception:
        return False
    return True if prefix is None else str(v).startswith(prefix)


def setup_environment_with_progress():
    """Prepara o ambiente. Etapas já satisfeitas (binários, pacotes, GNINA validado) são puladas, de modo que
    re-executar a célula na mesma VM do Colab não baixa nem instala nada. O LD_LIBRARY_PATH do GNINA é
    sempre recomputado (é por processo e define se o GNINA enxerga a GPU)."""
    t_start = time.time()
    steps = [
        ("Verificando resíduos de instalações legadas", "cleanup"),
        ("Verificando utilitários de sistema (OpenBabel, Patchelf e Pango/fontes do PDF)", "apt"),
        ("Verificando bibliotecas quimioinformáticas, OpenMM e PDBFixer", "pip"),
        ("Configurando CUDA 12 e NVTX para GPU NVIDIA A100", "cuda"),
        ("Verificando binário oficial GNINA 1.3.2 para A100", "gnina"),
    ]
    pbar = tqdm(total=len(steps), desc="⚙️ Inicializando Ambiente Colab", bar_format="{l_bar}{bar:30}{r_bar}")
    gnina_env = os.environ.copy()
    gnina_path = "/usr/local/bin/gnina"
    gnina_meta_path = "/usr/local/bin/gnina.setup.json"   # {"size", "sha256"} do binário já validado
    gnina_version_str = "unknown"
    gnina_sha256 = "n/d"
    cached = []

    for desc, step_code in steps:
        pbar.set_description(f"⚙️ {desc}")

        if step_code == "cleanup":
            # Só remove resíduos de instalações antigas; NÃO apaga mais o binário válido do GNINA.
            removed = False
            for p in ["/usr/local/lib/gnina_compat", "/etc/ld.so.conf.d/gnina_cuda12.conf"]:
                if os.path.isdir(p):
                    shutil.rmtree(p, ignore_errors=True)
                    removed = True
                elif os.path.exists(p):
                    os.remove(p)
                    removed = True
            if removed:
                _sh(["ldconfig"])
            else:
                cached.append("legado")

        elif step_code == "apt":
            missing_pkgs = [pkg for pkg, exe in (("openbabel", "obabel"), ("patchelf", "patchelf"))
                            if shutil.which(exe) is None]
            try:                                  # WeasyPrint (PDF do relatório) usa Pango; fontes Liberation = métricas do Arial
                import ctypes.util
                if ctypes.util.find_library("pango-1.0") is None:
                    missing_pkgs += ["libpango-1.0-0", "libpangoft2-1.0-0"]
                if not glob.glob("/usr/share/fonts/**/LiberationSans-Regular.ttf", recursive=True):
                    missing_pkgs.append("fonts-liberation")
            except Exception:
                pass
            if missing_pkgs:
                _sh(["apt-get", "-qq", "update"])
                _sh(["apt-get", "-qq", "install", "-y"] + missing_pkgs)
            else:
                cached.append("apt")

        elif step_code == "pip":
            to_install, soft_install = [], []
            for dist, prefix, spec in PIP_REQUIREMENTS:
                if _dist_ok(dist, prefix):
                    continue
                if dist in SOFT_PIP:
                    soft_install.append((dist, spec))
                    continue
                if dist in OPTIONAL_PIP and os.path.exists(f"/tmp/.dd_pip_tried_{dist}"):
                    continue                      # opcional já tentado nesta VM: não repete
                to_install.append(spec)
            if to_install or soft_install:
                if to_install:
                    _sh([sys.executable, "-m", "pip", "install", "-q"] + to_install)
                for dist, spec in soft_install:   # .docx/.pdf do relatório: se o pip falhar, o docking segue e o relatório avisa
                    _sh([sys.executable, "-m", "pip", "install", "-q", spec])
                importlib.invalidate_caches()
                for dist, _, spec in PIP_REQUIREMENTS:
                    if dist in OPTIONAL_PIP and spec in to_install:
                        try:
                            Path(f"/tmp/.dd_pip_tried_{dist}").touch()
                        except Exception:
                            pass
            else:
                cached.append("pip")
            # PDBFixer: tenta PyPI e, se necessário, o repositório oficial (somente se ausente).
            if importlib.util.find_spec("pdbfixer") is None:
                _sh([sys.executable, "-m", "pip", "install", "-q", "pdbfixer"])
                importlib.invalidate_caches()
                if importlib.util.find_spec("pdbfixer") is None:
                    _sh([sys.executable, "-m", "pip", "install", "-q",
                         "git+https://github.com/openmm/pdbfixer.git"])
                    importlib.invalidate_caches()
            try:
                importlib.import_module("pdbfixer")
                importlib.import_module("openmm")
            except Exception as e:
                raise RuntimeError(
                    "Falha ao instalar OpenMM/PDBFixer (necessários para o preparo do receptor). "
                    f"Detalhe: {e}"
                )

        elif step_code == "cuda":
            import torch
            site_pkgs = os.path.dirname(os.path.dirname(torch.__file__))
            nvtx_candidates = glob.glob(f"{site_pkgs}/nvidia/nvtx*/lib/libnvToolsExt.so.1*")
            nvtx_dest = "/usr/lib/x86_64-linux-gnu/libnvToolsExt.so.1"
            if nvtx_candidates and not os.path.exists(nvtx_dest):
                shutil.copy(nvtx_candidates[0], nvtx_dest)
                _sh(["ldconfig"])
            lib_dirs = [
                "/usr/lib/x86_64-linux-gnu",
                "/usr/local/cuda/lib64",
                "/usr/local/cuda/targets/x86_64-linux/lib",
                os.path.join(os.path.dirname(torch.__file__), "lib"),
            ]
            for p in glob.glob(f"{site_pkgs}/nvidia/*/lib"):
                if os.path.isdir(p) and p not in lib_dirs:
                    lib_dirs.append(p)
            # Preserva o LD_LIBRARY_PATH herdado e o diretório do driver NVIDIA do Colab (libcuda.so.1):
            # sobrescrevê-lo faz o GNINA reportar "No GPU detected" e cair para CPU.
            _driver = [d for d in ("/usr/lib64-nvidia", "/usr/local/nvidia/lib64") if os.path.isdir(d)]
            _prev = [d for d in os.environ.get("LD_LIBRARY_PATH", "").split(":") if d]
            gnina_env["LD_LIBRARY_PATH"] = ":".join(dict.fromkeys(
                _driver + [d for d in lib_dirs if os.path.isdir(d)] + _prev))
            gnina_env["CUDA_VISIBLE_DEVICES"] = "0"

        elif step_code == "gnina":
            def _gnina_version():
                if not (os.path.isfile(gnina_path) and os.access(gnina_path, os.X_OK)):
                    return None
                pr = subprocess.run([gnina_path, "--version"], capture_output=True, text=True, env=gnina_env)
                return pr if pr.returncode == 0 else None

            def _sha256_file(path):
                h = hashlib.sha256()
                with open(path, "rb") as fh:
                    for chunk in iter(lambda: fh.read(1 << 20), b""):
                        h.update(chunk)
                return h.hexdigest()

            def _write_meta():
                try:
                    with open(gnina_meta_path, "w") as fh:
                        json.dump({"size": os.path.getsize(gnina_path), "sha256": gnina_sha256}, fh)
                except Exception:
                    pass

            proc = _gnina_version()
            if proc is not None:
                # Binário já presente e funcional: reaproveita (sem download). O SHA-256 vem do arquivo auxiliar
                # quando o tamanho confere; caso contrário é recalculado uma única vez.
                try:
                    meta = json.load(open(gnina_meta_path))
                    if meta.get("size") == os.path.getsize(gnina_path) and meta.get("sha256"):
                        gnina_sha256 = meta["sha256"]
                except Exception:
                    pass
                if gnina_sha256 == "n/d":
                    gnina_sha256 = _sha256_file(gnina_path)
                    _write_meta()
                cached.append("gnina")
            else:
                if os.path.exists(gnina_path):
                    os.remove(gnina_path)
                url_132 = "https://github.com/gnina/gnina/releases/download/v1.3.2/gnina.1.3.2"
                subprocess.run(["wget", "-q", "-O", gnina_path, url_132], check=True)
                os.chmod(gnina_path, 0o755)
                gnina_sha256 = _sha256_file(gnina_path)
                proc = _gnina_version()
                if proc is None:
                    _sh(["patchelf", "--remove-needed", "libnvToolsExt.so.1", gnina_path])
                    proc = _gnina_version()
                if proc is None:
                    raise RuntimeError("Falha ao validar GNINA v1.3.2 na GPU A100 (binário baixado não executa).")
                _write_meta()
            gnina_version_str = proc.stdout.strip().splitlines()[0] if proc.stdout else "v1.3.2"

        pbar.update(1)

    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    dt = time.time() - t_start
    pbar.set_description(f"✅ Ambiente pronto em {dt:.1f}s ({gnina_version_str} | {gpu_name} | Python {sys.version.split()[0]})")
    pbar.close()
    if cached:
        print(f"⚡ Etapas reaproveitadas (sem download/instalação): {', '.join(cached)}")
    return gnina_path, gnina_env, gnina_version_str, gnina_sha256


GNINA_BIN, GNINA_ENV, GNINA_VERSION, GNINA_SHA256 = setup_environment_with_progress()


# ==============================================================================
# 2. IMPORTS CIENTÍFICOS
# ==============================================================================
import gradio as gr
import pandas as pd
import numpy as np
try:
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns
except Exception as e_viz:
    plt = sns = None
    print(f"⚠️ matplotlib/seaborn indisponíveis ({type(e_viz).__name__}): os gráficos de publicação serão omitidos.")
try:
    from adjustText import adjust_text
except Exception:
    adjust_text = None
    print("⚠️ adjustText indisponível: rótulos do gráfico de LE ficam nas posições iniciais.")
import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
import spyrmsd
from spyrmsd import rmsd as spy_rmsd
from spyrmsd import molecule as spy_mol
import posebusters
from posebusters import PoseBusters
import MDAnalysis as mda
import prolif as plf
import gc
import torch


def _rss_mb() -> float:
    """RAM residente do processo do kernel (MB), lida de /proc (sem depender de psutil)."""
    try:
        with open("/proc/self/status") as fh:
            for line in fh:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 1024.0
    except Exception:
        pass
    return -1.0


def _free_memory():
    """Coleta de lixo agressiva: Python, cache CUDA e devolução das páginas livres do malloc ao SO."""
    gc.collect()
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass
    gc.collect()


# ------------------------------------------------------------------------------
# Isolamento de etapas pesadas em processo filho (anti-queda do kernel)
# ------------------------------------------------------------------------------
class IsolatedMemoryError(MemoryError):
    """O processo filho excedeu o limite de RAM, morreu (SIGKILL/SIGSEGV) ou estourou o tempo."""


def _meminfo_mb(key: str) -> float:
    try:
        with open("/proc/meminfo") as fh:
            for line in fh:
                if line.startswith(key + ":"):
                    return int(line.split()[1]) / 1024.0
    except Exception:
        pass
    return -1.0


def _proc_rss_mb(pid: int) -> float:
    try:
        with open(f"/proc/{pid}/status") as fh:
            for line in fh:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 1024.0
    except Exception:
        pass
    return 0.0


def _default_rss_cap_mb() -> float:
    """Teto de RAM do filho: 60% da RAM total, sem passar da RAM disponível menos 1,5 GB (mín. 1,5 GB)."""
    total, avail = _meminfo_mb("MemTotal"), _meminfo_mb("MemAvailable")
    if total <= 0 or avail <= 0:
        return 4000.0
    return max(1500.0, min(0.60 * total, avail - 1500.0))


def _mem_log(tag: str):
    print(f"[memória] {tag}: kernel {_rss_mb():.0f} MB | livre no sistema {_meminfo_mb('MemAvailable'):.0f} MB")


class _IsoBootError(Exception):
    """O isolamento por 'spawn' não pôde iniciar (cloudpickle ausente, objeto não serializável ou filho sem conseguir carregar)."""


_ISO_CHILD_SRC = '''
def _dd_child(conn, payload, crash):
    _fh = None
    try:
        import faulthandler
        _fh = open(crash, "w")
        faulthandler.enable(file=_fh, all_threads=True)
    except Exception:
        pass
    try:
        import cloudpickle
        with open(payload, "rb") as fh:
            func, args = cloudpickle.load(fh)
    except BaseException as e:
        try:
            conn.send(("boot", f"{type(e).__name__}: {e}"))
        finally:
            conn.close()
        return
    try:
        conn.send(("ok", func(*args)))
    except BaseException as e:
        try:
            conn.send(("err", f"{type(e).__name__}: {e}"))
        except Exception:
            pass
    finally:
        conn.close()
_dd_child(conn, payload, crash)
'''


def _run_isolated_impl(func, args, max_rss_mb, timeout_s, label, method):
    import multiprocessing
    import tempfile
    cap = float(max_rss_mb) if max_rss_mb else _default_rss_cap_mb()
    ctx = multiprocessing.get_context(method)
    stamp = f"{os.getpid()}_{int(time.time() * 1000)}"
    crash_path = Path(tempfile.gettempdir()) / f"dd_crash_{stamp}.log"
    payload_path = Path(tempfile.gettempdir()) / f"dd_payload_{stamp}.pkl"
    parent_conn, child_conn = ctx.Pipe(duplex=False)

    def _target(conn):                          # só usado no 'fork'
        _fh = None
        try:
            import faulthandler
            _fh = open(crash_path, "w")
            faulthandler.enable(file=_fh, all_threads=True)
        except Exception:
            pass
        try:
            conn.send(("ok", func(*args)))
        except BaseException as e:      # noqa: BLE001 (o filho precisa relatar qualquer falha)
            try:
                conn.send(("err", f"{type(e).__name__}: {e}"))
            except Exception:
                pass
        finally:
            conn.close()

    try:
        if method == "spawn":
            # 'spawn' não herda o contexto CUDA do PyTorch. Como func/_target vivem no __main__ do notebook (o filho não
            # os importa), a função e os argumentos viajam por valor via cloudpickle e o alvo do processo é o exec embutido.
            if getattr(sys.modules.get("__main__"), "__file__", None):
                raise _IsoBootError("__main__ é um arquivo .py (o 'spawn' o reexecutaria)")
            try:
                import cloudpickle
                with open(payload_path, "wb") as fh:
                    cloudpickle.dump((func, tuple(args)), fh)
            except Exception as e:
                raise _IsoBootError(f"{type(e).__name__}: {str(e)[:160]}")
            proc = ctx.Process(target=exec, args=(_ISO_CHILD_SRC, {"conn": child_conn, "payload": str(payload_path),
                                                                   "crash": str(crash_path)}), daemon=True)
        else:
            proc = ctx.Process(target=_target, args=(child_conn,), daemon=True)
        try:
            proc.start()
        except Exception as e:
            if method == "spawn":
                raise _IsoBootError(f"{type(e).__name__}: {str(e)[:160]}")
            raise
    except BaseException:
        parent_conn.close()
        child_conn.close()
        for _p in (payload_path, crash_path):
            try:
                _p.unlink()
            except Exception:
                pass
        raise
    child_conn.close()
    t0, peak, msg, status, payload = time.time(), 0.0, None, None, None
    try:
        while True:
            if parent_conn.poll(0.4):
                break                                   # resultado disponível ou filho encerrado (EOF)
            rss = _proc_rss_mb(proc.pid)
            peak = max(peak, rss)
            if rss > cap:
                msg = f"{label}: processo isolado passou de {cap:.0f} MB de RAM (pico {peak:.0f} MB) e foi encerrado; o kernel foi preservado."
                break
            if time.time() - t0 > timeout_s:
                msg = f"{label}: tempo limite de {timeout_s:.0f} s excedido; processo isolado encerrado."
                break
            if not proc.is_alive() and not parent_conn.poll(0.2):
                break
        if msg is None:
            try:
                status, payload = parent_conn.recv()
            except (EOFError, OSError):
                proc.join(2)
                tail = ""
                try:
                    if crash_path.exists():
                        _txt = crash_path.read_text(errors="ignore")
                        _cut = _txt.find("Extension modules:")      # o Python imprime a pilha ANTES da lista de módulos
                        tail = (_txt[:_cut] if _cut > 0 else _txt).strip()[:3500]
                        if _cut > 0:
                            tail += "\n(lista de módulos de extensão omitida)"
                except Exception:
                    pass
                msg = (f"{label}: o processo isolado terminou sem resultado (código {proc.exitcode}; pico {peak:.0f} MB). "
                       "Código -9 = SIGKILL (RAM); -11 = SIGSEGV (falha em código nativo)."
                       + (f"\nTraceback nativo (faulthandler):\n{tail}" if tail else ""))
    finally:
        if proc.is_alive():
            proc.kill()
        proc.join(5)
        parent_conn.close()
        try:
            proc.close()                        # libera o sentinel (fd) do Process
        except Exception:
            pass
        for _p in (crash_path, payload_path):
            try:
                _p.unlink()
            except Exception:
                pass
    if msg is not None:
        raise IsolatedMemoryError(msg)
    if status == "boot":
        raise _IsoBootError(str(payload)[:200])
    if status == "err":
        raise RuntimeError(f"{label}: {payload}")
    print(f"[isolado] {label}: concluído ({method}); pico de RAM do filho {peak:.0f} MB (teto {cap:.0f} MB)")
    return payload, peak


def run_isolated(func, args=(), max_rss_mb=None, timeout_s=600.0, label="etapa"):
    """Executa func(*args) em um processo filho vigiado ('spawn': sem herdar o contexto CUDA do kernel). Se o filho passar do
    teto de RAM, morrer ou demorar demais, ele é encerrado e uma IsolatedMemoryError é levantada, sem derrubar o kernel.
    Se o 'spawn' não puder iniciar (cloudpickle ausente, objeto não serializável, execução como .py), recorre ao 'fork'
    (os filhos não usam CUDA). Retorna (resultado, pico_de_RAM_do_filho_em_MB). O resultado precisa ser serializável."""
    try:
        return _run_isolated_impl(func, args, max_rss_mb, timeout_s, label, "spawn")
    except _IsoBootError as e:
        print(f"⚠️ {label}: 'spawn' indisponível ({e}); usando 'fork' (o filho não usa CUDA).")
        return _run_isolated_impl(func, args, max_rss_mb, timeout_s, label, "fork")

RDLogger.DisableLog("rdApp.*")


def _pkg_ver(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except Exception:
        return "n/d"


def run_obabel(cmd, what):
    try:
        return subprocess.run(cmd, check=True, capture_output=True, text=True, errors="ignore")
    except subprocess.CalledProcessError as e:
        tail = "\n".join((e.stderr or "").strip().splitlines()[-3:]) or "(sem mensagem do OpenBabel)"
        raise RuntimeError(f"OpenBabel falhou em '{what}' (código {e.returncode}):\n{tail}") from e
    except FileNotFoundError as e:
        raise RuntimeError("OpenBabel (obabel) não encontrado no PATH.") from e


def _obabel_version() -> str:
    try:
        out = subprocess.run(["obabel", "-V"], capture_output=True, text=True).stdout.strip()
        return out.splitlines()[0] if out else "n/d"
    except Exception:
        return "n/d"


# ==============================================================================
# 3. FUNÇÕES PURAS (PDB / CAIXA / QC)  --  sem dependência de RDKit/OpenMM
# ==============================================================================
def _pdb_atom_record(line: str) -> dict:
    name = line[12:16].strip()
    el = line[76:78].strip().upper() if len(line) >= 78 else ""
    if not el:
        m = re.match(r"^\d*([A-Za-z]{1,2})", name)
        el = (m.group(1)[0] if m else "").upper()
        # HETATM de metal sem coluna de elemento: nome encostado na coluna 13 ("FE  ") indica elemento de 2 letras
        if line[:6].strip() == "HETATM" and line[12] != " " and name[:2].upper() in METAL_ELEMENTS:
            el = name[:2].upper()
    return {
        "rec": line[:6].strip(),
        "name": name,
        "res": line[17:20].strip(),
        "chain": line[21],
        "resseq": line[22:26].strip(),
        "icode": line[26] if len(line) > 26 else " ",
        "xyz": np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])], dtype=float),
        "el": el,
    }


def _read_pdb_records(path) -> list:
    recs = []
    for ln in Path(path).read_text().splitlines():
        if ln.startswith(("ATOM  ", "HETATM")) and len(ln) >= 54:
            try:
                recs.append(_pdb_atom_record(ln))
            except ValueError:
                continue
    return recs


_AROMATIC_RING_C = {
    "PHE": {"CD1", "CD2", "CE1", "CE2", "CZ"},
    "TYR": {"CD1", "CD2", "CE1", "CE2"},
    "TRP": {"CD1", "CE3", "CZ2", "CZ3", "CH2"},
    "HIS": {"CD2", "CE1"}, "HID": {"CD2", "CE1"}, "HIE": {"CD2", "CE1"}, "HIP": {"CD2", "CE1"},
}


def qc_receptor_pdb(pdb_path, metal_cutoff: float = METAL_COORD_CUTOFF) -> dict:
    """Controle de qualidade da protonação do receptor (detecta os defeitos do `obabel -p`)."""
    recs = _read_pdb_records(pdb_path)
    prot = [r for r in recs if r["rec"] == "ATOM"]
    metals = _metal_records_from(recs)             # íons livres + metais dentro de cofatores (FE do HEM)
    metal_xyz = np.array([m["xyz"] for m in metals]) if metals else np.empty((0, 3))

    residues = OrderedDict()
    for r in prot:
        residues.setdefault((r["chain"], r["resseq"], r["icode"], r["res"]), []).append(r)

    issues, his_states = [], Counter()
    n_h = 0
    for (chain, resseq, icode, resname), atoms in residues.items():
        heavy = [a for a in atoms if a["el"] != "H"]
        hyd = [a for a in atoms if a["el"] == "H"]
        n_h += len(hyd)
        cnt = Counter()
        for h in hyd:
            if not heavy:
                continue
            d = [np.linalg.norm(h["xyz"] - a["xyz"]) for a in heavy]
            j = int(np.argmin(d))
            if d[j] < 1.25:
                cnt[heavy[j]["name"]] += 1
        label = f"{resname}{resseq}{chain}"

        names = [a["name"] for a in atoms]
        dups = [n for n, c in Counter(names).items() if c > 1]
        if dups:
            issues.append(f"{label}: nomes atômicos duplicados {dups}")

        for cname in _AROMATIC_RING_C.get(resname, set()):
            if cnt.get(cname, 0) > 1:
                issues.append(f"{label}: carbono aromático {cname} com {cnt[cname]} H (inválido)")

        if resname in HIS_NAMES:
            nd1, ne2 = cnt.get("ND1", 0), cnt.get("NE2", 0)
            state = "HIP" if (nd1 and ne2) else ("HID" if nd1 else ("HIE" if ne2 else "SEM_NH"))
            his_states[state] += 1
            if state == "SEM_NH":
                issues.append(f"{label}: histidina sem N–H (nem ND1 nem NE2)")
            for nname in ("ND1", "NE2"):
                n_atom = [a for a in heavy if a["name"] == nname]
                if n_atom and metal_xyz.shape[0] > 0:
                    dm = np.min(np.linalg.norm(metal_xyz - n_atom[0]["xyz"], axis=1))
                    if dm < metal_cutoff and cnt.get(nname, 0) > 0:
                        issues.append(f"{label}: {nname} coordenado a metal ({dm:.2f} Å) mas protonado")

    return {
        "n_residues": len(residues),
        "n_h": n_h,
        "n_metals": len(metals),
        "his_states": dict(his_states),
        "issues": issues,
        "ok": len(issues) == 0,
    }


def compute_autobox(native_pdb, ligand_diameter: float, add: float = AUTOBOX_ADD):
    """Caixa explícita (regra do autobox do GNINA): bounding box do ligante nativo + `add` Å por lado,
    cada aresta estendida até o diâmetro do ligante a ser docado. Retorna (centro, tamanho, centróide, span)."""
    coords = []
    for r in _read_pdb_records(native_pdb):
        if r["el"] != "H":
            coords.append(r["xyz"])
    if not coords:
        raise ValueError("Ligante nativo sem coordenadas válidas.")
    arr = np.array(coords, dtype=float)
    lo, hi = arr.min(axis=0), arr.max(axis=0)
    center = (lo + hi) / 2.0
    span = hi - lo
    size = np.maximum(span + 2.0 * add, float(ligand_diameter))
    return center, size, arr.mean(axis=0), span


def renumber_receptor_lines(atom_lines_in_order: list, metal_lines: list) -> list:
    """Receptor final: ATOM sequenciais ➔ TER ➔ HETATM metálicos ➔ END (numeração contínua, sem lacunas)."""
    out, serial = [], 1
    for ln in atom_lines_in_order:
        out.append(f"ATOM  {serial:5d}{ln[11:]}")
        serial += 1
    out.append("TER")
    for ln in metal_lines:
        out.append(f"HETATM{serial:5d}{ln[11:]}")
        serial += 1
    out.append("END")
    return out


def parse_box_echo(text: str):
    """Procura a caixa ecoada pelo GNINA no log (se existir). Retorna dict ou None."""
    vals = {}
    for key in ("center_x", "center_y", "center_z", "size_x", "size_y", "size_z"):
        m = re.search(rf"{key}\s*[=:]?\s*(-?\d+(?:\.\d+)?)", text)
        if not m:
            return None
        vals[key] = float(m.group(1))
    return vals



# ==============================================================================
# 3b. FUNÇÕES PURAS: PoseBusters oficial, geometria de metais, HTML
# ==============================================================================
# --- PoseBusters ---------------------------------------------------------------
# Testes oficiais do modo "dock" (PoseBusters >= 0.6; Buttenschoen et al., 2024).
# Cada coluna é um TESTE (True = passou). Lista usada como contingência caso o relatório
# resumido da biblioteca (full_report=False) não possa ser obtido.
PB_DOCK_OFFICIAL_TESTS = [
    "mol_pred_loaded", "mol_cond_loaded", "sanitization", "inchi_convertible",
    "all_atoms_connected", "no_radicals", "bond_lengths", "bond_angles",
    "internal_steric_clash", "aromatic_ring_flatness", "non-aromatic_ring_non-flatness",
    "double_bond_flatness", "internal_energy", "protein-ligand_maximum_distance",
    "minimum_distance_to_protein", "minimum_distance_to_organic_cofactors",
    "minimum_distance_to_inorganic_cofactors", "minimum_distance_to_waters",
    "volume_overlap_with_protein", "volume_overlap_with_organic_cofactors",
    "volume_overlap_with_inorganic_cofactors", "volume_overlap_with_waters",
]
# Colunas booleanas AUXILIARES (detalhes de módulos). NÃO são testes de validade:
#   most_extreme_clash_*   -> True = HÁ choque no par mais extremo; False = conforme
#   not_too_far_away_*     -> False quando o cofator/água não existe (distância = NaN)
#   passes_*, no_radicals_before_sanitization -> detalhes do carregamento de mol_true
PB_DIAGNOSTIC_REGEX = re.compile(r"^(most_extreme_clash_|not_too_far_away_|passes_|no_radicals_before_)")


def pb_bool_columns(df) -> list:
    """Colunas booleanas do relatório PoseBusters (inclui colunas object com NaN + bool)."""
    cols = []
    for c in df.columns:
        s = df[c]
        if s.dtype == bool:
            cols.append(c)
            continue
        nn = s.dropna()
        if len(nn) > 0 and all(isinstance(v, (bool, np.bool_)) for v in nn):
            cols.append(c)
    return cols


def pb_is_diagnostic(col: str) -> bool:
    return bool(PB_DIAGNOSTIC_REGEX.match(col))


def pb_select_official_columns(df_full, short_cols=None):
    """Define quais colunas compõem o veredito PB_Valid.
    1) preferencial: colunas do relatório resumido da própria biblioteca (full_report=False),
       sem as auxiliares; 2) contingência: lista oficial embutida do modo dock.
    Retorna (oficiais, auxiliares, descrição_da_fonte)."""
    bool_cols = pb_bool_columns(df_full)
    official, source = [], ""
    if short_cols:
        official = [c for c in short_cols if c in bool_cols and not pb_is_diagnostic(c)]
        source = "relatório resumido oficial da biblioteca (full_report=False)"
    if len(official) < 10:
        official = [c for c in PB_DOCK_OFFICIAL_TESTS if c in bool_cols]
        source = "lista oficial embutida do modo dock (contingência)"
    diagnostic = [c for c in bool_cols if c not in official]
    return official, diagnostic, source


def pb_evaluate(df_full, official_cols):
    """Por pose: PASS se nenhum teste oficial é False. NaN = não aplicável (não reprova)."""
    status, failed, na = [], [], []
    for i in range(len(df_full)):
        row = df_full.iloc[i]
        f = [c for c in official_cols if pd.notna(row[c]) and not bool(row[c])]
        n = [c for c in official_cols if pd.isna(row[c])]
        status.append("FAIL" if f else "PASS")
        failed.append(", ".join(f))
        na.append(", ".join(n))
    return status, failed, na


# --- Metais --------------------------------------------------------------------
def _metal_records_from(recs) -> list:
    """Metais de uma lista de registros: (1) íons livres (resíduo em CATALYTIC_METALS) e (2) átomos metálicos
    DENTRO de cofatores HETATM com >= 2 átomos (FE do HEM, MG da clorofila, CO da B12, FE do SF4...)."""
    sizes = Counter((r["chain"], r["resseq"], r["res"]) for r in recs if r["rec"] == "HETATM")
    out = []
    for r in recs:
        if r["rec"] != "HETATM":
            continue
        if r["res"].upper() in CATALYTIC_METALS:
            out.append(r)
        elif sizes[(r["chain"], r["resseq"], r["res"])] >= 2 and r["el"].upper() in METAL_ELEMENTS \
                and r["res"].upper() not in SOLVENTS_AND_IONS:
            out.append(r)
    return out


def metal_records(pdb_path) -> list:
    """Átomos de metais catalíticos de um PDB, inclusive os presos em cofatores orgânicos."""
    return _metal_records_from(_read_pdb_records(pdb_path))


def metal_symbol(r) -> str:
    """Símbolo do metal: pelo nome do resíduo (íon livre) ou pelo elemento (metal dentro de cofator)."""
    res = r["res"].upper()
    return METAL_SYMBOL[res] if res in METAL_SYMBOL else r["el"].capitalize()


# --- Cofatores: separação, "íons substitutos" e colagem -------------------------
def is_bypass_cofactor(resname: str, elements) -> bool:
    """Cofator que NÃO deve passar pelo OpenMM: lista ORGANIC_COFACTORS ou complexo metálico (>= 2 átomos com metal)."""
    rn = resname.strip().upper()
    if rn in CATALYTIC_METALS or rn in SOLVENTS_AND_IONS:
        return False
    els = [str(e).upper() for e in elements]
    return rn in ORGANIC_COFACTORS or (len(els) >= 2 and any(e in METAL_ELEMENTS for e in els))


def make_metal_standins(cofactor_lines) -> tuple:
    """Para cada átomo metálico de um cofator cria uma linha de ÍON LIVRE (resíduo = símbolo do metal), que é a
    única parte do cofator enviada ao PDBFixer: assim a His/Cys axial é tratada como coordenada ao metal.
    Retorna (linhas_substitutas, [coordenadas])."""
    standins, coords = [], []
    for ln in cofactor_lines:
        rec = _pdb_atom_record(ln)
        sym = rec["el"].upper()
        if sym in METAL_ELEMENTS and sym in CATALYTIC_METALS:
            name4 = f"{sym:<4}"
            standins.append(ln[:12] + name4 + ln[16] + f"{sym:>3}" + ln[20:])
            coords.append(rec["xyz"])
    return standins, coords


def merge_cofactors_into_receptor(rec_pdb, cofactor_lines, standin_coords, tol: float = 0.02) -> int:
    """Abre o receptor protonado, remove os íons substitutos (por coordenada) e cola os cofatores INTACTOS no final,
    renumerando os átomos de forma contínua. Retorna o nº de linhas de cofator coladas."""
    atoms, hets = [], []
    for ln in Path(rec_pdb).read_text().splitlines():
        if ln.startswith("ATOM  "):
            atoms.append(ln)
        elif ln.startswith("HETATM"):
            hets.append(ln)
    if standin_coords:
        sc = np.array(standin_coords, dtype=float)
        kept = []
        for ln in hets:
            rec = _pdb_atom_record(ln)
            if rec["res"].upper() in CATALYTIC_METALS and float(np.min(np.linalg.norm(sc - rec["xyz"], axis=1))) < tol:
                continue                                  # íon substituto: o metal real vem dentro do cofator
            kept.append(ln)
        hets = kept
    Path(rec_pdb).write_text("\n".join(renumber_receptor_lines(atoms, hets + list(cofactor_lines))) + "\n")
    return len(cofactor_lines)


def strip_cofactors_from_pdb(src, dst, resnames) -> int:
    """Copia o receptor sem os HETATM dos cofatores indicados (usado nos fallbacks do GNINA/PoseBusters)."""
    names = {str(n).upper() for n in resnames}
    kept, removed = [], 0
    for ln in Path(src).read_text().splitlines():
        if ln.startswith("HETATM") and ln[17:20].strip().upper() in names:
            removed += 1
            continue
        kept.append(ln)
    Path(dst).write_text("\n".join(kept) + "\n")
    return removed


def inchain_modified_residue(resname, atom_names, chain, resseq, residue_numbers_by_chain) -> bool:
    """HETATM com N, CA e C cujo vizinho de sequência (resseq +-1) é aminoácido da mesma cadeia = resíduo
    modificado (MSE, SEP, TPO, PTR, HYP, CSO...). Ligantes aminoácido livres têm numeração isolada."""
    if not {"N", "CA", "C"} <= set(atom_names):
        return False
    try:
        k = int(str(resseq).strip())
    except ValueError:
        return False
    nums = residue_numbers_by_chain.get(chain, set())
    return (k - 1) in nums or (k + 1) in nums


def pdb_has_amino_acids(prot_lines) -> int:
    """Nº de resíduos com C-alfa (0 para cadeias de DNA/RNA)."""
    return sum(1 for l in prot_lines if l[12:16].strip() == "CA")


# --- Sanidade do ligante ---------------------------------------------------------
def ligand_safety_report(mol, hac: int, torsdof) -> tuple:
    """(bloqueio | None, [avisos]) para ligantes exóticos. Heurísticas conservadoras, documentadas nos metadados."""
    elems = sorted({a.GetSymbol() for a in mol.GetAtoms()})
    odd = [e for e in elems if e.upper() not in ORGANIC_LIGAND_ELEMENTS]
    warns = []
    if hac < 2:
        return ("Ligante com menos de 2 átomos pesados (íon ou átomo isolado): docking não se aplica. "
                "Íons do sítio devem ser mantidos no receptor (Passo 1).", warns)
    if hac > MAX_LIGAND_HEAVY_ATOMS:
        return (f"Ligante com {hac} átomos pesados (limite {MAX_LIGAND_HEAVY_ATOMS}): tempo/RAM inviáveis.", warns)
    if odd:
        warns.append(f"Elementos fora do conjunto orgânico padrão ({', '.join(odd)}): MMFF94/Gasteiger podem não ter "
                     "parâmetros e o GNINA pode usar tipos atômicos genéricos; interprete os escores com cautela.")
    if hac > WARN_LIGAND_HEAVY_ATOMS:
        warns.append(f"Ligante grande ({hac} átomos pesados): aumente a exaustividade e espere tempo maior.")
    try:
        if int(torsdof) > WARN_LIGAND_TORSIONS:
            warns.append(f"{int(torsdof)} torções ativas (> {WARN_LIGAND_TORSIONS}): espaço de busca muito amplo; "
                         "a amostragem pode não convergir.")
    except (TypeError, ValueError):
        pass
    return None, warns


def metal_tier(d) -> str:
    """Faixa de distância O/N/S–metal. Distâncias de coordenação cristalográficas ficam em torno de 2 Å
    (Fig. 4c, Zhang et al., JACS 2026); 2,8 Å é o raio de contato do ProLIF."""
    if d is None:
        return "sem doador"
    return "coordenação provável (≤ 2,5 Å)" if d <= 2.5 else ("contato próximo (≤ 2,8 Å)" if d <= METAL_COORD_CUTOFF else "sem contato (> 2,8 Å)")


def element_labels(symbols) -> list:
    """Rótulos por elemento (O1, O2, ...) na mesma ordem usada nos nomes do PDB do complexo."""
    cnt, out = Counter(), []
    for s in symbols:
        cnt[s] += 1
        out.append(f"{s}{cnt[s]}")
    return out


def metal_contacts(lig_xyz, symbols, metals, cutoff: float = METAL_COORD_CUTOFF):
    """Distância mínima O/N/S(ligante)–metal e lista textual de contatos <= cutoff.
    Retorna (dmin | None, texto)."""
    donors = [i for i, s in enumerate(symbols) if str(s).upper() in ("O", "N", "S")]
    if not metals or not donors:
        return None, ""
    labels = element_labels(symbols)
    m_xyz = np.array([m["xyz"] for m in metals], dtype=float)
    l_xyz = np.asarray(lig_xyz, dtype=float)[donors]
    d = np.linalg.norm(l_xyz[:, None, :] - m_xyz[None, :, :], axis=-1)
    hits = []
    for a in range(d.shape[0]):
        for b in range(d.shape[1]):
            if d[a, b] <= cutoff:
                m = metals[b]
                hits.append(f"{labels[donors[a]]}→{m['res']}{m['resseq']}: {d[a, b]:.2f} Å")
    return float(d.min()), "; ".join(hits)


def build_metal_rdmol(pdb_path):
    """Mol RDKit só com os metais do receptor (átomos isolados, sem ligações, com info de resíduo PDB).
    Evita a inferência de ligações metal–N/O do MDAnalysis (causa do FALLBACK 1 da v7)."""
    from rdkit.Geometry import Point3D
    recs = metal_records(pdb_path)
    if not recs:
        return None
    rw = Chem.RWMol()
    conf = Chem.Conformer(len(recs))
    for i, r in enumerate(recs):
        at = Chem.Atom(metal_symbol(r))
        at.SetNoImplicit(True)
        info = Chem.AtomPDBResidueInfo()
        info.SetName(f" {r['name']:<3}"[:4])
        info.SetResidueName(f"{r['res']:<3}"[:3])
        info.SetResidueNumber(int(r["resseq"]))
        info.SetChainId(r["chain"])
        info.SetSerialNumber(i + 1)
        info.SetIsHeteroAtom(True)
        at.SetMonomerInfo(info)
        idx = rw.AddAtom(at)
        conf.SetAtomPosition(idx, Point3D(float(r["xyz"][0]), float(r["xyz"][1]), float(r["xyz"][2])))
    mol = rw.GetMol()
    mol.AddConformer(conf, assignId=True)
    return mol


def count_metal_atoms(mol) -> int:
    return sum(1 for a in mol.GetAtoms() if a.GetSymbol() in METAL_SYMBOLS_SET)


# --- Bolsão de ligação (ProLIF sem SIGSEGV) e contatos geométricos de contingência ----------------------------------------
POCKET_RADIUS_A = 12.0             # Å: resíduos da proteína considerados no ProLIF (qualquer átomo a <= raio de qualquer átomo do ligante)
POCKET_FALLBACK_RADIUS_A = 16.0    # Å: esfera em torno do centro da grid box, se a seleção por ligante vier vazia
GEO_HB_CUTOFF, GEO_PI_CUTOFF, GEO_HF_CUTOFF = 3.5, 4.5, 4.0     # Å: N/O···N/O; centróide···centróide; C···C
_HF_RESIDUES = {"ALA", "VAL", "LEU", "ILE", "MET", "PHE", "TRP", "PRO", "CYS", "TYR"}
_BACKBONE_ATOMS = {"N", "CA", "C", "O", "OXT"}
_PROT_RINGS = {"PHE": [("CG", "CD1", "CD2", "CE1", "CE2", "CZ")], "TYR": [("CG", "CD1", "CD2", "CE1", "CE2", "CZ")],
               "TRP": [("CG", "CD1", "NE1", "CE2", "CD2"), ("CD2", "CE2", "CZ2", "CH2", "CZ3", "CE3")],
               "HIS": [("CG", "ND1", "CD2", "CE1", "NE2")], "HID": [("CG", "ND1", "CD2", "CE1", "NE2")],
               "HIE": [("CG", "ND1", "CD2", "CE1", "NE2")], "HIP": [("CG", "ND1", "CD2", "CE1", "NE2")]}
_GEO_COLNAME = {"HB": "HBDonor", "PiPi": "PiStacking", "Hf": "Hydrophobic", "Metal": "MetalAcceptor"}   # nomes de coluna no estilo ProLIF


def pocket_resindices(prot_xyz, prot_resindices, lig_xyz_list, radius: float = POCKET_RADIUS_A):
    """Índices (únicos, ordenados) dos resíduos com ao menos um átomo a <= radius de QUALQUER átomo de qualquer pose do ligante.
    Equivale a 'protein and (same residue as around <radius> group ligand)' do MDAnalysis, sem exigir um Universe do ligante."""
    from scipy.spatial import cKDTree
    prot_xyz = np.asarray(prot_xyz, dtype=float)
    res = np.asarray(prot_resindices)
    if not len(prot_xyz) or not len(lig_xyz_list):
        return np.array([], dtype=int)
    tree = cKDTree(prot_xyz)
    hit = set()
    for xyz in lig_xyz_list:
        xyz = np.asarray(xyz, dtype=float).reshape(-1, 3)
        for idxs in tree.query_ball_point(xyz, r=float(radius)):
            hit.update(res[list(idxs)].tolist())
    return np.array(sorted(hit), dtype=int)


def read_pose_geometries(poses_sdf) -> list:
    """Poses do SDF como dicts {sym, xyz, apolar_c, rings} (RDKit, hidrogênios mantidos); poses ilegíveis viram None."""
    out = []
    for m in Chem.SDMolSupplier(str(poses_sdf), removeHs=False, sanitize=True):
        if m is None or m.GetNumConformers() == 0:
            out.append(None)
            continue
        xyz = np.array(m.GetConformer().GetPositions(), dtype=float)
        sym = [a.GetSymbol() for a in m.GetAtoms()]
        apolar = [a.GetSymbol() == "C" and not any(n.GetSymbol() in ("N", "O", "S", "F", "Cl", "Br", "I", "P") for n in a.GetNeighbors())
                  for a in m.GetAtoms()]
        rings = [list(r) for r in m.GetRingInfo().AtomRings() if all(m.GetAtomWithIdx(i).GetIsAromatic() for i in r)]
        out.append({"sym": sym, "xyz": xyz, "apolar_c": apolar, "rings": rings})
    return out


def _geo_receptor_atoms(recs, metal_set=None):
    """Átomos pesados do receptor (proteína + cofatores orgânicos; sem água/íons simples) e os metais, com rótulo 'RES123.A'."""
    atoms, metals = [], []
    for r in recs:
        res = str(r["res"]).upper()
        if r["el"] in ("H", "D") or res in ("HOH", "DOD", "WAT"):
            continue
        lab = f"{res}{r['resseq']}.{(r['chain'] or 'A').strip() or 'A'}"
        is_metal = r["el"] in METAL_ELEMENTS and (r["rec"] == "HETATM")
        rec = {"lab": lab, "res": res, "name": str(r["name"]).upper(), "el": r["el"], "xyz": np.asarray(r["xyz"], dtype=float),
               "prot": r["rec"] == "ATOM"}
        (metals if is_metal else atoms).append(rec)
    return atoms, metals


def geometric_contacts(recs, pose) -> dict:
    """Contatos por proximidade (sem ProLIF) de UMA pose: {rótulo 'RES123.A': {'HB'|'PiPi'|'Hf'|'Metal', ...}}.
    Pontes de H: N/O(ligante)···N/O(receptor) <= 3,5 Å; π: centróides de anéis aromáticos <= 4,5 Å; hidrofóbico: C apolar do ligante ···
    C de cadeia lateral hidrofóbica <= 4,0 Å; metal: N/O/S(ligante)···metal <= 2,8 Å."""
    from scipy.spatial import cKDTree
    found = {}
    if pose is None:
        return found
    atoms, metals = _geo_receptor_atoms(recs)
    lig_xyz, lig_sym = np.asarray(pose["xyz"], dtype=float), list(pose["sym"])
    heavy = [i for i, s in enumerate(lig_sym) if s != "H"]

    def add(lab, k):
        found.setdefault(lab, set()).add(k)
    # --- pontes de hidrogênio e contatos hidrofóbicos
    if atoms and heavy:
        rxyz = np.array([a["xyz"] for a in atoms])
        tree = cKDTree(rxyz)
        for i in heavy:
            s = lig_sym[i]
            if s in ("N", "O"):
                for j in tree.query_ball_point(lig_xyz[i], r=GEO_HB_CUTOFF):
                    if atoms[j]["el"] in ("N", "O"):
                        add(atoms[j]["lab"], "HB")
            if pose.get("apolar_c", [False] * len(lig_sym))[i]:
                for j in tree.query_ball_point(lig_xyz[i], r=GEO_HF_CUTOFF):
                    a = atoms[j]
                    if a["el"] == "C" and a["prot"] and a["res"] in _HF_RESIDUES and a["name"] not in _BACKBONE_ATOMS:
                        add(a["lab"], "Hf")
    # --- empilhamento π: centróides de anéis
    lig_c = [lig_xyz[r].mean(axis=0) for r in pose.get("rings", []) if len(r)]
    if lig_c:
        by_res = {}
        for a in atoms:
            if a["prot"] and a["res"] in _PROT_RINGS:
                by_res.setdefault(a["lab"], {})[a["name"]] = a["xyz"]
        for lab, d in by_res.items():
            res = lab[:3]
            for ring in _PROT_RINGS.get(res, []):
                if all(n in d for n in ring):
                    c = np.mean([d[n] for n in ring], axis=0)
                    if any(np.linalg.norm(c - lc) <= GEO_PI_CUTOFF for lc in lig_c):
                        add(lab, "PiPi")
    # --- coordenação metálica
    for m in metals:
        for i in heavy:
            if lig_sym[i] in ("N", "O", "S") and np.linalg.norm(lig_xyz[i] - m["xyz"]) <= METAL_COORD_CUTOFF:
                add(m["lab"], "Metal")
    return found


def geometric_interactions_df(rec_path, poses_sdf, n_poses: int, poses=None):
    """DataFrame no mesmo formato do ProLIF (Pose_Rank + colunas booleanas 'RES123.A_Interacao') a partir de contatos geométricos.
    Alimenta 'Interacoes_Chave', figuras, planilha e relatório quando o ProLIF falha. Levanta RuntimeError se nada puder ser calculado."""
    recs = _read_pdb_records(rec_path)
    poses = poses if poses is not None else read_pose_geometries(poses_sdf)
    if not recs or not poses:
        raise RuntimeError("receptor ou poses ilegíveis para o cálculo geométrico")
    per_pose = [geometric_contacts(recs, p) for p in poses]
    cols = sorted({(lab, k) for d in per_pose for lab, ks in d.items() for k in ks},
                  key=lambda t: (int(re.search(r"(-?\d+)", t[0]).group(1)) if re.search(r"(-?\d+)", t[0]) else 0, t[0], t[1]))
    data = {"Pose_Rank": [f"Pose_{i + 1}" for i in range(len(per_pose))]}
    for lab, k in cols:
        data[f"{lab}_{_GEO_COLNAME[k]}"] = [bool(k in d.get(lab, ())) for d in per_pose]
    df = pd.DataFrame(data)
    if len(df) != n_poses:                       # alinha com o nº de poses esperado (poses ilegíveis = linhas sem contatos)
        df = df.reindex(range(n_poses))
        df["Pose_Rank"] = [f"Pose_{i + 1}" for i in range(n_poses)]
        for c in df.columns[1:]:
            df[c] = df[c].fillna(False).astype(bool)
    return df


GEO_BRANCH = ("FALLBACK GEOMÉTRICO: contatos por proximidade (H <= 3,5 Å; π <= 4,5 Å; hidrofóbico <= 4,0 Å; metal <= 2,8 Å)"
              " — ProLIF indisponível nesta corrida")


# --- HTML (interface) ----------------------------------------------------------
def esc(x) -> str:
    return _html.escape(str(x), quote=True)


def pill(kind: str, text: str) -> str:
    """kind: ok (verde) | info (azul) | warn (âmbar) | err (vermelho)."""
    return f'<span class="dd-pill dd-pill-{kind}">{esc(text)}</span>'


def status_box(kind: str, title: str, body_html: str = "") -> str:
    icons = {"ok": "✓", "info": "i", "warn": "!", "err": "×"}
    body = f'<div class="dd-status-text">{body_html}</div>' if body_html else ""
    return (f'<div class="dd-status dd-status-{kind}"><div class="dd-status-icon">{icons.get(kind, "i")}</div>'
            f'<div class="dd-status-body"><div class="dd-status-title">{esc(title)}</div>{body}</div></div>')


def metric_tiles(items) -> str:
    """items: lista de (rótulo, valor, tipo)."""
    cells = "".join(
        f'<div class="dd-tile dd-tile-{k}"><div class="dd-tile-value">{esc(v)}</div>'
        f'<div class="dd-tile-label">{esc(l)}</div></div>' for l, v, k in items)
    return f'<div class="dd-tiles">{cells}</div>'


# --- Nomes reais (PDB / ligante) ----------------------------------------------
_LOWER_WORDS = {"THE", "OF", "IN", "AND", "WITH", "FROM", "BY", "TO", "AT", "ON", "FOR", "A", "AN", "AS", "OR"}


def pretty_pdb_text(text: str) -> str:
    """Títulos do PDB vêm em CAIXA ALTA; converte para caixa de sentença preservando siglas/códigos."""
    text = " ".join(str(text).split())
    if not text or text != text.upper():
        return text
    out, prev = [], ""
    for w in text.split(" "):
        core = re.sub(r"[^A-Z0-9]", "", w)
        if w == "A" and prev[:1].isdigit():      # "2.3 A RESOLUTION" -> "2.3 Å resolution"
            out.append("Å")
            prev = w
            continue
        prev = w
        if any(ch.isdigit() for ch in core) or (len(core) <= 3 and core not in _LOWER_WORDS and len(core) > 0):
            out.append(w)
        else:
            out.append(w.lower())
    res = " ".join(out)
    return res[:1].upper() + res[1:]


def _pdb_record_text(pdb_text: str, record: str) -> str:
    """Concatena as linhas de continuação de um registro (TITLE, COMPND, SOURCE)."""
    parts = []
    for line in pdb_text.splitlines():
        if line.startswith(record):
            parts.append(line[10:].strip())
    return " ".join(p for p in parts if p)


def _parse_mol_blocks(text: str) -> list:
    blocks = []
    for seg in re.split(r"MOL_ID:\s*\d+;?", text):
        seg = seg.strip()
        if not seg:
            continue
        d = {}
        for m in re.finditer(r"([A-Z_]+):\s*([^;]*);?", seg):
            d[m.group(1)] = m.group(2).strip()
        if d:
            blocks.append(d)
    return blocks


def parse_pdb_names(pdb_text: str) -> dict:
    """Extrai do cabeçalho: título, nome da molécula por cadeia (COMPND) e organismo (SOURCE)."""
    title = pretty_pdb_text(_pdb_record_text(pdb_text, "TITLE"))
    compnd = _parse_mol_blocks(_pdb_record_text(pdb_text, "COMPND"))
    source = _parse_mol_blocks(_pdb_record_text(pdb_text, "SOURCE"))
    chains, organisms = {}, {}
    for i, blk in enumerate(compnd):
        mol = pretty_pdb_text(blk.get("MOLECULE", ""))
        org = source[i].get("ORGANISM_SCIENTIFIC", "") if i < len(source) else ""
        for ch in [c.strip() for c in blk.get("CHAIN", "").split(",") if c.strip()]:
            if mol:
                chains[ch] = mol
            if org:
                organisms[ch] = org.capitalize() if org == org.upper() else org
    return {"title": title, "chains": chains, "organisms": organisms}


# --------------------------------------------------------------------------------------------------
# Autodescoberta de domínio e função biológica a partir do cabeçalho do PDB (HEADER, COMPND/EC e metais)
# --------------------------------------------------------------------------------------------------
GENERIC_TARGET_DOMAIN = "cavidade ortostérica do sítio catalítico"
GENERIC_DOMAIN_FUNCTION = "atividade biológica e interação com substratos"
_CU_RES = {"CU", "CU1", "CU2", "CU3"}
_OTHER_METAL_RES = {"FE", "FE2", "FE3", "ZN", "MN", "NI", "CO", "MO"}      # Fe/Zn (e afins) => "dependente de metal"


def _pdb_header_class(pdb_text: str) -> str:
    """Classe da proteína: registro HEADER, colunas 11-50 (ex.: 'OXIDOREDUCTASE'). '' se ausente."""
    for ln in str(pdb_text or "").splitlines():
        if ln.startswith("HEADER"):
            return re.sub(r"\s+", " ", ln[10:50]).strip().upper()
    return ""


def _pdb_metal_counts(pdb_text: str, chain: str) -> tuple:
    """(n átomos de Cu, n de outros metais redox/estruturais) nos HETATM da cadeia (ou do arquivo, se a cadeia não tiver metais)."""
    rows = []
    for ln in str(pdb_text or "").splitlines():
        if ln.startswith("HETATM") and len(ln) >= 21:
            res = ln[17:20].strip().upper()
            el = ln[76:78].strip().upper() if len(ln) >= 78 else ""
            rows.append((ln[21] if len(ln) > 21 else " ", res, el))
    def count(sel):
        cu = sum(1 for _, r, e in sel if r in _CU_RES or (e == "CU" and r not in _OTHER_METAL_RES))
        ot = sum(1 for _, r, e in sel if r in _OTHER_METAL_RES)
        return cu, ot
    own = [x for x in rows if x[0] == chain]
    cu, ot = count(own)
    if not (cu or ot):
        cu, ot = count(rows)
    return cu, ot


def infer_domain_and_function(pdb_id, chain, pdb_text, pdb_names=None) -> dict:
    """Infere `target_domain` e `domain_function` (prosa científica em português) do próprio arquivo PDB:
    classe (HEADER), código EC e nome da molécula (COMPND) da cadeia, além de Cu/Fe/Zn nos HETATM.
    Nunca levanta exceção: sem informação reconhecível devolve o texto neutro padrão.
    Retorno: {target_domain, domain_function, category, ec, protein_class, molecule}."""
    cls, ec, mol, cu, ot = "", "", "", 0, 0
    pdb_text, ch = str(pdb_text or ""), str(chain or "").strip()
    try:                                                                               # cada fonte falha isoladamente
        cls = _pdb_header_class(pdb_text)
    except Exception:
        pass
    try:
        blocks = _parse_mol_blocks(_pdb_record_text(pdb_text, "COMPND"))
        sel = next((b for b in blocks if ch and ch in [c.strip() for c in b.get("CHAIN", "").split(",")]), None)
        sel = sel or next((b for b in blocks if b.get("EC")), None) or (blocks[0] if blocks else {})
        ec = re.sub(r"\s+", "", str(sel.get("EC", "")).split(",")[0])
        try:
            mol = pretty_pdb_text(sel.get("MOLECULE", ""))
        except Exception:
            mol = str(sel.get("MOLECULE", "")).strip()
        mol = mol or str(((pdb_names or {}).get("chains") or {}).get(ch, ""))
    except Exception:
        pass
    try:
        cu, ot = _pdb_metal_counts(pdb_text, ch)
    except Exception:
        pass
    ecm = re.match(r"^(\d+)(?:\.(\d+|-))?", ec)
    ec1, ec2 = (ecm.group(1), ecm.group(2) or "") if ecm else ("", "")
    txt = f"{cls} {mol}".upper()
    has = lambda *w: any(x in txt for x in w)                                          # noqa: E731
    if ec1 == "3" and ec2 == "4" or has("PROTEASE", "PEPTIDASE", "PROTEINASE", "TRYPSIN", "THROMBIN", "CASPASE", "CATHEPSIN"):
        cat, td, df = "protease", "fenda catalítica da protease", "clivagem proteolítica de ligações peptídicas"
    elif ec1 == "1" or "OXIDOREDUCTASE" in cls:
        cat, df = "oxidorredutase", "catálise de oxirredução e transferência de elétrons no sítio ativo"
        td = "domínio catalítico oxirredutor"
        if cu >= 2:
            td += " binuclear dependente de cobre"
        elif cu == 1:
            td += " dependente de cobre"
        elif ot:
            td += " dependente de metal"
    elif (ec1 == "2" and ec2 == "7") or has("KINASE"):
        cat, td, df = "quinase", "domínio quinase catalítico (KD)", "fosforilação de resíduos e transdução de sinal celular"
    elif ec1 == "2" or "TRANSFERASE" in cls:
        cat, td, df = "transferase", "domínio catalítico transferase", "transferência de grupos funcionais entre substratos"
    elif ec1 == "3" or "HYDROLASE" in cls:
        cat, td, df = "hidrolase", "domínio hidrolítico catalítico", "hidrólise de ligações covalentes do substrato"
    elif ec1 == "4" or "LYASE" in cls:
        cat, td, df = "liase", "domínio catalítico liase", "clivagem e formação de ligações sem hidrólise nem oxirredução"
    elif ec1 == "5" or "ISOMERASE" in cls:
        cat, td, df = "isomerase", "domínio catalítico isomerase", "rearranjo intramolecular do substrato"
    elif ec1 == "6" or "LIGASE" in cls:
        cat, td, df = "ligase", "domínio catalítico ligase", "formação de ligações covalentes com consumo de ATP"
    elif has("RECEPTOR", "NUCLEAR HORMONE", "TRANSCRIPTION FACTOR"):
        cat, td, df = "receptor", "domínio de ligação ao ligante (LBD) ortostérico", "reconhecimento molecular e modulação conformacional"
    else:
        cat, td, df = "", GENERIC_TARGET_DOMAIN, GENERIC_DOMAIN_FUNCTION
    return {"target_domain": td, "domain_function": df, "category": cat, "ec": ec, "protein_class": cls, "molecule": mol}


def make_local_pdb_id(filename) -> str:
    """Identificador seguro para arquivos/pastas da sessão a partir do nome do arquivo enviado."""
    stem = re.sub(r"[^A-Za-z0-9]+", "_", Path(str(filename)).stem).strip("_").upper()[:28]
    return LOCAL_ID_PREFIX + (stem or "MODELO")


def normalize_local_pdb_text(text: str) -> str:
    """BOM/quebras de linha do Windows; se NENHUM átomo tem cadeia (coluna 22 em branco), atribui a cadeia 'A'
    (evita uma cadeia ' ' invisível no menu). Arquivos com cadeias definidas não são alterados."""
    text = text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    lines = text.split("\n")
    idx = [i for i, ln in enumerate(lines) if ln.startswith(("ATOM  ", "HETATM")) and len(ln) > 21]
    if idx and all(lines[i][21] == " " for i in idx):
        for i in idx:
            lines[i] = lines[i][:21] + "A" + lines[i][22:]
    return "\n".join(lines)


def apply_local_placeholders(names: dict, chains) -> dict:
    """Título/organismo do cabeçalho do próprio arquivo quando existirem; senão, placeholders (sem rede)."""
    out = dict(names)
    out["title"] = out.get("title") or LOCAL_TITLE_PLACEHOLDER
    orgs = dict(out.get("organisms", {}))
    for ch in chains:
        orgs.setdefault(ch, LOCAL_ORG_PLACEHOLDER)
    out["organisms"] = orgs
    out.setdefault("chains", {})
    return out


def identity_card(prot_id, prot_title, prot_mol, prot_org, lig_name, lig_sub) -> str:
    """Cartão de identificação (proteína + ligante) exibido no resumo final."""
    p_sub = " · ".join(x for x in [prot_mol, prot_org] if x)
    return (
        '<div class="dd-ident">'
        f'<div class="dd-ident-col"><div class="dd-ident-tag">Proteína · PDB {esc(prot_id)}</div>'
        f'<div class="dd-ident-name">{esc(prot_title or "título não disponível")}</div>'
        f'<div class="dd-ident-sub">{esc(p_sub) if p_sub else "&nbsp;"}</div></div>'
        f'<div class="dd-ident-col"><div class="dd-ident-tag">Ligante</div>'
        f'<div class="dd-ident-name">{esc(lig_name)}</div>'
        f'<div class="dd-ident-sub">{esc(lig_sub) if lig_sub else "&nbsp;"}</div></div></div>')



# --- CNNvariance (incerteza do ensemble 3D-CNN) ---------------------------------
_GNINA_BLOCK_RE = re.compile(
    r"Affinity:\s*(-?[\d.]+)\s+(-?[\d.]+)\s*\(kcal/mol\)\s*"
    r"RMSD:\s*(-?[\d.]+)\s*"
    r"CNNscore:\s*(-?[\d.]+)\s*"
    r"CNNaffinity:\s*(-?[\d.]+)\s*"
    r"CNNvariance:\s*(-?[\d.]+)")


def parse_gnina_cnn_blocks(text: str) -> list:
    """Blocos 'Affinity/RMSD/CNNscore/CNNaffinity/CNNvariance' do stdout do GNINA (etapa --minimize)."""
    return [{"vina": float(m.group(1)), "cnn_score": float(m.group(4)),
             "cnn_aff": float(m.group(5)), "cnn_var": float(m.group(6))}
            for m in _GNINA_BLOCK_RE.finditer(text or "")]


def assign_cnn_variance(pose_props, blocks, tol: float = 1e-4, vina_tol: float = 0.02):
    """CNNvariance por pose. 1) tag `CNNvariance` do SDF, se existir; 2) bloco do log casado por
    (CNNscore, CNNaffinity) com tolerância `tol` (o log imprime 5 casas: erro de arredondamento <= ~1e-5).
    A ORDEM dos blocos do log não é a do SDF nem é estável entre execuções; por isso o casamento é por valores.
    Mais de um candidato: desempata pelo Vina (`minimizedAffinity`); persistindo, NÃO adivinha ("ambíguo").
    Retorna (valores | None, fontes 'SDF'|'log'|'ambíguo'|'n/d')."""
    used, vals, srcs = set(), [], []
    for p in pose_props:
        v = p.get("CNNvariance")
        try:
            if v is not None and str(v) != "":
                vals.append(float(v)); srcs.append("SDF"); continue
        except (TypeError, ValueError):
            pass
        try:
            sc, af = float(p.get("CNNscore")), float(p.get("CNNaffinity"))
        except (TypeError, ValueError):
            vals.append(None); srcs.append("n/d"); continue
        cands = [j for j, b in enumerate(blocks)
                 if j not in used and abs(b["cnn_score"] - sc) + abs(b["cnn_aff"] - af) < tol]
        if len(cands) > 1:
            try:
                vn = float(p.get("minimizedAffinity"))
                cands = [j for j in cands if abs(blocks[j]["vina"] - vn) < vina_tol]
            except (TypeError, ValueError):
                pass
        if len(cands) == 1:
            used.add(cands[0]); vals.append(blocks[cands[0]]["cnn_var"]); srcs.append("log")
        elif len(cands) > 1:
            vals.append(None); srcs.append("ambíguo")
        else:
            vals.append(None); srcs.append("n/d")
    return vals, srcs


def collect_cnn_variance(mols, log_paths):
    blocks = []
    for lp in log_paths:
        try:
            blocks += parse_gnina_cnn_blocks(Path(lp).read_text(errors="ignore"))
        except Exception:
            pass
    return assign_cnn_variance([m.GetPropsAsDict() for m in mols], blocks)


# ==============================================================================
# 4. IONIZAÇÃO DO LIGANTE EM pH 7.4 (SUBESTRUTURAS RDKIT, SEM REAÇÕES SMARTS)
# ==============================================================================
# (rótulo, SMARTS, posição do O-H dentro da correspondência)
IONIZATION_RULES = [
    ("Ácido carboxílico ➔ carboxilato (pKa ~4-5)",
     "[CX3](=[OX1])[OX2H1]", 2),
    ("Tropolona (α-hidroxi-tropona, anel de 7) ➔ tropolonato (pKa ~7; forma quelante)",
     "[#6r7;X3](=[OX1])[#6r7;X3;$([#6]=,:[#6])][OX2H1]", 3),
    ("7-OH de flavonoide/cromona ➔ fenolato (pKa ~7.0-7.2)",
     "[OX2H1]c1cc2[o,O]~[#6]~[#6]~[#6](=O)c2cc1", 0),
]
_IONIZATION_PATTERNS = [(lab, Chem.MolFromSmarts(sma), idx) for lab, sma, idx in IONIZATION_RULES]
for _lab, _patt, _ in _IONIZATION_PATTERNS:
    if _patt is None:
        raise RuntimeError(f"SMARTS inválido na regra de ionização: {_lab}")


def apply_ph74_ionization(mol_heavy):
    """Recebe molécula SEM H explícitos; desprotona diretamente os grupos identificados.
    Retorna (mol, lista de regras aplicadas). Moléculas sem esses grupos passam intactas."""
    Chem.GetSymmSSSR(mol_heavy)                       # garante informação de anéis (consultas r7)
    rw = Chem.RWMol(mol_heavy)
    applied, done = [], set()
    for label, patt, o_pos in _IONIZATION_PATTERNS:
        n_hit = 0
        for match in mol_heavy.GetSubstructMatches(patt):   # correspondências na molécula original
            o_idx = match[o_pos]
            if o_idx in done:
                continue
            ref = mol_heavy.GetAtomWithIdx(o_idx)
            if ref.GetAtomicNum() != 8 or ref.GetFormalCharge() != 0 or ref.GetTotalNumHs() < 1:
                continue
            at = rw.GetAtomWithIdx(o_idx)
            at.SetFormalCharge(-1)
            at.SetNumExplicitHs(0)
            at.SetNoImplicit(False)
            done.add(o_idx)
            n_hit += 1
        if n_hit:
            applied.append(f"{label} x{n_hit}")
    out = rw.GetMol()
    if applied:
        Chem.SanitizeMol(out)
        for o_idx in done:
            a = out.GetAtomWithIdx(o_idx)
            if a.GetFormalCharge() != -1 or a.GetTotalNumHs() != 0:
                raise ValueError("Ionização inconsistente (O desprotonado ainda possui H).")
    return out, applied


def run_ionization_self_test() -> list:
    """Autoteste das regras (executa no Colab). Retorna lista de avisos (vazia = tudo OK)."""
    cases = [
        ("quercetina (7-OH)", "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12", -1),
        ("tropolona", "O=c1cccccc1O", -1),
        ("ácido acético", "CC(=O)O", -1),
        ("ibuprofeno", "CC(C)Cc1ccc(cc1)C(C)C(=O)O", -1),
        ("fenol (inalterado)", "Oc1ccccc1", 0),
        ("α-arbutina (inalterada)", "OC[C@H]1O[C@H](Oc2ccc(O)cc2)[C@H](O)[C@@H](O)[C@@H]1O", 0),
        ("crisina (7-OH)", "O=c1cc(-c2ccccc2)oc2cc(O)cc(O)c12", -1),
        ("tropolonato já ionizado (inalterado)", "O=c1cccccc1[O-]", -1),
    ]
    warns = []
    for nome, smi, esperado in cases:
        try:
            m = Chem.MolFromSmiles(smi)
            m2, _ = apply_ph74_ionization(m)
            q = Chem.GetFormalCharge(m2)
            if q != esperado:
                warns.append(f"{nome}: carga {q:+d} (esperado {esperado:+d})")
        except Exception as e:
            warns.append(f"{nome}: erro {e}")
    return warns


# ==============================================================================
# 5. MOTOR DO PIPELINE
# ==============================================================================
# --- BENCHMARK vs. LIGANTE NATIVO ---------------------------------------------------------------------------
# Camada SOMENTE-LEITURA. Lê as métricas JÁ calculadas da pose exportada e as compara com o "controle positivo" (o ligante
# cristalográfico re-dockado no mesmo receptor, com os mesmos parâmetros). Nada aqui escreve em _Complexo.pdb, .sdf,
# PoseBusters ou ProLIF, e qualquer exceção é capturada: no pior caso o cartão mostra "benchmark indisponível".
BM_BAND_KCAL = 0.5                                  # banda de equivalência (±kcal/mol), da ordem do ruído do método
BM_KT = 1.364                                       # kcal/mol por unidade de pK a 298 K (mesma constante do LE CNN)
BM_NO_CONTROL = "n/d (execute o redocking do nativo primeiro)"
BM_COLS = ["Métrica", "Ligante de teste", "Controle nativo", "Δ (teste − nativo)"]


def _bm_num(x):
    """float ou None (aceita os 'n/d' e '—' das tabelas)."""
    try:
        v = float(x)
        return v if np.isfinite(v) else None
    except (TypeError, ValueError):
        return None


def _bm_file_sig(path) -> str:
    """Assinatura de CONTEÚDO (não de data/serial): nome, resíduo, cadeia e coordenadas dos registros ATOM/HETATM."""
    h = hashlib.sha1()
    for ln in Path(path).read_text(errors="ignore").splitlines():
        if ln.startswith(("ATOM  ", "HETATM")):
            h.update((ln[12:27] + ln[30:54]).encode())
    return h.hexdigest()[:16]


def bm_native_resname(native_pdb):
    """O arquivo do nativo é '<PDB>_<cadeia>_<RES>_native.pdb' (ver extract_receptor_and_native)."""
    stem = Path(str(native_pdb)).stem
    parts = stem[:-len("_native")].split("_") if stem.endswith("_native") else stem.split("_")
    return parts[-1] if parts else ""


def bm_classify(delta_kcal) -> str:
    d = round(float(delta_kcal), 2)
    if d <= -BM_BAND_KCAL:
        return "Superior"
    if d >= BM_BAND_KCAL:
        return "Inferior"
    return "Equivalente"


def bm_make_ctx(pdb_id, chain, native_pdb, rec_path, center, size, exhaustiveness, seed, prep_mode, device):
    return {"pdb_id": str(pdb_id), "chain": str(chain), "resname": bm_native_resname(native_pdb),
            "rec_sig": _bm_file_sig(rec_path), "nat_sig": _bm_file_sig(native_pdb),
            "center": [round(float(v), 3) for v in center], "size": [round(float(v), 3) for v in size],
            "exh": int(exhaustiveness), "seed": int(seed), "prep": str(prep_mode)[:2].strip(". "),
            "device": str(device)}


def bm_make_record(ctx, sel, cnn_var, hac, lig_name, pose_rank, rmsd, status_pb_note=""):
    """sel: linha (dict) da tabela Resumo_Executivo da pose exportada."""
    rec = {
        "ctx": ctx, "lig_name": str(lig_name), "pose": int(pose_rank), "hac": int(hac),
        "vina": _bm_num(sel.get("Vina score (kcal/mol)")), "cnn_s": _bm_num(sel.get("CNNscore (0-1)")),
        "pk": _bm_num(sel.get("CNNaffinity (pK)")), "cnn_var": _bm_num(cnn_var),
        "cnn_vs": _bm_num(sel.get("CNN_VS (score×pK)")),
        "le_v": _bm_num(sel.get("LE Vina (-Vina/HAC)")), "le_c": _bm_num(sel.get("LE CNN (1.364·pK/HAC)")),
        "pb": str(sel.get("PB_Valid (PoseBusters)", "n/d")), "rmsd": _bm_num(rmsd),
        "dmetal": _bm_num(sel.get("Dist. mín. metal–ligante (Å)")),
    }
    if rec["vina"] is None or rec["pk"] is None:
        raise ValueError("Vina/CNNaffinity ausentes na pose exportada.")
    return rec


def bm_context_diff(ctx_t, ctx_c):
    """(motivos que INVALIDAM a comparação, avisos que a mantêm). Chaves duras: o mesmo alvo, o mesmo receptor preparado,
    o mesmo ligante nativo e a mesma semente. Caixa/exaustividade são avisos (o modo de preparo NÃO é: controle no modo 2 e teste no modo 1 é o fluxo correto); a caixa automática depende do
    diâmetro de cada ligante, então exigir caixa idêntica invalidaria quase todas as comparações."""
    hard, soft = [], []
    for k, lab in (("pdb_id", "PDB"), ("chain", "cadeia"), ("resname", "ligante nativo")):
        if ctx_t[k] != ctx_c[k]:
            hard.append(f"{lab} difere (controle: {ctx_c[k]}; esta corrida: {ctx_t[k]})")
    if ctx_t["rec_sig"] != ctx_c["rec_sig"]:
        hard.append("o receptor preparado mudou desde o controle (novo 'Fixar sítio ativo' com outras opções)")
    if ctx_t["nat_sig"] != ctx_c["nat_sig"]:
        hard.append("as coordenadas do ligante nativo mudaram desde o controle")
    if ctx_t["seed"] != ctx_c["seed"]:
        hard.append("semente de amostragem diferente")
    if ctx_t["exh"] != ctx_c["exh"]:
        soft.append(f"exaustividade diferente (controle {ctx_c['exh']}; teste {ctx_t['exh']})")
    dc = float(np.linalg.norm(np.array(ctx_t["center"]) - np.array(ctx_c["center"])))
    ds = float(np.max(np.abs(np.array(ctx_t["size"]) - np.array(ctx_c["size"]))))
    if dc > 0.5 or ds > 2.0:
        soft.append(f"caixa diferente (Δcentro {dc:.2f} Å; Δaresta máx. {ds:.1f} Å)")
    return hard, soft


# --- Taxonomia estrutural automática -----------------------------------------------------
PERFIL_FALLBACK = {"termo_substantivo": "o composto orgânico", "classe_estrutural": "composto orgânico",
                   "perfil_medchem": "n/d", "mw": None, "hac": None}
_SMARTS_FLAVONOIDE = "[#6]1:[#6]:[#6]:[#6]2:[#6](:[#6]:1)~[#8]~[#6](~[#6]~[#6]2)~[#6]3:[#6]:[#6]:[#6]:[#6]:[#6]:3"
_SMARTS_FENOL = "[OX2H,OX1-][c]"
_SMARTS_AZOL = "[n;r5]"


def classificar_perfil_ligante(mol) -> dict:
    """Função pura: porte pelo peso molecular (pequeno < 250 Da; médio 250-450; grande > 450) e classe química simples por SMARTS. Fenol inclui o fenolato (pH 7.4)."""
    hac = int(mol.GetNumHeavyAtoms())
    mw = float(Descriptors.MolWt(mol))
    if mw < 250:
        porte = "pequeno porte"
    elif mw <= 450:
        porte = "médio porte"
    else:
        porte = "grande porte"
    n_fenol = len({m[0] for m in mol.GetSubstructMatches(Chem.MolFromSmarts(_SMARTS_FENOL))})
    if mol.HasSubstructMatch(Chem.MolFromSmarts(_SMARTS_FLAVONOIDE)):
        classe, termo = "flavonoide", "o flavonoide"
    elif n_fenol >= 2:
        classe, termo = "composto polifenólico", "o polifenol"
    elif n_fenol == 1:
        classe, termo = "composto fenólico", "o derivado fenólico"
    elif mol.HasSubstructMatch(Chem.MolFromSmarts(_SMARTS_AZOL)):
        classe, termo = "composto azólico", "o derivado azólico"
    else:
        classe, termo = "composto orgânico", "o composto orgânico"
    return {"termo_substantivo": termo, "classe_estrutural": classe, "perfil_medchem": porte, "mw": mw, "hac": hac}


def bm_veredicto(t, c, cmp) -> str:
    cv, cc = cmp["class_vina"], cmp["class_cnn"]
    le_t, dle_v = t.get("le_v"), cmp.get("dle_v")
    if t.get("pb") != "PASS":
        return "Inviável (PoseBusters FAIL)"
    if (cv == "Superior" and cc == "Inferior") or (cv == "Inferior" and cc == "Superior"):
        return "Inconclusivo (Discordância Vina/CNN)"
    if cv == "Superior":
        if cc in ("Superior", "Equivalente"):
            if le_t is not None and dle_v is not None and le_t < 0.28 and dle_v < -0.05:
                return "Superior (Ressalva: Baixa LE / Efeito Volume)"
            return "Superior (Consenso Robusto)"
        return "Inconclusivo (Apoio Fraco da CNN)"
    if cv == "Equivalente":
        if cc == "Superior" and dle_v is not None and dle_v > 0.02:
            return "Promissor (LE/CNN Superior ao Nativo)"
        if cc == "Inferior":
            return "Equivalente (Alerta: CNN Desfavorável)"
        return "Equivalente ao Controle"
    if le_t is not None and le_t >= 0.35:
        return "Bioativo Eficiente (Potencial Lead-like)"
    return "Inferior ao Nativo"


def bm_veredicto_kind(v) -> str:
    v = str(v)
    if v.startswith("Inviável"):
        return "err"
    if v.startswith(("Inconclusivo", "Inferior")) or "Alerta" in v:
        return "warn"
    if v.startswith(("Superior (Consenso", "Promissor")):
        return "ok"
    return "info"


def bm_compare(t, c):
    dv = t["vina"] - c["vina"]
    dpk = t["pk"] - c["pk"]
    out = {"dvina": dv, "dpk": dpk, "ratio": 10.0 ** dpk,
           "class_vina": bm_classify(dv), "class_cnn": bm_classify(-BM_KT * dpk),
           "dle_v": (t["le_v"] - c["le_v"]) if None not in (t["le_v"], c["le_v"]) else None,
           "dle_c": (t["le_c"] - c["le_c"]) if None not in (t["le_c"], c["le_c"]) else None}
    out["concordam"] = out["class_vina"] == out["class_cnn"]
    out["veredito"] = bm_veredicto(t, c, out)
    return out


def _bm_fmt(v, nd=2, sign=False):
    if v is None:
        return "n/d"
    return f"{v:+.{nd}f}" if sign else f"{v:.{nd}f}"


def _bm_ratio(r):
    return f"{r:.2f}×" if 0.01 <= r < 1000 else f"{r:.1e}×"


def bm_table_df(t=None, c=None, cmp=None, status=""):
    """Aba 'Benchmark_Nativo': tabela lado a lado (teste × controle) seguida das métricas relativas."""
    rows = [["Status", status, "", ""]]
    def line(lab, k, nd=2):
        tv = t.get(k) if t else None
        cv = c.get(k) if c else None
        d = (tv - cv) if (tv is not None and cv is not None and not isinstance(tv, str)) else None
        rows.append([lab, tv if tv is not None else ("—" if t is None else "n/d"),
                     cv if cv is not None else ("—" if c is None else "n/d"),
                     round(d, nd) if d is not None else ""])
    if t or c:
        rows.append(["Ligante", t["lig_name"] if t else "—", c["lig_name"] if c else "—", ""])
        rows.append(["Pose exportada (rank)", t["pose"] if t else "—", c["pose"] if c else "—", ""])
        for lab, k, nd in (("Vina score (kcal/mol)", "vina", 2), ("CNNscore (0-1)", "cnn_s", 3),
                           ("CNNaffinity (pK)", "pk", 2), ("CNNvariance", "cnn_var", 3),
                           ("CNN_VS (score×pK)", "cnn_vs", 2), ("LE Vina", "le_v", 3), ("LE CNN", "le_c", 3),
                           ("Átomos pesados (HAC)", "hac", 0), ("RMSD simétrico ao cristal (Å)", "rmsd", 3),
                           ("Dist. mín. metal–ligante (Å)", "dmetal", 2)):
            line(lab, k, nd)
        rows.append(["PoseBusters (pose exportada)", t["pb"] if t else "—", c["pb"] if c else "—", ""])
    if cmp:
        rows += [["ΔVina = Vina_teste − Vina_nativo (kcal/mol)", "", "", round(cmp["dvina"], 2)],
                 ["ΔpK = pK_teste − pK_nativo", "", "", round(cmp["dpk"], 2)],
                 ["Razão de potência prevista 10^ΔpK", "", "", round(cmp["ratio"], 3)],
                 ["ΔLE Vina", "", "", round(cmp["dle_v"], 3) if cmp["dle_v"] is not None else "n/d"],
                 ["ΔLE CNN", "", "", round(cmp["dle_c"], 3) if cmp["dle_c"] is not None else "n/d"],
                 [f"Classe (Vina, banda ±{BM_BAND_KCAL} kcal/mol)", "", "", cmp["class_vina"]],
                 ["Classe (CNN, mesma banda em ΔG = −1,364·ΔpK)", "", "", cmp["class_cnn"]],
                 ["Vina e CNN concordam?", "", "", "sim" if cmp["concordam"] else "não"],
                 ["Veredito multiparamétrico (PoseBusters + Vina + CNN + LE)", "", "", cmp["veredito"]]]
    return pd.DataFrame(rows, columns=BM_COLS)


def _bm_side_table(t, c):
    def cell(r, k, nd):
        return esc(_bm_fmt(r.get(k), nd)) if r else "—"
    spec = (("Vina (kcal/mol)", "vina", 2), ("CNNscore", "cnn_s", 3), ("CNNaffinity (pK)", "pk", 2),
            ("CNNvariance", "cnn_var", 3), ("LE Vina", "le_v", 3), ("LE CNN", "le_c", 3), ("HAC", "hac", 0))
    body = "".join(f"<tr><td>{esc(lab)}</td><td>{cell(t, k, nd)}</td><td>{cell(c, k, nd)}</td></tr>" for lab, k, nd in spec)
    th_t = esc(t["lig_name"][:28]) if t else "teste"
    th_c = esc(c["lig_name"][:28]) if c else "nativo"
    return ('<table style="width:100%;border-collapse:collapse;font-size:.85rem;margin:6px 0">'
            f'<tr><th style="text-align:left">Métrica (pose exportada)</th><th>{th_t}</th><th>{th_c} (nativo)</th></tr>'
            f'{body}</table>')


BM_CAVEAT = ("Escores de docking <b>não são afinidades experimentais</b>: o controle é o nativo re-dockado, não o seu Ki/IC50. "
             "A classe é um ranking computacional dentro da banda de ruído do método (da ordem de 1–2 kcal/mol para o Vina), "
             "não prova de equivalência biológica. A comparação só vale com o mesmo receptor, a mesma semente e parâmetros "
             "parecidos; a LE depende do número de átomos pesados.")


def bm_card(mode, t=None, c=None, cmp=None, hard=(), soft=(), extra=""):
    title = "Benchmark vs. ligante nativo (controle positivo)"
    wrap = lambda inner: f'<div class="dd-card" style="margin-top:12px">{inner}</div>'
    if mode == "compared":
        k = {"Superior": "ok", "Equivalente": "info", "Inferior": "warn"}[cmp["class_vina"]]
        kc = {"Superior": "ok", "Equivalente": "info", "Inferior": "warn"}[cmp["class_cnn"]]
        tiles = metric_tiles([
            ("ΔVina (kcal/mol)", _bm_fmt(cmp["dvina"], 2, True), k),
            ("ΔpK · potência", f"{_bm_fmt(cmp['dpk'], 2, True)} · {_bm_ratio(cmp['ratio'])}", kc),
            ("ΔLE Vina", _bm_fmt(cmp["dle_v"], 3, True), "info"),
            ("ΔLE CNN", _bm_fmt(cmp["dle_c"], 3, True), "info"),
            ("Classe (Vina)", cmp["class_vina"], k),
            ("Veredito", cmp["veredito"], bm_veredicto_kind(cmp["veredito"])),
        ])
        notes = [f"<b>{esc(t['lig_name'])}</b> vs. nativo <b>{esc(c['ctx']['resname'])}</b>: ΔVina = Vina_teste − Vina_nativo "
                 f"(negativo = teste melhor); potência prevista = 10^ΔpK (&gt;1 = teste mais potente). "
                 f"Classe pela CNN (mesma banda em ΔG = −1,364·ΔpK): {esc(cmp['class_cnn'])} — "
                 + ("concorda com o Vina." if cmp["concordam"] else "difere da classe do Vina.")
                 + f" <b>Veredito:</b> {esc(cmp['veredito'])}."]
        notes += ["<b>Atenção:</b> " + esc(s) + "." for s in soft]
        if c.get("rmsd") is not None and c["rmsd"] >= 2.0:
            notes.append(f"<b>Atenção:</b> o redocking do nativo teve RMSD = {c['rmsd']:.2f} Å (≥ 2,0 Å); a referência é menos confiável.")
        if c.get("pb") != "PASS":
            notes.append("<b>Atenção:</b> a pose do controle não obteve PB_Valid = PASS.")
        pill_html = pill("ok" if not soft else "warn", "Comparação válida" if not soft else "Comparação com ressalvas")
        return wrap(f'<div class="dd-status-title">{esc(title)} {pill_html}</div>' + tiles + _bm_side_table(t, c)
                    + '<div class="dd-detail">' + "<br>".join(notes) + "</div>"
                    + f'<details><summary>Como ler</summary><div class="dd-detail">{BM_CAVEAT}</div></details>')
    if mode == "control_saved":
        warn = []
        if c.get("rmsd") is not None and c["rmsd"] >= 2.0:
            warn.append(f"<b>Atenção:</b> RMSD do redocking = {c['rmsd']:.2f} Å (≥ 2,0 Å); a pose do controle pode não reproduzir o cristal.")
        if c.get("pb") != "PASS":
            warn.append("<b>Atenção:</b> a pose exportada do controle não obteve PB_Valid = PASS.")
        return wrap(f'<div class="dd-status-title">{esc(title)} {pill("ok", "Controle registrado")}</div>'
                    + metric_tiles([("Vina do nativo", _bm_fmt(c["vina"], 2), "info"), ("CNNaffinity do nativo (pK)", _bm_fmt(c["pk"], 2), "info"),
                                    ("LE Vina do nativo", _bm_fmt(c["le_v"], 3), "info"), ("LE CNN do nativo", _bm_fmt(c["le_c"], 3), "info")])
                    + '<div class="dd-detail">Esta corrida re-dockou o ligante cristalográfico <b>' + esc(c["ctx"]["resname"])
                    + '</b> (identidade confirmada pelo grafo e pela composição, via RMSD simétrico). Ele ficou guardado como '
                    + 'controle: dê agora o docking dos seus ligantes de teste <b>sem refixar o sítio ativo</b> e este cartão '
                    + 'mostrará o benchmark relativo.' + ("<br>" + "<br>".join(warn) if warn else "") + "</div>")
    if mode == "stale":
        return wrap(f'<div class="dd-status-title">{esc(title)} {pill("warn", "Controle invalidado")}</div>'
                    '<div class="dd-detail"><b>' + esc(BM_NO_CONTROL) + "</b><br>O controle guardado não vale para esta corrida: "
                    + esc("; ".join(hard)) + ".</div>")
    if mode == "no_control":
        return wrap(f'<div class="dd-status-title">{esc(title)} {pill("info", "Sem controle")}</div>'
                    '<div class="dd-detail"><b>' + esc(BM_NO_CONTROL) + "</b><br>Para calibrar: dê docking no ligante nativo "
                    "(Passo 2: envie o arquivo <code>" + esc(extra) + "</code> gerado no Passo 1; Passo 3: modo "
                    "<i>Redocking cristalográfico</i>) com este mesmo receptor. Depois rode seus ligantes de teste.</div>")
    if mode == "no_native":
        return wrap(f'<div class="dd-status-title">{esc(title)} {pill("info", "Não aplicável")}</div>'
                    '<div class="dd-detail"><b>n/d (sem ligante nativo fixado no sítio)</b>: o benchmark exige um ligante '
                    "cristalográfico selecionado no Passo 1.</div>")
    return wrap(f'<div class="dd-status-title">{esc(title)} {pill("warn", "Indisponível")}</div>'
                '<div class="dd-detail">O benchmark falhou nesta corrida (' + esc(extra) + "). "
                "O docking, o PoseBusters, o ProLIF e os arquivos exportados <b>não foram afetados</b>.</div>")


def bm_process(engine, ctx, native_pdb, sel, cnn_var, hac, lig_name, pose_rank, rmsd, is_native_identity):
    """Devolve {'mode','html','df','meta','short'}. Só lê; a única escrita é engine.native_benchmark (um dict em memória)."""
    t = bm_make_record(ctx, sel, cnn_var, hac, lig_name, pose_rank, rmsd)
    ctrl = getattr(engine, "native_benchmark", None)
    if is_native_identity and getattr(engine, "_lock_control", False) and getattr(engine, "native_benchmark", None) is not None:
        is_native_identity = False
    if is_native_identity:
        engine.native_benchmark = t
        mode, cmp, hard, soft, c = "control_saved", None, [], [], t
        df = bm_table_df(None, t, None, "Controle nativo registrado nesta corrida")
        meta = [("Benchmark_Nativo_Status", f"controle registrado ({ctx['resname']}; Vina {t['vina']:.2f}; pK {t['pk']:.2f})")]
        short = "<b>Benchmark:</b> controle nativo registrado. "
    elif ctrl is None:
        mode, cmp, hard, soft, c = "no_control", None, [], [], None
        df = bm_table_df(t, None, None, BM_NO_CONTROL)
        meta = [("Benchmark_Nativo_Status", BM_NO_CONTROL)]
        short = "<b>Benchmark:</b> sem controle nativo. "
    else:
        hard, soft = bm_context_diff(ctx, ctrl["ctx"])
        if hard:
            mode, cmp, c = "stale", None, None
            df = bm_table_df(t, None, None, BM_NO_CONTROL + " — controle invalidado: " + "; ".join(hard))
            meta = [("Benchmark_Nativo_Status", BM_NO_CONTROL + " — controle invalidado: " + "; ".join(hard))]
            short = "<b>Benchmark:</b> controle invalidado (contexto mudou). "
        else:
            mode, c = "compared", ctrl
            cmp = bm_compare(t, c)
            st = "Comparação válida" if not soft else "Comparação com ressalvas: " + "; ".join(soft)
            df = bm_table_df(t, c, cmp, st)
            meta = [("Benchmark_Nativo_Status", st),
                    ("Benchmark_Controle", f"{c['lig_name']} ({c['ctx']['resname']}); Vina {c['vina']:.2f}; pK {c['pk']:.2f}; pose {c['pose']}"),
                    ("Benchmark_Preparo (informativo)", f"controle: modo {c['ctx']['prep']}; teste: modo {ctx['prep']}"),
                    ("Benchmark_dVina (kcal/mol)", round(cmp["dvina"], 2)), ("Benchmark_dpK", round(cmp["dpk"], 2)),
                    ("Benchmark_Razao_Potencia (10^dpK)", round(cmp["ratio"], 3)),
                    ("Benchmark_dLE_Vina", round(cmp["dle_v"], 3) if cmp["dle_v"] is not None else "n/d"),
                    ("Benchmark_dLE_CNN", round(cmp["dle_c"], 3) if cmp["dle_c"] is not None else "n/d"),
                    ("Benchmark_Classe_Vina", cmp["class_vina"]), ("Benchmark_Classe_CNN", cmp["class_cnn"]),
                    ("Benchmark_Veredito", cmp["veredito"])]
            short = f"<b>Benchmark:</b> {esc(cmp['veredito'])} (ΔVina {cmp['dvina']:+.2f} kcal/mol). "
    return {"mode": mode, "df": df, "meta": meta, "short": short, "rec": t,
            "html": bm_card(mode, t, c, cmp, hard, soft, extra=str(native_pdb))}


def bm_failsafe(native_pdb, err):
    """Qualquer exceção do benchmark vira este resultado inerte."""
    msg = f"{type(err).__name__}: {str(err)[:120]}"
    return {"mode": "erro", "short": "", "html": bm_card("erro", extra=msg),
            "df": pd.DataFrame([["Status", "Benchmark indisponível: " + msg, "", ""]], columns=BM_COLS),
            "meta": [("Benchmark_Nativo_Status", "indisponível: " + msg)], "rec": None}


def bm_no_native():
    return {"mode": "no_native", "short": "", "html": bm_card("no_native"),
            "df": pd.DataFrame([["Status", "n/d (sem ligante nativo fixado no sítio)", "", ""]], columns=BM_COLS),
            "meta": [("Benchmark_Nativo_Status", "n/d (sem ligante nativo fixado no sítio)")], "rec": None}


# --- FILA MULTI-LIGANTE, LEADERBOARD E EXCEL DE ARTIGO ------------------------------------------------------
# Funções puras: não tocam o docking. O motor chama _run_docking_core uma vez por ligante e só lê o resultado.
MAX_BATCH_LIGANDS = 50
ROLE_CTRL, ROLE_TEST = "Controle nativo", "Teste"
ROLE_CTRL_NV = "Controle nativo (não validado)"


def is_ctrl(role):
    return role in (ROLE_CTRL, ROLE_CTRL_NV)

NO_NATIVE_TXT = "n/d (sem ligante nativo fixado no sítio)"
INTER_ABBR = {"HBDonor": "HB", "HBAcceptor": "HB", "PiStacking": "PiPi", "PiCation": "PiCat", "CationPi": "PiCat",
              "Hydrophobic": "Hf", "Cationic": "Ion", "Anionic": "Ion", "MetalAcceptor": "Metal"}
INTER_ORDER = ["HB", "PiPi", "PiCat", "Hf", "Ion", "Metal"]
ARTIGO_COLS = ["Composto", "Papel", "HAC", "Vina score (kcal/mol)", "ΔVina (kcal/mol)", "CNNscore",
               "CNNaffinity (pK)", "LE Vina", "RMSD (Å)", "PoseBusters", "Interações Chave"]


def parse_ligand_queries(text):
    """'5280343, 158637' / 'Quercetin, Rutin' / uma por linha / ';' -> lista sem repetições.
    Vírgula só separa quando seguida de espaço ou entre CIDs numéricos; assim 'N,N-dimetilglicina' e '1,2-diol' não quebram."""
    if text is None or not str(text).strip():
        return []
    out = []
    for part in re.split(r"[\n;]+", str(text)):
        part = part.strip()
        if not part:
            continue
        toks = re.split(r"[\s,]+", part) if re.fullmatch(r"[\d\s,]+", part) else re.split(r",\s+", part)
        out += [t.strip().strip("\"'") for t in toks if t.strip().strip("\"'")]
    seen, res = set(), []
    for q in out:
        if q.lower() not in seen:
            seen.add(q.lower())
            res.append(q)
    return res


def _iter_paths(obj):
    if obj is None:
        return []
    if isinstance(obj, (list, tuple)):
        return [p for o in obj for p in _iter_paths(o)]
    if isinstance(obj, dict):
        p = obj.get("path") or obj.get("name")
        return [str(p)] if p else []
    p = getattr(obj, "name", obj)
    return [str(p)] if p else []


def _info_list(lig_info):
    if not lig_info:
        return []
    return [x for x in (lig_info if isinstance(lig_info, (list, tuple)) else [lig_info]) if isinstance(x, dict)]


def _info_for(path, infos):
    for inf in infos:
        if inf.get("file") and Path(str(inf["file"])).name == Path(path).name:
            return inf
    return None


def _sdf_names(path, limit=MAX_BATCH_LIGANDS + 1):
    """Leitura leve (sem sanitizar) dos nomes das moléculas de um SDF, para a prévia da fila."""
    names = []
    for i, m in enumerate(Chem.SDMolSupplier(str(path), sanitize=False, removeHs=False)):
        if i >= limit:
            break
        nm = m.GetProp("_Name").strip() if (m is not None and m.HasProp("_Name")) else ""
        names.append(nm or f"{Path(path).stem}#{i + 1}")
    return names


def preview_queue(user_ligand_file, lig_info, pubchem_query="", auto_native=False, native_pdb=None, prep_mode=""):
    try:
        tests = []
        for p in _iter_paths(user_ligand_file):
            tests += _sdf_names(p) if Path(p).suffix.lower() == ".sdf" else [Path(p).stem]
        if not tests:
            tests = [f"{t} (PubChem)" for t in parse_ligand_queries(pubchem_query)]
        has_native = bool(auto_native and native_pdb and Path(str(native_pdb)).is_file())
        if not has_native and len(tests) < 2:
            return ""
        if len(tests) > MAX_BATCH_LIGANDS:
            return status_box("err", f"Fila grande demais ({len(tests)} ligantes de teste)",
                              f"O limite é {MAX_BATCH_LIGANDS} por corrida. Divida em lotes menores.")
        mt = str(prep_mode or "1")[:1]
        lines = ([f"<b>#0 [Controle - Modo 2]</b> {esc(bm_native_resname(native_pdb))} (ligante nativo, coordenadas de cristal)"] if has_native else [])
        lines += [f"<b>#{k} [Teste - Modo {esc(mt)}]</b> {esc(nm)}" for k, nm in enumerate(tests[:12], 1)]
        if len(tests) > 12:
            lines.append(f"… (+{len(tests) - 12} ligantes de teste)")
        if has_native and not tests:
            lines.append("Nenhum ligante de teste informado: será executado somente o controle nativo.")
        title = f"Fila de triagem: {len(tests) + (1 if has_native else 0)} ligante(s)" if tests or has_native else "Fila"
        return status_box("info", title, "<br>".join(lines) + "<br>Processamento sequencial, com o mesmo receptor e a mesma caixa.")
    except Exception:
        return ""


def key_interactions_summary(df_prolif, idx, max_per_type=6):
    """'HB: Ser289, Asn260; PiCat: Arg288; Hf: Leu330' a partir da linha da pose exportada."""
    try:
        if df_prolif is None or len(df_prolif) <= idx:
            return "—"
        if "Status_Contatos" in df_prolif.columns:
            return "n/d (ProLIF sem contatos nesta corrida)"
        row, found = df_prolif.iloc[idx], {}
        for col in df_prolif.columns:
            if "_" not in str(col):
                continue
            res, inter = str(col).rsplit("_", 1)
            ab = INTER_ABBR.get(inter)
            v = row[col]
            if ab is None or not isinstance(v, (bool, np.bool_)) or not bool(v):
                continue
            m = re.match(r"^(.*?)(-?\d+)(?:\.[A-Za-z0-9]+)?$", res)
            if not m:
                continue
            found.setdefault(ab, {})[int(m.group(2))] = m.group(1).capitalize() + m.group(2)
        parts = []
        for ab in INTER_ORDER:
            if ab in found:
                labs = [found[ab][k] for k in sorted(found[ab])]
                extra = f" (+{len(labs) - max_per_type})" if len(labs) > max_per_type else ""
                parts.append(f"{ab}: {', '.join(labs[:max_per_type])}{extra}")
        return "; ".join(parts) if parts else "—"
    except Exception:
        return "—"


def bm_posthoc(rec, ctrl):
    """Compara um ligante com o controle FINAL do lote (independe da ordem em que o nativo foi processado)."""
    if rec is None or rec.get("ctx") is None:
        return "no_native", None, [], []
    if ctrl is None:
        return "no_control", None, [], []
    if rec is ctrl:
        return "control_saved", None, [], []
    hard, soft = bm_context_diff(rec["ctx"], ctrl["ctx"])
    if hard:
        return "stale", None, hard, soft
    return "compared", bm_compare(rec, ctrl), [], soft


def _delta_status(mode, hard):
    if mode == "no_native":
        return NO_NATIVE_TXT
    if mode == "stale":
        return BM_NO_CONTROL + " — controle invalidado: " + "; ".join(hard)
    return BM_NO_CONTROL


def _ranked(results, ctrl):
    """[(resultado, papel, modo, cmp, hard, soft)]: controle primeiro; testes por PB=PASS, Vina e CNNaffinity."""
    ctrl_rows, tests = [], []
    for r in results:
        if not r.get("ok"):
            continue
        rec = r["rec"]
        mode, cmp, hard, soft = bm_posthoc(rec, ctrl)
        if mode == "control_saved":
            ctrl_rows.append((r, ROLE_CTRL, mode, cmp, hard, soft))
        elif r.get("is_native"):
            ctrl_rows.append((r, ROLE_CTRL_NV, "unvalidated", None, [], []))
        else:
            tests.append((r, ROLE_TEST, mode, cmp, hard, soft))
    tests.sort(key=lambda it: (it[0]["rec"]["pb"] != "PASS", it[0]["rec"]["vina"], -it[0]["rec"]["pk"]))
    return ctrl_rows + tests


def _na(v, nd):
    return round(v, nd) if v is not None else "n/d"


def build_leaderboard(results, ctrl):
    rows, rank = [], 0
    for r, role, mode, cmp, hard, soft in _ranked(results, ctrl):
        t = r["rec"]
        if is_ctrl(role):
            d = {"Rank": "ref"}
        else:
            rank += 1
            d = {"Rank": rank}
        nd = _delta_status(mode, hard) if cmp is None and not is_ctrl(role) else None
        d.update({"Composto": r["lig_name"], "Papel": role, "HAC": t["hac"], "Vina (kcal/mol)": _na(t["vina"], 2),
                  "ΔVina (kcal/mol)": "—" if is_ctrl(role) else (round(cmp["dvina"], 2) if cmp else "n/d"),
                  "CNNscore": _na(t["cnn_s"], 3), "CNNaffinity (pK)": _na(t["pk"], 2),
                  "ΔpK": "—" if is_ctrl(role) else (round(cmp["dpk"], 2) if cmp else "n/d"),
                  "Potência relativa (10^ΔpK)": "—" if is_ctrl(role) else (round(cmp["ratio"], 2) if cmp else "n/d"),
                  "LE Vina": _na(t["le_v"], 3),
                  "ΔLE Vina": "—" if is_ctrl(role) else (_na(cmp["dle_v"], 3) if cmp else "n/d"),
                  "LE CNN": _na(t["le_c"], 3),
                  "ΔLE CNN": "—" if is_ctrl(role) else (_na(cmp["dle_c"], 3) if cmp else "n/d"),
                  "PoseBusters": t["pb"],
                  "Classe vs. nativo": ("Controle" if role == ROLE_CTRL else "Controle (RMSD não validado)") if is_ctrl(role) else (cmp["veredito"] if cmp else (nd or "n/d")),
                  "Pose exportada": f"#{t['pose']}"})
        rows.append(d)
    for r in results:
        if not r.get("ok"):
            rows.append({"Rank": "—", "Composto": r["name"], "Papel": "Falha", "Classe vs. nativo": "FALHA: " + str(r["reason"])[:90]})
    return pd.DataFrame(rows)


def build_tabela_artigo(results, ctrl):
    rows = []
    for r, role, mode, cmp, hard, soft in _ranked(results, ctrl):
        t = r["rec"]
        rows.append([r["lig_name"], role, t["hac"], _na(t["vina"], 2),
                     "—" if is_ctrl(role) else (round(cmp["dvina"], 2) if cmp else "n/d"),
                     _na(t["cnn_s"], 3), _na(t["pk"], 2), _na(t["le_v"], 3),
                     (f"{t['rmsd']:.2f}" if t.get("rmsd") is not None else "n/d") if is_ctrl(role) else "—",
                     "PASS" if t["pb"] == "PASS" else "FAIL", r.get("key_inter", "—")])
    return pd.DataFrame(rows, columns=ARTIGO_COLS)


def artigo_notes(results, ctrl):
    ok = [r for r in results if r.get("ok")]
    n_fail = len(results) - len(ok)
    first = ok[0] if ok else {}
    notes = [
        "Pose apresentada = a de maior CNNscore entre as aprovadas no PoseBusters (22 testes oficiais, modo dock); sem pose aprovada, "
        "usa-se a Pose 1 e a coluna PoseBusters indica FAIL.",
        f"ΔVina = Vina(composto) − Vina(controle nativo); valores negativos indicam energia estimada mais favorável. Banda de "
        f"equivalência: ±{BM_BAND_KCAL} kcal/mol. Escores de docking não equivalem a afinidade experimental.",
        "LE Vina = −Vina/HAC (kcal/mol por átomo pesado). HAC = número de átomos pesados do ligante preparado.",
        "Veredito (coluna 'Classe vs. nativo' do Leaderboard), nesta ordem: (1) Inviável (PoseBusters FAIL) = pose sem PB_Valid; "
        "(2) Inconclusivo (Discordância Vina/CNN) = Vina e CNN em classes opostas; (3) Superior (Consenso Robusto) = ΔVina ≤ −0,5 kcal/mol com a "
        "CNN Superior ou Equivalente; Superior (Ressalva: Baixa LE / Efeito Volume) = o mesmo, mas com LE Vina < 0,28 e ΔLE Vina < −0,05 "
        "(ganho de energia explicado por tamanho molecular); (4) Equivalente (|ΔVina| < 0,5): Promissor (LE/CNN Superior ao Nativo) se a CNN "
        "for Superior e ΔLE Vina > 0,02, Equivalente (Alerta: CNN Desfavorável) se a CNN for Inferior, senão Equivalente ao Controle; "
        "(5) Vina Inferior (ΔVina ≥ +0,5 kcal/mol): Bioativo Eficiente (Potencial Lead-like) se LE Vina ≥ 0,35, senão Inferior ao Nativo. A classe da CNN usa a mesma banda em ΔG = −1,364·ΔpK. Ranking computacional, não afinidade experimental.",
        "RMSD (Å): simétrico (spyrmsd, sem superposição prévia) da pose do controle nativo frente ao cristal; '—' para compostos de teste.",
        "Interações Chave (ProLIF, pose apresentada): HB = ligação de hidrogênio; PiPi = empilhamento π; PiCat = π–cátion; "
        "Hf = contato hidrofóbico; Ion = interação iônica; Metal = coordenação ao íon metálico.",
        f"Protocolo: GNINA 1.3.2 (semente {RANDOM_SEED}; exaustividade {first.get('exh', 'n/d')}; {GNINA_DOCK_PARAMS['num_modes']} modos; refino --minimize), "
        f"receptor {first.get('pdb_id', '')} cadeia {first.get('chain', '')}, dispositivo: {first.get('gnina_dev', 'n/d')}.",
    ]
    if ctrl is None:
        notes.append("Controle nativo ausente: os ΔVina aparecem como n/d. Rode primeiro o redocking do ligante cristalográfico.")
    if n_fail:
        notes.append(f"{n_fail} composto(s) do lote falharam e não constam nesta tabela (ver Metadados_Sessao).")
    return notes


# --- Parecer científico automático -------------------------------------------------------
def faixa_molar_pk(pk) -> str:
    """Ordem de grandeza do Kd implícita no pK da CNN (Kd = 10^-pK); estimativa, não medida."""
    if pk is None:
        return "n/d"
    if pk >= 8.5:
        return "nanomolar estrita (Kd ≤ 3 nM)"
    if pk >= 6.5:
        return "submicromolar (Kd ≈ 3 a 300 nM)"
    if pk >= 5.0:
        return "micromolar (Kd ≈ 0,3 a 10 µM)"
    return "interação lábil / fraca (Kd > 10 µM)"


def _kd_txt(pk) -> str:
    kd = 10.0 ** (-float(pk))
    if kd < 1e-6:
        return f"{kd * 1e9:.1f} nM"
    if kd < 1e-3:
        return f"{kd * 1e6:.1f} µM"
    return f"{kd * 1e3:.1f} mM"


_INTER_TXT = {
    "HB": "ligações de hidrogênio ({}), que dão estabilidade direcional e especificidade ao encaixe",
    "PiPi": "empilhamento π ({}), que posiciona o anel aromático na fenda",
    "PiCat": "interação π–cátion ({}), que reforça a ancoragem eletrostática",
    "Hf": "contatos hidrofóbicos ({}), que estabilizam o ligante na fenda apolar",
    "Ion": "interações iônicas ({}), que contribuem com atração eletrostática de longo alcance",
    "Metal": "contato/coordenação com o metal do sítio ({}), relevante para o centro catalítico",
}


def _interacoes_txt(key_inter) -> str:
    s = str(key_inter or "—").strip()
    if s in ("", "—") or s.startswith("n/d"):
        return "O ProLIF não registrou interações específicas para a pose exportada."
    partes = []
    for bloco in s.split(";"):
        if ":" not in bloco:
            continue
        ab, res = (x.strip() for x in bloco.split(":", 1))
        if ab in _INTER_TXT and res:
            partes.append(_INTER_TXT[ab].format(res))
    if not partes:
        return "O ProLIF não registrou interações específicas para a pose exportada."
    return "A pose exibe " + "; ".join(partes) + "."


def _cap(s):
    s = str(s)
    return s[:1].upper() + s[1:]


def veredito_final(r, ctrl, role=None, cmp=None):
    t = r["rec"]
    if r.get("is_native") or (ctrl is not None and t is ctrl):
        return "Controle nativo (referência)"
    if cmp is not None:
        return cmp["veredito"]
    return "n/d (sem controle nativo válido)"


def gerar_parecer_cientifico(r, ctrl, perfil) -> str:
    """Parágrafo analítico por ligante. Computacional: não substitui medida de afinidade, ADME ou toxicidade."""
    perfil = perfil or PERFIL_FALLBACK
    t = r["rec"]
    mode, cmp, hard, soft = bm_posthoc(t, ctrl)
    ver = veredito_final(r, ctrl, cmp=cmp)
    nome = r.get("lig_name", t.get("lig_name", "o ligante"))
    termo = _cap(perfil["termo_substantivo"])
    mw = perfil.get("mw")
    if mw is not None:
        p = [f"O {nome} enquadra-se como um {perfil['classe_estrutural']} de {perfil['perfil_medchem']} (MW {mw:.0f} Da; {perfil['hac']} átomos pesados)."]
    else:
        p = [f"O {nome} enquadra-se como um {perfil['classe_estrutural']}."]
    pk, le = t.get("pk"), t.get("le_v")
    p.append(f"{termo} tem CNNaffinity de {pk:.2f} pK, o que corresponde a Kd ≈ {_kd_txt(pk)} (faixa {faixa_molar_pk(pk)}; "
             f"ordem de grandeza prevista pela rede, não medida), com Vina de {t['vina']:.2f} kcal/mol e eficiência de ligante (LE Vina) de "
             f"{_bm_fmt(le, 2)} kcal/mol por átomo pesado.")
    p.append("A pose exportada " + ("passou" if t.get("pb") == "PASS" else "NÃO passou") + f" no PoseBusters (CNNscore {_bm_fmt(t.get('cnn_s'), 2)}). "
             + _interacoes_txt(r.get("key_inter")))
    grp = ver.split(" ")[0]
    if cmp is None:
        if ver.startswith("Controle"):
            rm = t.get("rmsd")
            p.append("Este é o ligante cristalográfico re-docado, usado como referência dos Δ"
                     + (f" (RMSD simétrico {rm:.2f} Å)." if rm is not None else "."))
        else:
            p.append("Sem controle nativo válido neste lote, a comparação com o cristal não é possível.")
    else:
        dv, dpk, dle = cmp["dvina"], cmp["dpk"], cmp["dle_v"]
        base = f"Frente ao nativo, ΔVina = {dv:+.2f} kcal/mol e ΔpK = {dpk:+.2f} (potência prevista ≈ {_bm_ratio(cmp['ratio'])}; ΔLE Vina = {_bm_fmt(dle, 3, True)}). "
        if grp == "Inviável":
            p.append(base + "Como a pose reprova nos testes físico-químicos, os escores não são interpretáveis: refaça o docking (maior exaustividade) antes de qualquer conclusão.")
        elif grp == "Inconclusivo":
            p.append(base + "Vina e CNN apontam direções opostas; trate o resultado como inconclusivo e confirme com maior exaustividade ou outro rescoring.")
        elif ver.startswith("Superior (Ressalva"):
            p.append(base + f"A vantagem energética vem acompanhada de LE baixa ({_bm_fmt(le, 2)}), isto é, parte do ganho decorre do tamanho molecular: "
                     "o composto merece otimização de eficiência (remoção de massa não contributiva) antes de ser priorizado.")
        elif grp == "Superior":
            p.append(base + "Vina e CNN concordam em superioridade e a LE sustenta o ganho: é um candidato a priorização para validação experimental.")
        elif grp == "Promissor":
            p.append(base + "Embora equivalente em energia pura, a CNN e a LE favorecem o composto frente ao nativo, o que o torna candidato à priorização.")
        elif grp == "Bioativo":
            p.append(base + f"Mesmo abaixo do nativo em energia pura, a LE de {_bm_fmt(le, 2)} (≥ 0,35) indica alta eficiência atômica: "
                     "um ponto de partida lead-like para crescimento ou ligação de fragmentos.")
        elif ver.startswith("Equivalente (Alerta"):
            p.append(base + "O Vina o iguala ao nativo, mas a CNN é desfavorável; vale confirmar com rescoring antes de seguir.")
        elif grp == "Equivalente":
            p.append(base + "Sem diferença relevante de energia frente ao nativo; o composto é viável como protótipo, e a escolha pode se apoiar em síntese, seletividade e propriedades ADME.")
        else:
            p.append(base + "A afinidade prevista é inferior à do nativo, mas isso não descarta o composto como protótipo: "
                     f"com LE de {_bm_fmt(le, 2)}, ele pode servir de andaime para otimização.")
        if soft:
            p.append("Ressalvas da comparação: " + "; ".join(soft) + ".")
    p.append("Docking é triagem computacional: toxicidade, ADME e seletividade não são avaliadas aqui.")
    return " ".join(p)


def parecer_html(r, ctrl, perfil=None) -> str:
    try:
        txt = gerar_parecer_cientifico(r, ctrl, perfil if perfil is not None else r.get("perfil"))
    except Exception as e_par:
        return f'<div class="dd-detail">Parecer automático indisponível: {esc(type(e_par).__name__)}: {esc(str(e_par)[:120])}</div>'
    return ('<div class="dd-explain dd-blue" style="margin-top:8px"><h3>Parecer científico (automático)</h3>'
            f'<p style="margin:0">{esc(txt)}</p></div>')


PARECER_COLS = ["Composto", "Perfil_Taxonômico", "Veredito_Final", "Faixa_Molar_Estimada", "Parecer_Descritivo"]


def build_parecer_df(results, ctrl):
    rows = []
    for r, role, mode, cmp, hard, soft in _ranked(results, ctrl):
        perfil = r.get("perfil") or PERFIL_FALLBACK
        try:
            txt = gerar_parecer_cientifico(r, ctrl, perfil)
        except Exception as e_par:
            txt = f"Parecer indisponível: {type(e_par).__name__}: {str(e_par)[:120]}"
        mw = perfil.get("mw")
        rows.append([r["lig_name"],
                     f"{perfil['classe_estrutural']}; {perfil['perfil_medchem']}" + (f" (MW {mw:.0f} Da; HAC {perfil['hac']})" if mw is not None else ""),
                     veredito_final(r, ctrl, cmp=cmp), faixa_molar_pk(r["rec"].get("pk")), txt])
    return pd.DataFrame(rows, columns=PARECER_COLS)


def _style_parecer(ws):
    from openpyxl.styles import Alignment, Font, PatternFill
    for j, wd in enumerate([28, 46, 36, 34, 120], 1):
        ws.column_dimensions[chr(64 + j)].width = wd
        c = ws.cell(row=1, column=j)
        c.font, c.fill = Font(bold=True, color="FFFFFF"), PatternFill("solid", fgColor="3F3D8F")
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in ws.iter_rows(min_row=2):
        for c in row:
            c.alignment = Alignment(vertical="top", wrap_text=True)


# --- Gráficos de publicação: Figura 1 (fingerprint ProLIF) e Figura 2 (paisagem de LE) --------
FIG_DIRNAME = "06_graficos_artigo"
FIG_LANGS = ("pt", "en")                          # as duas figuras são geradas automaticamente nos dois idiomas
FIG_I18N = {
    "pt": {"res_axis": "Resíduo do receptor", "leg_inter": "Interação",
           "cats": ["Sem contato", "Ligação de hidrogênio", "Contato hidrofóbico", "Empilhamento π / π–cátion",
                    "Coordenação metálica / iônica"],
           "hac_axis": "Átomos pesados (HAC)", "vina_axis": "Vina score (kcal/mol)", "ctrl": "Controle nativo",
           "le_line": "LE = 0,30", "leg_verdict": "Veredito (bolha = LE Vina)", "no_ctrl": "Sem controle",
           "no_prolif": "Sem interações ProLIF disponíveis neste lote", "no_data": "Sem dados de docking para plotar",
           "verd": {"Superior": "Superior", "Promissor": "Promissor", "Equivalente": "Equivalente", "Bioativo": "Bioativo",
                    "Inconclusivo": "Inconclusivo", "Inferior": "Inferior", "Inviável": "Inviável"}},
    "en": {"res_axis": "Receptor residue", "leg_inter": "Interaction",
           "cats": ["No contact", "Hydrogen bond", "Hydrophobic contact", "π-stacking / π-cation", "Metal coordination / Ionic"],
           "hac_axis": "Heavy atoms (HAC)", "vina_axis": "Vina score (kcal/mol)", "ctrl": "Native control",
           "le_line": "LE = 0.30", "leg_verdict": "Verdict (bubble = Vina LE)", "no_ctrl": "No control",
           "no_prolif": "No ProLIF interactions available in this batch", "no_data": "No docking data to plot",
           "verd": {"Superior": "Superior", "Promissor": "Promising", "Equivalente": "Equivalent", "Bioativo": "Bioactive",
                    "Inconclusivo": "Inconclusive", "Inferior": "Inferior", "Inviável": "Infeasible"}},
}
FIG_CAT_COLORS = ["#E2E8F0", "#10B981", "#FF7F50", "#8B5CF6", "#0EA5E9"]
FIG_CATS = list(zip(FIG_I18N["pt"]["cats"], FIG_CAT_COLORS))      # compatibilidade: rótulos em português
_AB_CAT = {"HB": 1, "Hf": 2, "PiPi": 3, "PiCat": 3, "Ion": 4, "Metal": 4}
_CAT_PRIORITY = [4, 1, 3, 2]                      # se um resíduo tem vários contatos, vale a categoria mais forte
VERDICT_COLORS = {"Superior": "#10B981", "Promissor": "#0EA5E9", "Equivalente": "#94A3B8", "Bioativo": "#14B8A6",
                  "Inconclusivo": "#F59E0B", "Inferior": "#FF7F50", "Inviável": "#EF4444"}
CONTROL_COLOR = "#1E293B"                         # cinza chumbo escuro: o controle nativo se destaca dos veredictos
INK = "#334155"                                   # cinza chumbo dos eixos, ticks e textos
FIG_DPI = 600


def prolif_contatos_pose(df_prolif, idx):
    """{(número, rótulo): categoria 1-4} para a pose idx; vazio se o ProLIF não trouxe contatos."""
    out = {}
    try:
        if df_prolif is None or len(df_prolif) <= idx or "Status_Contatos" in df_prolif.columns:
            return out
        row = df_prolif.iloc[idx]
        for col in df_prolif.columns:
            if "_" not in str(col):
                continue
            res, inter = str(col).rsplit("_", 1)
            cat = _AB_CAT.get(INTER_ABBR.get(inter))
            v = row[col]
            if cat is None or not isinstance(v, (bool, np.bool_)) or not bool(v):
                continue
            m = re.match(r"^(.*?)(-?\d+)(?:\.[A-Za-z0-9]+)?$", res)
            if not m:
                continue
            key = (int(m.group(2)), m.group(1).capitalize() + m.group(2))
            old = out.get(key)
            if old is None or _CAT_PRIORITY.index(cat) < _CAT_PRIORITY.index(old):
                out[key] = cat
    except Exception:
        return {}
    return out


_FIG_RC = {"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
           "axes.edgecolor": INK, "axes.linewidth": 1.5, "axes.labelcolor": INK, "axes.labelweight": "bold",
           "axes.titleweight": "bold", "axes.titlecolor": INK, "text.color": INK, "xtick.color": INK, "ytick.color": INK,
           "xtick.major.width": 1.5, "ytick.major.width": 1.5, "xtick.labelcolor": INK, "ytick.labelcolor": INK,
           "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
           "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none"}


def _fig_style():
    """Contexto de rcParams editorial; restaurado ao sair, sem poluir o notebook."""
    import logging
    logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
    return plt.rc_context(_FIG_RC)


def _tema():
    try:
        sns.set_theme(style="ticks", context="talk")
    except Exception:
        pass
    plt.rcParams.update(_FIG_RC)                      # o tema do seaborn redefine cores e fonte: reaplica a identidade visual


def _despine(fig, ax):
    try:
        sns.despine(fig=fig, ax=ax, trim=True, offset=10)
    except Exception:
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)


def _save_fig(fig, base_path):
    import warnings as _w
    try:
        with _w.catch_warnings():
            _w.simplefilter("ignore")
            fig.tight_layout(pad=2.0)
    except Exception:
        pass
    paths = []
    for ext in ("png", "svg", "pdf"):
        p = Path(f"{base_path}.{ext}")
        kw = {"dpi": FIG_DPI} if ext == "png" else {}
        fig.savefig(str(p), bbox_inches="tight", pad_inches=0.1, facecolor="white", **kw)
        paths.append(str(p))
    return paths


def _figura_interacoes(ranked, base_path, lang="pt"):
    L = FIG_I18N[lang]
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch
    names, contatos, n_ctrl = [], [], 0
    for r, role, mode, cmp, hard, soft in ranked:
        names.append(str(r["lig_name"])[:30])
        n_ctrl += 1 if is_ctrl(role) else 0
        contatos.append(prolif_contatos_pose(r.get("df_prolif"), r.get("selected_idx", 0)))
    cols = sorted({k for c in contatos for k in c})
    with _fig_style():
        _tema()
        if cols:
            rows = list(range(len(names)))
            if 0 < n_ctrl < len(names):                     # linha em branco separa o controle (topo) dos testes
                rows.insert(n_ctrl, None)
            mat = np.zeros((len(rows), len(cols)))
            mask = np.zeros(mat.shape, dtype=bool)
            ylab, bold = [], []
            for i, src_i in enumerate(rows):
                if src_i is None:
                    mask[i, :] = True
                    ylab.append("")
                    continue
                ylab.append(names[src_i])
                bold.append(i if src_i < n_ctrl else -1)
                for j, k in enumerate(cols):
                    mat[i, j] = contatos[src_i].get(k, 0)
            cell = 0.62
            fig, ax = plt.subplots(figsize=(max(5.0, cell * len(cols) + 5.5), max(3.0, cell * len(rows) + 2.4)))
            sns.heatmap(mat, mask=mask, cmap=ListedColormap(FIG_CAT_COLORS), vmin=-0.5, vmax=4.5, square=True,
                        linewidths=2.5, linecolor="white", cbar=False, ax=ax, xticklabels=False, yticklabels=False)
            _despine(fig, ax)                               # antes dos rótulos: trim=True redefine os ticks do eixo
            ax.set_xticks(np.arange(len(cols)) + 0.5)
            ax.set_xticklabels([k[1] for k in cols], rotation=45, ha="right", rotation_mode="anchor", fontsize=13)
            keep = [i for i, r_ in enumerate(rows) if r_ is not None]      # sem tick na linha-espaçadora
            ax.set_yticks([i + 0.5 for i in keep])
            ax.set_yticklabels([ylab[i] for i in keep], fontsize=13)
            for i, lab in zip(keep, ax.get_yticklabels()):
                if i in bold:
                    lab.set_fontweight("bold")
            ax.set_xlabel(L["res_axis"], fontweight="bold", fontsize=15)
            ax.set_ylabel("")
            ax.tick_params(length=4)
            ax.legend(handles=[Patch(facecolor=c, edgecolor="#E2E8F0", linewidth=1.0, label=l) for l, c in zip(L["cats"], FIG_CAT_COLORS)], loc="upper left",
                      bbox_to_anchor=(1.04, 1.0), frameon=False, fontsize=12, title=L["leg_inter"], title_fontsize=13, alignment="left")
        else:
            fig, ax = plt.subplots(figsize=(6, 2.6))
            ax.axis("off")
            ax.text(0.5, 0.5, L["no_prolif"], ha="center", va="center", fontsize=13)
        paths = _save_fig(fig, base_path)
        plt.close(fig)
    return paths


def _retangulo_circulo(rect, c, r):
    nx, ny = min(max(c[0], rect[0]), rect[2]), min(max(c[1], rect[1]), rect[3])
    return (nx - c[0]) ** 2 + (ny - c[1]) ** 2 < r * r


def _retangulos(a, b, pad=2.0):
    return not (a[2] + pad <= b[0] or b[2] + pad <= a[0] or a[3] + pad <= b[1] or b[3] + pad <= a[1])


def _geometria_bolhas(ax, xs, ys, radii_pt):
    k = ax.figure.dpi / 72.0
    return ax.transData.transform(np.column_stack([xs, ys])), (np.asarray(radii_pt, float) + 6.0) * k


def _rotulos_ok(ax, texts, xs, ys, radii_pt):
    """True se nenhum rótulo toca uma bolha (inclusive o diamante do controle), outro rótulo ou sai do eixo."""
    fig = ax.figure
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    cen, rad = _geometria_bolhas(ax, xs, ys, radii_pt)
    axbb = ax.get_window_extent(rend)
    rects = []
    for t in texts:
        bb = t.get_window_extent(rend)
        rects.append((bb.x0, bb.y0, bb.x1, bb.y1))
    for i, rc in enumerate(rects):
        if rc[0] < axbb.x0 - 2 or rc[2] > axbb.x1 + 2 or rc[1] < axbb.y0 - 2 or rc[3] > axbb.y1 + 2:
            return False
        if any(_retangulo_circulo(rc, cen[j], rad[j]) for j in range(len(cen))):
            return False
        if any(_retangulos(rc, rects[j]) for j in range(i)):
            return False
    return True


def _pos_rotulo(t):
    """Estado de posição de um rótulo (Text ou Annotation), para restaurar depois."""
    from matplotlib.text import Annotation
    if isinstance(t, Annotation):
        return ("ann", t.anncoords, tuple(t.xyann), t.get_ha(), t.get_va())
    return ("txt", None, tuple(t.get_position()), t.get_ha(), t.get_va())


def _repor_rotulo(t, st):
    kind, coords, xy, ha, va = st
    if kind == "ann":
        t.anncoords = coords
        t.xyann = xy
    else:
        t.set_position(xy)
    t.set_ha(ha); t.set_va(va)


def _mover_rotulo_dados(t, xy):
    """Move o rótulo para uma coordenada de dados (um Annotation com deslocamento em pontos passa a usar coordenadas de dados)."""
    from matplotlib.text import Annotation
    if isinstance(t, Annotation):
        t.anncoords = "data"
        t.xyann = (float(xy[0]), float(xy[1]))
    else:
        t.set_position((float(xy[0]), float(xy[1])))


def _colocar_rotulos(ax, texts, xs, ys, radii_pt):
    """Posiciona cada rótulo fora de todas as bolhas e dos outros rótulos: candidatos em 8 direções e 4 distâncias, do mais
    próximo ao mais distante. Rótulos afastados (>= 12 pt) ganham um conector fino."""
    fig = ax.figure
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    k = fig.dpi / 72.0
    cen, rad = _geometria_bolhas(ax, xs, ys, radii_pt)
    axbb = ax.get_window_extent(rend)
    inv = ax.transData.inverted()
    placed, extra = [], []
    for i in np.argsort(-rad):
        t = texts[i]
        t.set_ha("center"); t.set_va("center")
        bb = t.get_window_extent(rend)
        w, h = bb.width, bb.height
        best = None
        for d in (4, 12, 24, 40):
            for ang in (45, 0, 90, 135, 180, 315, 270, 225):
                dx, dy = np.cos(np.deg2rad(ang)), np.sin(np.deg2rad(ang))
                dist = rad[i] + d * k
                cx, cy = cen[i, 0] + dx * (dist + w / 2), cen[i, 1] + dy * (dist + h / 2)
                rc = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
                pen = sum(10 for j in range(len(cen)) if _retangulo_circulo(rc, cen[j], rad[j]))
                pen += sum(10 for p in placed if _retangulos(rc, p))
                if rc[0] < axbb.x0 or rc[2] > axbb.x1 or rc[1] < axbb.y0 or rc[3] > axbb.y1:
                    pen += 5
                if best is None or pen < best[0]:
                    best = (pen, cx, cy, d, rc)
                if pen == 0:
                    break
            if best[0] == 0:
                break
        pen, cx, cy, d, rc = best
        _mover_rotulo_dados(t, inv.transform((cx, cy)))
        placed.append(rc)
        if d >= 12:
            ex = min(max(cen[i, 0], rc[0]), rc[2]); ey = min(max(cen[i, 1], rc[1]), rc[3])
            v = np.array([ex - cen[i, 0], ey - cen[i, 1]]); nrm = float(np.hypot(*v)) or 1.0
            p0 = cen[i] + v / nrm * (rad[i] - 6.0 * k)
            (x0_, y0_), (x1_, y1_) = inv.transform(p0), inv.transform((ex, ey))
            extra += ax.plot([x0_, x1_], [y0_, y1_], color="#94A3B8", lw=1, zorder=2, solid_capstyle="round")
    return extra


def _ajustar_textos(ax, texts, xs, ys, radii_pt):
    """1) adjustText só para o ajuste fino ENTRE os textos (only_move: textos movem, pontos não) com repulsão forte contra os pontos
    (x, y explícitos); 2) valida; 3) se algum rótulo tocar uma bolha, outro rótulo ou sair do eixo, descarta o resultado e usa o
    posicionamento geométrico garantido (rótulo sempre fora da borda do marcador)."""
    if not texts:
        return
    xs, ys = np.asarray(xs, float), np.asarray(ys, float)
    antes = {id(c) for c in ax.get_children()}
    estados = [_pos_rotulo(t) for t in texts]
    keep = {id(t) for t in texts}

    def _limpar():
        for c in list(ax.get_children()):                              # remove setas/artistas criados pelo adjustText
            if id(c) not in antes and id(c) not in keep:
                try:
                    c.remove()
                except Exception:
                    pass
        for t, st in zip(texts, estados):
            _repor_rotulo(t, st)

    if adjust_text is not None:
        arrow = dict(arrowstyle="-", color="#94A3B8", lw=1)
        tentativas = (
            {"x": xs, "y": ys, "ax": ax, "arrowprops": arrow, "force_text": (1.5, 1.5), "force_static": (1.5, 1.5), "expand": (1.2, 1.2),
             "only_move": {"text": "xy", "static": ""}},
            {"x": xs, "y": ys, "ax": ax, "arrowprops": arrow, "force_points": 1.5, "force_text": 1.5,
             "expand_points": (1.5, 1.5), "expand_text": (1.2, 1.2), "only_move": {"text": "xy", "points": ""}},
            {"x": xs, "y": ys, "ax": ax, "arrowprops": arrow},
        )
        for kw in tentativas:
            try:
                adjust_text(texts, **kw)
                break
            except Exception:                                          # assinatura/valor incompatível com esta versão: próxima forma
                _limpar()
    try:
        ok = _rotulos_ok(ax, texts, xs, ys, radii_pt)
    except Exception:
        ok = False
    if ok:
        return
    _limpar()
    _colocar_rotulos(ax, texts, xs, ys, radii_pt)


def _figura_le(ranked, base_path, lang="pt"):
    from matplotlib.lines import Line2D
    L = FIG_I18N[lang]
    pts = []
    for r, role, mode, cmp, hard, soft in ranked:
        t = r["rec"]
        if t.get("vina") is None or not t.get("hac"):
            continue
        le = t.get("le_v") if t.get("le_v") is not None else -t["vina"] / t["hac"]
        ver = "Controle" if is_ctrl(role) else (cmp["veredito"] if cmp else "sem controle")
        pts.append((str(r["lig_name"])[:28], float(t["hac"]), float(t["vina"]), float(le), ver))
    with _fig_style():
        _tema()
        fig, ax = plt.subplots(figsize=(9.2, 6.4))
        if not pts:
            ax.axis("off")
            ax.text(0.5, 0.5, L["no_data"], ha="center", va="center", fontsize=13)
        else:
            xs = np.array([p[1] for p in pts]); ys = np.array([p[2] for p in pts])
            xspan = max(float(xs.max() - xs.min()), 10.0)
            x0, x1 = max(0.0, float(xs.min()) - 0.18 * xspan), float(xs.max()) + 0.18 * xspan
            ypad = max(1.0, 0.3 * float(ys.max() - ys.min()))
            ytop, ybot = float(ys.min()) - ypad, float(ys.max()) + ypad       # eixo invertido: mais negativo no topo
            xl = np.array([x0, x1])
            ax.plot(xl, -0.30 * xl, color="#CBD5E1", linestyle=":", lw=1.5, zorder=0)
            texts, used, radii = [], {}, []
            for nm, hac, vina, le, ver in pts:
                ctl = ver == "Controle"
                grp = ver.split(" ")[0]
                col = CONTROL_COLOR if ctl else VERDICT_COLORS.get(grp, "#94A3B8")
                area = (150 + 1500 * max(0.0, min(le, 1.0)) ** 1.5) * (1.5 if ctl else 1.0)
                ax.scatter([hac], [vina], s=area, c=col, marker="D" if ctl else "o", edgecolor="white", linewidth=2.5, alpha=0.85,
                           zorder=4 if ctl else 3)
                r_pt = float(np.sqrt(area) * 0.75) if ctl else float(np.sqrt(area / np.pi))      # raio físico efetivo do marcador, em pontos
                radii.append(r_pt)
                texts.append(ax.annotate(nm, xy=(hac, vina), xytext=(r_pt + 8.0, 0.0), textcoords="offset points", ha="left",
                                         va="center", fontsize=10, color=INK, zorder=6,
                                         bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.75, edgecolor="none")))
                used[L["ctrl"] if ctl else (L["verd"][grp] if grp in VERDICT_COLORS else L["no_ctrl"])] = col
            ax.set_xlim(x0, x1)
            ax.set_ylim(ybot, ytop)
            xl_lab = min(x1, -ytop / 0.30) * 0.985                              # onde a reta entra na janela visível
            if x0 < xl_lab and ytop <= -0.30 * xl_lab <= ybot:
                borda = xl_lab > x1 - 0.12 * (x1 - x0)           # perto da margem direita: rótulo à esquerda da reta
                ax.annotate(L["le_line"], (xl_lab, -0.30 * xl_lab), xytext=(-8 if borda else 8, -4), textcoords="offset points",
                            ha="right" if borda else "left", va="top", fontsize=10, color="#94A3B8")
            ax.set_xlabel(L["hac_axis"], fontweight="bold", fontsize=15)
            ax.set_ylabel(L["vina_axis"], fontweight="bold", fontsize=15)
            ax.legend(handles=[Line2D([0], [0], marker="D" if k == L["ctrl"] else "o", ls="", color=c, markeredgecolor="white",
                                      markeredgewidth=1.5, markersize=11, label=k) for k, c in used.items()],
                      loc="upper left", bbox_to_anchor=(1.03, 1.0), frameon=False, fontsize=12, title=L["leg_verdict"],
                      title_fontsize=12, alignment="left")
            _despine(fig, ax)
            _ajustar_textos(ax, texts, xs, ys, radii)
        paths = _save_fig(fig, base_path)
        plt.close(fig)
    return paths


def gerar_graficos_publicacao(results, ctrl, sdir, prefix_nome):
    """Salva Fig1 (fingerprint ProLIF) e Fig2 (LE), sem título interno, em PT e EN (sufixos _PT/_EN), cada uma em PNG 600 DPI + SVG + PDF.
    Nunca levanta exceção: devolve os caminhos gerados."""
    if plt is None:
        print("⚠️ Gráficos de publicação não gerados: matplotlib/seaborn indisponíveis.")
        return []
    out_dir = Path(sdir) / FIG_DIRNAME
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        ranked = [it for it in _ranked(results, ctrl) if it[0].get("rec")]
    except Exception as e:
        print(f"⚠️ Gráficos de publicação não gerados: {type(e).__name__}: {e}")
        return []
    if not ranked:
        return []
    files = []
    for lang in FIG_LANGS:
        for tag, fn in (("Fig1_Interacoes_ProLIF", _figura_interacoes), ("Fig2_LE_Landscape", _figura_le)):
            try:
                files += fn(ranked, str(out_dir / f"{prefix_nome}_{tag}_{lang.upper()}"), lang)
            except Exception as e:
                plt.close("all")
                print(f"⚠️ {tag}_{lang.upper()} não gerada: {type(e).__name__}: {e}")
    return files


# ==================================================================================================
# RELATÓRIO ACADÊMICO DINÂMICO (módulo academic_reporter v1.6 embutido): Metodologia + Resultados/Discussão em .docx, .pdf, .md, .tex, .txt
# Todo o texto vem de `config` (derivado do estado do app) e do leaderboard; nenhum nome de alvo/ligante no código.
# Nomes internos levam o prefixo _ar_/AR_ para não colidir com o restante da célula.
# ==================================================================================================
import io
import math
import tempfile
import html as _htmllib
from html.parser import HTMLParser as _HTMLParser
from typing import Any, Dict, List, Optional, Tuple

AR_REQUIRED_COLUMNS = ["Composto", "Papel", "HAC", "Vina_score", "CNNscore", "CNNaffinity_pK", "LE_Vina", "RMSD",
                    "PoseBusters", "Interacoes_Chave"]
AR_REQUIRED_CONFIG = ["protein_name", "pdb_id", "target_chain", "has_metals", "metals_list", "ph_value",
                   "box_padding_angstrom", "docking_engine", "ligands_info"]
AR_NUM_COLS = ["HAC", "Vina_score", "CNNscore", "CNNaffinity_pK", "LE_Vina", "RMSD"]

# RT ln(10) em kcal/mol a 298,15 K (R = 1,987204 cal/mol/K): converte pK em energia livre (LE_CNN, ΔΔG).
AR_RT_LN10 = 1.987204e-3 * 298.15 * math.log(10)

AR_PLDDT_HIGH = "Elevada confiabilidade estrutural (pLDDT >= 90)"
AR_PLDDT_MID = "Confiabilidade moderada a boa (70 <= pLDDT < 90)"
AR_PLDDT_LOW = "Baixa confiabilidade / Região potencialmente flexível ou desordenada (pLDDT < 70)"


def _ar_classify_plddt(mean_plddt: Any) -> str:
    """Classe de confiabilidade do bolsão pelo pLDDT médio (limiares do AlphaFold: 90 e 70). '' se não numérico."""
    try:
        v = float(mean_plddt)
    except (TypeError, ValueError):
        return ""
    if v != v:
        return ""
    return AR_PLDDT_HIGH if v >= 90.0 else (AR_PLDDT_MID if v >= 70.0 else AR_PLDDT_LOW)


# --------------------------------------------------------------------------------------------------------------------
# Banco BibTeX (mesmas chaves do referencias.bib do projeto). Só as chaves efetivamente citadas vão para o .bib exportado.
# --------------------------------------------------------------------------------------------------------------------
AR_BIB_DB: Dict[str, str] = {
"berman2000": r"""@article{berman2000,
  author    = {Berman, H. M. and Westbrook, J. and Feng, Z. and Gilliland, G. and Bhat, T. N. and Weissig, H. and Shindyalov, I. N. and Bourne, P. E.},
  title     = {The Protein Data Bank},
  journal   = {Nucleic Acids Research},
  volume    = {28},
  number    = {1},
  pages     = {235--242},
  year      = {2000},
  doi       = {10.1093/nar/28.1.235}
}""",
"landrum2016": r"""@misc{landrum2016,
  author    = {Landrum, Greg and others},
  title     = {{RDKit}: Open-source cheminformatics},
  url       = {https://www.rdkit.org},
  year      = {2016}
}""",
"oboyle2011": r"""@article{oboyle2011,
  author    = {O'Boyle, Noel M. and Banck, Michael and James, Craig A. and Morley, Chris and Vandermeersch, Tim and Hutchison, Geoffrey R.},
  title     = {Open Babel: An open chemical toolbox},
  journal   = {Journal of Cheminformatics},
  volume    = {3},
  number    = {1},
  pages     = {33},
  year      = {2011},
  doi       = {10.1186/1758-2946-3-33}
}""",
"riniker2015": r"""@article{riniker2015,
  author    = {Riniker, Sereina and Landrum, Gregory A.},
  title     = {Better Informed Distance Geometry: Using What We Know To Improve Conformation Generation},
  journal   = {Journal of Chemical Information and Modeling},
  volume    = {55},
  number    = {12},
  pages     = {2562--2574},
  year      = {2015},
  doi       = {10.1021/acs.jcim.5b00588}
}""",
"wang2020": r"""@article{wang2020,
  author    = {Wang, Shuzhe and Witek, Jagna and Landrum, Gregory A. and Riniker, Sereina},
  title     = {Improving Conformer Generation for Small Rings and Macrocycles Based on Distance Geometry and Experimental Torsional-Angle Preferences},
  journal   = {Journal of Chemical Information and Modeling},
  volume    = {60},
  number    = {4},
  pages     = {2044--2058},
  year      = {2020},
  doi       = {10.1021/acs.jcim.0c00025}
}""",
"halgren1996": r"""@article{halgren1996,
  author    = {Halgren, T. A.},
  title     = {Merck molecular force field. I. Basis, form, scope, parameterization, and performance of MMFF94},
  journal   = {Journal of Computational Chemistry},
  volume    = {17},
  number    = {5--6},
  pages     = {490--519},
  year      = {1996},
  doi       = {10.1002/(SICI)1096-987X(199604)17:5/6<490::AID-JCC1>3.0.CO;2-P}
}""",
"trott2009": r"""@article{trott2009,
  author    = {Trott, O. and Olson, A. J.},
  title     = {AutoDock Vina: Improving the speed and accuracy of docking with a new scoring function, efficient optimization, and multithreading},
  journal   = {Journal of Computational Chemistry},
  volume    = {31},
  number    = {2},
  pages     = {455--461},
  year      = {2009},
  doi       = {10.1002/jcc.21334}
}""",
"mcnutt2021": r"""@article{mcnutt2021,
  author    = {McNutt, Andrew T. and Francoeur, Paul and Aggarwal, Rishal and Masuda, Tomohide and Meli, Rocco and Ragoza, Matthew and Sunseri, Jocelyn and Koes, David Ryan},
  title     = {{GNINA} 1.0: molecular docking with deep learning},
  journal   = {Journal of Cheminformatics},
  volume    = {13},
  number    = {1},
  pages     = {43},
  year      = {2021},
  doi       = {10.1186/s13321-021-00522-2}
}""",
"mcnutt2025": r"""@article{mcnutt2025,
  author    = {McNutt, Andrew T. and Li, Yanjing and Meli, Rocco and Aggarwal, Rishal and Koes, David Ryan},
  title     = {{GNINA} 1.3: the next increment in molecular docking with deep learning},
  journal   = {Journal of Cheminformatics},
  volume    = {17},
  number    = {1},
  pages     = {28},
  year      = {2025},
  doi       = {10.1186/s13321-025-00973-x}
}""",
"sunseri2021": r"""@article{sunseri2021,
  author    = {Sunseri, Jocelyn and Koes, David Ryan},
  title     = {Virtual Screening with {GNINA} 1.0},
  journal   = {Molecules},
  volume    = {26},
  number    = {23},
  pages     = {7369},
  year      = {2021},
  doi       = {10.3390/molecules26237369}
}""",
"francoeur2020": r"""@article{francoeur2020,
  author    = {Francoeur, Paul G. and Masuda, Tomohide and Sunseri, Jocelyn and Jia, Andrew and Iovanisci, Richard B. and Snyder, Ian and Koes, David Ryan},
  title     = {Three-Dimensional Convolutional Neural Networks and a Cross-Docked Data Set for Structure-Based Drug Design},
  journal   = {Journal of Chemical Information and Modeling},
  volume    = {60},
  number    = {9},
  pages     = {4200--4215},
  year      = {2020},
  doi       = {10.1021/acs.jcim.0c00411}
}""",
"buttenschoen2024": r"""@article{buttenschoen2024,
  author    = {Buttenschoen, Martin and Morris, Garrett M. and Deane, Charlotte M.},
  title     = {{PoseBusters}: {AI}-based docking methods fail to generate physically valid poses or generalise to novel sequences},
  journal   = {Chemical Science},
  volume    = {15},
  number    = {9},
  pages     = {3130--3139},
  year      = {2024},
  doi       = {10.1039/D3SC04185A}
}""",
"meli2020": r"""@article{meli2020,
  author    = {Meli, Rocco and Biggin, Philip C.},
  title     = {{spyrmsd}: symmetry-corrected {RMSD} calculations in {Python}},
  journal   = {Journal of Cheminformatics},
  volume    = {12},
  number    = {1},
  pages     = {49},
  year      = {2020},
  doi       = {10.1186/s13321-020-00455-2}
}""",
"bouysset2021": r"""@article{bouysset2021,
  author    = {Bouysset, C{\'e}dric and Fiorucci, S{\'e}bastien},
  title     = {{ProLIF}: a library to encode molecular interactions as fingerprints},
  journal   = {Journal of Cheminformatics},
  volume    = {13},
  number    = {1},
  pages     = {72},
  year      = {2021},
  doi       = {10.1186/s13321-021-00548-6}
}""",
"hopkins2004": r"""@article{hopkins2004,
  author    = {Hopkins, A. L. and Groom, C. R. and Alex, A.},
  title     = {Ligand efficiency: a useful metric for lead selection},
  journal   = {Drug Discovery Today},
  volume    = {9},
  number    = {10},
  pages     = {430--431},
  year      = {2004},
  doi       = {10.1016/S1359-6446(04)03069-7}
}""",
"kenny2019": r"""@article{kenny2019,
  author    = {Kenny, P. W.},
  title     = {The nature of ligand efficiency},
  journal   = {Journal of Cheminformatics},
  volume    = {11},
  number    = {1},
  pages     = {8},
  year      = {2019},
  doi       = {10.1186/s13321-019-0330-2}
}""",
"abad2021": r"""@article{abad2021,
  author    = {Abad-Zapatero, C.},
  title     = {Ligand efficiency indices for effective drug discovery: a unifying vector formulation},
  journal   = {Expert Opinion on Drug Discovery},
  volume    = {16},
  number    = {7},
  pages     = {763--775},
  year      = {2021},
  doi       = {10.1080/17460441.2021.1884065}
}""",
"abramson2024": r"""@article{abramson2024,
  author    = {Abramson, Josh and Adler, Jonas and Dunger, Jack and Evans, Richard and Green, Tim and Pritzel, Alexander and others},
  title     = {Accurate structure prediction of biomolecular interactions with {AlphaFold} 3},
  journal   = {Nature},
  volume    = {630},
  number    = {8016},
  pages     = {493--500},
  year      = {2024},
  doi       = {10.1038/s41586-024-07487-w}
}""",
"jumper2021": r"""@article{jumper2021,
  author    = {Jumper, John and Evans, Richard and Pritzel, Alexander and Green, Tim and Figurnov, Michael and Ronneberger, Olaf and others},
  title     = {Highly accurate protein structure prediction with {AlphaFold}},
  journal   = {Nature},
  volume    = {596},
  number    = {7873},
  pages     = {583--589},
  year      = {2021},
  doi       = {10.1038/s41586-021-03819-2}
}""",
"he2026": r"""@article{he2026,
  author    = {He, Songlin and others},
  title     = {Transforming the Landscape of Drug Innovation in {AI} Era: {AlphaFold} 3 and Beyond},
  journal   = {Exploration},
  volume    = {6},
  number    = {5},
  pages     = {70236},
  year      = {2026},
  doi       = {10.1002/exp2.70236}
}""",
}


# --------------------------------------------------------------------------------------------------------------------
# Utilidades de BibTeX
# --------------------------------------------------------------------------------------------------------------------
def _ar_bib_fields(entry: str) -> Dict[str, str]:
    """Extrai campos `chave = {valor}` de uma entrada BibTeX (aceita chaves aninhadas)."""
    out: Dict[str, str] = {}
    for m in re.finditer(r"(\w+)\s*=\s*\{", entry):
        i, depth = m.end(), 1
        while i < len(entry) and depth:
            depth += {"{": 1, "}": -1}.get(entry[i], 0)
            i += 1
        out[m.group(1).lower()] = entry[m.end():i - 1]
    return out


def _ar_bib_plain(s: str) -> str:
    """BibTeX -> texto simples (acentos {\\'e} e chaves de proteção)."""
    s = re.sub(r"\\(?:textit|emph)\{([^{}]*)\}", r"\1", s)
    acc = {"'": {"e": "é", "a": "á", "i": "í", "o": "ó", "u": "ú"}, "`": {"e": "è", "a": "à"}}
    s = re.sub(r"\{\\(['`])([a-zA-Z])\}", lambda m: acc.get(m.group(1), {}).get(m.group(2), m.group(2)), s)
    s = s.replace("{", "").replace("}", "").replace("--", "–")
    return re.sub(r"\s+", " ", s).strip()


def _ar_bib_keys(text: str) -> List[str]:
    return re.findall(r"@\w+\s*\{\s*([^,\s]+)\s*,", text)


def _ar_bib_split(text: str) -> List[str]:
    """Separa um texto .bib em entradas (cada uma de '@tipo{chave,' até a chave '}' que a fecha)."""
    out: List[str] = []
    for m in re.finditer(r"@\w+\s*\{", text):
        i, depth = m.end(), 1
        while i < len(text) and depth:
            depth += {"{": 1, "}": -1}.get(text[i], 0)
            i += 1
        out.append(text[m.start():i].strip())
    return out


def _ar_register_bib(bibtex: Any) -> List[str]:
    """Registra entradas BibTeX (texto ou lista de textos) no banco de referências do módulo, p. ex. o artigo cristalográfico
    original do PDB (`target_paper_bib`). Devolve as chaves registradas. Entrada sem autor, título ou ano levanta ValueError.
    Chave já existente é substituída (a numeração a/b de autor-ano é recalculada)."""
    texts = [bibtex] if isinstance(bibtex, str) else list(bibtex or [])
    keys: List[str] = []
    for t_ in texts:
        for entry in _ar_bib_split(str(t_)):
            k = _ar_bib_keys(entry)[0]
            fl = _ar_bib_fields(entry)
            missing = [x for x in ("author", "title", "year") if not _ar_bib_plain(fl.get(x, ""))]
            if missing:
                raise ValueError(f"entrada BibTeX '{k}' sem os campos: {missing}")
            AR_BIB_DB[k] = entry
            keys.append(k)
    if keys:
        AR_YEAR_SUFFIX.clear()
        AR_YEAR_SUFFIX_READY[0] = False
    return keys


def _ar_bib_names(key: str) -> Tuple[List[Tuple[str, str]], bool]:
    """Autores de uma entrada do banco -> ([(sobrenome, prenomes)], há 'and others')."""
    raw = _ar_bib_fields(AR_BIB_DB[key]).get("author", "")
    names: List[Tuple[str, str]] = []
    others = False
    for part in re.split(r"\s+and\s+", raw.strip()):
        p = _ar_bib_plain(part)
        if not p:
            continue
        if p.lower() in ("others", "et al.", "et al"):
            others = True
        elif "," in p:
            last, first = [x.strip() for x in p.split(",", 1)]
            names.append((last, first))
        else:
            toks = p.split()
            names.append((toks[-1], " ".join(toks[:-1])))
    return names, others


def _ar_initials(first: str) -> str:
    out = []
    for tok in first.split():
        parts = [t for t in tok.split("-") if t]
        if parts:
            out.append("-".join(t[0].upper() + "." for t in parts))
    return " ".join(out)


def _ar_cite_base(key: str) -> str:
    """Parte de autoria da citação NBR 10520: 1 autor; 2-3 autores separados por ';'; mais de 3 -> 'PRIMEIRO et al.'."""
    names, others = _ar_bib_names(key)
    sn = [n[0].upper() for n in names]
    if not sn:
        return key.upper()
    if others or len(sn) > 3:
        return sn[0] + " et al."
    return "; ".join(sn)


AR_YEAR_SUFFIX: Dict[str, str] = {}
AR_YEAR_SUFFIX_READY: List[bool] = [False]


def _ar_year_label(key: str) -> str:
    """Ano da obra; se duas obras têm a mesma autoria e o mesmo ano, recebem a, b, ... (NBR 10520, 7.2)."""
    if not AR_YEAR_SUFFIX_READY[0]:
        groups: Dict[Tuple[str, str], List[str]] = {}
        for k in sorted(AR_BIB_DB):
            groups.setdefault((_ar_cite_base(k), _ar_bib_fields(AR_BIB_DB[k]).get("year", "s.d.")), []).append(k)
        for g in groups.values():
            if len(g) > 1:
                for i, k in enumerate(g):
                    AR_YEAR_SUFFIX[k] = chr(97 + i)
        AR_YEAR_SUFFIX_READY[0] = True
    return _ar_bib_fields(AR_BIB_DB[key]).get("year", "s.d.") + AR_YEAR_SUFFIX.get(key, "")


def _ar_abnt_cite(keys: Tuple[str, ...]) -> str:
    """(HALGREN, 1996) | (MELI; BIGGIN, 2020) | (MCNUTT et al., 2021). Várias obras: ';' e ordem alfabética."""
    items = sorted((_ar_cite_base(k), _ar_year_label(k)) for k in dict.fromkeys(keys))
    groups: List[Tuple[str, List[str]]] = []
    for base, yr in items:
        if groups and groups[-1][0] == base:
            groups[-1][1].append(yr)
        else:
            groups.append((base, [yr]))
    return "(" + "; ".join(f"{b}, {', '.join(ys)}" for b, ys in groups) + ")"


def _ar_ref_authors(key: str) -> str:
    """'SOBRENOME, P. I.; SOBRENOME, P.' (até 3 autores) ou 'SOBRENOME, P. et al.' (mais de 3)."""
    names, others = _ar_bib_names(key)

    def one(n: Tuple[str, str]) -> str:
        ini = _ar_initials(n[1])
        return n[0].upper() + (", " + ini if ini else "")
    if not names:
        return ""
    if others or len(names) > 3:
        return one(names[0]) + " et al."
    return "; ".join(one(n) for n in names)


def _ar_ref_sort_key(key: str) -> Tuple[str, str, str]:
    names, _ = _ar_bib_names(key)
    return ((names[0][0] if names else key).casefold(), _ar_year_label(key), _ar_bib_plain(_ar_bib_fields(AR_BIB_DB[key]).get("title", "")).casefold())


# --------------------------------------------------------------------------------------------------------------------
# Formatadores: o mesmo construtor de conteúdo alimenta Markdown e LaTeX
# --------------------------------------------------------------------------------------------------------------------
AR_TEX_SPECIAL = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#", "_": r"\_", "{": r"\{",
                "}": r"\}", "~": r"\textasciitilde{}", "^": r"\textasciicircum{}"}
AR_TEX_UNICODE = {"Å": r"\AA{}", "µ": r"$\mu$", "μ": r"$\mu$", "≥": r"$\geq$", "≤": r"$\leq$", "Δ": r"$\Delta$",
                "π": r"$\pi$", "±": r"$\pm$", "×": r"$\times$", "°": r"$^{\circ}$", "≈": r"$\approx$", "−": r"$-$",
                "→": r"$\rightarrow$", "²": r"$^{2}$", "⁺": r"$^{+}$"}


class _ar_Fmt:
    """Interface comum; `cited` acumula as chaves BibTeX na ordem da primeira citação."""
    kind = "base"

    def __init__(self) -> None:
        self.cited: List[str] = []

    def _note(self, keys: Tuple[str, ...]) -> None:
        for k in keys:
            if k not in AR_BIB_DB:
                raise KeyError(f"Referência desconhecida no banco interno: {k}")
            if k not in self.cited:
                self.cited.append(k)


class _ar_MdFmt(_ar_Fmt):
    kind = "md"

    def t(self, s: Any) -> str:                      # texto de dados -> Markdown seguro
        s = str(s)
        for a, b in (("\\", "\\\\"), ("|", "\\|"), ("*", "\\*"), ("_", "\\_"), ("`", "\\`"), ("$", "\\$")):
            s = s.replace(a, b)
        return s

    def b(self, s: str) -> str: return f"**{s}**"
    def i(self, s: str) -> str: return f"*{s}*"
    def m(self, s: str) -> str: return f"${s}$"
    def tt(self, s: str) -> str: return f"`{s}`"
    def unit_mu(self) -> str: return "µ"

    def cite(self, *keys: str) -> str:
        self._note(keys)
        return " " + _ar_abnt_cite(keys)


class _ar_TexFmt(_ar_Fmt):
    kind = "tex"

    def t(self, s: Any) -> str:
        return "".join(AR_TEX_SPECIAL.get(ch, ch) for ch in str(s))

    def b(self, s: str) -> str: return rf"\textbf{{{s}}}"
    def i(self, s: str) -> str: return rf"\textit{{{s}}}"
    def m(self, s: str) -> str: return f"${s}$"
    def tt(self, s: str) -> str: return rf"\texttt{{{self.t(s)}}}"
    def unit_mu(self) -> str: return r"$\mu$"

    def cite(self, *keys: str) -> str:
        self._note(keys)
        return r"~\parencite{" + ",".join(keys) + "}"


def _ar_tex2html(s: str) -> str:
    """Subconjunto de LaTeX usado nas fórmulas do relatório -> HTML simples (sub/sup, frações em linha)."""
    for _ in range(2):
        s = re.sub(r"\\(?:mathrm|text)\{([^{}]*)\}", r"\1", s)
    s = re.sub(r"\\frac\{((?:[^{}]|\{[^{}]*\})*)\}\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)/(\2)", s)
    for a, b in ((r"\Delta", "Δ"), (r"\times", "×"), (r"\cdot", "·"), (r"\approx", "≈"), (r"\ln", "ln"), (r"\,", "\u2009"),
                 ("{,}", ","), ("\\", "")):
        s = s.replace(a, b)
    s = re.sub(r"_\{([^{}]*)\}", r"<sub>\1</sub>", s)
    s = re.sub(r"\^\{([^{}]*)\}", r"<sup>\1</sup>", s)
    return re.sub(r"_(\w)", r"<sub>\1</sub>", s)


# --------------------------------------------------------------------------------------------------------------------
# Equações: subconjunto de LaTeX -> árvore -> OMML (equação nativa do Word) ou HTML (sub/sup e fração empilhada)
# --------------------------------------------------------------------------------------------------------------------
AR_EQ_SYM = {"Delta": "Δ", "cdot": "·", "approx": "≈", "times": "×", "pi": "π", "mu": "μ", "leq": "≤", "geq": "≥", "pm": "±",
           "rightarrow": "→", "alpha": "α", "beta": "β", "sigma": "σ"}
AR_EQ_FUNC = {"ln", "log", "exp", "min", "max"}
AR_EQ_SPACE = {",": " ", ";": " ", ":": " ", " ": " ", "quad": " ", "qquad": "  ", "!": ""}
AR_EQ_STYLE = {"mathrm": "p", "text": "nor", "mathit": "i", "mathbf": "b"}


def _ar_tex_parse(src: str) -> List[Any]:
    """Subconjunto de LaTeX (\\mathrm, \\text, \\frac, _ e ^, \\Delta, \\cdot, \\approx, \\ln, \\, ...) -> lista de nós:
    ("t", texto, estilo) | ("g", [nós]) | ("sub"|"sup", base, [nós]) | ("f", [num], [den]).  Estilos: i (itálico matemático),
    p (upright), nor (texto), b (negrito). Levanta ValueError para comandos fora do subconjunto."""
    pos = [0]

    def skip() -> None:
        while pos[0] < len(src) and src[pos[0]].isspace():
            pos[0] += 1

    def arg(style: str) -> List[Any]:
        skip()
        if pos[0] >= len(src):
            raise ValueError("argumento ausente")
        if src[pos[0]] == "{":
            pos[0] += 1
            return seq(style, True)
        ch = src[pos[0]]
        pos[0] += 1
        return [("t", ch, style if ch.isalpha() else "p")]

    def seq(style: str, closing: bool) -> List[Any]:
        out: List[Any] = []
        while pos[0] < len(src):
            ch = src[pos[0]]
            if ch == "}":
                if not closing:
                    raise ValueError("chave sem par")
                pos[0] += 1
                return out
            if ch.isspace():
                pos[0] += 1
            elif ch == "{":
                pos[0] += 1
                out.append(("g", seq(style, True)))
            elif ch in "_^":
                pos[0] += 1
                a = arg(style)
                base = out.pop() if out else ("t", "", style)
                out.append(("sub" if ch == "_" else "sup", base, a))
            elif ch == "\\":
                m = re.match(r"\\([A-Za-z]+|.)", src[pos[0]:])
                if not m:
                    raise ValueError("barra solta")
                name = m.group(1)
                pos[0] += m.end()
                if name in AR_EQ_STYLE:
                    out.append(("g", arg(AR_EQ_STYLE[name])))
                elif name == "frac":
                    num = arg(style)
                    out.append(("f", num, arg(style)))
                elif name in AR_EQ_SYM:
                    out.append(("t", AR_EQ_SYM[name], "p"))
                elif name in AR_EQ_FUNC:
                    out.append(("t", name + "\u2009", "p"))
                elif name in AR_EQ_SPACE:
                    out.append(("t", AR_EQ_SPACE[name], "p"))
                elif name in ("left", "right"):
                    pass
                else:
                    raise ValueError(f"comando LaTeX fora do subconjunto: \\{name}")
            else:
                pos[0] += 1
                if ch == "-":
                    out.append(("t", "−", "p"))
                elif ch.isalpha():
                    out.append(("t", ch, style))
                else:
                    out.append(("t", ch, "nor" if style == "nor" else "p"))
        if closing:
            raise ValueError("chave não fechada")
        return out
    return seq("i", False)


def _ar_merge_runs(nodes: List[Any]) -> List[Any]:
    """Une nós de texto adjacentes de mesmo estilo (aplica-se também aos grupos sem script)."""
    flat: List[Any] = []
    for n in nodes:
        if n[0] == "g":
            flat += _ar_merge_runs(n[1])
        else:
            flat.append(n)
    out: List[Any] = []
    for n in flat:
        if n[0] == "t" and out and out[-1][0] == "t" and out[-1][2] == n[2]:
            out[-1] = ("t", out[-1][1] + n[1], n[2])
        else:
            out.append(n)
    return out


def _ar_omml_nodes(nodes: List[Any]) -> str:
    esc = lambda x: _htmllib.escape(x, quote=False)                                       # noqa: E731
    sty = {"p": '<m:rPr><m:sty m:val="p"/></m:rPr>', "nor": "<m:rPr><m:nor/></m:rPr>", "b": '<m:rPr><m:sty m:val="b"/></m:rPr>', "i": ""}
    font = '<w:rPr><w:rFonts w:ascii="Cambria Math" w:hAnsi="Cambria Math"/></w:rPr>'
    parts: List[str] = []
    for n in _ar_merge_runs(nodes):
        if n[0] == "t":
            if n[1]:
                parts.append(f'<m:r>{sty[n[2]]}{font}<m:t xml:space="preserve">{esc(n[1])}</m:t></m:r>')
        elif n[0] in ("sub", "sup"):
            tag, arg_tag = ("sSub", "sub") if n[0] == "sub" else ("sSup", "sup")
            base = _ar_omml_nodes([n[1]]) or f'<m:r>{font}<m:t></m:t></m:r>'
            parts.append(f"<m:{tag}><m:e>{base}</m:e><m:{arg_tag}>{_ar_omml_nodes(n[2])}</m:{arg_tag}></m:{tag}>")
        elif n[0] == "f":
            parts.append(f"<m:f><m:num>{_ar_omml_nodes(n[1])}</m:num><m:den>{_ar_omml_nodes(n[2])}</m:den></m:f>")
    return "".join(parts)


def _ar_tex2omml(tex: str) -> str:
    """Equação LaTeX (subconjunto) -> bloco OMML `m:oMathPara`, centralizado, com fração, subscritos reais e variáveis em itálico."""
    return ('<m:oMathPara xmlns:m="http://schemas.openxmlformats.org/officeDocument/2006/math" '
            'xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main"><m:oMath>'
            + _ar_omml_nodes(_ar_tex_parse(tex)) + "</m:oMath></m:oMathPara>")


def _ar_tex2html_eq(tex: str, stacked: bool = True) -> str:
    """Equação LaTeX (subconjunto) -> HTML: variáveis em <i>, <sub>/<sup> reais e fração empilhada (ou 'a/b' se `stacked=False`)."""
    esc = lambda x: _htmllib.escape(x, quote=False)                                       # noqa: E731

    def one(nodes: List[Any]) -> str:
        out = []
        for n in _ar_merge_runs(nodes):
            if n[0] == "t":
                out.append(f"<i>{esc(n[1])}</i>" if n[2] == "i" else (f"<b>{esc(n[1])}</b>" if n[2] == "b" else esc(n[1])))
            elif n[0] in ("sub", "sup"):
                t_ = n[0]
                out.append(one([n[1]]) + f"<{t_}>{one(n[2])}</{t_}>")
            elif n[0] == "f":
                if stacked:
                    out.append('<span style="display:inline-block;vertical-align:middle;text-align:center">'
                               f'<span style="display:block;border-bottom:1px solid currentColor;padding:0 .25em">{one(n[1])}</span>'
                               f'<span style="display:block;padding:0 .25em">{one(n[2])}</span></span>')
                else:
                    out.append(f"({one(n[1])})/({one(n[2])})")
        return "".join(out)
    return one(_ar_tex_parse(tex))


class _ar_HtmlFmt(_ar_Fmt):
    kind = "html"

    def t(self, s: Any) -> str:
        import html as _h
        return _h.escape(str(s), quote=False)

    def b(self, s: str) -> str: return f"<b>{s}</b>"
    def i(self, s: str) -> str: return f"<i>{s}</i>"
    def m(self, s: str) -> str: return f"<i>{_ar_tex2html(s)}</i>"
    def tt(self, s: str) -> str: return f"<code>{self.t(s)}</code>"
    def unit_mu(self) -> str: return "µ"

    def cite(self, *keys: str) -> str:
        self._note(keys)
        return " " + self.t(_ar_abnt_cite(keys))


def _ar_hcell(f: "_ar_Fmt", h: Any) -> str:
    """Rótulo de coluna: LE_Vina / LE_CNN viram LE com subscrito real (HTML, DOCX e LaTeX); Markdown mantém o texto simples."""
    m_ = re.fullmatch(r"LE_(Vina|CNN)", str(h))
    if m_ and f.kind in ("html", "tex"):
        return f.m(rf"\mathrm{{LE}}_{{\mathrm{{{m_.group(1)}}}}}")
    return f.t(h)


def _ar_tex_unicode(s: str) -> str:
    for a, b in AR_TEX_UNICODE.items():
        s = s.replace(a, b)
    return s


# --------------------------------------------------------------------------------------------------------------------
# Auxiliares numéricos
# --------------------------------------------------------------------------------------------------------------------
def _ar_fin(x: Any) -> bool:
    try:
        return x is not None and math.isfinite(float(x))
    except (TypeError, ValueError):
        return False


def _ar_n(x: Any, d: int = 2, sign: bool = False) -> str:
    """Número com vírgula decimal (padrão pt-BR); 'n/d' se ausente."""
    if not _ar_fin(x):
        return "n/d"
    s = f"{float(x):+.{d}f}" if sign else f"{float(x):.{d}f}"
    return s.replace(".", ",").replace("-", "−")


def _ar_nz(x: Any, d: int = 3, min_d: int = 1) -> str:
    """Decimal com até `d` casas (mínimo `min_d`), vírgula decimal e sinal '−': -10.021 -> '−10,021'; 20.0 -> '20,0'."""
    if not _ar_fin(x):
        return "n/d"
    v = float(x)
    if abs(v) < 0.5 * 10 ** (-d):
        v = 0.0
    a, b = f"{v:.{d}f}".split(".") if d > 0 else (f"{v:.0f}", "")
    b = b.rstrip("0").ljust(min_d, "0") if d > 0 else ""
    return (a + ("." + b if b else "")).replace(".", ",").replace("-", "−")


def _ar_vec3(v: Any) -> Optional[List[float]]:
    try:
        a = [float(x) for x in v]
    except (TypeError, ValueError):
        return None
    return a if len(a) == 3 and all(math.isfinite(x) for x in a) else None


def _ar_pl(n: int, sing: str, plur: str) -> str:
    return sing if n == 1 else plur


def _ar_norm_res(s: str) -> str:
    return re.sub(r"[\s.:\-_]", "", str(s)).lower()


def _ar_split_interactions(s: Any) -> List[str]:
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return []
    parts = re.split(r"[;|\n]+|,(?![^()]*\))", str(s))
    return [p.strip() for p in parts if p.strip() and p.strip().lower() not in {"nan", "none", "-", "—"}]


def _ar_res_key(item: str) -> str:
    m = re.match(r"\s*([A-Za-z]{1,4})\s*-?(\d+)", item)
    return (m.group(1) + m.group(2)).lower() if m else _ar_norm_res(item.split("(")[0])


def _ar_join_pt(items: List[str], conj: str = "e") -> str:
    items = [i for i in items if i]
    if not items:
        return ""
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + f" {conj} " + items[-1]


AR_TD_FEM = {"cavidade", "região", "regiao", "fenda", "bolsa", "interface", "subunidade", "cadeia", "hélice", "helice",
           "superfície", "superficie", "enzima", "proteína", "proteina", "câmara", "camara", "cavidades", "fendas"}
AR_DF_MASC = ("mento", "ismo", "porte", "tor", "dor", "ato")


def _ar_art_domain(td: str) -> Tuple[str, str]:
    """('no', 'do') ou ('na', 'da') conforme o gênero provável da primeira palavra do domínio (padrão masculino)."""
    w = (re.findall(r"[^\W\d_]+", str(td).casefold()) or [""])[0]
    return ("na", "da") if w in AR_TD_FEM else ("no", "do")


def _ar_art_function(df_: str) -> str:
    """'pela' (padrão, feminino) ou 'pelo' para substantivos tipicamente masculinos ('metabolismo', 'transporte')."""
    w = (re.findall(r"[^\W\d_]+", str(df_).casefold()) or [""])[0]
    return "pelo" if w.endswith(AR_DF_MASC) else "pela"


AR_CAT_ITEM_RE = re.compile(r"metal|axial|catal[ií]t|cat[aá]lise", re.I)


# ====================================================================================================================
class AcademicReporter:
    """Gera Metodologia e Resultados/Discussão a partir de `config` + `leaderboard_df`. Veja o docstring do módulo."""

    def __init__(self, config: Dict[str, Any], leaderboard_df: pd.DataFrame, *, le_threshold: float = 0.30,
                 rmsd_threshold: float = 2.0, pk_resolution: float = 1.0, max_secondary_text: int = 5,
                 leader_requires_pb_pass: bool = False) -> None:
        missing = [k for k in AR_REQUIRED_CONFIG if k not in config]
        if missing:
            raise ValueError(f"config incompleto; chaves ausentes: {missing}")
        if not isinstance(leaderboard_df, pd.DataFrame):
            raise TypeError("leaderboard_df deve ser um pandas.DataFrame")
        miss_c = [c for c in AR_REQUIRED_COLUMNS if c not in leaderboard_df.columns]
        if miss_c:
            raise ValueError(f"leaderboard_df sem as colunas obrigatórias: {miss_c}")
        self.cfg = dict(config)
        self.le_thr, self.rmsd_thr = float(le_threshold), float(rmsd_threshold)
        self.pk_res, self.max_sec = float(pk_resolution), int(max_secondary_text)
        self.leader_requires_pb_pass = bool(leader_requires_pb_pass)
        self.cfg["metals_list"] = [str(m).strip() for m in (self.cfg.get("metals_list") or []) if str(m).strip()]
        self.cfg["has_metals"] = bool(self.cfg["has_metals"])
        if self.cfg["has_metals"] and not self.cfg["metals_list"]:
            warnings.warn("has_metals=True com metals_list vazia: o texto usará a expressão genérica 'íons metálicos do sítio'.")
        if not self.cfg["has_metals"] and self.cfg["metals_list"]:
            warnings.warn("has_metals=False: metals_list será ignorada.")
            self.cfg["metals_list"] = []
        hm = self.cfg["has_metals"]
        td = re.sub(r"\s+", " ", str(self.cfg.get("target_domain") or "")).strip()
        dfn = re.sub(r"\s+", " ", str(self.cfg.get("domain_function") or "")).strip()
        self.cfg["target_domain"] = td or "cavidade ortostérica do sítio catalítico"
        self.cfg["domain_function"] = dfn or "atividade biológica e interação com substratos"
        self.cfg["catalytic_residues"] = [str(x).strip() for x in (self.cfg.get("catalytic_residues") or []) if str(x).strip()]
        self.cfg["docking_engine"] = self._clean_engine(self.cfg.get("docking_engine"))
        # Modelo predito (AlphaFold DB): SOMENTE source_kind == "alphafold" ativa o texto de modelo predito. Estruturas
        # experimentais (RCSB ou PDB local) nunca recebem discussão sobre limitações de modelos generativos.
        self.is_predicted = str(self.cfg.get("source_kind") or "").strip().lower() == "alphafold"
        def _fl(k):
            try:
                v = float(self.cfg.get(k))
                return v if v == v else None
            except (TypeError, ValueError):
                return None
        self.plddt_mean, self.plddt_min = _fl("mean_pocket_plddt"), _fl("min_pocket_plddt")
        self.uniprot_id = re.sub(r"[^A-Za-z0-9]", "", str(self.cfg.get("uniprot_id") or "")).upper()
        # artigo cristalográfico original do PDB: texto BibTeX (registrado no banco) ou chave já existente; falha nunca interrompe
        self.target_paper_key = ""
        key = str(self.cfg.get("target_paper_key") or "").strip()
        try:
            if self.cfg.get("target_paper_bib"):
                ks = _ar_register_bib(self.cfg["target_paper_bib"])
                key = key or (ks[0] if ks else "")
        except Exception as e_bib:
            warnings.warn(f"target_paper_bib ignorado ({type(e_bib).__name__}: {e_bib}); o artigo cristalográfico não será citado.")
            key = ""
        if key and key not in AR_BIB_DB:
            warnings.warn(f"target_paper_key '{key}' não existe no banco de referências; o artigo cristalográfico não será citado.")
            key = ""
        self.target_paper_key = key
        self._prepare(leaderboard_df)
        self._make_labels()

    # ------------------------------------------------------------------ preparo dos dados
    @staticmethod
    def _is_control(role: Any) -> bool:
        r = str(role).strip().casefold()
        return r.startswith(("controle", "control", "nativ", "native", "referên", "referen"))

    @staticmethod
    def _pb_state(v: Any) -> str:
        if isinstance(v, (bool, np.bool_)):
            return "PASS" if bool(v) else "FAIL"
        s = str(v).strip().upper()
        if s in {"PASS", "TRUE", "1", "SIM", "YES", "OK", "VÁLIDO", "VALIDO", "VALID"}:
            return "PASS"
        if s in {"", "NAN", "NONE", "N/D", "NA", "—", "-"}:
            return "n/d"
        return "FAIL"

    def _prepare(self, raw: pd.DataFrame) -> None:
        df = raw.copy().reset_index(drop=True)
        for c in AR_NUM_COLS:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = df[c].astype(str).str.replace(",", ".", regex=False).str.replace("−", "-", regex=False)
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["Composto"] = df["Composto"].astype(str).str.strip()
        df["_le_calc"] = df["LE_Vina"].isna() & df["Vina_score"].notna() & (df["HAC"] > 0)
        df.loc[df["_le_calc"], "LE_Vina"] = -df["Vina_score"] / df["HAC"]
        le_cnn_calc = np.where((df["HAC"] > 0) & df["CNNaffinity_pK"].notna(), AR_RT_LN10 * df["CNNaffinity_pK"] / df["HAC"], np.nan)
        if "LE_CNN" in df.columns:                              # valor do app (se houver); calcula só os que faltam
            df["LE_CNN"] = pd.to_numeric(df["LE_CNN"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
            df["LE_CNN"] = df["LE_CNN"].fillna(pd.Series(le_cnn_calc, index=df.index))
        else:
            df["LE_CNN"] = le_cnn_calc
        df["_pb"] = df["PoseBusters"].map(self._pb_state)
        df["_ctrl"] = df["Papel"].map(self._is_control)
        ctrl_idx = list(df.index[df["_ctrl"]])
        self.ctrl: Optional[pd.Series] = df.loc[ctrl_idx[0]] if ctrl_idx else None
        if len(ctrl_idx) > 1:
            warnings.warn("Mais de uma linha de controle nativo: apenas a primeira é usada na validação.")
        tests = df[~df["_ctrl"]].copy()
        if tests.empty:
            raise ValueError("leaderboard_df não contém nenhum composto teste (Papel diferente de controle).")
        valid = tests.dropna(subset=["Vina_score", "CNNaffinity_pK"])
        if self.leader_requires_pb_pass and (valid["_pb"] == "PASS").any():
            valid = valid[valid["_pb"] == "PASS"]
        if valid.empty:
            raise ValueError("Nenhum composto teste com Vina_score e CNNaffinity_pK válidos para eleger o líder.")
        valid = valid.assign(_rv=valid["Vina_score"].rank(ascending=True, method="min"),
                             _rk=valid["CNNaffinity_pK"].rank(ascending=False, method="min"))
        valid["_score"] = valid["_rv"] + valid["_rk"]
        valid = valid.sort_values(["_score", "CNNaffinity_pK", "Vina_score"], ascending=[True, False, True])
        self.ranked = valid                                   # compostos teste ordenados, o primeiro é o líder
        self.leader: pd.Series = valid.iloc[0]
        self.secondary: pd.DataFrame = valid.iloc[1:]
        self.consensus = bool(self.leader["Vina_score"] == tests["Vina_score"].min()
                              and self.leader["CNNaffinity_pK"] == tests["CNNaffinity_pK"].max())
        self.best_vina_name = str(valid.loc[valid["Vina_score"].idxmin(), "Composto"])
        self.best_pk_name = str(valid.loc[valid["CNNaffinity_pK"].idxmax(), "Composto"])
        self.tests = tests
        ordered = [self.ctrl.name] if self.ctrl is not None else []
        ordered += list(valid.index) + [i for i in tests.index if i not in valid.index]
        ordered += [i for i in ctrl_idx[1:]]
        self.table_df = df.loc[ordered]
        self.df = df
        # nomes de compostos presentes na planilha mas ausentes em ligands_info (e vice-versa)
        info_names = {str(d.get("name", "")).strip().casefold() for d in self.cfg["ligands_info"]}
        extra = [n for n in df["Composto"] if n.casefold() not in info_names]
        if extra:
            warnings.warn(f"Compostos do leaderboard sem entrada em ligands_info (CID 'n/d'): {extra}")

    # ------------------------------------------------------------------ helpers de texto
    @staticmethod
    def _kd(f: _ar_Fmt, pk: float) -> str:
        kd = 10.0 ** (-float(pk))                      # molar
        for lim, mul, unit in ((1e-9, 1e12, "pM"), (1e-6, 1e9, "nM"), (1e-3, 1e6, None), (np.inf, 1e3, "mM")):
            if kd < lim:
                val = kd * mul
                d = 0 if val >= 100 else (1 if val >= 10 else 2)
                u = unit if unit else f.unit_mu() + "M"
                return f"{_ar_n(val, d)} {u}"
        return "n/d"

    @staticmethod
    def _fold(f: _ar_Fmt, x: float) -> str:
        if x < 1e3:
            return _ar_n(x, 0 if x >= 100 else 1)
        e = int(math.floor(math.log10(x)))
        return f.m(rf"{_ar_n(x / 10 ** e, 1)}\times 10^{{{e}}}")

    def _metal_phrase(self) -> str:
        return ", ".join(self.cfg["metals_list"])

    def _mentions_metal(self, item: str) -> bool:
        if re.search(r"metal", item, re.I):
            return True
        if self.cfg["has_metals"]:
            return _ar_res_key(item) in {_ar_res_key(m) for m in self.cfg["metals_list"]}
        return False

    def _catalytic_keys(self) -> set:
        """Chaves de resíduo (nome+número do resíduo, em minúsculas) que definem o centro catalítico: metais, catalytic_residues e, se houver
        metais, os resíduos com estado de protonação explícito (his_states = esfera de coordenação)."""
        c = self.cfg
        keys = {_ar_res_key(m) for m in c["metals_list"]} | {_ar_res_key(r) for r in c["catalytic_residues"]}
        if c["has_metals"]:
            keys |= {_ar_res_key(k) for k in (c.get("his_states") or {})}
        return {k for k in keys if k}

    def _touches_catalytic(self, item: str) -> bool:
        """True se a interação ProLIF toca o centro catalítico: contato metálico, resíduo axial/catalítico anotado ou rótulo
        'metal/axial/catalítico' no próprio item."""
        return bool(AR_CAT_ITEM_RE.search(item)) or self._mentions_metal(item) or _ar_res_key(item) in self._catalytic_keys()

    def _compartment(self, interacoes: Any) -> str:
        """'core' (toca o centro catalítico), 'vestibulo' (só contatos periféricos) ou 'nd' (sem interações registradas)."""
        its = _ar_split_interactions(interacoes)
        if not its:
            return "nd"
        return "core" if any(self._touches_catalytic(i) for i in its) else "vestibulo"

    @staticmethod
    def _clean_method(raw: Any, default: str = "PDBFixer/OpenMM") -> str:
        """Nome do método de preparo sem texto de log/código: remove parênteses com parâmetros, `a=b` e chamadas `Obj.metodo`."""
        t_ = re.sub(r"\([^)]*\)?", " ", str(raw or ""))
        t_ = re.sub(r"\S*=\S*|\b\w+\.\w+\b", " ", t_)
        t_ = re.sub(r"\s+", " ", t_).strip(" ,;:.-")
        return t_ or default

    @staticmethod
    def _clean_engine(raw: Any) -> str:
        """Rótulo do motor de docking sem metadados de compilação nem opções de linha de comando:
        'gnina v1.3.2 master:f23dd2b Built Jul 8 2025.' -> 'GNINA v1.3.2'; 'GNINA v1.3.2 com AutoDock Vina e refino --minimize' ->
        'GNINA v1.3.2 com AutoDock Vina'."""
        t_ = str(raw or "").strip()
        t_ = re.sub(r"\b(?:master|main|dev|HEAD)[:@][0-9A-Fa-f]{5,}\b", " ", t_)
        t_ = re.sub(r"\bBuilt\s+[A-Za-z]{3,9}\s+\d{1,2}\s+\d{4}\.?", " ", t_, flags=re.I)
        t_ = re.sub(r"\bcommit\s+[0-9A-Fa-f]{5,}\b", " ", t_, flags=re.I)
        t_ = re.sub(r"\s+e\s+refino\b(?:\s+--[\w-]+)?", " ", t_, flags=re.I)
        t_ = re.sub(r"\(?\s*--[\w-]+(?:[ =]\S+)?\s*\)?", " ", t_)
        t_ = re.sub(r"(?i)\bgnina\b\s*v?(\d+(?:\.\d+)+)", lambda m: f"GNINA v{m.group(1)}", t_)
        return re.sub(r"\s+", " ", t_).strip(" ,;.") or "GNINA"

    def _make_labels(self) -> None:
        """Rótulos 'Alvo 1', 'Alvo 2', ... na ordem de `ligands_info` (ordem da fila); o controle nativo é 'Controle Nativo'."""
        order = [str(d.get("name", "")).strip() for d in self.cfg["ligands_info"] if not self._is_control(d.get("role", ""))]
        names = list(self.tests["Composto"])
        by_cf = {n.casefold(): n for n in names}
        seq: List[str] = []
        for n in order:
            if n.casefold() in by_cf and by_cf[n.casefold()] not in seq:
                seq.append(by_cf[n.casefold()])
        seq += [n for n in names if n not in seq]
        self.labels: Dict[str, str] = {n: f"Alvo {i}" for i, n in enumerate(seq, 1)}

    def _alvo(self, f: _ar_Fmt, name: Any, bold: bool = False) -> str:
        """'Alvo 1 (Nome)' (com 'Alvo 1' em negrito se `bold`); nomes fora dos testes (controle) saem só com o nome."""
        nm = str(name)
        lab = self.labels.get(nm)
        if lab is None:
            return f.b(f.t(nm)) if bold else f.t(nm)
        return (f.b(lab) if bold else lab) + f" ({f.t(nm)})"

    def _ctrl_label(self, f: _ar_Fmt) -> str:
        return f"{f.b('Controle Nativo')} ({f.t(self.ctrl['Composto'])})" if self.ctrl is not None else "Controle Nativo"

    def _role_label(self, row: Any) -> str:
        return "Controle Nativo" if row["_ctrl"] else self.labels.get(str(row["Composto"]), str(row["Papel"]))

    def _prot(self, f: _ar_Fmt) -> str:
        org = str(self.cfg.get("organism") or "").strip()
        return f.t(self.cfg["protein_name"]) + (f" ({f.i(f.t(org))})" if org else "")

    # ------------------------------------------------------------------ construção de blocos
    def _build(self, f: _ar_Fmt) -> List[Tuple[str, Any]]:
        B: List[Tuple[str, Any]] = []
        B += self._methods(f)
        B += self._results(f)
        return B

    # ---------------------------------------------------------------- caixa de busca (grid box)
    def _box_sentence(self, f: _ar_Fmt, ctrl_present: bool) -> str:
        """Frase da caixa de busca (v10.1). Com `grid_center` e `grid_size` informa as coordenadas cartesianas efetivas (x, y, z)
        com 3 casas e as dimensões com 1 casa, em Å; sem elas, recorre a `box_description` ou à regra do autobox (v9.x)."""
        c = self.cfg
        gc, gs = _ar_vec3(c.get("grid_center")), _ar_vec3(c.get("grid_size"))
        pad = _ar_nz(c["box_padding_angstrom"], 2)
        mode = str(c.get("box_mode") or "").strip().lower()
        if gc is None or gs is None:
            if c.get("box_description"):
                return f"A caixa de busca foi definida como: {f.t(c['box_description'])}."
            ref_lig = "ligante nativo" if ctrl_present else "ligante de referência"
            return (f"A caixa de busca foi definida a partir do {ref_lig} (regra do autobox: caixa envolvente do ligante mais "
                    f"{f.b(pad)} Å por lado).")
        xyz = f"x = {_ar_nz(gc[0], 3)} Å, y = {_ar_nz(gc[1], 3)} Å e z = {_ar_nz(gc[2], 3)} Å"
        dim = f"{_ar_nz(gs[0], 1)} Å × {_ar_nz(gs[1], 1)} Å × {_ar_nz(gs[2], 1)} Å"
        cover = ("assegurando a cobertura tridimensional completa da fenda catalítica e do espaço conformacional acessível aos "
                 "ligantes durante a busca.")
        if mode == "manual":
            return (f"A caixa de busca (grid box) foi centrada em {xyz}, coordenadas definidas manualmente a partir do centroide do "
                    f"sítio catalítico, com dimensões de {dim}, {cover}")
        ext = bool(c.get("grid_size_extended"))
        expan = (f"expansão uniforme de {pad} Å por lado a partir do bounding box de referência" if not ext else
                 f"expansão uniforme de {pad} Å por lado a partir do bounding box de referência, com as arestas estendidas até o "
                 f"diâmetro molecular quando este era maior")
        s = f"A caixa de busca (grid box) foi centrada em {xyz}, com dimensões de {dim} ({expan}), {cover}"
        by = c.get("grid_size_by_ligand") or {}
        diff = []
        for nm, v in by.items():
            v3 = _ar_vec3(v)
            if v3 is not None and any(round(a, 1) != round(b, 1) for a, b in zip(v3, gs)):
                diff.append(f"{f.t(nm)} ({_ar_nz(v3[0], 1)} × {_ar_nz(v3[1], 1)} × {_ar_nz(v3[2], 1)} Å)")
        if diff:
            more = f" e outros {len(diff) - 6}" if len(diff) > 6 else ""
            s += (f" Para {_ar_pl(len(diff), 'o ligante', 'os ligantes')} {_ar_join_pt(diff[:6])}{more}, cujo diâmetro molecular excede as "
                  f"arestas dessa caixa, o centro foi mantido e as arestas foram estendidas até o diâmetro do ligante.")
        return s

    def _site_sentence(self, f: _ar_Fmt) -> str:
        """Localização funcional da cavidade de ancoramento (v10.1): usa target_domain e domain_function (ou o texto neutro padrão, sem metatexto)."""
        td, dfn = self.cfg["target_domain"], self.cfg["domain_function"]
        no, _ = _ar_art_domain(td)
        return (f"A cavidade de ancoramento situa-se {no} {f.t(td)}, responsável {_ar_art_function(dfn)} {f.t(dfn)}.")

    # ---------------------------------------------------------------- METODOLOGIA
    def _methods(self, f: _ar_Fmt) -> List[Tuple[str, Any]]:
        c = self.cfg
        B: List[Tuple[str, Any]] = [("h2", "Metodologia")]
        app = f.t(c.get("app_name") or "DL-Docking Studio") + (f" v{f.t(c['app_version'])}" if c.get("app_version") else "")
        chain, pdb = f.t(c["target_chain"]), f.t(str(c["pdb_id"]).upper())
        ph = _ar_n(c["ph_value"], 1)
        exh, seed = c.get("exhaustiveness", 32), c.get("seed", 42)
        sub10, subd = f.m("_{10}"), f.m("_{d}")
        le_v = f.m(r"\mathrm{LE}_{\mathrm{Vina}}")
        rt_tex = _ar_n(AR_RT_LN10, 3).replace(",", "{,}")

        # --- receptor
        B.append(("h3", "Preparo do receptor"))
        rmeth = f.t(self._clean_method(c.get("receptor_method")))
        ph_txt = f"em pH fisiológico ({f.b(ph)})" if abs(float(c["ph_value"]) - 7.4) <= 0.15 else f"em pH {f.b(ph)}"
        taut = ("ajustando os estados de protonação e as variantes tautoméricas específicas para a estabilização da esfera de "
                "coordenação do centro metálico" if c["has_metals"] else
                "ajustando os estados de protonação dos resíduos ionizáveis")
        cite_pdb = ("berman2000", self.target_paper_key) if self.target_paper_key else ("berman2000",)
        if self.is_predicted:                                          # único ramo com AlphaFold
            uni = f.b(f.t(self.uniprot_id)) if self.uniprot_id else f.b(pdb)
            src = (f"A estrutura tridimensional da proteína-alvo {self._prot(f)} foi obtida do AlphaFold Database "
                   f"(código UniProt {uni}; modelo predito){f.cite('abramson2024', 'jumper2021')}.")
        else:
            src = (f"A estrutura cristalográfica da proteína-alvo {self._prot(f)} foi obtida do Protein Data Bank "
                   f"(RCSB PDB; código {f.b(pdb)}){f.cite(*cite_pdb)}.")
        p = (f"{src} Foi isolada a cadeia {f.b(chain)}, que constitui o receptor em todas as "
             f"etapas seguintes. Átomos pesados ausentes foram reconstruídos quando possível com o {rmeth} e os átomos de hidrogênio "
             f"foram adicionados {ph_txt}, {taut}.")
        B.append(("p", p))
        if c["has_metals"]:
            metals = self._metal_phrase() or "íons metálicos do sítio"
            coord = f.t(c.get("coordinating_residues") or "histidinas")
            his = c.get("his_states") or {}
            his_txt = (f" Os estados explícitos atribuídos foram: {f.t('; '.join(f'{k}: {v}' for k, v in sorted(his.items())))}."
                       if his else "")
            B.append(("p", f"O sítio ativo contém cofatores metálicos essenciais, que foram {f.i('preservados')} no receptor: "
                           f"{f.t(metals)}. Os resíduos que participam da esfera de coordenação desses íons (ex., {coord}) receberam "
                           f"tautômero neutro explícito, de modo a conservar a geometria de coordenação observada no cristal.{his_txt} "
                           f"Moléculas de água, co-solventes, íons de cristalização e demais heteroátomos fora do sítio catalítico foram "
                           f"removidos."))
        else:
            B.append(("p", "O sítio ativo não possui cofatores metálicos mantidos no modelo. Foram removidas, de forma padronizada, as "
                           "moléculas de água, os co-solventes, os íons de cristalização e todos os heteroátomos não catalíticos, "
                           "preservando-se apenas a cadeia proteica selecionada."))

        cofs = [str(x) for x in (c.get("cofactors_list") or []) if str(x).strip()]
        if cofs:
            B.append(("p", f"{_ar_pl(len(cofs), 'O cofator orgânico', 'Os cofatores orgânicos')} {f.t(_ar_join_pt(cofs))} "
                           f"{_ar_pl(len(cofs), 'foi retido', 'foram retidos')} no receptor, sem hidrogênios adicionados."))

        # --- ligantes
        B.append(("h3", "Preparo dos ligantes"))
        by_name = {str(d.get("name", "")).strip().casefold(): d for d in c["ligands_info"]}
        itens, sem_cid = [], 0
        for nm_, lab_ in self.labels.items():
            d = by_name.get(nm_.casefold(), {})
            cid = str(d.get("cid", "") or "").strip()
            if cid and cid.lower() not in {"nan", "none", "n/d"} and not cid.upper().startswith("PDB"):
                itens.append(f"{lab_}: {f.t(nm_)}, CID PubChem {f.t(cid)}")
            else:
                itens.append(f"{lab_}: {f.t(nm_)}")
                sem_cid += 1
        n_lig = len(itens)
        ctrl_present = any(self._is_control(d.get("role", "")) for d in c["ligands_info"]) or self.ctrl is not None
        origem = ("foram obtidas do banco de dados PubChem" if sem_cid == 0 else
                  "foram fornecidas em arquivo pelo usuário" if sem_cid == n_lig else
                  "foram obtidas do banco de dados PubChem ou fornecidas em arquivo pelo usuário")
        ctrl_nm = (str(self.ctrl["Composto"]) if self.ctrl is not None else
                   next((str(d.get("name", "")) for d in c["ligands_info"] if self._is_control(d.get("role", ""))), ""))
        s1 = f"As estruturas tridimensionais {_ar_pl(n_lig, 'do ligante avaliado', 'dos ligantes avaliados')} ({'; '.join(itens)}) {origem}"
        s1 += (f", enquanto o controle nativo ({f.t(ctrl_nm)}) foi extraído diretamente das coordenadas cristalográficas." if ctrl_present else ".")
        nconf = c.get("n_conformers")
        conf_txt = f", com {int(nconf)} conformeros amostrados," if nconf else ""
        B.append(("p", f"{s1} Após a dessalinização, com retenção do maior fragmento pelo RDKit{f.cite('landrum2016')}, os estados de "
                       f"protonação fisiológicos foram determinados em pH {f.b(ph)} utilizando as bibliotecas Open Babel{f.cite('oboyle2011')} "
                       f"e RDKit. A conformação tridimensional inicial foi gerada pelo algoritmo ETKDGv3{f.cite('riniker2015', 'wang2020')}"
                       f"{conf_txt} seguida de relaxamento geométrico e minimização de energia sob o campo de força MMFF94"
                       f"{f.cite('halgren1996')} com critério de convergência de gradiente de força. Cargas parciais atômicas de "
                       f"Gasteiger–Marsili foram atribuídas a cada estrutura, e as ligações rotacionáveis ativas foram mantidas flexíveis "
                       f"durante a busca conformacional."))

        # --- docking
        B.append(("h3", "Docking molecular e reavaliação por redes neurais"))
        eng = f.t(c["docking_engine"])
        box_src = self._site_sentence(f) + " " + self._box_sentence(f, ctrl_present)
        gp = c.get("gnina_params") or {}
        gp_txt = ""
        if gp:
            bits = []
            if gp.get("num_modes") is not None:
                bits.append(f"até {f.t(gp['num_modes'])} modos por ligante")
            if gp.get("min_rmsd_filter") is not None:
                bits.append(f"filtro de diversidade de {_ar_n(gp['min_rmsd_filter'], 1)} Å entre poses")
            if gp.get("num_mc_saved") is not None:
                bits.append(f"{f.t(gp['num_mc_saved'])} poses salvas por simulação de Monte Carlo")
            gp_txt = f" Foram retidos {_ar_join_pt(bits)}." if bits else ""
        sort_txt = (f" As poses foram ordenadas por {f.b(f.t(gp['pose_sort_order']))}." if gp.get("pose_sort_order") else "")
        B.append(("p", f"O docking foi conduzido com {eng}{f.cite('mcnutt2021', 'mcnutt2025', 'trott2009')}, com receptor rígido. "
                       f"{box_src} A busca conformacional combina amostragem estocástica de Monte Carlo com otimização local do "
                       f"escore empírico do Vina, com exaustividade {f.b(f.t(exh))} e semente fixa ({f.t(seed)}) para "
                       f"reprodutibilidade.{gp_txt} As poses foram reavaliadas por um conjunto de redes neurais convolucionais "
                       f"3D para refinamento de escore{f.cite('francoeur2020')}, que fornecem o {f.b('CNNscore')} (probabilidade de a pose ser "
                       f"correta, entre 0 e 1) e a {f.b('CNNaffinity')} (afinidade prevista, expressa como "
                       f"pK = −log{sub10}\u00a0K{subd}){f.cite('mcnutt2021', 'sunseri2021')}.{sort_txt}"))
        B.append(("p", f"Em seguida, as poses foram submetidas a uma etapa subsequente de minimização local de energia no campo de "
                       f"força do receptor rígido, mantendo a reavaliação 3D-CNN. Para cada composto exportou-se a "
                       f"pose de maior CNNscore entre as aprovadas no PoseBusters; se nenhuma foi aprovada, manteve-se a pose de "
                       f"maior ranking com alerta."))

        # --- validação
        B.append(("h3", "Validação física, RMSD e interações"))
        B.append(("p", f"A plausibilidade físico-química das poses foi auditada com o PoseBusters em modo {f.tt('dock')} (choques "
                       f"estéricos, comprimentos e ângulos de ligação, planaridade de anéis e energia interna){f.cite('buttenschoen2024')}; "
                       f"o critério {f.b('PoseBusters = PASS')} indica aprovação em todos os testes. O desvio quadrático médio "
                       f"({f.b('RMSD')}) do ligante nativo redocado foi calculado com correção de simetria e sem superposição prévia "
                       f"(spyrmsd){f.cite('meli2020')}, adotando-se {_ar_n(self.rmsd_thr, 1)} Å "
                       f"como limiar de sucesso{f.cite('buttenschoen2024')}. As interações intermoleculares (ligações de hidrogênio, "
                       f"empilhamento π, contatos hidrofóbicos, interações iônicas e contatos com íons metálicos) foram mapeadas com o "
                       f"ProLIF{f.cite('bouysset2021')}."))

        # --- eficiência de ligante
        B.append(("h3", "Eficiência de ligante"))
        B.append(("p", f"A eficiência de ligante normaliza a afinidade pelo tamanho molecular{f.cite('hopkins2004')}. Para o escore do "
                       f"Vina e para a afinidade prevista pela rede neural, definiram-se, respectivamente (HAC = número de átomos pesados):"))
        B.append(("eq", r"\mathrm{LE}_{\mathrm{Vina}} = \frac{-\Delta G_{\mathrm{Vina}}}{N_{\mathrm{HAC}}}"))
        B.append(("eq", r"\mathrm{LE}_{\mathrm{CNN}} = \frac{RT\,\ln 10\cdot \mathrm{p}K}{N_{\mathrm{HAC}}} \approx \frac{" + rt_tex + r"\,\mathrm{p}K}{N_{\mathrm{HAC}}}"))
        B.append(("p", f"ambas em kcal/mol por átomo pesado (T = 298,15 K). Valores de {le_v} "
                       f"≥ {_ar_n(self.le_thr, 2)} foram tomados como indicativo de que o escore não decorre apenas do tamanho do ligante "
                       f"({f.i('size bias')}){f.cite('kenny2019', 'abad2021')}."))
        B.append(("p", f"Todo o fluxo foi executado no {app}."))
        notes = [str(x) for x in (c.get("run_notes") or []) if str(x).strip()]
        if notes:
            B.append(("p", f"{f.b('Observações da corrida:')}"))
            B.append(("ul", [f.t(x) for x in notes]))
        return B

    # ---------------------------------------------------------------- RESULTADOS E DISCUSSÃO
    def _results(self, f: _ar_Fmt) -> List[Tuple[str, Any]]:
        """Seção 2 em texto corrido contínuo, sem subtítulos: quatro parágrafos encadeados (validação por redocking; ancoramento
        dos ligantes prospectivos; eficiência de ligante; interações no sítio ativo), intercalados pelas Tabelas 1 e 2."""
        c = self.cfg
        A = "Å"
        B: List[Tuple[str, Any]] = [("h2", "Resultados e Discussão")]
        L, tests = self.leader, self.tests
        lead = self._alvo(f, L["Composto"], bold=True)
        sub10, subd, subvina = f.m("_{10}"), f.m("_{d}"), f.m(r"_{\mathrm{Vina}}")
        le_v, le_c = f.m(r"\mathrm{LE}_{\mathrm{Vina}}"), f.m(r"\mathrm{LE}_{\mathrm{CNN}}")
        pk_lead, pk_sec = f.m(r"_{\text{líder}}"), f.m(r"_{\text{secundário}}")
        n_t = len(tests)
        labs_all = [self.labels.get(str(n), str(n)) for n in self.ranked["Composto"]] + \
                   [self.labels.get(str(n), str(n)) for n in tests["Composto"] if n not in set(self.ranked["Composto"])]
        labs_all = sorted(set(labs_all), key=lambda x: (len(x), x))

        # ---- parágrafo 1: validação por redocking do controle nativo
        if self.ctrl is not None:
            C = self.ctrl
            cn = self._ctrl_label(f)
            rm = C["RMSD"]
            if _ar_fin(rm):
                ok = bool(rm <= self.rmsd_thr)
                s = (f"O {cn} foi redocado com RMSD simétrico de {f.b(_ar_n(rm, 2) + ' ' + A)}, "
                     f"{'abaixo' if ok else 'acima'} do limiar de {_ar_n(self.rmsd_thr, 1)} {A}, "
                     f"{'reproduzindo' if ok else 'sem reproduzir'} o modo de ligação cristalográfico{f.cite('buttenschoen2024')}. ")
            else:
                ok = None
                s = f"O {cn} foi incluído como controle, mas o RMSD não está disponível no leaderboard. "
            pb = C["_pb"]
            if pb == "PASS":
                s += ("A pose foi aprovada integralmente na auditoria físico-química do PoseBusters (PASS), o que indica plausibilidade "
                      "estereoquímica e ausência de choques estéricos. ")
            elif pb == "FAIL":
                s += (f"O PoseBusters resultou em {f.b('FAIL')}, de modo que a validação física do controle é inconclusiva ou desfavorável "
                      f"e deve ser examinada antes de interpretar os demais resultados. ")
            else:
                s += "O resultado do PoseBusters não está disponível para o controle. "
            if ok is True and pb == "PASS":
                s += "O protocolo é, portanto, considerado validado para este alvo."
            elif ok is False or pb == "FAIL":
                s += "O protocolo não satisfaz integralmente os critérios de validação; as comparações a seguir devem ser tratadas com cautela."
            if _ar_fin(C["Vina_score"]):
                s += (f" O controle apresentou Vina = {_ar_n(C['Vina_score'], 2)} kcal/mol, CNNscore = {_ar_n(C['CNNscore'], 2)} e "
                      f"CNNaffinity = {_ar_n(C['CNNaffinity_pK'], 2)} pK (LE{subvina} = {_ar_n(C['LE_Vina'], 2)}).")
            B.append(("p", s.strip()))
        else:
            B.append(("p", "O leaderboard não contém ligante nativo como controle; portanto, não é possível reportar o RMSD de redocking "
                           "nem comparar os ligantes prospectivos com uma referência cristalográfica. Os resultados a seguir correspondem a "
                           "uma triagem prospectiva, em que a plausibilidade das poses depende do critério PoseBusters = PASS."))

        # ---- parágrafo 2: ancoramento dos ligantes prospectivos, composto líder e potência relativa
        n_pass = int((tests["_pb"] == "PASS").sum())
        shown = labs_all[:6]
        labs_txt = _ar_join_pt(shown) + (f" e outros {len(labs_all) - 6}" if len(labs_all) > 6 else "")
        if n_t == 1:
            s = (f"O único ligante prospectivo ({labs_txt}) {'foi aprovado' if n_pass else 'não foi aprovado'} no PoseBusters. "
                 f"O {lead} apresentou escore do Vina de {_ar_n(L['Vina_score'], 2)} kcal/mol e afinidade prevista de "
                 f"{_ar_n(L['CNNaffinity_pK'], 2)} pK (K{subd} previsto ≈ {self._kd(f, L['CNNaffinity_pK'])}), com CNNscore de "
                 f"{_ar_n(L['CNNscore'], 2)}. ")
        else:
            s = (f"Entre os {n_t} ligantes prospectivos ({labs_txt}), {n_pass} {_ar_pl(n_pass, 'foi aprovado', 'foram aprovados')} no "
                 f"PoseBusters. ")
            if self.consensus:
                s += (f"O {lead} é o composto líder e lidera simultaneamente os dois critérios: menor escore do Vina "
                      f"({_ar_n(L['Vina_score'], 2)} kcal/mol) e maior afinidade prevista pela 3D-CNN ({_ar_n(L['CNNaffinity_pK'], 2)} pK; "
                      f"K{subd} previsto ≈ {self._kd(f, L['CNNaffinity_pK'])}), com CNNscore de {_ar_n(L['CNNscore'], 2)}. ")
            else:
                s += (f"Os dois critérios não convergem para um único composto: o menor escore do Vina pertence ao "
                      f"{self._alvo(f, self.best_vina_name, bold=True)} e a maior CNNaffinity ao {self._alvo(f, self.best_pk_name, bold=True)}. "
                      f"Adotando a soma das posições nos dois rankings, o composto líder é o {lead}, com Vina = {_ar_n(L['Vina_score'], 2)} kcal/mol, "
                      f"CNNaffinity = {_ar_n(L['CNNaffinity_pK'], 2)} pK (K{subd} previsto ≈ {self._kd(f, L['CNNaffinity_pK'])}) e "
                      f"CNNscore = {_ar_n(L['CNNscore'], 2)}; a discordância entre o escore empírico e o neural indica que a hierarquia deve "
                      f"ser lida com cautela. ")
        s += (f"O PoseBusters do composto líder foi {f.b(L['_pb'])}"
              + (". " if L["_pb"] == "PASS" else ", o que compromete a plausibilidade física da pose e exige inspeção antes de qualquer "
                                                    "conclusão estrutural. "))
        if self.ctrl is not None and _ar_fin(self.ctrl["Vina_score"]):
            C = self.ctrl
            dv = L["Vina_score"] - C["Vina_score"]
            dk = L["CNNaffinity_pK"] - C["CNNaffinity_pK"] if _ar_fin(C["CNNaffinity_pK"]) else np.nan
            s += (f"Em relação ao Controle Nativo, o composto líder apresenta ΔVina = {_ar_n(dv, 2, sign=True)} kcal/mol "
                  f"({'mais' if dv < 0 else 'menos'} favorável)"
                  + (f" e ΔCNNaffinity = {_ar_n(dk, 2, sign=True)} pK ({'equivalente' if abs(dk) < 0.005 else 'maior' if dk > 0 else 'menor'} afinidade prevista)" if _ar_fin(dk) else "")
                  + f"; o número de átomos pesados é {_ar_n(L['HAC'], 0)} contra {_ar_n(C['HAC'], 0)} do controle. ")
        if len(self.secondary):
            sec = self.secondary.head(self.max_sec)
            parts = []
            for _, r in sec.iterrows():
                d = L["CNNaffinity_pK"] - r["CNNaffinity_pK"]
                fold = 10.0 ** abs(d)
                ddg = AR_RT_LN10 * abs(d)
                rl = self._alvo(f, r["Composto"], bold=True)
                if d >= 0:
                    core = (f"{rl} apresenta pK = {_ar_n(r['CNNaffinity_pK'], 2)} e Vina = {_ar_n(r['Vina_score'], 2)} kcal/mol, "
                            f"com ΔpK = {_ar_n(d, 2)}, isto é, a concentração prevista para saturar metade do sítio é ≈ {self._fold(f, fold)} vezes "
                            f"menor para o líder (ΔΔG ≈ {_ar_n(ddg, 2)} kcal/mol a 298 K)")
                else:
                    core = (f"{rl} apresenta pK = {_ar_n(r['CNNaffinity_pK'], 2)} e Vina = {_ar_n(r['Vina_score'], 2)} kcal/mol, "
                            f"com ΔpK = {_ar_n(d, 2)}, ou seja, afinidade neural prevista ≈ {self._fold(f, fold)} vezes maior que a do líder, "
                            f"embora o conjunto dos dois rankings favoreça o líder")
                if abs(d) < self.pk_res:
                    core += f", diferença inferior à resolução adotada ({_ar_n(self.pk_res, 1)} unidade de pK) e sem hierarquia robusta"
                parts.append(core)
            s += (f"Definindo ΔpK = pK{pk_lead} − pK{pk_sec} e lembrando que pK = −log{sub10}\u00a0K{subd}, cada unidade de ΔpK "
                  f"corresponde a uma razão de 10 vezes na concentração (dose) necessária para o mesmo grau de saturação do sítio e a "
                  f"≈ {_ar_n(AR_RT_LN10, 2)} kcal/mol de energia livre: {'; '.join(parts)}. ")
            if len(self.secondary) > self.max_sec:
                k_ = len(self.secondary) - self.max_sec
                s += f"{_ar_pl(k_, 'O', 'Os')} {k_} {_ar_pl(k_, 'demais ligante consta', 'demais ligantes constam')} na Tabela 1. "
        s += ("Os escores de todos os ligantes estão reunidos na Tabela 1. Ressalta-se que a CNNaffinity é uma predição do modelo, não uma "
              "medida experimental; as razões acima descrevem a ordenação prevista e não substituem a determinação experimental de "
              "constantes de inibição.")
        B.append(("p", s.strip()))
        hdr = ["Composto", "Papel", "HAC", "Vina", "CNNscore", "pK", "LE_Vina", "LE_CNN", "RMSD (Å)", "PoseBusters"]
        rows = []
        for _, r in self.table_df.iterrows():
            nm = f.t(r["Composto"])
            rows.append([f.b(nm) if r["Composto"] == L["Composto"] and not r["_ctrl"] else nm, f.t(self._role_label(r)), _ar_n(r["HAC"], 0),
                         _ar_n(r["Vina_score"], 2), _ar_n(r["CNNscore"], 2), _ar_n(r["CNNaffinity_pK"], 2), _ar_n(r["LE_Vina"], 2),
                         _ar_n(r["LE_CNN"], 2), _ar_n(r["RMSD"], 2), f.t(r["_pb"])])
        note1 = ("Nota: HAC = número de átomos pesados; Vina = escore empírico do AutoDock Vina (kcal/mol); CNNscore = probabilidade de "
                 f"pose correta estimada pela 3D-CNN (0 a 1); pK = −log{sub10}\u00a0K{subd} previsto pela 3D-CNN (CNNaffinity); {le_v} e {le_c} = "
                 "eficiências de ligante (kcal/mol por átomo pesado); RMSD = desvio quadrático médio simétrico de redocking (Å); "
                 "PoseBusters = auditoria físico-química da pose (PASS/FAIL). O composto líder está em negrito.")
        B.append(("table", {"caption": "Parâmetros termodinâmicos e escores de docking molecular.", "note": note1,
                            "label": "tab:resultados", "header": hdr, "rows": rows, "align": "l l r r r r r r r l", "wide": False}))

        # ---- parágrafo 3: eficiência de ligante e viés de tamanho
        le = tests["LE_Vina"]
        n_ok = int((le >= self.le_thr).sum())
        n_le = int(le.notna().sum())
        names_low = [self._alvo(f, x) for x in tests.loc[le < self.le_thr, "Composto"]]
        rng = ("n/d" if not n_le else f"valor {_ar_n(le.min(), 2)}" if _ar_n(le.min(), 2) == _ar_n(le.max(), 2) else f"faixa {_ar_n(le.min(), 2)}–{_ar_n(le.max(), 2)}")
        thr = _ar_n(self.le_thr, 2)
        if n_le == 0:
            s = "A eficiência de ligante não pôde ser avaliada por ausência de dados de Vina e HAC."
        elif n_ok == n_le:
            quant = "O único ligante prospectivo apresentou" if n_le == 1 else f"Todos os {n_le} ligantes prospectivos apresentaram"
            s = (f"{quant} {le_v} ≥ {thr} kcal/mol por átomo pesado ({rng}), o que descarta o viés de tamanho molecular "
                 f"({f.i('size bias')}) como explicação da afinidade prevista{f.cite('hopkins2004', 'kenny2019')}.")
        elif n_ok == 0:
            s = (f"Nenhum ligante prospectivo atingiu {le_v} ≥ {thr} kcal/mol por átomo pesado ({rng}); os escores favoráveis podem "
                 f"refletir, em boa parte, o tamanho molecular ({f.i('size bias')}), e a priorização deve se apoiar em outros critérios"
                 f"{f.cite('kenny2019', 'abad2021')}.")
        else:
            nl = len(names_low)
            s = (f"{n_ok} de {n_le} ligantes prospectivos {_ar_pl(n_ok, 'atingiu', 'atingiram')} {le_v} ≥ {thr} kcal/mol por átomo pesado "
                 f"({rng}); abaixo do limiar {_ar_pl(nl, 'ficou', 'ficaram')} {_ar_join_pt(names_low)}, "
                 f"{_ar_pl(nl, 'cujo escore deve', 'cujos escores devem')} ser interpretado{'' if nl == 1 else 's'} com cautela por possível "
                 f"viés de tamanho ({f.i('size bias')}){f.cite('kenny2019', 'abad2021')}.")
        if _ar_fin(L["LE_Vina"]):
            s += (f" O composto líder apresenta {le_v} = {_ar_n(L['LE_Vina'], 2)}"
                  f"{' (≥ ' + thr + ')' if L['LE_Vina'] >= self.le_thr else ' (< ' + thr + ')'}"
                  + (f" e {le_c} = {_ar_n(L['LE_CNN'], 2)}" if _ar_fin(L["LE_CNN"]) else "")
                  + " kcal/mol por átomo pesado.")
        if self.df["_le_calc"].any():
            s += " Para os compostos sem valor de LE no leaderboard, ele foi calculado como −Vina/HAC."
        B.append(("p", s))

        # ---- parágrafo 4: mecanismo de interação no sítio ativo (ProLIF)
        items = _ar_split_interactions(L["Interacoes_Chave"])
        metal_items = [i for i in items if self._mentions_metal(i)]
        other_items = [i for i in items if i not in metal_items]
        s = self._compartment_text(f)
        if not items:
            s += f" Não há interações registradas para o composto líder {lead}, de modo que o seu modo de ligação não pôde ser caracterizado."
        elif metal_items:
            s += (f" No composto líder {lead}, o mapeamento evidencia {f.b('coordenação/quelação direta ao cofator metálico')} "
                  f"({f.t(_ar_join_pt(metal_items))})"
                  + (f", acompanhada de contatos adicionais com {f.t(_ar_join_pt(other_items))}" if other_items else "")
                  + ". Esse perfil é compatível com ocupação do centro metálico do sítio catalítico, mecanismo típico de inibidores que "
                    "competem com o substrato pela esfera de coordenação.")
        else:
            s += (f" O composto líder {lead} não estabelece contato com cofator metálico"
                  + (f" (apesar de o sítio conter {f.t(self._metal_phrase())})" if c["has_metals"] and self._metal_phrase() else "")
                  + f"; suas interações restringem-se a {f.t(_ar_join_pt(items))}. Esse perfil, baseado em ligações de hidrogênio e contatos "
                    f"hidrofóbicos com resíduos que revestem a fenda, é compatível com {f.b('estabilização por oclusão estérica')}, isto é, "
                    f"com impedimento de entrada do substrato na fenda do sítio, e não com coordenação direta ao cofator.")
        if self.ctrl is not None:
            ck = {_ar_res_key(i) for i in _ar_split_interactions(self.ctrl["Interacoes_Chave"])}
            lk = {_ar_res_key(i): i for i in items}
            shared = [lk[k] for k in lk if k in ck]
            if ck and items:
                lab = ", ".join(sorted({re.match(r"\s*([A-Za-z]{1,4}\s*-?\d+)", s_).group(1).replace(" ", "")
                                        if re.match(r"\s*([A-Za-z]{1,4}\s*-?\d+)", s_) else s_ for s_ in shared}))
                s += (f" Comparado ao Controle Nativo, o líder compartilha {len(shared)} de {len(lk)} {_ar_pl(len(lk), 'resíduo', 'resíduos')} de contato"
                      + (f" ({f.t(lab)})" if shared else "") + f", enquanto o controle contacta {len(ck)} {_ar_pl(len(ck), 'resíduo', 'resíduos')}.")
        s += " As interações-chave de cada ligante estão listadas na Tabela 2."
        B.append(("p", s.strip()))
        rows2 = [[f.t(f"{self._role_label(r)}: {r['Composto']}"), f.t("; ".join(_ar_split_interactions(r["Interacoes_Chave"])) or "—")]
                 for _, r in self.table_df.iterrows()]
        codes = {"HB": "ligação de hidrogênio", "Hf": "contato hidrofóbico", "PiPi": "empilhamento π", "PiCat": "interação π–cátion",
                 "Metal": "contato com íon metálico", "Ionic": "interação iônica", "Anion": "interação aniônica", "Cation": "interação catiônica",
                 "XB": "ligação de halogênio", "Hb": "ligação de hidrogênio"}
        present = [k for k in codes if any(re.search(rf"\({k}\)", str(x)) for x in self.df["Interacoes_Chave"])]
        note2 = ("Nota: " + "; ".join(f"{k} = {codes[k]}" for k in present) + ".") if present else ""
        B.append(("table", {"caption": "Interações intermoleculares no sítio ativo mapeadas via ProLIF.", "note": note2,
                            "label": "tab:interacoes", "header": ["Ligante", "Interações-chave"], "rows": rows2, "align": "l X", "wide": True}))
        if self.is_predicted:                                          # parágrafo dedicado, somente para modelo predito
            B.append(("p", self._predicted_paragraph(f)))
        return B

    def _predicted_paragraph(self, f: _ar_Fmt) -> str:
        """Confiabilidade do bolsão pelo pLDDT médio (coluna do fator B do modelo) e justificativa do refino físico do pipeline."""
        if self.plddt_mean is not None:
            mn = f" (mínimo de {f.b(_ar_n(self.plddt_min, 1))})" if self.plddt_min is not None else ""
            cls = _ar_classify_plddt(self.plddt_mean)
            s = (f"Por tratar-se de um modelo predito, a confiabilidade do sítio de ligação foi avaliada pelo pLDDT, métrica de confiança "
                 f"por resíduo (0 a 100) registrada na coluna do fator B do arquivo. No bolsão de ligação, o pLDDT médio dos átomos pesados "
                 f"foi {f.b(_ar_n(self.plddt_mean, 1))}{mn}, o que corresponde a: {f.t(cls)}.")
            if self.plddt_mean < 70.0:
                s += " Os resultados de docking nesse bolsão devem, portanto, ser interpretados com cautela e, preferencialmente, corroborados por dinâmica molecular."
        else:
            s = ("Por tratar-se de um modelo predito, a confiabilidade do sítio de ligação deve ser avaliada pelo pLDDT; o valor do bolsão não "
                 "pôde ser calculado nesta corrida.")
        s += (" Embora o modelo predito forneça o arcabouço conformacional para o docking, a dinâmica local do sítio e a ausência de cálculo "
              "direto de energia livre de ligação no AlphaFold 3 justificam o refino físico e a triagem energética conduzidos pelo presente "
              f"pipeline{f.cite('he2026')}.")
        return s

    # ---------------------------------------------------------------- compartimentos (ProLIF)
    def _compartment_text(self, f: _ar_Fmt) -> str:
        """Abertura do parágrafo ProLIF + classificação por ligante: 'núcleo ortostérico catalítico profundo' (toca metal/resíduos
        axiais catalíticos) versus 'vestíbulo periférico / canal de acesso' (só contatos periféricos)."""
        c = self.cfg
        td, dfn = c["target_domain"], c["domain_function"]
        _, do = _ar_art_domain(td)
        groups: Dict[str, List[str]] = {"core": [], "vestibulo": [], "nd": []}
        for _, r in self.ranked.iterrows():
            its = _ar_split_interactions(r["Interacoes_Chave"])
            lab = self._alvo(f, r["Composto"], bold=r["Composto"] == self.leader["Composto"])
            if its:
                res = [f.t(re.sub(r"\s*\(.*$", "", i).strip()) for i in its[:4]]
                lab = lab[:-1] + f"; contatos: {_ar_join_pt(res)}{' e outros' if len(its) > 4 else ''})"
            groups[self._compartment(r["Interacoes_Chave"])].append(lab)
        n_comp = sum(1 for k in ("core", "vestibulo") if groups[k])
        dom = f"{do} {f.t(td)} ({f.t(dfn)})"
        cite = f.cite("bouysset2021")
        if n_comp == 0:
            return (f"Não há interações registradas para os ligantes prospectivos; os compartimentos {dom} não puderam ser "
                    f"discriminados pelo ProLIF.")
        if n_comp == 2:
            op = f"A análise tridimensional revela que os compostos exploram compartimentos distintos {dom}{cite}."
        elif len(self.ranked) == 1:
            op = f"A análise tridimensional revela que o ligante prospectivo explora um compartimento específico {dom}{cite}."
        else:
            op = (f"A análise tridimensional revela que todos os ligantes prospectivos exploram o mesmo compartimento {dom}{cite}, "
                  f"sem segregação entre compartimentos.")
        parts = [op]

        def lst(names: List[str]) -> str:
            more = f" e outros {len(names) - 8}" if len(names) > 8 else ""
            return "; ".join(names[:8]) + more
        if groups["core"]:
            n = len(groups["core"])
            parts.append(f"{lst(groups['core'])} {_ar_pl(n, 'ancora-se', 'ancoram-se')} no {f.b('núcleo ortostérico catalítico profundo')} "
                         f"{do} {f.t(td)}, em contato com o centro catalítico (quelação do cofator metálico ou resíduos axiais), o que "
                         f"é compatível com inibição direta do maquinário de {f.t(dfn)}.")
        if groups["vestibulo"]:
            n = len(groups["vestibulo"])
            parts.append(f"{lst(groups['vestibulo'])} {_ar_pl(n, 'ancora-se', 'ancoram-se')} no {f.b('vestíbulo periférico / canal de acesso')} "
                         f"{do} {f.t(td)}, apenas com contatos polares e/ou hidrofóbicos sem tocar o centro catalítico, "
                         f"atuando por oclusão estérica da entrada de substratos.")
        if groups["nd"]:
            parts.append(f"{lst(groups['nd'])} {_ar_pl(len(groups['nd']), 'não apresentou', 'não apresentaram')} interações registradas e "
                         f"{_ar_pl(len(groups['nd']), 'permanece', 'permanecem')} sem classificação de compartimento.")
        if not self._catalytic_keys():
            parts.append("Como nenhum cofator metálico ou resíduo catalítico foi anotado, o centro catalítico só foi reconhecido "
                         "quando a própria interação era designada como metálica, axial ou catalítica; a atribuição de compartimento é, "
                         "portanto, indicativa.")
        return " ".join(parts)

    # ------------------------------------------------------------------ renderização
    def _title(self) -> str:
        c = self.cfg
        pdb = str(c["pdb_id"]).upper()
        name = str(c["protein_name"])
        who = f"cadeia {c['target_chain']}" if name.upper() == f"PDB {pdb}" else f"PDB {pdb}, cadeia {c['target_chain']}"
        return str(c.get("title") or f"Docking molecular com aprendizado profundo: {name} ({who})")

    def _source_line(self) -> str:
        """'Fonte: Elaborado pelo autor (<ano corrente>).' (NBR 14724); `table_source` no config substitui o texto."""
        return "Fonte: " + str(self.cfg.get("table_source") or f"Elaborado pelo autor ({time.strftime('%Y')}).").strip()

    @staticmethod
    def _sorted_refs(cited: List[str]) -> List[str]:
        """Lista de referências em ordem alfabética de autoria (NBR 6023)."""
        return sorted(cited, key=_ar_ref_sort_key)

    @staticmethod
    def _heading_labels(blocks: List[Tuple[str, Any]]) -> List[str]:
        """Numeração progressiva das seções (NBR 6024): '1', '1.1', ... alinhada à lista de blocos."""
        a = b = 0
        out: List[str] = []
        for kind, _ in blocks:
            if kind == "h2":
                a, b = a + 1, 0
                out.append(f"{a}")
            elif kind == "h3":
                b += 1
                out.append(f"{a}.{b}")
            else:
                out.append("")
        return out

    def to_markdown(self) -> str:
        """Relatório em Markdown (tabelas pipe, equações em $...$, citações autor-data ABNT e referências NBR 6023)."""
        f = _ar_MdFmt()
        blocks = self._build(f)
        out = [f"# {f.t(self._title())}", ""]
        for kind, p in blocks:
            if kind == "h2":
                out += [f"## {p}", ""]
            elif kind == "h3":
                out += [f"### {p}", ""]
            elif kind == "p":
                out += [p, ""]
            elif kind == "ul":
                out += [f"- {x}" for x in p] + [""]
            elif kind == "eq":
                out += [f"$${p}$$", ""]
            elif kind == "table":
                out += [f"**Tabela {1 if p['label'] == 'tab:resultados' else 2} –** {p['caption']}", "",
                        "| " + " | ".join(p["header"]) + " |", "|" + "|".join(["---"] * len(p["header"])) + "|"]
                out += ["| " + " | ".join(r) + " |" for r in p["rows"]] + ["", f"{self._source_line()}", ""]
                if p.get("note"):
                    out += [p["note"], ""]
        out += ["## Referências", ""]
        for k in self._sorted_refs(f.cited):
            out += [self._format_ref(k, "md"), ""]
        return "\n".join(out).rstrip() + "\n"

    # CSS de impressão (WeasyPrint): A4, margens ABNT (sup 3 cm, dir 2 cm, inf 2 cm, esq 3 cm), Arial 12 pt, entrelinha 1,5.
    _PRINT_CSS = """
@page { size: A4; margin: 3cm 2cm 2cm 3cm;
        @top-right { content: counter(page); font-family: Arial, "Liberation Sans", Helvetica, sans-serif; font-size: 10pt; } }
html { font-family: Arial, "Liberation Sans", Helvetica, sans-serif; font-size: 12pt; color: #000; }
body { margin: 0; line-height: 1.5; text-align: justify; }
p { margin: 0; text-indent: 1.25cm; orphans: 2; widows: 2; }
h1.ar-title { font-size: 14pt; font-weight: bold; text-align: center; text-transform: uppercase; margin: 0 0 1.2em 0; }
h2 { font-size: 12pt; font-weight: bold; text-transform: uppercase; text-align: left; margin: 1.5em 0 .8em 0; page-break-after: avoid; }
h3 { font-size: 12pt; font-weight: bold; text-align: left; margin: 1.2em 0 .6em 0; page-break-after: avoid; }
h2.ar-refs { text-align: center; page-break-before: always; margin-top: 0; }
ul { margin: .2em 0 .6em 0; padding-left: 1.9cm; }
li { text-align: justify; }
div.ar-eq { text-align: center; margin: .5em 0; }
p.ar-cap { font-size: 10pt; line-height: 1.15; text-indent: 0; text-align: left; margin: 1.1em 0 .4em 0; page-break-after: avoid; }
p.ar-src { font-size: 10pt; line-height: 1.15; text-indent: 0; text-align: left; margin: .35em 0 .35em 0; }
table { margin: 0 auto; border-collapse: collapse; font-size: 9pt; line-height: 1.2; width: 100%; }
table.ar-t10 { font-size: 8pt; }
thead { display: table-header-group; }
tr { page-break-inside: avoid; }
th { font-weight: bold; text-align: center; padding: 3pt 4pt; border-top: 1pt solid #000; border-bottom: .75pt solid #000; }
td { padding: 2pt 4pt; vertical-align: top; text-align: left; }
td.r { text-align: right; }
tbody tr:last-child td { border-bottom: 1pt solid #000; }
p.ar-ref { text-indent: 0; text-align: left; line-height: 1.15; margin: 0 0 12pt 0; }
"""

    def to_html(self, *, print_mode: bool = False) -> str:
        """Fragmento HTML para painéis (Gradio, IPython.display.HTML). Com `print_mode=True` devolve um documento HTML completo
        (UTF-8, seções numeradas, CSS de impressão ABNT), que é a entrada de `to_pdf`."""
        f = _ar_HtmlFmt()
        blocks = self._build(f)
        labels = self._heading_labels(blocks) if print_mode else [""] * len(blocks)
        if print_mode:
            css_p = css_h = css_tbl = ""
            cell = lambda al: (' class="r"' if al == "r" else "")                       # noqa: E731
            out = [f'<h1 class="ar-title">{f.t(self._title())}</h1>']
        else:
            css_p = css_h = ""
            css_tbl = ' style="border-collapse:collapse;font-size:.88em;margin:.4em 0 1em 0;width:100%"'
            cell_css = "border-bottom:1px solid rgba(127,127,127,.35);padding:3px 8px;text-align:left"
            cell = lambda al: f' style="{cell_css}{";text-align:right" if al == "r" else ""}"'   # noqa: E731
            out = ['<div class="ar-report" style="font-size:.93em;line-height:1.5;color:inherit">', f"<h2>{f.t(self._title())}</h2>"]
        for (kind, p), lab in zip(blocks, labels):
            num = f"{lab} " if lab else ""
            if kind == "h2":
                out.append(f"<h2>{num}{p}</h2>" if print_mode else f"<h3>{p}</h3>")
            elif kind == "h3":
                out.append(f"<h3>{num}{p}</h3>" if print_mode else f"<h4>{p}</h4>")
            elif kind == "p":
                out.append(f"<p>{p}</p>")
            elif kind == "ul":
                out.append("<ul>" + "".join(f"<li>{x}</li>" for x in p) + "</ul>")
            elif kind == "eq":
                try:
                    out.append(f'<div class="ar-eq">{_ar_tex2html_eq(p)}</div>')
                except ValueError:
                    out.append(f'<div class="ar-eq"><i>{_ar_tex2html(p)}</i></div>')
            elif kind == "table":
                n = 1 if p["label"] == "tab:resultados" else 2
                als = p["align"].split()
                head = "".join(f"<th{'' if print_mode else ' style=' + chr(34) + 'border-bottom:1px solid rgba(127,127,127,.35);padding:3px 8px;text-align:left' + chr(34)}>{_ar_hcell(f, h)}</th>" for h in p["header"])
                body = "".join("<tr>" + "".join(f"<td{cell(als[j] if j < len(als) else 'l')}>{c}</td>" for j, c in enumerate(r)) + "</tr>"
                               for r in p["rows"])
                tcls = ' class="ar-t10"' if print_mode and len(p["header"]) >= 8 else ""
                if print_mode:
                    out.append(f'<p class="ar-cap"><b>Tabela {n} –</b> {p["caption"]}</p><table{tcls}><thead><tr>{head}</tr></thead>'
                               f'<tbody>{body}</tbody></table><p class="ar-src">{f.t(self._source_line())}</p>'
                               + (f'<p class="ar-src">{p["note"]}</p>' if p.get("note") else ""))
                else:
                    out.append(f'<p><b>Tabela {n} –</b> {p["caption"]}</p><div style="overflow-x:auto"><table{css_tbl}>'
                               f'<thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'
                               f'<p style="font-size:.85em;opacity:.8">{f.t(self._source_line())}</p>'
                               + (f'<p style="font-size:.85em;opacity:.8">{p["note"]}</p>' if p.get("note") else ""))
        refs = "".join(f'<p class="ar-ref"{"" if print_mode else " style=" + chr(34) + "margin:.3em 0" + chr(34)}>{self._format_ref(k, "html")}</p>'
                       for k in self._sorted_refs(f.cited))
        if print_mode:
            out.append(f'<h2 class="ar-refs">Referências</h2>{refs}')
            return ('<!DOCTYPE html>\n<html lang="pt-BR"><head><meta charset="utf-8">'
                    f"<title>{f.t(self._title())}</title><style>{self._PRINT_CSS}</style></head><body>\n" + "\n".join(out) + "\n</body></html>\n")
        out.append(f"<h3>Referências</h3>{refs}</div>")
        return "\n".join(out)

    @staticmethod
    def _format_ref(key: str, fmt: str = "md") -> str:
        """Referência NBR 6023:  SOBRENOME, P. et al. Título. **Periódico**, v. X, n. Y, p. Z–W, ano. DOI: https://doi.org/...
        `fmt`: 'md' (periódico em **negrito**), 'html' (<b>) ou 'plain' (sem marcação)."""
        d = _ar_bib_fields(AR_BIB_DB[key])
        if fmt == "html":
            esc = lambda s: _htmllib.escape(s, quote=False)                                   # noqa: E731
            bold, ital = "<b>{}</b>", ("<i>", "</i>")
        elif fmt == "md":
            esc = lambda s: _ar_MdFmt().t(s).replace("<", "&lt;").replace(">", "&gt;")          # noqa: E731
            bold, ital = "**{}**", ("*", "*")
        else:
            esc = lambda s: s                                                                   # noqa: E731
            bold, ital = "{}", ("", "")

        def end(x: str) -> str:
            return x if x.endswith((".", "?", "!")) else x + "."
        au = _ar_ref_authors(key)
        # nomes científicos em itálico: \textit{...} / \emph{...} viram marcadores antes da limpeza e voltam como marcação depois
        title = _ar_bib_plain(re.sub(r"\\(?:textit|emph)\{([^{}]*)\}", "\u0001\\1\u0002", d.get("title", "")))
        year = _ar_year_label(key)
        jr = _ar_bib_plain(d.get("journal", ""))
        vol, num, pg = _ar_bib_plain(d.get("volume", "")), _ar_bib_plain(d.get("number", "")), _ar_bib_plain(d.get("pages", ""))
        head = (esc(end(au)) + " ") if au else ""
        if jr:
            loc = [bold.format(esc(jr))] + ([f"v. {esc(vol)}"] if vol else []) + ([f"n. {esc(num)}"] if num else []) \
                + ([f"p. {esc(pg)}"] if pg else []) + [year]
            body = f"{head}{esc(end(title))} {', '.join(loc)}."
        else:
            body = f"{head}{esc(end(title))} [S. l.], {year}."
            if d.get("url"):
                body += f" Disponível em: {esc(d['url'])}."
        if d.get("doi"):
            body += f" DOI: https://doi.org/{esc(d['doi'])}"
        return body.replace("\u0001", ital[0]).replace("\u0002", ital[1])

    def to_latex(self) -> str:
        """Documento .tex completo para abntex2 + biblatex-abnt (citações com \\parencite, \\printbibliography).
        Compilar: pdflatex -> biber -> pdflatex (x2)."""
        f = _ar_TexFmt()
        blocks = self._build(f)
        body: List[str] = []
        src = f.t(self._source_line().replace("Fonte: ", "", 1))
        for kind, p in blocks:
            if kind == "h2":
                body += [rf"\section{{{p}}}", ""]
            elif kind == "h3":
                body += [rf"\subsection{{{p}}}", ""]
            elif kind == "p":
                body += [p, ""]
            elif kind == "ul":
                body += [r"\begin{itemize}"] + [rf"  \item {x}" for x in p] + [r"\end{itemize}", ""]
            elif kind == "eq":
                body += [r"\begin{equation}", f"  {p}", r"\end{equation}", ""]
            elif kind == "table":
                cols = "".join(("l" if a == "l" else "r" if a == "r" else r">{\raggedright\arraybackslash}X") for a in p["align"].split())
                hdr = " & ".join(rf"\textbf{{{_ar_hcell(f, h)}}}" for h in p["header"])
                rows = " \\\\\n    ".join(" & ".join(r) for r in p["rows"])
                if p["wide"]:
                    tab = (rf"\begin{{tabularx}}{{\textwidth}}{{{cols}}}" + "\n    \\toprule\n    " + hdr + " \\\\\n    \\midrule\n    " +
                           rows + " \\\\\n    \\bottomrule\n  \\end{tabularx}")
                else:
                    tab = (r"\resizebox{\textwidth}{!}{%" + "\n  " + rf"\begin{{tabular}}{{{cols}}}" + "\n    \\toprule\n    " + hdr +
                           " \\\\\n    \\midrule\n    " + rows + " \\\\\n    \\bottomrule\n  \\end{tabular}}")
                body += [r"\begin{table}[htbp]", r"  \centering\small", rf"  \caption{{{p['caption']}}}", rf"  \label{{{p['label']}}}",
                         "  " + tab, rf"  \par\vspace{{2pt}}{{\footnotesize Fonte: {src}}}"]
                if p.get("note"):
                    body += [rf"  \par{{\footnotesize {p['note']}}}"]
                body += [r"\end{table}", ""]
        authors = self.cfg.get("authors")
        au = (" \\and ".join(f.t(a) for a in authors) if isinstance(authors, (list, tuple)) else f.t(authors or ""))
        head = "\n".join([
            r"\documentclass[12pt,openright,oneside,a4paper,english,brazil]{abntex2}",
            r"\usepackage[utf8]{inputenc}",
            r"\usepackage[T1]{fontenc}",
            r"\usepackage{amsmath,booktabs,array,tabularx,graphicx}",
            r"\usepackage[backend=biber,style=abnt,justify,noslsn,extrayear]{biblatex}",
            r"\addbibresource{referencias.bib}",
            "",
            rf"\title{{{f.t(self._title())}}}",
            rf"\author{{{au}}}",
            r"\date{\today}",
            "",
            r"\begin{document}",
            r"\maketitle",
            "",
        ])
        tail = "\n".join(["", r"\printbibliography", "", r"\end{document}", ""])
        return _ar_tex_unicode(head + "\n".join(body) + tail)

    def to_bibtex(self, existing_bib: Optional[str] = None) -> str:
        """referencias.bib atualizado: entradas já existentes são preservadas; faltantes citadas são acrescentadas."""
        f = _ar_TexFmt()
        self._build(f)
        base = (existing_bib or "").rstrip()
        have = set(_ar_bib_keys(base))
        add = [AR_BIB_DB[k] for k in f.cited if k not in have]
        return (base + ("\n\n" if base and add else "") + "\n\n".join(add)).strip() + "\n"

    def to_plaintext(self) -> str:
        """Versão em texto simples (sem marcações Markdown/LaTeX) para colar em editores de texto."""
        md = self.to_markdown()
        md = re.sub(r"\$\$(.*?)\$\$", r"[eq] \1", md)
        md = re.sub(r"\$(.*?)\$", r"\1", md)
        md = md.replace("**", "").replace("\\|", "|").replace("\\_", "_").replace("\\*", "*").replace("\\$", "$")
        md = re.sub(r"(?<!\w)\*(.+?)\*(?!\w)", r"\1", md)
        return _htmllib.unescape(md.replace("`", ""))

    # ------------------------------------------------------------------ Word (.docx)
    def _build_docx(self, omml: bool = True) -> Any:
        """Monta o documento python-docx: A4, margens ABNT (sup 3, esq 3, inf 2, dir 2 cm), Arial 12, entrelinha 1,5, recuo de
        primeira linha de 1,25 cm, títulos em negrito (nível 1 em caixa alta), tabelas centralizadas e quebra antes de REFERÊNCIAS."""
        try:
            from docx import Document
            from docx.enum.style import WD_STYLE_TYPE
            from docx.enum.table import WD_TABLE_ALIGNMENT
            from docx.enum.text import WD_ALIGN_PARAGRAPH
            from docx.oxml import OxmlElement
            from docx.oxml.ns import qn
            from docx.shared import Cm, Pt, RGBColor
        except ImportError as e:                                    # pragma: no cover - depende do ambiente
            raise RuntimeError("python-docx não está instalado (pip install python-docx)") from e

        def fonts(style: Any, size: float, bold: Optional[bool] = None, italic: Optional[bool] = None) -> None:
            style.font.name, style.font.size = "Arial", Pt(size)
            if bold is not None:
                style.font.bold = bold
            if italic is not None:
                style.font.italic = italic
            style.font.color.rgb = RGBColor(0, 0, 0)
            rpr = style.element.get_or_add_rPr()
            rf = rpr.get_or_add_rFonts()
            for att in ("w:asciiTheme", "w:hAnsiTheme", "w:eastAsiaTheme", "w:cstheme"):
                rf.attrib.pop(qn(att), None)
            rf.set(qn("w:eastAsia"), "Arial")
            rf.set(qn("w:cs"), "Arial")
            col = rpr.find(qn("w:color"))
            if col is not None:
                for att in ("w:themeColor", "w:themeShade", "w:themeTint"):
                    col.attrib.pop(qn(att), None)

        def para_fmt(style: Any, *, align: Any = None, first: Optional[float] = None, spacing: float = 1.5, before: float = 0.0,
                     after: float = 0.0, keep_next: Optional[bool] = None) -> None:
            pf = style.paragraph_format
            pf.line_spacing, pf.space_before, pf.space_after = spacing, Pt(before), Pt(after)
            if align is not None:
                pf.alignment = align
            if first is not None:
                pf.first_line_indent = Cm(first)
            if keep_next is not None:
                pf.keep_with_next = keep_next

        doc = Document()
        sec = doc.sections[0]
        sec.page_width, sec.page_height = Cm(21.0), Cm(29.7)
        sec.top_margin, sec.left_margin, sec.bottom_margin, sec.right_margin = Cm(3), Cm(3), Cm(2), Cm(2)
        sec.header_distance = sec.footer_distance = Cm(1.25)
        S = doc.styles
        fonts(S["Normal"], 12)
        para_fmt(S["Normal"], align=WD_ALIGN_PARAGRAPH.JUSTIFY)

        def custom(name: str, size: float = 12, bold: Optional[bool] = None, **kw: Any) -> Any:
            st = S.add_style(name, WD_STYLE_TYPE.PARAGRAPH)
            st.base_style = S["Normal"]
            fonts(st, size, bold)
            para_fmt(st, **kw)
            return st
        custom("Texto ABNT", first=1.25, align=WD_ALIGN_PARAGRAPH.JUSTIFY)
        custom("Título do trabalho", 14, True, align=WD_ALIGN_PARAGRAPH.CENTER, spacing=1.5, after=12)
        custom("Equação ABNT", align=WD_ALIGN_PARAGRAPH.CENTER, spacing=1.5, before=6, after=6)
        custom("Legenda de tabela", 10, align=WD_ALIGN_PARAGRAPH.LEFT, spacing=1.0, before=12, after=4, keep_next=True)
        custom("Fonte de tabela", 10, align=WD_ALIGN_PARAGRAPH.LEFT, spacing=1.0, before=4, after=12)
        custom("Nota de tabela", 10, align=WD_ALIGN_PARAGRAPH.LEFT, spacing=1.0, before=0, after=12)
        custom("Texto de tabela", 9, align=WD_ALIGN_PARAGRAPH.LEFT, spacing=1.0)
        custom("Referência ABNT", align=WD_ALIGN_PARAGRAPH.LEFT, spacing=1.0, after=12)
        for nm, sz in (("Heading 1", 12), ("Heading 2", 12)):
            fonts(S[nm], sz, True, False)
            para_fmt(S[nm], align=WD_ALIGN_PARAGRAPH.LEFT, spacing=1.5, before=18 if nm == "Heading 1" else 12, after=6, keep_next=True,
                     first=0)
        fonts(S["List Bullet"], 12)
        para_fmt(S["List Bullet"], align=WD_ALIGN_PARAGRAPH.JUSTIFY, spacing=1.5)

        class _Inline(_HTMLParser):                                  # HTML do _ar_HtmlFmt -> runs do Word
            def __init__(self, par: Any, size: Optional[float] = None, bold: bool = False, italic: bool = False) -> None:
                super().__init__(convert_charrefs=True)
                self.par, self.size, self.stack = par, size, []
                self.base_b, self.base_i = bold, italic

            def handle_starttag(self, tag: str, attrs: Any) -> None:
                if tag in ("b", "strong", "i", "em", "sub", "sup", "code"):
                    self.stack.append(tag)

            def handle_endtag(self, tag: str) -> None:
                if tag in self.stack:
                    self.stack.reverse()
                    self.stack.remove(tag)
                    self.stack.reverse()

            def handle_data(self, data: str) -> None:
                if not data:
                    return
                r = self.par.add_run(data)
                st = self.stack
                if self.base_b or "b" in st or "strong" in st:
                    r.bold = True
                if self.base_i or "i" in st or "em" in st:
                    r.italic = True
                if "sub" in st:                                  # (atribuir None apagaria o vertAlign: só se define quando True)
                    r.font.subscript = True
                elif "sup" in st:
                    r.font.superscript = True
                if "code" in st:
                    r.font.name = "Courier New"
                if self.size:
                    r.font.size = Pt(self.size)

        def put(par: Any, html_text: str, **kw: Any) -> Any:
            ps = _Inline(par, **kw)
            ps.feed(html_text)
            ps.close()
            return par

        f = _ar_HtmlFmt()
        blocks = self._build(f)
        labels = self._heading_labels(blocks)
        put(doc.add_paragraph(style="Título do trabalho"), f.t(self._title()).upper())
        au = self.cfg.get("authors")
        if au:
            ap = doc.add_paragraph(style="Normal")
            ap.alignment = WD_ALIGN_PARAGRAPH.CENTER
            put(ap, f.t("; ".join(map(str, au)) if isinstance(au, (list, tuple)) else str(au)))
        # numeração de página no canto superior direito (NBR 14724)
        hp = sec.header.paragraphs[0]
        hp.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        run = hp.add_run()
        run.font.size = Pt(10)
        for typ, txt in (("begin", None), (None, "PAGE"), ("end", None)):
            if typ:
                el = OxmlElement("w:fldChar")
                el.set(qn("w:fldCharType"), typ)
            else:
                el = OxmlElement("w:instrText")
                el.set(qn("xml:space"), "preserve")
                el.text = txt
            run._r.append(el)

        def set_borders(table: Any) -> None:                         # só linhas horizontais: topo, rodapé e abaixo do cabeçalho
            tbl_pr = table._tbl.tblPr
            bd = OxmlElement("w:tblBorders")
            for edge, val in (("top", "single"), ("left", "nil"), ("bottom", "single"), ("right", "nil"), ("insideH", "nil"), ("insideV", "nil")):
                e = OxmlElement(f"w:{edge}")
                e.set(qn("w:val"), val)
                e.set(qn("w:sz"), "8" if val == "single" else "0")
                e.set(qn("w:space"), "0")
                e.set(qn("w:color"), "000000")
                bd.append(e)
            anchor = next((tbl_pr.find(qn(t)) for t in ("w:shd", "w:tblLayout", "w:tblCellMar", "w:tblLook") if tbl_pr.find(qn(t)) is not None), None)
            if anchor is not None:
                anchor.addprevious(bd)
            else:
                tbl_pr.append(bd)

        def add_table(p: dict, n: int) -> None:
            cap = doc.add_paragraph(style="Legenda de tabela")
            put(cap, f"<b>Tabela {n} –</b> {p['caption']}")
            hdr, rows = p["header"], p["rows"]
            als = p["align"].split()
            tb = doc.add_table(rows=1 + len(rows), cols=len(hdr))
            tb.alignment = WD_TABLE_ALIGNMENT.CENTER
            tb.autofit = False
            plain = lambda h: re.sub(r"<[^>]+>", "", h)                                            # noqa: E731
            total = 16.0                                           # largura útil (cm): 21 − 3 − 2
            size = 8 if len(hdr) >= 8 else 9
            cw = 0.0195 * size                                     # largura média de um caractere (cm) em Arial, tamanho `size` pt
            need: List[Optional[float]] = []
            for j, h in enumerate(hdr):
                if als[j] == "X":
                    need.append(None)
                else:
                    longest = max(len(w) for w in h.split()) * cw * 1.12            # cabeçalho em negrito
                    cells = max([len(plain(r[j])) for r in rows] + [0]) * cw
                    need.append(max(longest, min(cells, 2.4)) + 0.3)
            fixed = sum(x for x in need if x is not None)
            nx = sum(1 for x in need if x is None)
            if nx:
                rem = max(total - fixed, 3.0 * nx)
                cm = [x if x is not None else rem / nx for x in need]
            else:
                cm = [x * total / fixed for x in need]
            sc = total / sum(cm)
            widths = [Cm(x * sc) for x in cm]
            for j, w in enumerate(widths):                          # gridCol (LibreOffice/Google Docs) + largura total fixa
                tb.columns[j].width = w
            tw = tb._tbl.tblPr.find(qn("w:tblW"))
            tw.set(qn("w:type"), "dxa")
            tw.set(qn("w:w"), str(int(total / 2.54 * 1440)))
            set_borders(tb)
            mar = OxmlElement("w:tblCellMar")                       # margens laterais das células: 0,1 cm
            for side in ("left", "right"):
                e = OxmlElement(f"w:{side}")
                e.set(qn("w:w"), "57")
                e.set(qn("w:type"), "dxa")
                mar.append(e)
            look = tb._tbl.tblPr.find(qn("w:tblLook"))
            if look is not None:
                look.addprevious(mar)
            else:
                tb._tbl.tblPr.append(mar)
            for i in range(len(rows) + 1):
                row = tb.rows[i]
                trpr = row._tr.get_or_add_trPr()
                cs = OxmlElement("w:cantSplit")
                trpr.append(cs)
                if i == 0:
                    th = OxmlElement("w:tblHeader")
                    trpr.append(th)
                for j in range(len(hdr)):
                    cell = row.cells[j]
                    cell.width = widths[j]
                    cp = cell.paragraphs[0]
                    cp.style = S["Texto de tabela"]
                    if i == 0:
                        cp.alignment = WD_ALIGN_PARAGRAPH.CENTER
                        put(cp, _ar_hcell(f, hdr[j]), size=size, bold=True)
                        tcpr = cell._tc.get_or_add_tcPr()
                        bd = OxmlElement("w:tcBorders")
                        b = OxmlElement("w:bottom")
                        b.set(qn("w:val"), "single")
                        b.set(qn("w:sz"), "6")
                        b.set(qn("w:space"), "0")
                        b.set(qn("w:color"), "000000")
                        bd.append(b)
                        tcpr.append(bd)
                    else:
                        cp.alignment = WD_ALIGN_PARAGRAPH.RIGHT if als[j] == "r" else WD_ALIGN_PARAGRAPH.LEFT
                        put(cp, rows[i - 1][j], size=size)
            src_p = doc.add_paragraph(style="Fonte de tabela")             # NBR 14724: fonte alinhada à esquerda, 10 pt
            put(src_p, f.t(self._source_line()), size=10)
            if p.get("note"):
                src_p.paragraph_format.space_after = Pt(2)
                put(doc.add_paragraph(style="Nota de tabela"), p["note"], size=10)

        for (kind, p), lab in zip(blocks, labels):
            if kind == "h2":
                put(doc.add_paragraph(style="Heading 1"), f"{lab} {p}".upper())
            elif kind == "h3":
                put(doc.add_paragraph(style="Heading 2"), f"{lab} {p}")
            elif kind == "p":
                put(doc.add_paragraph(style="Texto ABNT"), p)
            elif kind == "ul":
                for x in p:
                    put(doc.add_paragraph(style="List Bullet"), x)
            elif kind == "eq":
                eq_par = doc.add_paragraph(style="Equação ABNT")
                try:                                                  # equação nativa do Word (OMML): fração, subscritos e itálico reais
                    if not omml:
                        raise RuntimeError("OMML desativado (conversão por LibreOffice sem módulo Math)")
                    from docx.oxml import parse_xml
                    eq_par._p.append(parse_xml(_ar_tex2omml(p)))
                except Exception:                                     # alternativa: runs formatados (itálico + subscritos verdadeiros)
                    for r_ in list(eq_par.runs):
                        r_._r.getparent().remove(r_._r)
                    put(eq_par, _ar_tex2html_eq(p, stacked=False))
            elif kind == "table":
                add_table(p, 1 if p["label"] == "tab:resultados" else 2)
        hr = doc.add_paragraph(style="Heading 1")
        hr.alignment = WD_ALIGN_PARAGRAPH.CENTER
        hr.paragraph_format.page_break_before = True
        put(hr, "REFERÊNCIAS")
        for k in self._sorted_refs(f.cited):
            put(doc.add_paragraph(style="Referência ABNT"), self._format_ref(k, "html"))
        cp_ = doc.core_properties
        cp_.title = self._title()
        cp_.author = (", ".join(map(str, au)) if isinstance(au, (list, tuple)) else str(au or self.cfg.get("app_name") or "DL-Docking Studio"))
        cp_.subject = "Metodologia e Resultados e Discussão de docking molecular"
        cp_.language = "pt-BR"
        return doc

    def to_docx(self, output_path: str) -> str:
        """Grava o relatório em Word (.docx), compatível com Google Docs. Levanta RuntimeError se python-docx faltar."""
        out = Path(output_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        self._build_docx().save(str(out))
        return str(out)

    # ------------------------------------------------------------------ PDF
    def to_pdf(self, output_path: str) -> str:
        """Grava o relatório em PDF A4 (WeasyPrint sobre `to_html(print_mode=True)`, com @page de margens ABNT).
        Se o WeasyPrint não estiver utilizável e houver LibreOffice (`soffice`), converte o .docx como alternativa.
        Levanta RuntimeError se nenhum caminho funcionar."""
        out = Path(output_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        try:
            from weasyprint import HTML                              # importação tardia: dependência opcional
            import logging
            for _lg in ("weasyprint", "fontTools"):                  # evita centenas de linhas de aviso de CSS/fontes no log da célula
                logging.getLogger(_lg).setLevel(logging.ERROR)
            HTML(string=self.to_html(print_mode=True)).write_pdf(str(out))
            return str(out)
        except Exception as e_w:                                     # ImportError, OSError (pango ausente), erros de layout
            soffice = shutil.which("soffice") or shutil.which("libreoffice")
            if not soffice:
                raise RuntimeError(f"WeasyPrint indisponível ({type(e_w).__name__}: {e_w}); sem LibreOffice para alternativa") from e_w
            try:
                with tempfile.TemporaryDirectory() as td:
                    src = Path(td) / "relatorio.docx"
                    self._build_docx(omml=False).save(str(src))      # sem OMML: o LibreOffice sem módulo Math descartaria as equações
                    subprocess.run([soffice, "--headless", "--convert-to", "pdf", "--outdir", td, str(src)], check=True,
                                   capture_output=True, timeout=180, env={**os.environ, "HOME": td})
                    pdf = Path(td) / "relatorio.pdf"
                    if not pdf.exists():
                        raise RuntimeError("LibreOffice não gerou o PDF")
                    shutil.copyfile(pdf, out)
                return str(out)
            except Exception as e_l:
                raise RuntimeError(f"WeasyPrint indisponível ({type(e_w).__name__}: {e_w}); alternativa LibreOffice falhou "
                                   f"({type(e_l).__name__}: {e_l})") from e_l

    # ------------------------------------------------------------------ pacote
    def export_bundle(self, output_dir: str, basename: str = "relatorio_docking", existing_bib: Optional[str] = None,
                      make_zip: bool = True, include_txt: bool = True, include_docx: bool = True,
                      include_pdf: bool = True) -> Dict[str, Path]:
        """Grava .docx, .pdf, .md, .tex, referencias.bib e .txt (e .zip) em `output_dir` e devolve {extensão: Path}.
        `existing_bib` aceita um caminho ou o texto do .bib. A geração de .docx e .pdf é isolada: se falhar, a chave simplesmente
        não aparece no dicionário, o motivo fica em `self.export_errors` (e em um aviso) e nenhuma exceção sai desta função por
        causa deles."""
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        bib_text = None
        if existing_bib:
            p = Path(existing_bib) if len(str(existing_bib)) < 1024 and "@" not in str(existing_bib) else None
            bib_text = p.read_text(encoding="utf-8") if p is not None and p.exists() else str(existing_bib)
        self.export_errors: Dict[str, str] = {}
        files: Dict[str, Path] = {}
        for kind, flag, fn in (("docx", include_docx, self.to_docx), ("pdf", include_pdf, self.to_pdf)):
            if not flag:
                continue
            target = out / f"{basename}.{kind}"
            try:
                fn(str(target))
                if target.exists() and target.stat().st_size > 0:
                    files[kind] = target
                else:
                    raise RuntimeError("arquivo vazio")
            except BaseException as e:                                # noqa: BLE001 - nunca interromper o pipeline por causa do docx/pdf
                if isinstance(e, (KeyboardInterrupt, SystemExit)):
                    raise
                self.export_errors[kind] = f"{type(e).__name__}: {e}"
                warnings.warn(f"Relatório .{kind} não gerado: {self.export_errors[kind]}")
                if target.exists():
                    try:
                        target.unlink()
                    except OSError:
                        pass
        files["md"] = out / f"{basename}.md"
        files["tex"] = out / f"{basename}.tex"
        files["bib"] = out / "referencias.bib"
        files["md"].write_text(self.to_markdown(), encoding="utf-8")
        files["tex"].write_text(self.to_latex(), encoding="utf-8")
        files["bib"].write_text(self.to_bibtex(bib_text), encoding="utf-8")
        if include_txt:
            files["txt"] = out / f"{basename}.txt"
            files["txt"].write_text(self.to_plaintext(), encoding="utf-8")
        if make_zip:
            files["zip"] = out / f"{basename}_pacote.zip"
            files["zip"].write_bytes(self.to_zip_bytes(basename, bib_text, _extra={k: files[k] for k in ("docx", "pdf") if k in files}))
        return files

    def to_zip_bytes(self, basename: str = "relatorio_docking", existing_bib: Optional[str] = None,
                     _extra: Optional[Dict[str, Path]] = None) -> bytes:
        """Pacote .zip em memória (útil para st.download_button). Inclui .docx/.pdf quando puderem ser gerados."""
        buf = io.BytesIO()
        with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
            z.writestr(f"{basename}.tex", self.to_latex())
            z.writestr("referencias.bib", self.to_bibtex(existing_bib))
            z.writestr(f"{basename}.md", self.to_markdown())
            z.writestr(f"{basename}.txt", self.to_plaintext())
            try:
                if _extra:
                    for kind, pth in _extra.items():
                        z.write(str(pth), f"{basename}.{kind}")
                else:
                    with tempfile.TemporaryDirectory() as td:
                        for kind, fn in (("docx", self.to_docx), ("pdf", self.to_pdf)):
                            try:
                                fn(str(Path(td) / f"x.{kind}"))
                                z.write(str(Path(td) / f"x.{kind}"), f"{basename}.{kind}")
                            except Exception:                           # noqa: BLE001 - opcional
                                pass
            except Exception:                                           # noqa: BLE001
                pass
        return buf.getvalue()


# --- Adaptador: estado do app (results/ctrl/engine) -> AcademicReporter -----------------------------
# ==================================================================================================
# ALPHAFOLD DB (UniProt) + AUDITORIA DE pLDDT DO BOLSÃO + GERADOR DO SCRIPT TURNKEY DE DINÂMICA MOLECULAR (OpenMM)
# pLDDT = coluna do fator B dos modelos do AlphaFold (0-100). Calculado sobre o TEXTO ORIGINAL do modelo: o receptor tratado pelo
# PDBFixer não preserva o fator B. Bolsão = resíduos com ao menos um átomo pesado a < PLDDT_POCKET_RADIUS do centro da grid box.
# ==================================================================================================
AFDB_URL_TMPL = "https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_{ver}.pdb"
AFDB_API_TMPL = "https://alphafold.ebi.ac.uk/api/prediction/{uid}"
AFDB_VERSIONS = ("v4", "v3")                          # 404 em v4 -> tenta v3 antes de reportar erro
UNIPROT_RE = re.compile(r"^[A-Z0-9]{6,10}$")
AF_ID_PREFIX = "AF-"
PLDDT_POCKET_RADIUS = 10.0                            # Å do centro da grid box
PLDDT_HIGH, PLDDT_MID, PLDDT_LOW = 90.0, 70.0, 0.0
PLDDT_CLASS_HIGH = "Elevada confiabilidade estrutural (pLDDT >= 90)"
PLDDT_CLASS_MID = "Confiabilidade moderada a boa (70 <= pLDDT < 90)"
PLDDT_CLASS_LOW = "Baixa confiabilidade / Região potencialmente flexível ou desordenada (pLDDT < 70)"
MD_DIRNAME = "08_dinamica_molecular"
MD_SCRIPT_NAME = "run_md_openmm.py"
SOURCE_LABELS = {"rcsb": "RCSB PDB (estrutura experimental)", "local": "PDB local enviado pelo usuário",
                 "alphafold": "AlphaFold DB (modelo predito)"}


def classify_plddt(mean_plddt) -> str:
    """Classe de confiabilidade do bolsão: >= 90 elevada; 70 <= x < 90 moderada a boa; < 70 baixa. '' se não numérico."""
    try:
        v = float(mean_plddt)
    except (TypeError, ValueError):
        return ""
    if v != v:
        return ""
    return PLDDT_CLASS_HIGH if v >= 90.0 else (PLDDT_CLASS_MID if v >= 70.0 else PLDDT_CLASS_LOW)


def _heavy_atoms_bfactor(pdb_text: str, chain=None):
    """Lista (chave_do_resíduo, x, y, z, fator_B) dos átomos pesados de ATOM (conformação A/1/' ' do 1º modelo)."""
    out = []
    for line in pdb_text.splitlines():
        if line.startswith("ENDMDL"):
            break
        if not line.startswith("ATOM  ") or len(line) < 66 or line[16] not in (" ", "A", "1"):
            continue
        if chain and line[21] != chain:
            continue
        el = (line[76:78].strip() if len(line) >= 78 else "") or line[12:16].strip()[:1]
        if el.upper() in ("H", "D"):
            continue
        try:
            out.append(((line[21], line[22:27].strip(), line[17:20].strip()),
                        float(line[30:38]), float(line[38:46]), float(line[46:54]), float(line[60:66])))
        except ValueError:
            continue
    return out


def compute_pocket_plddt(pdb_text: str, center, chain=None, radius: float = PLDDT_POCKET_RADIUS):
    """pLDDT do bolsão: átomos pesados da proteína dos resíduos a < `radius` Å do centro da grid box (qualquer átomo pesado do
    resíduo dentro do raio). Devolve dict(mean, min, max, n_atoms, n_residues, radius, classe, n_low) ou None se o bolsão está vazio."""
    atoms = _heavy_atoms_bfactor(pdb_text, chain)
    if not atoms:
        return None
    xyz = np.array([[a[1], a[2], a[3]] for a in atoms], dtype=float)
    b = np.array([a[4] for a in atoms], dtype=float)
    d = np.linalg.norm(xyz - np.asarray(center, dtype=float)[None, :], axis=1)
    keys = [a[0] for a in atoms]
    near = {keys[i] for i in np.where(d < float(radius))[0]}
    if not near:
        return None
    sel = np.array([k in near for k in keys])
    vals = b[sel]
    per_res = {}
    for k, v in zip([k for k, s in zip(keys, sel) if s], vals):
        per_res.setdefault(k, []).append(v)
    mean = float(np.mean(vals))
    return {"mean": round(mean, 2), "min": round(float(np.min(vals)), 2), "max": round(float(np.max(vals)), 2),
            "n_atoms": int(sel.sum()), "n_residues": len(near), "radius": float(radius), "classe": classify_plddt(mean),
            "n_low": int(sum(1 for v in per_res.values() if np.mean(v) < 70.0)),
            "residues": sorted(f"{k[2]}{k[1]}" for k in near)}


def global_plddt(pdb_text: str, chain=None):
    """pLDDT da cadeia inteira (média, mínimo e fração de átomos com pLDDT >= 70); None se não houver átomos."""
    atoms = _heavy_atoms_bfactor(pdb_text, chain)
    if not atoms:
        return None
    b = np.array([a[4] for a in atoms], dtype=float)
    return {"mean": round(float(b.mean()), 2), "min": round(float(b.min()), 2), "frac_ge70": round(float((b >= 70.0).mean()), 3),
            "n_atoms": int(b.size)}


def plddt_html(site, glob=None) -> str:
    """Linhas HTML do Passo 1/4 com o pLDDT do bolsão (e o global, se houver)."""
    parts = []
    if glob:
        parts.append(f"<b>pLDDT global da cadeia:</b> média {glob['mean']:.1f} · mínimo {glob['min']:.1f} · "
                     f"{100 * glob['frac_ge70']:.0f}% dos átomos pesados com pLDDT ≥ 70")
    if site:
        parts.append(f"<b>pLDDT do sítio (&lt; {site['radius']:.0f} Å do centro da caixa; {site['n_residues']} resíduos, "
                     f"{site['n_atoms']} átomos pesados):</b> média {site['mean']:.1f} · mínimo {site['min']:.1f} → {esc(site['classe'])}"
                     + (f". {site['n_low']} resíduo(s) do bolsão com pLDDT médio &lt; 70." if site["n_low"] else "."))
    else:
        parts.append("<b>pLDDT do sítio:</b> calculado ao definir a grid box (Passo 3); o valor entra no resumo do Passo 4 e na aba "
                     "Metadados_Sessao.")
    return "<br>".join(parts)


def download_alphafold_pdb(uniprot_id: str, timeout: int = 30):
    """Baixa o modelo do AFDB: model_v4 e, se HTTP 404, model_v3; por último consulta a API do AFDB (pdbUrl). Devolve
    (texto, versão, url). Levanta FileNotFoundError (404 em todas as versões) ou a exceção de rede original."""
    uid = str(uniprot_id).strip().upper()
    tried = []
    for ver in AFDB_VERSIONS:
        url = AFDB_URL_TMPL.format(uid=uid, ver=ver)
        try:
            req = urllib.request.Request(url, headers={"User-Agent": f"DL-Docking-Studio/{PIPELINE_VERSION}"})
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return resp.read().decode("utf-8"), ver, url
        except urllib.error.HTTPError as e:
            tried.append(f"{ver}: HTTP {e.code}")
            if e.code != 404:
                raise
    try:                                              # contingência: o AFDB publica versões novas (v5, v6...) sob a mesma API
        areq = urllib.request.Request(AFDB_API_TMPL.format(uid=uid), headers={"User-Agent": f"DL-Docking-Studio/{PIPELINE_VERSION}"})
        with urllib.request.urlopen(areq, timeout=timeout) as aresp:
            entry = json.loads(aresp.read().decode("utf-8"))
        entry = entry[0] if isinstance(entry, list) and entry else entry
        purl = entry.get("pdbUrl") if isinstance(entry, dict) else None
        if purl:
            preq = urllib.request.Request(purl, headers={"User-Agent": f"DL-Docking-Studio/{PIPELINE_VERSION}"})
            with urllib.request.urlopen(preq, timeout=timeout) as presp:
                m = re.search(r"model_(v\d+)", purl)
                return presp.read().decode("utf-8"), (m.group(1) if m else "API"), purl
    except Exception:
        pass
    raise FileNotFoundError(f"Modelo AlphaFold de {uid} não encontrado ({'; '.join(tried)}).")


# --------------------------------------------------------------------------------------------------------------------
# Gerador do script turnkey de MD (OpenMM). O texto abaixo é o PRÓPRIO run_md_openmm.py; marcadores __XXX__ são substituídos.
# --------------------------------------------------------------------------------------------------------------------
OPENMM_SCRIPT_TEMPLATE = r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""run_md_openmm.py -- MD turnkey da pose líder (gerado por __PLATFORM__ em __STAMP__).

Protocolo: Amber14 + TIP3P-FB; ligante via openmmforcefields (OpenFF/GAFF); caixa cúbica (1,0 nm) + NaCl 0,15 M (neutralizado);
minimização (LocalEnergyMinimizer); aquecimento e NVT/NPT com restrições nos átomos pesados; produção NPT de __SIM_NS__ ns (2 fs,
ligações com H restritas); DCD a cada 10 ps; log com tempo, energia potencial, energia cinética e temperatura.

Dependências:  conda install -c conda-forge openmm openmmforcefields openff-toolkit
Uso:           python run_md_openmm.py --ns __SIM_NS__ --temp __TEMP_K__ [--platform CUDA|OpenCL|CPU] [--outdir md_out]
Sem openmmforcefields/openff-toolkit (ou se a parametrização do ligante falhar) o script SIMULA SÓ A PROTEÍNA, com restrição
harmônica de posição nos Cα do bolsão (resíduos a 8 Å do ligante), e avisa. Metais/cofatores sem parâmetros Amber14 (ex.: Cu) impedem
o createSystem: parametrize-os (MCPB.py) ou use --strip-hetero (remove-os; ALTERA a física do sítio).
"""
import argparse
import sys
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
DEF_COMPLEX = "__COMPLEX__"
DEF_LIGAND_SDF = "__LIGSDF__"
DEF_RESNAME = "__RESNAME__"
DEF_NS = __SIM_NS__
DEF_TEMP = __TEMP_K__
TIMESTEP_FS = 2.0
REPORT_PS = 10.0
K_REST = 1000.0                      # kJ/mol/nm^2
POCKET_CUTOFF_A = 8.0
STD_RES = {"ALA", "ARG", "ASN", "ASP", "ASH", "CYS", "CYX", "CYM", "GLN", "GLU", "GLH", "GLY", "HIS", "HID", "HIE", "HIP", "ILE", "LEU",
           "LYS", "LYN", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL", "ACE", "NME", "NMA", "HOH", "WAT"}


def parse_args():
    ap = argparse.ArgumentParser(description="MD turnkey (OpenMM) da pose líder")
    ap.add_argument("--complex", default=str(HERE / DEF_COMPLEX), help="PDB do complexo proteína-ligante (resíduo LIG)")
    ap.add_argument("--ligand-sdf", default=(str(HERE / DEF_LIGAND_SDF) if DEF_LIGAND_SDF else ""), help="SDF 3D da pose (com H)")
    ap.add_argument("--resname", default=DEF_RESNAME)
    ap.add_argument("--ns", type=float, default=DEF_NS, help="tempo de produção (ns)")
    ap.add_argument("--temp", type=float, default=DEF_TEMP, help="temperatura (K)")
    ap.add_argument("--outdir", default=str(HERE / "md_out"))
    ap.add_argument("--platform", default="auto", help="auto (CUDA > OpenCL > CPU), CUDA, OpenCL ou CPU")
    ap.add_argument("--ligand-ff", default="openff-2.1.0", help="campo de força do ligante: openff-2.1.0 ou gaff-2.11")
    ap.add_argument("--protein-only", action="store_true", help="não parametrizar o ligante (modo de contingência)")
    ap.add_argument("--strip-hetero", action="store_true", help="remove metais/cofatores sem parâmetros Amber14 (altera a física)")
    ap.add_argument("--seed", type=int, default=42)
    return ap.parse_args()


def pick_platform(name):
    from openmm import Platform
    for n in ([name] if name != "auto" else ["CUDA", "OpenCL", "CPU"]):
        try:
            return Platform.getPlatformByName(n), ({"Precision": "mixed"} if n in ("CUDA", "OpenCL") else {})
        except Exception:
            continue
    raise RuntimeError(f"Plataforma OpenMM indisponível: {name}")


def parametrize_ligand(a, app, unit, modeller, resname):
    """Tenta openmmforcefields (SystemGenerator). Devolve (generator, molecule) ou (None, motivo)."""
    try:
        try:
            from openff.toolkit import Molecule
        except ImportError:
            from openff.toolkit.topology import Molecule
        from openmmforcefields.generators import SystemGenerator
    except Exception as e:
        return None, f"openmmforcefields/openff-toolkit ausentes ({type(e).__name__}: {e})"
    try:
        mol = Molecule.from_file(a.ligand_sdf, file_format="sdf", allow_undefined_stereo=True)
        mol = mol[0] if isinstance(mol, (list, tuple)) else mol
        lig_top = mol.to_topology().to_openmm()
        pos = mol.conformers[0]
        lig_pos = pos.to_openmm() if hasattr(pos, "to_openmm") else pos
        for r in lig_top.residues():
            r.name = resname
        gen = SystemGenerator(
            forcefields=["amber14-all.xml", "amber14/tip3pfb.xml"], small_molecule_forcefield=a.ligand_ff, molecules=[mol],
            forcefield_kwargs={"constraints": app.HBonds, "rigidWater": True, "removeCMMotion": True},
            periodic_forcefield_kwargs={"nonbondedMethod": app.PME, "nonbondedCutoff": 1.0 * unit.nanometer})
        modeller.add(lig_top, lig_pos)
        return gen, mol
    except Exception as e:
        return None, f"falha ao parametrizar o ligante ({type(e).__name__}: {e})"


def main():
    a = parse_args()
    import numpy as np
    import openmm as mm
    from openmm import app, unit
    out = Path(a.outdir)
    out.mkdir(parents=True, exist_ok=True)
    kJ, nm = unit.kilojoule_per_mole, unit.nanometer
    T = a.temp * unit.kelvin
    dt = TIMESTEP_FS * unit.femtoseconds
    n_prod = int(round(a.ns * 1000.0 * 1000.0 / TIMESTEP_FS))          # ns -> fs -> passos
    rep_steps = int(round(REPORT_PS * 1000.0 / TIMESTEP_FS))           # 10 ps = 5000 passos

    # 1. Leitura do complexo
    pdb = app.PDBFile(a.complex)
    modeller = app.Modeller(pdb.topology, pdb.positions)
    lig_res = [r for r in modeller.topology.residues() if r.name == a.resname]
    pos0 = np.array(pdb.positions.value_in_unit(nm))
    idx_all = {at.index: at for at in modeller.topology.atoms()}
    lig_xyz = np.array([pos0[at.index] for r in lig_res for at in r.atoms() if at.element is not None and at.element.symbol != "H"])
    pocket_ca = []
    if len(lig_xyz):
        for r in modeller.topology.residues():
            if r.name in STD_RES and r.name not in ("HOH", "WAT"):
                heavy = [pos0[at.index] for at in r.atoms() if at.element is not None and at.element.symbol != "H"]
                ca = [at.index for at in r.atoms() if at.name == "CA"]
                if ca and heavy and np.min(np.linalg.norm(np.array(heavy)[:, None, :] - lig_xyz[None, :, :], axis=-1)) < POCKET_CUTOFF_A / 10.0:
                    pocket_ca.append(ca[0])
    modeller.delete(lig_res)                                            # o ligante volta com topologia/parâmetros próprios
    if a.strip_hetero:
        drop = [r for r in modeller.topology.residues() if r.name not in STD_RES]
        if drop:
            print("AVISO --strip-hetero: removidos " + ", ".join(sorted({r.name for r in drop})) + " (física do sítio alterada).")
            modeller.delete(drop)

    # 2. Campo de força: Amber14 (proteína/água) + ligante (OpenFF/GAFF) com contingência
    gen, mol, why = None, None, ""
    if a.protein_only or not lig_res or not a.ligand_sdf or not Path(a.ligand_sdf).is_file():
        why = "modo --protein-only, sem resíduo/SDF do ligante" if not (a.protein_only) else "--protein-only"
    else:
        gen, mol = parametrize_ligand(a, app, unit, modeller, a.resname)
        if gen is None:
            why, mol = mol, None
    if gen is None:
        print(f"AVISO: ligante NÃO parametrizado ({why}). Contingência: simulação da proteína com restrição harmônica "
              f"(k = {K_REST:.0f} kJ/mol/nm^2) nos Cα do bolsão ({len(pocket_ca)} resíduos). Instale openmmforcefields e openff-toolkit.")
        ff = app.ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
        solv_ff = ff
    else:
        solv_ff = gen.forcefield

    # 3. Solvatação: caixa cúbica (1,0 nm de margem) + NaCl 0,15 M neutralizando
    kw = dict(model="tip3p", padding=1.0 * nm, ionicStrength=0.15 * unit.molar, positiveIon="Na+", negativeIon="Cl-", neutralize=True)
    try:
        modeller.addSolvent(solv_ff, boxShape="cube", **kw)
    except TypeError:
        modeller.addSolvent(solv_ff, **kw)
    print(f"Sistema solvatado: {modeller.topology.getNumAtoms()} átomos; caixa = {modeller.topology.getPeriodicBoxVectors()}")
    app.PDBFile.writeFile(modeller.topology, modeller.positions, open(out / "sistema_solvatado.pdb", "w"), keepIds=True)

    try:
        if gen is not None:
            system = gen.create_system(modeller.topology, molecules=[mol])
        else:
            system = ff.createSystem(modeller.topology, nonbondedMethod=app.PME, nonbondedCutoff=1.0 * nm,
                                     constraints=app.HBonds, rigidWater=True, removeCMMotion=True)
    except ValueError as e:
        print(f"ERRO ao criar o sistema: {e}\nResíduo sem parâmetros Amber14 (metal/cofator?). Parametrize-o (ex.: MCPB.py) "
              "ou rode com --strip-hetero (remove-o e ALTERA a física do sítio).", file=sys.stderr)
        sys.exit(2)

    # 4. Restrições: (i) átomos pesados solutos durante a equilibração (k_rest -> 0 na produção); (ii) Cα do bolsão na contingência
    pos = np.array(modeller.positions.value_in_unit(nm))
    rest = mm.CustomExternalForce("k_rest*periodicdistance(x,y,z,x0,y0,z0)^2")
    rest.addGlobalParameter("k_rest", K_REST)
    for p in ("x0", "y0", "z0"):
        rest.addPerParticleParameter(p)
    n_solute = 0
    for at in modeller.topology.atoms():
        if at.residue.name not in ("HOH", "WAT", "Na+", "Cl-", "NA", "CL") and at.element is not None and at.element.symbol != "H":
            rest.addParticle(at.index, pos[at.index].tolist())
            n_solute += 1
    system.addForce(rest)
    if gen is None and pocket_ca:
        pk = mm.CustomExternalForce("k_pocket*periodicdistance(x,y,z,x0,y0,z0)^2")
        pk.addGlobalParameter("k_pocket", K_REST)
        for p in ("x0", "y0", "z0"):
            pk.addPerParticleParameter(p)
        names = {(at.residue.chain.index, at.residue.id, at.residue.name): at.index for at in modeller.topology.atoms() if at.name == "CA"}
        keys = {(at.residue.chain.index, at.residue.id, at.residue.name) for i in pocket_ca for at in [idx_all[i]]}
        for k in keys:
            if k in names:
                pk.addParticle(names[k], pos[names[k]].tolist())
        system.addForce(pk)
        print(f"Restrição de bolsão: {pk.getNumParticles()} Cα.")

    # 5. Simulação: Langevin (middle), 2 fs, minimização local
    integ = mm.LangevinMiddleIntegrator(50.0 * unit.kelvin, 1.0 / unit.picosecond, dt)
    integ.setRandomNumberSeed(a.seed)
    platform, props = pick_platform(a.platform)
    print(f"Plataforma: {platform.getName()} {props}")
    sim = app.Simulation(modeller.topology, system, integ, platform, props)
    sim.context.setPositions(modeller.positions)
    t0 = time.time()
    mm.LocalEnergyMinimizer.minimize(sim.context, 10.0 * kJ / nm, 10000)
    print(f"Minimizado: E_pot = {sim.context.getState(getEnergy=True).getPotentialEnergy()}")

    # 6. Termalização (50 K -> T em 6 degraus de 10 ps, NVT) + NVT 100 ps + NPT 200 ps com restrições
    sim.context.setVelocitiesToTemperature(50.0 * unit.kelvin, a.seed)
    for temp in np.linspace(50.0, a.temp, 6):
        integ.setTemperature(temp * unit.kelvin)
        sim.step(rep_steps)
    sim.step(int(100.0 * 1000.0 / TIMESTEP_FS))
    system.addForce(mm.MonteCarloBarostat(1.0 * unit.atmosphere, T, 25))
    sim.context.reinitialize(preserveState=True)
    sim.step(int(200.0 * 1000.0 / TIMESTEP_FS))
    sim.context.setParameter("k_rest", 0.0)                             # produção sem restrição nos átomos pesados
    sim.step(int(50.0 * 1000.0 / TIMESTEP_FS))                          # 50 ps de relaxamento livre antes de gravar
    print(f"Equilibração concluída em {(time.time() - t0) / 60:.1f} min.")

    # 7. Produção: DCD a cada 10 ps + log (tempo, energia potencial, energia cinética, temperatura)
    sim.reporters.append(app.DCDReporter(str(out / "trajetoria.dcd"), rep_steps))
    sim.reporters.append(app.StateDataReporter(str(out / "md_producao.log"), rep_steps, step=True, time=True, potentialEnergy=True,
                                               kineticEnergy=True, temperature=True, speed=True, remainingTime=True,
                                               totalSteps=n_prod, separator="\t"))
    sim.reporters.append(app.CheckpointReporter(str(out / "estado.chk"), rep_steps * 10))
    print(f"Produção: {a.ns} ns = {n_prod} passos de {TIMESTEP_FS} fs a {a.temp} K.")
    sim.step(n_prod)
    st = sim.context.getState(getPositions=True, enforcePeriodicBox=True)
    app.PDBFile.writeFile(modeller.topology, st.getPositions(), open(out / "final.pdb", "w"), keepIds=True)
    print(f"Concluído em {(time.time() - t0) / 3600:.2f} h. Saídas em {out}")


if __name__ == "__main__":
    main()
'''


def generate_openmm_script(complex_pdb_path, output_py_path, sim_time_ns=10.0, temp_k=300.0,
                           ligand_sdf_path=None, ligand_resname="LIG"):
    """Grava o `run_md_openmm.py` autônomo (OpenMM) para a MD da pose líder e devolve o caminho. O script lê `complex_pdb_path`
    (nome relativo à pasta do script), usa Amber14 + TIP3P-FB, parametriza o ligante via openmmforcefields (contingência: proteína
    com restrição harmônica no bolsão), solvata (cubo 1,0 nm, NaCl 0,15 M), minimiza, equilibra NVT/NPT e roda `sim_time_ns` ns
    (2 fs; DCD a cada 10 ps; log de tempo/Epot/Ecin/T). A sintaxe do texto gerado é verificada com ast antes de gravar."""
    import ast
    sim_time_ns, temp_k = float(sim_time_ns), float(temp_k)
    if not (sim_time_ns > 0 and 0 < temp_k < 1000):
        raise ValueError(f"sim_time_ns > 0 e 0 < temp_k < 1000 K; recebido {sim_time_ns} ns, {temp_k} K")
    rep = {"__PLATFORM__": PLATFORM_NAME, "__STAMP__": time.strftime("%Y-%m-%d %H:%M:%S"),
           "__COMPLEX__": Path(complex_pdb_path).name, "__LIGSDF__": (Path(ligand_sdf_path).name if ligand_sdf_path else ""),
           "__RESNAME__": str(ligand_resname), "__SIM_NS__": repr(sim_time_ns), "__TEMP_K__": repr(temp_k)}
    text = OPENMM_SCRIPT_TEMPLATE
    for k, v in rep.items():
        text = text.replace(k, v)
    ast.parse(text)
    out = Path(output_py_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(text, encoding="utf-8")
    try:
        out.chmod(0o755)
    except Exception:
        pass
    return out


def stage_md_folder(dest_dir, complex_pdb_path, ligand_mol=None, ligand_sdf_src=None, sim_time_ns=10.0, temp_k=300.0):
    """Monta `dest_dir` (…/08_dinamica_molecular[/ligante]) com: complexo pré-alinhado (_Complexo.pdb), SDF 3D da pose (com H) e
    run_md_openmm.py. A pose vem de `ligand_mol` (RDKit) ou de `ligand_sdf_src`. Devolve dict(dir, script, complex, ligand_sdf, files)."""
    dest = Path(dest_dir)
    dest.mkdir(parents=True, exist_ok=True)
    cplx = dest / Path(complex_pdb_path).name
    if Path(complex_pdb_path).resolve() != cplx.resolve():
        shutil.copy(str(complex_pdb_path), str(cplx))
    sdf = None
    if ligand_mol is not None:
        m = ligand_mol if any(at.GetAtomicNum() == 1 for at in ligand_mol.GetAtoms()) else Chem.AddHs(ligand_mol, addCoords=True)
        sdf = dest / (cplx.stem.replace("_Complexo", "") + "_pose_lider.sdf")
        w = Chem.SDWriter(str(sdf))
        w.write(m)
        w.close()
    elif ligand_sdf_src and Path(ligand_sdf_src).is_file():
        sdf = dest / Path(ligand_sdf_src).name
        if Path(ligand_sdf_src).resolve() != sdf.resolve():
            shutil.copy(str(ligand_sdf_src), str(sdf))
    script = generate_openmm_script(cplx, dest / MD_SCRIPT_NAME, sim_time_ns, temp_k, sdf, "LIG")
    files = [script, cplx] + ([sdf] if sdf else [])
    return {"dir": str(dest), "script": str(script), "complex": str(cplx), "ligand_sdf": (str(sdf) if sdf else ""),
            "files": [str(f) for f in files]}


def md_notice_html(md, leader_name="") -> str:
    """Aviso do Passo 4: o script turnkey de dinâmica molecular está disponível para download."""
    if not md:
        return ""
    who = f" da pose líder (<i>{esc(leader_name)}</i>)" if leader_name else " da pose exportada"
    return ('<div class="dd-detail"><b>Dinâmica molecular (OpenMM):</b> o script turnkey <code>run_md_openmm.py</code>' + who +
            f" está disponível para download e na pasta <code>{MD_DIRNAME}/</code> do .zip (com o complexo pré-alinhado e o SDF da pose). "
            "Execute <code>python run_md_openmm.py --ns 10</code> (requer openmm; openmmforcefields + openff-toolkit parametrizam o "
            "ligante, senão roda só a proteína com restrição no bolsão). Metais sem parâmetros Amber14 (ex.: Cu) exigem parametrização "
            "própria ou <code>--strip-hetero</code>.</div>")



REPORT_DIRNAME = "07_relatorio_academico"
REPORT_DISPLAY_IPYTHON = True      # tenta exibir o Markdown na saída da célula (IPython.display) além do painel Gradio
PH_FISIOLOGICO = 7.4               # espelha o pH usado no PDBFixer/OpenMM e no obabel -p desta célula


def _ar_gnina_label(raw):
    """Sanitiza a versão do GNINA para o texto científico: 'gnina v1.3.2 master:f23dd2b Built Jul 8 2025.' -> 'GNINA v1.3.2'
    (sem hash do commit, data de compilação nem pontuação final). Sem número de versão legível devolve 'GNINA'."""
    m = re.search(r"(?<![\w.])v?(\d+\.\d+(?:\.\d+)*)", str(raw or ""))
    return f"GNINA v{m.group(1)}" if m else "GNINA"


# Artigo cristalográfico primário por PDB ID (BibTeX curado; tem prioridade sobre o lido dos registros JRNL do arquivo PDB).
AR_PDB_PRIMARY_BIB = {
    "2Y9X": r"""@article{ismaya2011,
  author    = {Ismaya, W. T. and Rozeboom, H. J. and Weijn, A. and Mes, J. J. and Fusetti, F. and Wichers, H. J. and Dijkstra, B. W.},
  title     = {Crystal structure of {\textit{Agaricus bisporus}} mushroom tyrosinase: identity of the tetramer subunits and interaction with tropolone},
  journal   = {Biochemistry},
  volume    = {50},
  number    = {24},
  pages     = {5477--5486},
  year      = {2011},
  doi       = {10.1021/bi200395t}
}""",
}


def _ar_jrnl_bibtex(pdb_text):
    """Registros JRNL de um arquivo PDB (AUTH, TITL, REF, DOI) -> BibTeX do artigo cristalográfico original. Devolve '' se o
    artigo não foi publicado ('TO BE PUBLISHED'), se faltar autor/título/ano ou em qualquer erro. Capitalização do título e das
    siglas do periódico é aproximada: para conferência, use AR_PDB_PRIMARY_BIB ou config['target_paper_bib']."""
    try:
        auth, titl, ref, doi = [], [], "", ""
        for ln in str(pdb_text).splitlines():
            if not ln.startswith("JRNL"):
                continue
            tag, txt = ln[12:16].strip(), ln[19:].rstrip()
            if tag == "AUTH":
                auth.append(txt.strip())
            elif tag == "TITL":
                titl.append(txt.strip())
            elif tag == "REF":
                ref = txt.strip()
            elif tag == "DOI":
                doi = txt.strip().lower()
        m = re.match(r"(.+?)\s+V\.\s*(\S+)\s+(\S+)\s+(\d{4})\s*$", ref)
        if not auth or not titl or not m:
            return ""
        names = []
        for a in "".join(auth).split(","):
            a = a.strip()
            mm = re.match(r"^((?:[A-Z]{1,2}\.)+)\s*(.+)$", a)
            if not mm:
                continue
            ini = " ".join(x + "." for x in mm.group(1).split(".") if x)
            names.append(f"{mm.group(2).strip().title()}, {ini}")
        if not names:
            return ""
        tex = lambda x: re.sub(r"([&%#_])", r"\\\1", x)                                # noqa: E731
        title = " ".join(titl).strip().rstrip(".").capitalize()
        jr, vol, page, year = m.group(1).strip().title(), m.group(2), m.group(3), m.group(4)
        surname = re.sub(r"[^a-z]", "", names[0].split(",")[0].lower()) or "pdb"
        fields = [f"  author    = {{{' and '.join(names)}}}", f"  title     = {{{tex(title)}}}", f"  journal   = {{{tex(jr)}}}"]
        if vol.isdigit():
            fields.append(f"  volume    = {{{vol}}}")
        if page.isdigit():
            fields.append(f"  pages     = {{{page}}}")
        fields.append(f"  year      = {{{year}}}")
        if doi:
            fields.append(f"  doi       = {{{doi}}}")
        return "@article{" + f"{surname}{year},\n" + ",\n".join(fields) + "\n}"
    except Exception:
        return ""


def _ar_interacoes_app(txt):
    """'HB: Ser289, Asn260 (+2); Metal: Cu400' (key_interactions_summary) -> 'Ser289 (HB); Asn260 (HB); Cu400 (Metal)'."""
    t = str(txt or "").strip()
    if not t or t.startswith(("—", "n/d")):
        return ""
    out = []
    for part in t.split(";"):
        if ":" not in part:
            continue
        ab, res = part.split(":", 1)
        for tok in res.split(","):
            tok = re.sub(r"\(\+\d+\)", "", tok).strip()
            if tok:
                out.append(f"{tok} ({ab.strip()})")
    return "; ".join(out)


def _ar_meta(r):
    """df_meta (Parâmetro/Valor) de um resultado -> dict; vazio se ausente."""
    try:
        dm = r.get("df_meta")
        return {str(a): b for a, b in zip(dm["Parâmetro"], dm["Valor"])}
    except Exception:
        return {}


def _ar_triplet(txt):
    """'-10.021, -28.823, -43.596' -> [-10.021, -28.823, -43.596]; None se não houver três números."""
    try:
        v = [float(str(a).strip()) for a in str(txt).split(",")]
    except ValueError:
        return None
    return v if len(v) == 3 else None


def _ar_grid_info(entries):
    """Caixa EFETIVA passada ao GNINA (centro e arestas em Å). Fonte estruturada: rec['ctx'] (os mesmos valores do --center_*/--size_*);
    alternativa: a string 'Centro: (...) | Tamanho: (...)' do df_meta. entries = [(nome, é_controle, ctx, meta)].
    Devolve {grid_center, grid_size, box_mode, grid_size_by_ligand, grid_size_extended} ou {} se nada for legível."""
    boxes = []
    for nm, ctl, ctx, meta in entries:
        c, s = (ctx or {}).get("center"), (ctx or {}).get("size")
        bs = str(meta.get("Grid_Box_Efetiva (passada ao GNINA via CLI)", "") or "")
        if not (c and s) and bs:
            m1, m2 = re.search(r"Centro:\s*\(([^)]*)\)", bs), re.search(r"Tamanho:\s*\(([^)]*)\)", bs)
            c, s = (_ar_triplet(m1.group(1)) if m1 else None), (_ar_triplet(m2.group(1)) if m2 else None)
        try:
            c, s = [float(v) for v in c], [float(v) for v in s]
        except (TypeError, ValueError):
            continue
        if len(c) != 3 or len(s) != 3:
            continue
        try:
            diam = float(meta.get("Diametro_Maximo_Ligante (Å)"))
        except (TypeError, ValueError):
            diam = None
        boxes.append((nm, ctl, c, s, bs.startswith("Manual"), diam))
    if not boxes:
        return {}
    ref = next((b for b in boxes if b[1]), boxes[0])              # a caixa do controle (ligante nativo) é a de referência
    out = {"grid_center": [round(float(v), 3) for v in ref[2]],           # contrato: centro com 3 casas (Å)
           "grid_size": [round(float(v), 1) for v in ref[3]],             # arestas com 1 casa (Å); o motor recebeu os valores a 3 casas
           "box_mode": "manual" if ref[4] else "native",
           "grid_size_by_ligand": {b[0]: [round(float(v), 1) for v in b[3]] for b in boxes}}
    if not ref[4] and ref[5] is not None:                         # aresta == diâmetro do ligante => foi estendida pela regra do autobox
        out["grid_size_extended"] = any(abs(v - ref[5]) < 0.006 for v in ref[3])   # testado nos valores NÃO arredondados
    return out


def _ar_annotation(engine, chain=""):
    """Anotação funcional do alvo -> (target_domain, domain_function). Prioridade: texto do Passo 1 (autopreenchido ou refinado pelo
    usuário); campo vazio => infer_domain_and_function (cabeçalho do PDB); sem classe reconhecida => texto neutro padrão."""
    ann = getattr(engine, "target_annotation", None) or {}
    _s = lambda v: v if isinstance(v, str) else ""                                  # só texto; qualquer outro objeto = campo vazio
    td = re.sub(r"\s+", " ", _s(ann.get("target_domain"))).strip()
    dfn = re.sub(r"\s+", " ", _s(ann.get("domain_function"))).strip()
    if not (td and dfn):
        pn = getattr(engine, "pdb_names", None)
        pn = pn if isinstance(pn, dict) else {}
        inf = infer_domain_and_function(pn.get("pdb_id", ""), chain, getattr(engine, "pdb_text", None) or "", pn)
        td, dfn = td or inf["target_domain"], dfn or inf["domain_function"]
    return td, dfn


def _ar_append_to_zip(zip_path, files):
    """1 ligante: o .zip do núcleo já foi fechado; acrescenta a ele a pasta 07_relatorio_academico/ (falha não é fatal)."""
    try:
        zp = Path(str(zip_path or ""))
        if zp.suffix.lower() != ".zip" or not zp.is_file() or not files:
            return
        with zipfile.ZipFile(zp, "a", zipfile.ZIP_DEFLATED) as z:
            have = set(z.namelist())
            for fp in files:
                nm = f"{REPORT_DIRNAME}/{Path(fp).name}"
                if Path(fp).exists() and nm not in have:
                    z.write(str(fp), nm)
    except Exception as e_z:
        print(f"⚠️ Relatório acadêmico não acrescentado ao .zip ({type(e_z).__name__}: {e_z})")


def gerar_relatorio_academico(engine, results, ctrl, sdir, prefix_nome, pdb_id, chain, exhaustiveness, qnotes=()):
    """Monta config + leaderboard_df a partir do estado da corrida e grava .docx, .pdf, .md, .tex, referencias.bib e .txt em
    sdir/07_relatorio_academico. Devolve {"files", "html", "md", "error", "warnings"}. NUNCA levanta exceção: falhas de .docx/.pdf
    ficam em out["warnings"] (o restante do relatório e o docking seguem normalmente). A memória é liberada em `finally`."""
    out = {"files": [], "html": "", "md": "", "error": "", "warnings": []}
    reporter = None
    try:
        ranked = [it for it in _ranked(results, ctrl) if it[0].get("rec")]
        if not ranked:
            return out
        rows, ligs, box_entries = [], [], []
        qpos = {id(r_): i_ for i_, r_ in enumerate(results)}                  # ordem da fila: define "Alvo 1", "Alvo 2", ...
        for r, role, mode, cmp, hard, soft in ranked:
            t, meta, ctl = r["rec"], _ar_meta(r), is_ctrl(role)
            rows.append({"Composto": r["lig_name"], "Papel": role, "HAC": t["hac"], "Vina_score": t["vina"], "CNNscore": t["cnn_s"],
                         "CNNaffinity_pK": t["pk"], "LE_Vina": t["le_v"], "LE_CNN": t["le_c"],
                         "RMSD": t["rmsd"] if ctl else None, "PoseBusters": t["pb"],
                         "Interacoes_Chave": _ar_interacoes_app(r.get("key_inter"))})
            box_entries.append((r["lig_name"], ctl, t.get("ctx"), meta))
            cid = str(meta.get("Ligante_CID_PubChem", "") or "").strip()
            cid = "" if cid.upper().startswith("PDB") or cid.lower() in ("n/d", "nan", "none") else cid
            try:
                chg = f"carga formal {int(float(meta.get('Carga_Formal_pH7.4'))):+d} em pH {PH_FISIOLOGICO}"
            except (TypeError, ValueError):
                chg = "carga formal n/d"
            ion = str(meta.get("Ionizacao_Aplicada", "") or "").strip()
            ligs.append({"name": r["lig_name"], "cid": cid, "role": role, "_q": qpos.get(id(r), 0),
                         "deprotonated_state": chg + (f"; {ion}" if ion and ion not in ("n/d", "—") else "")})
        ligs.sort(key=lambda d_: d_["_q"])
        for d_ in ligs:
            d_.pop("_q", None)
        pn = getattr(engine, "pdb_names", None) or {}
        qc = getattr(engine, "receptor_qc", None) or {}
        pdb_u = str(pdb_id).strip().upper()
        metals = []
        try:
            rf = sorted(Path(sdir).glob("*_receptor_h.pdb"))
            for m in (metal_records(rf[0]) if rf else []):
                lab = f"{metal_symbol(m)}{m['resseq']}"
                if lab not in metals:
                    metals.append(lab)
        except Exception as e_m:
            print(f"⚠️ Relatório: metais do receptor não lidos ({type(e_m).__name__}: {e_m})")
        grid = _ar_grid_info(box_entries)                                 # centro/tamanho EFETIVOS (mesmos valores enviados ao GNINA)
        bx = ""
        if not grid:                                                      # sem coordenadas legíveis: descrição textual
            bx = next((str(_ar_meta(r).get("Grid_Box_Efetiva (passada ao GNINA via CLI)", "")) for r in results if r.get("ok")
                       and _ar_meta(r).get("Grid_Box_Efetiva (passada ao GNINA via CLI)")), "")
        notes = [str(x) for x in (qnotes or [])]
        for r in results:
            if not r.get("ok"):
                notes.append(f"{r.get('name')} não foi processado ({str(r.get('reason'))[:140]}) e não consta nas tabelas.")
                continue
            if not str(r.get("gnina_dev", "")).startswith("GPU"):
                notes.append(f"{r['lig_name']}: CNN avaliado em {r.get('gnina_dev', 'n/d')}.")
            if "Status_Contatos" in getattr(r.get("df_prolif"), "columns", []):
                notes.append(f"{r['lig_name']}: sem interações ProLIF nesta corrida.")
            elif str(r.get("prolif_branch", "")).startswith("FALLBACK GEOMÉTRICO"):
                notes.append(f"{r['lig_name']}: interações calculadas por proximidade geométrica (ProLIF indisponível nesta corrida).")
        td, dfn = _ar_annotation(engine, chain)
        cfg = {"protein_name": (pn.get("chains", {}) or {}).get(chain) or pn.get("title") or f"PDB {pdb_u}",
               "organism": (pn.get("organisms", {}) or {}).get(chain, ""), "pdb_id": pdb_u, "target_chain": str(chain),
               "has_metals": bool(metals), "metals_list": metals, "ph_value": PH_FISIOLOGICO,
               "box_padding_angstrom": AUTOBOX_ADD, "box_description": bx,
               "docking_engine": f"{_ar_gnina_label(GNINA_VERSION)} (função de escore AutoDock Vina)",   # sem build/commit nem opções de CLI
               "ligands_info": ligs, "exhaustiveness": int(exhaustiveness), "seed": RANDOM_SEED,
               "n_conformers": LIG_N_CONFORMERS, "gnina_params": dict(GNINA_DOCK_PARAMS),
               "receptor_method": qc.get("method") or "PDBFixer/OpenMM",
               "his_states": {str(k): str(v) for k, v in (qc.get("his_states") or {}).items()},
               "cofactors_list": list(getattr(engine, "cofactor_names", None) or []),
               "app_name": PLATFORM_NAME, "run_notes": notes,         # PLATFORM_NAME já traz "v12.0"
               "target_domain": td, "domain_function": dfn,
               "target_paper_bib": AR_PDB_PRIMARY_BIB.get(pdb_u) or str(pn.get("primary_bib") or ""),   # artigo cristalográfico do PDB
               "catalytic_residues": [str(x) for x in (getattr(engine, "catalytic_residues", None) or [])]}
        sk = str(getattr(engine, "source_kind", "rcsb") or "rcsb")           # "rcsb" | "local" | "alphafold"
        cfg["source_kind"] = sk
        if sk == "alphafold":                                                # SOMENTE aqui o relatório ganha o texto de modelo predito
            pls = [r_["plddt"] for r_ in results if r_.get("ok") and r_.get("plddt")]
            cfg["uniprot_id"] = str(pn.get("uniprot_id") or "")
            if pls:
                cfg["mean_pocket_plddt"] = round(float(np.mean([q["mean"] for q in pls])), 2)
                cfg["min_pocket_plddt"] = round(float(min(q["min"] for q in pls)), 2)
        cfg.update(grid)          # grid_center [3 casas], grid_size [1 casa], box_mode, grid_size_by_ligand, grid_size_extended
        cfg["box_padding_angstrom"] = float(AUTOBOX_ADD)
        reporter = AcademicReporter(cfg, pd.DataFrame(rows))
        rdir = Path(sdir) / REPORT_DIRNAME
        # .docx e .pdf são isolados dentro de export_bundle: se falharem, só faltam na lista e o motivo vai para out["warnings"]
        fl = reporter.export_bundle(str(rdir), basename=f"{prefix_nome}_Relatorio_Academico_{VERSION_TAG}", make_zip=False,
                                    include_txt=True)
        out["files"] = [str(fl[k]) for k in ("docx", "pdf", "md", "tex", "bib", "txt") if k in fl]
        out["warnings"] = [f".{k} não gerado ({v[:140]})" for k, v in (getattr(reporter, "export_errors", None) or {}).items()]
        for w_ in out["warnings"]:
            print(f"⚠️ Relatório acadêmico: {w_}")
        out["md"] = reporter.to_markdown()
        pre = "<pre style=\"white-space:pre-wrap;font-size:.8em\">"
        fmts = ", ".join(Path(f).suffix.lstrip(".") for f in out["files"])
        out["html"] = ("<details><summary><b>Relatório acadêmico (prévia): Metodologia e Resultados/Discussão</b></summary>"
                       + reporter.to_html()
                       + "<details><summary>LaTeX (.tex; abntex2 + biblatex-abnt)</summary>" + pre + esc(reporter.to_latex()) + "</pre></details>"
                       + "<details><summary>referencias.bib</summary>" + pre + esc(reporter.to_bibtex()) + "</pre></details>"
                       + f"<div class=\"dd-detail\">Arquivos ({esc(fmts)}) salvos em <code>{esc(rdir)}</code> e na pasta "
                         f"<code>{REPORT_DIRNAME}/</code> do .zip."
                       + (f" <b>Atenção:</b> {esc('; '.join(out['warnings']))}." if out["warnings"] else "") + "</div></details>")
        if REPORT_DISPLAY_IPYTHON:
            try:
                from IPython.display import display, Markdown
                display(Markdown(out["md"]))
            except Exception:
                pass
    except Exception as e:
        import traceback
        out["error"] = f"{type(e).__name__}: {e}"
        print(f"⚠️ Relatório acadêmico não gerado: {out['error']}\n{traceback.format_exc()[-800:]}")
    finally:
        reporter = None                                                   # libera DataFrames/blocos do relatório
        try:
            _free_memory()
        except Exception:
            pass
    return out


def _concat_named(results, key, post=None):
    parts = []
    for r in results:
        if r.get("ok") and r.get(key) is not None:
            d = r[key].copy()
            d.insert(0, "Composto", r["lig_name"])
            if post:
                d = post(d, r)
            parts.append(d)
    if not parts:
        return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True, sort=False)
    for c in df.columns:
        col = df[c].dropna()
        if len(col) and col.map(lambda v: isinstance(v, (bool, np.bool_))).all():
            df[c] = df[c].fillna(False)
    return df


def _style_artigo(ws, df, notes):
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.utils import get_column_letter
    thin = Side(style="thin", color="999999")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    ncol = len(df.columns)
    widths = [30, 16, 7, 14, 13, 10, 13, 9, 10, 12, 52]
    fmts = {4: "0.00", 5: '+0.00;-0.00;0.00', 6: "0.000", 7: "0.00", 8: "0.000"}
    for j in range(1, ncol + 1):
        ws.column_dimensions[get_column_letter(j)].width = widths[j - 1] if j <= len(widths) else 14
        c = ws.cell(row=1, column=j)
        c.font, c.fill = Font(bold=True, color="FFFFFF"), PatternFill("solid", fgColor="3F3D8F")
        c.alignment, c.border = Alignment(horizontal="center", vertical="center", wrap_text=True), border
    ws.row_dimensions[1].height = 32
    for i in range(2, len(df) + 2):
        is_ctrl_row = ws.cell(row=i, column=2).value == ROLE_CTRL
        for j in range(1, ncol + 1):
            c = ws.cell(row=i, column=j)
            c.border = border
            c.alignment = Alignment(horizontal="left" if j in (1, ncol) else "center", vertical="top", wrap_text=True)
            if j in fmts and isinstance(c.value, (int, float)):
                c.number_format = fmts[j]
            if is_ctrl_row:
                c.fill = PatternFill("solid", fgColor="E6F4EA")
    r0 = len(df) + 3
    ws.cell(row=r0, column=1, value="Notas").font = Font(bold=True)
    for k, txt in enumerate(notes, 1):
        row = r0 + k
        ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=ncol)
        c = ws.cell(row=row, column=1, value=f"{k}. {txt}")
        c.alignment, c.font = Alignment(wrap_text=True, vertical="top"), Font(italic=True, size=9)
        ws.row_dimensions[row].height = 14 * (1 + len(txt) // 150)
    ws.freeze_panes = "A2"


def write_leaderboard_xlsx(path, results, ctrl, batch_meta):
    """<PDB>_<cadeia>_LEADERBOARD_v<versão>_<data>.xlsx: Tabela_Artigo (1ª aba) + abas completas consolidadas de todos os ligantes."""
    art = build_tabela_artigo(results, ctrl)
    roles = {r["stem"]: role for r, role, *_ in _ranked(results, ctrl)}

    def _post_resumo(d, r):
        d.insert(1, "Papel", roles.get(r["stem"], ROLE_TEST))
        d.insert(2, "★ exportada", ["★" if int(k) == r["selected_idx"] + 1 else "" for k in d["Rank da Pose"]])
        return d

    resumo = _concat_named(results, "df_exec", _post_resumo)
    pbs, plf_all = _concat_named(results, "pb_df"), _concat_named(results, "df_prolif")
    bm_parts = []
    for r, role, mode, cmp, hard, soft in _ranked(results, ctrl):
        if is_ctrl(role):
            d = bm_table_df(None, r["rec"], None, "Controle nativo (referência do lote)" if role == ROLE_CTRL else "Controle nativo NÃO validado (RMSD simétrico indisponível)")
        elif cmp is not None:
            d = bm_table_df(r["rec"], ctrl, cmp, "Comparação válida" if not soft else "Comparação com ressalvas: " + "; ".join(soft))
        else:
            d = bm_table_df(r["rec"], None, None, _delta_status(mode, hard))
        d.insert(0, "Composto", r["lig_name"])
        bm_parts.append(d)
    bmdf = pd.concat(bm_parts, ignore_index=True) if bm_parts else pd.DataFrame(
        [["—", "Status", BM_NO_CONTROL, "", ""]], columns=["Composto"] + BM_COLS)
    meta_rows = [["Lote", k, v] for k, v in batch_meta.items()]
    for r in results:
        if r.get("ok") and r.get("df_meta") is not None:
            meta_rows += [[r["lig_name"], row["Parâmetro"], row["Valor"]] for _, row in r["df_meta"].iterrows()]
        elif not r.get("ok"):
            meta_rows.append([r["name"], "Status", "FALHA: " + str(r["reason"])[:300]])
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        art.to_excel(w, sheet_name="Tabela_Artigo", index=False)
        _style_artigo(w.sheets["Tabela_Artigo"], art, artigo_notes(results, ctrl))
        build_parecer_df(results, ctrl).to_excel(w, sheet_name="Parecer_Cientifico", index=False)
        _style_parecer(w.sheets["Parecer_Cientifico"])
        resumo.to_excel(w, sheet_name="Resumo_Executivo", index=False)
        pbs.to_excel(w, sheet_name="Auditoria_PoseBusters", index=False)
        plf_all.to_excel(w, sheet_name="Interacoes_ProLIF", index=False)
        bmdf.to_excel(w, sheet_name="Benchmark_Nativo", index=False)
        pd.DataFrame(meta_rows, columns=["Escopo", "Parâmetro", "Valor"]).to_excel(w, sheet_name="Metadados_Sessao", index=False)


def build_batch_zip(zip_path, results, xlsx_path, figs=None, reports=None, md_files=None):
    """<PDB>_<cadeia>_Pacote_Completo_v<versão>_<data>.zip: planilha consolidada + gráficos + relatório acadêmico + 00_receptor/ (uma vez) + uma subpasta por ligante."""
    rec_done = False
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zout:
        if xlsx_path and Path(xlsx_path).exists():
            zout.write(str(xlsx_path), Path(xlsx_path).name)
        for fp in (figs or []):
            if Path(fp).exists():
                zout.write(str(fp), f"{FIG_DIRNAME}/{Path(fp).name}")
        for fp in (reports or []):                       # relatório acadêmico: .docx, .pdf, .md, .tex e referencias.bib
            if Path(fp).exists():
                zout.write(str(fp), f"{REPORT_DIRNAME}/{Path(fp).name}")
        for fp in (md_files or []):                      # 08_dinamica_molecular/: script turnkey + complexo pré-alinhado (+ SDF) da pose líder
            if Path(fp).exists():
                zout.write(str(fp), f"{MD_DIRNAME}/{Path(fp).name}")
        for i, r in enumerate([x for x in results if x.get("ok")], 1):
            zp = r.get("zip_path")
            if not zp or not Path(zp).exists():
                continue
            folder = f"{i:02d}_{re.sub(r'[^A-Za-z0-9_.-]', '_', str(r['stem']))[:50]}"
            wrote_rec = False
            with zipfile.ZipFile(zp) as zin:
                for info in zin.infolist():
                    if info.is_dir():
                        continue
                    if md_files and info.filename.startswith(MD_DIRNAME + "/"):
                        continue                            # só a pose líder leva a pasta de MD na raiz do .zip do lote
                    if info.filename.startswith("00_receptor/"):
                        if not rec_done:
                            zout.writestr(info.filename, zin.read(info.filename))
                            wrote_rec = True
                        continue
                    zout.writestr(f"{folder}/{info.filename}", zin.read(info.filename))
            rec_done = rec_done or wrote_rec


def _strip_tags(h):
    return re.sub(r"\s+", " ", _html.unescape(re.sub(r"<[^>]+>", " ", str(h)))).strip()


def leaderboard_html(results, ctrl):
    """Passo 4 no modo lote: faixa de contagens, aviso do controle e um bloco recolhível por ligante."""
    ranked = _ranked(results, ctrl)
    n, ok = len(results), len([r for r in results if r.get("ok")])
    cls = Counter(it[3]["veredito"].split(" ")[0] for it in ranked if it[3] is not None)
    tiles = metric_tiles([("Ligantes na fila", n, "info"), ("Concluídos", ok, "ok" if ok == n else "warn"),
                          ("Superiores", cls.get("Superior", 0), "ok"), ("Promissores", cls.get("Promissor", 0), "ok"),
                          ("Equivalentes", cls.get("Equivalente", 0), "info"),
                          ("Bioativos eficientes", cls.get("Bioativo", 0), "info"),
                          ("Inconclusivos", cls.get("Inconclusivo", 0), "warn"),
                          ("Inviáveis (PB FAIL)", cls.get("Inviável", 0), "err"),
                          ("Inferiores", cls.get("Inferior", 0), "warn")])
    if ctrl is not None:
        head = status_box("info", f"Controle nativo: {ctrl['ctx']['resname']}",
                          f"Vina {ctrl['vina']:.2f} kcal/mol · CNNaffinity {ctrl['pk']:.2f} pK · LE Vina {_bm_fmt(ctrl['le_v'], 3)}. "
                          f"Veredito multiparamétrico (PoseBusters, Vina na banda de ±{BM_BAND_KCAL} kcal/mol, CNN e LE); computacional, não experimental.")
    else:
        head = status_box("warn", "Sem controle nativo neste lote",
                          esc(BM_NO_CONTROL) + ". Os Δ aparecem como n/d. Dê docking no ligante cristalográfico (modo 2) antes do lote.")
    blocks = []
    for k, (r, role, mode, cmp, hard, soft) in enumerate(ranked, 1):
        t, html = r["rec"], r.get("summary_html", "")
        if mode not in ("no_native", "erro", "unvalidated") and r.get("bm") and r["bm"].get("html") and r["bm"]["html"] in html:
            html = html.replace(r["bm"]["html"], bm_card(mode, t, t if mode == "control_saved" else ctrl, cmp, hard, soft,
                                                         extra=r.get("native_pdb", "")))
        lab = "controle" if is_ctrl(role) else (cmp["veredito"] if cmp else "n/d")
        blocks.append(f"<details><summary><b>{esc(r['lig_name'])}</b> — Vina {t['vina']:.2f} · pK {t['pk']:.2f} · {esc(lab)}</summary>{html}{parecer_html(r, ctrl)}</details>")
    for r in results:
        if not r.get("ok"):
            blocks.append(f"<details><summary><b>{esc(r['name'])}</b> — FALHA</summary>"
                          + status_box("err", "Ligante não processado", esc(str(r["reason"])[:600])) + "</details>")
    return head + tiles + "".join(blocks)


class DockingInterfaceEngine:
    def __init__(self):
        self._auto_annotation = ("", "")                     # último par inferido do PDB (para não sobrescrever edição manual)
        self.target_annotation = {"target_domain": "", "domain_function": ""}   # anotação funcional do alvo (Passo 1, opcional)
        self.cofactor_names = []
        self.cofactor_lines_cache = []
        self.source_kind = "rcsb"                      # "rcsb" | "local" | "alphafold"
        self.local_source_copy = None
        self.base_workdir = Path("gradio_docking_sessions")
        self.base_workdir.mkdir(parents=True, exist_ok=True)
        self.pubchem_dir = self.base_workdir / "pubchem_downloads"
        self.pubchem_dir.mkdir(parents=True, exist_ok=True)
        self.pdb_text = None
        self.ligands_by_chain = {}
        self.metals_by_chain = {}
        self.cofactors_by_chain = {}
        self.pdb_names = {"pdb_id": "", "title": "", "chains": {}, "organisms": {}}
        self.native_centroid = None
        self.native_box_span = None
        self.native_bbox_center = None
        self.receptor_qc = None
        self.native_benchmark = None                   # controle positivo (dict em memória)
        self._lock_control = False                     # controle da corrida em curso não é substituído por um teste idêntico
        self._last_result = None                       # resultado estruturado do último ligante (lido pelo lote)
        self.receptor_prep_log = ""
        try:
            self.no_gpu = subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL,
                                         stderr=subprocess.DEVNULL).returncode != 0
        except Exception:
            self.no_gpu = True

    def _get_session_dir(self, pdb_id: str, chain: str, new: bool = False) -> Path:
        cache = self.__dict__.setdefault("_session_dirs", {})
        key = (str(pdb_id), str(chain))
        if new or key not in cache or not cache[key].exists():
            sdir = self.base_workdir / f"{pdb_id}_{chain}_{int(time.time())}"
            k = 1
            while sdir.exists():
                k += 1
                sdir = self.base_workdir / f"{pdb_id}_{chain}_{int(time.time())}_{k}"
            sdir.mkdir(parents=True, exist_ok=True)
            cache[key] = sdir
        return cache[key]

    # --------------------------------------------------------------------------
    # PubChem
    # --------------------------------------------------------------------------
    def _pubchem_download(self, query: str, progress):
        """Baixa o SDF do PubChem. Retorna (caminho | None, mensagem em texto puro, info do composto)."""
        if not query or not query.strip():
            return None, "Digite um Nome (ex.: Quercetin) ou CID (ex.: 5280343) para buscar no PubChem.", {}

        q = query.strip()
        is_cid = q.isdigit()
        namespace = "cid" if is_cid else "name"
        encoded_q = urllib.parse.quote(q)
        progress(0.3, desc=f"Consultando PubChem ({namespace.upper()}: {q})...")

        urls_to_try = [
            (f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/{namespace}/{encoded_q}/SDF?record_type=3d",
             "3D oficial do PubChem"),
            (f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/{namespace}/{encoded_q}/SDF",
             "2D do PubChem (convertido em 3D no preparo)"),
        ]
        sdf_bytes, source_dim_type = None, ""
        for url, dim_label in urls_to_try:
            try:
                req = urllib.request.Request(url, headers={"User-Agent": f"DL-Docking-Studio/{PIPELINE_VERSION}"})
                with urllib.request.urlopen(req, timeout=20) as resp:
                    sdf_bytes = resp.read()
                    source_dim_type = dim_label
                    break
            except Exception:
                continue

        if not sdf_bytes:
            return None, f"Composto '{q}' não encontrado no PubChem. Tente o CID numérico.", {}

        progress(0.8, desc="Validando estrutura baixada do PubChem...")
        safe_q = re.sub(r"[^a-zA-Z0-9_-]", "_", q)[:30]
        temp_sdf = self.pubchem_dir / f"PubChem_{safe_q}.sdf"
        temp_sdf.write_bytes(sdf_bytes)

        suppl = [m for m in Chem.SDMolSupplier(str(temp_sdf), removeHs=False) if m is not None]
        if not suppl:
            return None, "Falha ao decodificar a estrutura baixada do PubChem.", {}

        mol = suppl[0]
        props = mol.GetPropsAsDict()
        cid_val = props.get("PUBCHEM_COMPOUND_CID", q if is_cid else "N/A")
        iupac_val = props.get("PUBCHEM_IUPAC_NAME", "N/A")
        formula = rdMolDescriptors.CalcMolFormula(mol)
        hac = mol.GetNumHeavyAtoms()

        # Nome comum (campo Title do PUG REST); contingência: texto digitado ou nome IUPAC
        common = ""
        if str(cid_val).isdigit():
            try:
                turl = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_val}/property/Title/JSON"
                treq = urllib.request.Request(turl, headers={"User-Agent": f"DL-Docking-Studio/{PIPELINE_VERSION}"})
                with urllib.request.urlopen(treq, timeout=10) as tresp:
                    common = json.loads(tresp.read().decode("utf-8"))["PropertyTable"]["Properties"][0].get("Title", "") or ""
            except Exception:
                common = ""
        if not common and not is_cid:
            common = q
        display_name = common or (str(iupac_val) if iupac_val != "N/A" else f"CID {cid_val}")

        final_sdf_path = self.pubchem_dir / f"CID_{cid_val}_{safe_q}.sdf"
        shutil.move(str(temp_sdf), str(final_sdf_path))
        progress(1.0, desc="Ligante baixado do PubChem com sucesso!")
        msg = (f"{display_name} | CID {cid_val} ({source_dim_type}) | Fórmula: {formula} | "
               f"Átomos pesados (HAC): {hac} | IUPAC: {str(iupac_val)[:70]}")
        info = {"file": str(final_sdf_path), "name": str(display_name), "common": str(common),
                "iupac": str(iupac_val), "cid": str(cid_val), "formula": formula, "source": "PubChem"}
        return str(final_sdf_path), msg, info

    def fetch_ligand_from_pubchem(self, query: str, progress=gr.Progress()):
        """Baixa 1 ou vários compostos (CIDs/nomes separados por vírgula+espaço, ';' ou quebra de linha)."""
        tokens = parse_ligand_queries(query)
        if not tokens:
            return None, status_box("err", "Ligante não obtido",
                                    "Digite um Nome (ex.: Quercetin) ou CID (ex.: 5280343). Vários: separe por vírgula ou quebra de linha."), None
        n = len(tokens)
        paths, infos, lines, fails = [], [], [], []
        for k, tok in enumerate(tokens):
            def sub(frac=0.0, desc="", _k=k):
                progress((_k + max(0.0, min(1.0, float(frac)))) / n, desc=(f"[{_k + 1}/{n}] {desc}" if n > 1 else desc))
            path, msg, info = self._pubchem_download(tok, sub)
            if path is None and "," in tok:                      # 'Quercetin,Rutin' (sem espaço): tenta cada parte
                for part in [x.strip() for x in tok.split(",") if x.strip()]:
                    p2, m2, i2 = self._pubchem_download(part, sub)
                    if p2 is None:
                        fails.append(f"{part}: {m2}")
                    elif p2 not in paths:
                        paths.append(p2)
                        infos.append(i2)
                        lines.append(m2)
                continue
            if path is None:
                fails.append(f"{tok}: {msg}")
            elif path not in paths:
                paths.append(path)
                infos.append(info)
                lines.append(msg)
        if not paths:
            return None, status_box("err", "Ligante não obtido", "<br>".join(esc(f) for f in fails)), None
        if len(paths) > MAX_BATCH_LIGANDS:
            return None, status_box("err", f"Fila grande demais ({len(paths)})", f"O limite é {MAX_BATCH_LIGANDS} ligantes por corrida."), None
        if len(paths) == 1 and not fails:                       # 1 molécula sem falhas: mensagem direta do ligante (sem lista numerada)
            title = f"Ligante: {infos[0]['name']}" if infos[0].get("name") else "Ligante baixado do PubChem"
            return paths, status_box("ok", title, esc(lines[0])), infos
        body = "<br>".join(f"{i + 1}. {esc(m)}" for i, m in enumerate(lines))
        html = status_box("ok", f"{len(paths)} ligante(s) na fila (PubChem)", body)
        if fails:
            html += status_box("warn", f"{len(fails)} item(ns) não obtido(s)", "<br>".join(esc(f) for f in fails))
        return paths, html, infos

    def _split_sdf(self, path, qdir):
        """SDF multi-molécula -> um SDF por molécula (3D e propriedades preservados). None se for molécula única."""
        mols = list(Chem.SDMolSupplier(str(path), removeHs=False))
        if len(mols) <= 1:
            return None, 0
        valid = [(i, m) for i, m in enumerate(mols) if m is not None]
        if len(valid) > MAX_BATCH_LIGANDS:
            raise ValueError(f"{Path(path).name} tem {len(valid)} moléculas (limite {MAX_BATCH_LIGANDS} por corrida).")
        out = []
        for i, m in valid:
            props = m.GetPropsAsDict()
            nm = (m.GetProp("_Name").strip() if m.HasProp("_Name") else "") or str(props.get("PUBCHEM_COMPOUND_CID", "")) or f"mol{i + 1}"
            safe = re.sub(r"[^A-Za-z0-9_-]", "_", nm)[:40] or f"mol{i + 1}"
            fp = qdir / f"{Path(path).stem}_{i + 1:03d}_{safe}.sdf"
            with Chem.SDWriter(str(fp)) as w:
                w.write(m)
            out.append((fp, nm, {"file": str(fp), "name": nm, "common": nm, "iupac": "N/A",
                                 "cid": str(props.get("PUBCHEM_COMPOUND_CID", "n/d")),
                                 "formula": rdMolDescriptors.CalcMolFormula(m), "source": "SDF multi-molécula"}))
        return out, len(mols) - len(valid)

    def _build_queue(self, user_ligand_file, pubchem_query, lig_info, progress, native_pdb=None, auto_native=False, prep_mode=None):
        """Fila: [#0 nativo (modo 2), se pedido e existente] + ligantes de teste (arquivos, SDF multi-molécula ou PubChem),
        cada item com o seu modo de preparo. Retorna (itens, avisos, erro)."""
        qdir = self.base_workdir / "_queue"
        qdir.mkdir(parents=True, exist_ok=True)
        infos, items, notes, used = _info_list(lig_info), [], [], set()
        has_native = bool(auto_native and native_pdb and Path(str(native_pdb)).is_file())
        tests_given = bool(_iter_paths(user_ligand_file)) or bool(parse_ligand_queries(pubchem_query))
        if has_native:
            used.add(Path(str(native_pdb)).stem)

        def add(path, info, label=None):
            p, stem = Path(path), Path(path).stem
            if stem in used:                                     # evita colisão dos nomes das saídas
                k = 2
                while f"{stem}_{k}" in used:
                    k += 1
                newp = qdir / f"{stem}_{k}{p.suffix}"
                shutil.copy(str(p), str(newp))
                p, stem = newp, f"{stem}_{k}"
                if info:
                    info = dict(info, file=str(p))
            used.add(stem)
            items.append({"path": str(p), "info": info, "label": label or (info or {}).get("name") or stem,
                          "prep": prep_mode, "is_native": False})

        paths = list(dict.fromkeys(_iter_paths(user_ligand_file)))
        if paths:
            for p in paths:
                suf = Path(p).suffix.lower()
                if suf not in (".sdf", ".mol2", ".pdb", ".pdbqt"):
                    return [], notes, ("Formato não suportado", f"{Path(p).name}: utilize SDF, MOL2, PDB ou PDBQT.")
                if suf == ".sdf":
                    try:
                        parts, skipped = self._split_sdf(p, qdir)
                    except Exception as e:
                        return [], notes, ("SDF inválido", str(e))
                    if parts:
                        if skipped:
                            notes.append(f"{Path(p).name}: {skipped} molécula(s) ilegível(is) ignorada(s).")
                        for fp, nm, inf in parts:
                            add(fp, inf, nm)
                        continue
                add(p, _info_for(p, infos))
        else:
            tokens = parse_ligand_queries(pubchem_query)
            if not tokens and not has_native:
                return [], notes, ("Nenhum ligante informado",
                                   "Envie um ou mais arquivos (.sdf, .mol2, .pdb, .pdbqt) ou baixe compostos do PubChem no Passo 2.")
            for k, tok in enumerate(tokens):
                def sub(frac=0.0, desc="", _k=k):
                    progress(0.02 * (_k + max(0.0, min(1.0, float(frac)))) / len(tokens), desc=desc)
                path, msg, info = self._pubchem_download(tok, sub)
                if path is None:
                    if len(tokens) == 1:
                        return [], notes, ("Ligante não obtido", msg)
                    notes.append(f"{tok}: {msg}")
                    continue
                add(path, info)
            if tokens and not items:
                return [], notes, ("Nenhum ligante obtido", " | ".join(notes))
        if len(items) > MAX_BATCH_LIGANDS:
            return [], notes, ("Fila grande demais", f"{len(items)} ligantes de teste; o limite é {MAX_BATCH_LIGANDS} por corrida. Divida em lotes.")
        if has_native:
            rn = bm_native_resname(native_pdb)
            nat = {"path": str(native_pdb), "label": f"{rn} (controle nativo)", "prep": PREP_MODES[1], "is_native": True,
                   "info": {"file": str(native_pdb), "name": f"{rn} (nativo)", "common": rn, "iupac": "N/A",
                            "cid": f"PDB {self.pdb_names.get('pdb_id', '')}".strip(), "formula": "", "source": "ligante cristalográfico (PDB)"}}
            items.insert(0, nat)
            if not tests_given:
                notes.append("Nenhum ligante de teste informado: executado somente o controle nativo.")
        return items, notes, None

    # --------------------------------------------------------------------------
    # RCSB PDB
    # --------------------------------------------------------------------------
    def _metals_update(self, chain):
        """Marca/desmarca o checkbox de cofatores conforme os metais catalíticos da cadeia."""
        if chain and self.pdb_text:
            return gr.update(value=bool(self.metals_by_chain.get(chain) or self.cofactors_by_chain.get(chain)))
        return gr.update()

    def fetch_and_analyze_pdb(self, pdb_id, progress=gr.Progress()):
        a, b, c = self._fetch_and_analyze_pdb_core(pdb_id, progress)
        chain = a.get("value") if isinstance(a, dict) else getattr(a, "value", None)
        return a, b, c, self._metals_update(chain)

    def _load_local_pdb_core(self, pdb_file, progress=gr.Progress()):
        """PDB enviado pelo usuário (ex.: modelo SWISS-MODEL): lê, normaliza, guarda cópia e usa a MESMA varredura do RCSB."""
        empty = gr.update(choices=[], value=None)
        path = pdb_file.get("path") or pdb_file.get("name") if isinstance(pdb_file, dict) else getattr(pdb_file, "name", pdb_file)
        if not path or not Path(str(path)).is_file():
            return empty, empty, status_box("err", "Nenhum arquivo enviado", "Envie um arquivo <b>.pdb</b> e clique em Carregar.")
        p = Path(str(path))
        if p.suffix.lower() not in (".pdb", ".ent"):
            return empty, empty, status_box("err", "Formato não suportado", f"Recebi <b>{esc(p.name)}</b>; envie um arquivo .pdb.")
        if p.stat().st_size > MAX_LOCAL_PDB_MB * 1024 * 1024:
            return empty, empty, status_box("err", "Arquivo grande demais", f"Limite de {MAX_LOCAL_PDB_MB} MB.")
        progress(0.2, desc=f"Lendo o PDB local ({p.name})...")
        try:
            text = normalize_local_pdb_text(p.read_text(encoding="utf-8", errors="replace"))
        except Exception as e:
            return empty, empty, status_box("err", "Falha ao ler o arquivo", esc(e))
        atom_lines = [ln for ln in text.split("\n") if ln.startswith("ATOM  ") and len(ln) > 21]
        if not atom_lines:
            return empty, empty, status_box("err", "Arquivo sem registros ATOM",
                                            "Não encontrei átomos de proteína. O arquivo está em formato PDB?")
        local_id = make_local_pdb_id(p.name)
        names = apply_local_placeholders(parse_pdb_names(text), {ln[21] for ln in atom_lines})   # sem consulta ao RCSB
        names["pdb_id"] = local_id
        self.pdb_names = names
        up_dir = self.base_workdir / "_uploads"
        up_dir.mkdir(parents=True, exist_ok=True)
        copy_path = up_dir / f"{local_id}.pdb"
        copy_path.write_text(text)
        self.source_kind, self.local_source_copy = "local", str(copy_path)
        return self._scan_pdb_text(text, local_id, names, progress, f"PDB local carregado: {p.name}")

    def _load_alphafold_core(self, uniprot_id, progress=gr.Progress()):
        """[v12] Modelo do AlphaFold DB por UniProt: model_v4 (HTTP 404 -> model_v3), normaliza a cadeia (coluna 22 em branco -> 'A'),
        guarda a cópia em _uploads/AF_<ID>.pdb, define source_kind = "alphafold" e pdb_names["pdb_id"] = "AF-<ID>"; mesma varredura do RCSB."""
        empty = gr.update(choices=[], value=None)
        uid = re.sub(r"\s+", "", str(uniprot_id or "")).upper()
        if not UNIPROT_RE.match(uid):
            return empty, empty, status_box("err", "UniProt ID inválido", "Use o código de acesso do UniProt, por exemplo <b>P00533</b> ou <b>P04150</b>.")
        progress(0.2, desc=f"Baixando o modelo do AlphaFold DB ({uid})...")
        try:
            raw, ver, url = download_alphafold_pdb(uid)
        except FileNotFoundError as e:
            return empty, empty, status_box("err", "Modelo não encontrado no AlphaFold DB", esc(e))
        except Exception as e:
            return empty, empty, status_box("err", "Falha ao baixar o modelo do AlphaFold DB", esc(e))
        text = normalize_local_pdb_text(raw)
        atom_lines = [ln for ln in text.split("\n") if ln.startswith("ATOM  ") and len(ln) > 21]
        if not atom_lines:
            return empty, empty, status_box("err", "Arquivo sem registros ATOM", f"O AlphaFold DB não devolveu átomos para {esc(uid)}.")
        af_id = f"{AF_ID_PREFIX}{uid}"
        names = parse_pdb_names(text)
        names["title"] = names.get("title") or f"Modelo AlphaFold DB de {uid}"
        names = apply_local_placeholders(names, {ln[21] for ln in atom_lines})
        names["pdb_id"], names["uniprot_id"], names["af_version"], names["af_url"] = af_id, uid, ver, url
        self.pdb_names = names
        up_dir = self.base_workdir / "_uploads"
        up_dir.mkdir(parents=True, exist_ok=True)
        copy_path = up_dir / f"AF_{uid}.pdb"
        copy_path.write_text(text)
        self.source_kind, self.local_source_copy = "alphafold", str(copy_path)
        return self._scan_pdb_text(text, af_id, names, progress, f"Modelo AlphaFold {af_id} carregado (model_{ver})")

    def load_receptor_source(self, mode, pdb_id, pdb_file, uniprot_id="", progress=gr.Progress()):
        """Etapa 1: despacha para RCSB (ID de 4 letras), arquivo local ou AlphaFold DB (UniProt). Saídas:
        (cadeia, ligante nativo, status, checkbox de cofatores, caixa de texto do ID)."""
        if mode in (INPUT_LOCAL, INPUT_AFDB):
            a, b, c = (self._load_alphafold_core(uniprot_id, progress) if mode == INPUT_AFDB
                       else self._load_local_pdb_core(pdb_file, progress))
            chain = a.get("value") if isinstance(a, dict) else getattr(a, "value", None)
            if not chain:
                self.pdb_text = None          # falha: um PDB anterior não pode "vazar" para a extração
                return a, b, c, gr.update(), gr.update(), gr.update(), gr.update()
            return (a, b, c, self._metals_update(chain), gr.update(value=self.pdb_names["pdb_id"]),
                    *self._annotation_update(chain, force=True))
        a, b, c, d = self.fetch_and_analyze_pdb(pdb_id, progress)
        chain = a.get("value") if isinstance(a, dict) else getattr(a, "value", None)
        if not chain:
            return a, b, c, d, gr.update(), gr.update(), gr.update()
        return a, b, c, d, gr.update(), *self._annotation_update(chain, force=True)

    def _annotation_update(self, chain, cur_td="", cur_df="", force=False):
        """v10.3: infere domínio/função do PDB carregado e devolve (update target_domain, update domain_function).
        Só sobrescreve o campo se ele estiver vazio, se ainda tiver o valor automático anterior ou se `force` (novo PDB);
        texto editado pelo usuário é preservado ao trocar de cadeia."""
        try:
            pn = self.pdb_names if isinstance(self.pdb_names, dict) else {}
            inf = infer_domain_and_function(pn.get("pdb_id", ""), chain, self.pdb_text or "", pn)
        except Exception:
            return gr.update(), gr.update()
        old = tuple(getattr(self, "_auto_annotation", ("", "")))
        self._auto_annotation = (inf["target_domain"], inf["domain_function"])
        keep = lambda cur, prev: (not force) and bool(str(cur or "").strip()) and str(cur).strip() != prev   # noqa: E731
        return (gr.update() if keep(cur_td, old[0]) else gr.update(value=inf["target_domain"]),
                gr.update() if keep(cur_df, old[1]) else gr.update(value=inf["domain_function"]))

    def update_ligands_dropdown(self, chain, cur_td="", cur_df=""):
        a, b = self._update_ligands_dropdown_core(chain)
        if not chain or not self.pdb_text:
            return a, b, self._metals_update(chain), gr.update(), gr.update()
        return (a, b, self._metals_update(chain), *self._annotation_update(chain, cur_td, cur_df))

    def _fetch_and_analyze_pdb_core(self, pdb_id, progress=gr.Progress()):
        pdb_id = (pdb_id or "").strip().upper()
        empty = gr.update(choices=[], value=None)
        if not re.match(r"^[0-9A-Z]{4}$", pdb_id):
            return empty, empty, status_box("err", "PDB ID inválido", "Use 4 caracteres, por exemplo <b>2Y9X</b>.")

        progress(0.2, desc=f"Baixando estrutura do RCSB PDB ({pdb_id})...")
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        try:
            req = urllib.request.Request(url, headers={"User-Agent": f"DL-Docking-Studio/{PIPELINE_VERSION}"})
            with urllib.request.urlopen(req, timeout=30) as response:
                pdb_text = response.read().decode("utf-8")
        except Exception as e:
            return empty, empty, status_box("err", "Falha ao baixar o PDB", esc(e))

        names = parse_pdb_names(pdb_text)
        names["primary_bib"] = _ar_jrnl_bibtex(pdb_text)     # artigo cristalográfico original (registros JRNL; '' se não publicado)
        if not names["title"]:      # contingência: API de dados do RCSB
            try:
                api = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
                areq = urllib.request.Request(api, headers={"User-Agent": f"DL-Docking-Studio/{PIPELINE_VERSION}"})
                with urllib.request.urlopen(areq, timeout=10) as aresp:
                    names["title"] = pretty_pdb_text(json.loads(aresp.read().decode("utf-8"))["struct"]["title"])
            except Exception:
                pass
        names["pdb_id"] = pdb_id
        self.pdb_names = names
        self.source_kind, self.local_source_copy = "rcsb", None
        return self._scan_pdb_text(pdb_text, pdb_id, names, progress, f"PDB {pdb_id} carregado")

    def _scan_pdb_text(self, pdb_text, pdb_id, names, progress, status_title):
        """Mapeia cadeias, ligantes nativos, metais e cofatores de um texto PDB (RCSB OU arquivo local)."""
        progress(0.6, desc="Filtrando conformações e mapeando metais catalíticos...")
        chains, ligands_by_chain, metals_by_chain = set(), {}, {}
        _hets = {}                                    # (cadeia, resname, resseq) -> elementos (para achar cofatores)
        for line in pdb_text.splitlines():
            if line.startswith("ENDMDL"):
                break
            if line.startswith("ATOM  "):
                if line[16] not in (" ", "A", "1"):
                    continue
                chains.add(line[21])
            elif line.startswith("HETATM"):
                if line[16] not in (" ", "A", "1"):
                    continue
                chain = line[21]
                resname = line[17:20].strip()
                _hets.setdefault((chain, resname, line[22:26].strip()), []).append(
                    (line[76:78].strip() or line[12:16].strip()[:1]).upper())
                if resname in CATALYTIC_METALS:
                    metals_by_chain.setdefault(chain, set()).add(resname)
                elif resname not in SOLVENTS_AND_IONS:
                    ligands_by_chain.setdefault(chain, set()).add(resname)

        self.cofactors_by_chain = {}
        for (ch, rn, _rs), els in _hets.items():
            if is_bypass_cofactor(rn, els):
                self.cofactors_by_chain.setdefault(ch, set()).add(rn)
        chain_list = sorted(chains)
        self.pdb_text = pdb_text
        self.ligands_by_chain = ligands_by_chain
        self.metals_by_chain = metals_by_chain

        default_chain = chain_list[0] if chain_list else None
        ligs = sorted(ligands_by_chain.get(default_chain, [])) if default_chain else []
        metals = sorted(set(metals_by_chain.get(default_chain, [])) | set(self.cofactors_by_chain.get(default_chain, []))) if default_chain else []
        progress(1.0, desc="Estrutura processada com sucesso!")
        mol_name = names["chains"].get(default_chain, "")
        title_html = (f"<b>{esc(names['title'])}</b><br>" if names["title"] else "")
        if mol_name:
            org = names["organisms"].get(default_chain, "")
            title_html += f"Cadeia {esc(default_chain)}: {esc(mol_name)}{(' (' + esc(org) + ')') if org else ''}<br>"
        body = (title_html + f"Cadeias: <b>{esc(', '.join(chain_list))}</b> &nbsp;|&nbsp; "
                f"Cadeia <b>{esc(default_chain)}</b>: ligantes {esc(ligs) if ligs else '—'}, "
                f"metais/cofatores {esc(', '.join(metals)) if metals else '—'}")
        if self.source_kind == "alphafold":          # confiabilidade do modelo predito (pLDDT = fator B)
            try:
                body += "<br>" + plddt_html(None, global_plddt(pdb_text, default_chain))
            except Exception as e_pl:
                print(f"⚠️ pLDDT global não calculado: {type(e_pl).__name__}: {e_pl}")
        return (gr.update(choices=chain_list, value=default_chain),
                gr.update(choices=ligs, value=ligs[0] if ligs else None),
                status_box("ok", status_title, body))

    def _update_ligands_dropdown_core(self, chain):
        if not chain or not self.pdb_text:
            return gr.update(choices=[], value=None), status_box("info", "Nenhuma cadeia selecionada")
        ligs = sorted(self.ligands_by_chain.get(chain, []))
        metals = sorted(set(self.metals_by_chain.get(chain, [])) | set(self.cofactors_by_chain.get(chain, [])))
        metal_info = f" | metais/cofatores: {', '.join(metals)}" if metals else ""
        mol_name = self.pdb_names.get("chains", {}).get(chain, "")
        if mol_name:
            metal_info += f" | molécula: {mol_name}"
        if not ligs:
            return (gr.update(choices=[], value=None),
                    status_box("warn", f"Cadeia {chain} sem ligantes orgânicos",
                               f"Use a caixa manual no Passo 3.{esc(metal_info)}"))
        return (gr.update(choices=ligs, value=ligs[0]),
                status_box("info", f"Cadeia {chain}", f"Ligantes: {esc(', '.join(ligs))}{esc(metal_info)}"))


    def _prepare_receptor_with_pdbfixer(self, raw_pdb: Path, out_pdb: Path, ph: float = 7.4) -> dict:
        """Envoltório anti-OOM: garante a devolução da RAM mesmo se o preparo falhar."""
        try:
            try:
                qc, peak = run_isolated(self._prepare_receptor_with_pdbfixer_impl, (raw_pdb, out_pdb, ph, False),
                                        timeout_s=480, label="PDBFixer/OpenMM")
            except IsolatedMemoryError as e:
                print(f"⚠️ {e}\n   Nova tentativa SEM reconstrução de átomos ausentes (modo econômico).")
                qc, peak = run_isolated(self._prepare_receptor_with_pdbfixer_impl, (raw_pdb, out_pdb, ph, True),
                                        timeout_s=480, label="PDBFixer/OpenMM (modo econômico)")
                qc["memory_guard"] = ("1ª tentativa excedeu a RAM e foi encerrada; refeito sem reconstruir átomos "
                                      "ausentes. " + (qc.get("memory_guard") or ""))
            qc["pdbfixer_peak_mb"] = round(peak, 0)
            return qc
        finally:
            _free_memory()

    def _prepare_receptor_with_pdbfixer_impl(self, raw_pdb: Path, out_pdb: Path, ph: float = 7.4,
                                             skip_missing: bool = False) -> dict:
        """Átomos ausentes + hidrogênios em pH `ph`. Histidinas/cisteínas coordenadas a metal recebem
        tautômero/estado explícito; H em conflito (tioéter His–Cys, esfera do metal) são removidos."""
        try:
            from pdbfixer import PDBFixer
            from openmm import unit
            from openmm.app import PDBFile, Modeller
        except Exception as e:
            raise RuntimeError(f"OpenMM/PDBFixer indisponíveis: {e}")

        fixer = PDBFixer(filename=str(raw_pdb))
        try:
            fixer.findNonstandardResidues()
            fixer.replaceNonstandardResidues()
        except Exception:
            pass
        fixer.findMissingResidues()
        fixer.missingResidues = {}               # não reconstruir alças ausentes no cristal
        fixer.findMissingAtoms()
        # TRAVA DE RAM: lacunas cristalográficas grandes fazem o addMissingAtoms/addHydrogens
        # construir estruturas gigantes (OOM). Acima do limite, a reconstrução é ignorada.
        n_missing_heavy = sum(len(v) for v in fixer.missingAtoms.values())
        memory_guard = ""
        if skip_missing or n_missing_heavy > PDBFIXER_MAX_MISSING_ATOMS:
            fixer.missingAtoms = {}
            memory_guard = (f"{n_missing_heavy} átomos pesados ausentes"
                            + (f" (> {PDBFIXER_MAX_MISSING_ATOMS})" if n_missing_heavy > PDBFIXER_MAX_MISSING_ATOMS else "")
                            + ": reconstrução ignorada para poupar RAM; resíduos incompletos permanecem como no cristal.")
            print("⚠️ [PDBFixer] " + memory_guard)
        try:
            fixer.addMissingAtoms(seed=RANDOM_SEED)
        except TypeError:
            fixer.addMissingAtoms()

        top = fixer.topology
        pos = np.array(fixer.positions.value_in_unit(unit.angstrom), dtype=float)
        atoms = list(top.atoms())
        res_list = list(top.residues())

        def _is_h(a):
            return a.element is not None and a.element.atomic_number == 1

        metal_atoms = [a for a in atoms if a.residue.name.strip().upper() in CATALYTIC_METALS]
        metal_pos = pos[[a.index for a in metal_atoms]] if metal_atoms else np.empty((0, 3))
        heavy_idx = np.array([a.index for a in atoms if not _is_h(a)], dtype=int)

        def _near_metal(idx: int) -> bool:
            return metal_pos.shape[0] > 0 and float(np.min(np.linalg.norm(metal_pos - pos[idx], axis=1))) < METAL_COORD_CUTOFF

        variants, notes = [], []
        for res in res_list:
            v = None
            rname = res.name.strip().upper()
            by_name = {a.name: a for a in res.atoms()}
            if rname in HIS_NAMES and "ND1" in by_name and "NE2" in by_name:
                nd1_m, ne2_m = _near_metal(by_name["ND1"].index), _near_metal(by_name["NE2"].index)
                if ne2_m and not nd1_m:
                    v = "HID"          # NE2 coordena ➔ H em ND1
                    notes.append(f"HIS{res.id}: NE2–metal ➔ HID")
                elif nd1_m and not ne2_m:
                    v = "HIE"          # ND1 coordena ➔ H em NE2
                    notes.append(f"HIS{res.id}: ND1–metal ➔ HIE")
                elif nd1_m and ne2_m:
                    notes.append(f"HIS{res.id}: ND1 e NE2 próximos a metal (tautômero automático)")
            elif rname == "CYS" and "SG" in by_name:
                sg = by_name["SG"]
                sg_xyz = pos[sg.index]
                own = {a.index for a in res.atoms()}
                thioether = False
                for j in heavy_idx:
                    if j in own:
                        continue
                    if np.linalg.norm(pos[j] - sg_xyz) < 2.1:   # S–C (tioéter) ou S–S (dissulfeto)
                        thioether = True
                        break
                if thioether or _near_metal(sg.index):
                    v = "CYX"          # sem H no enxofre
                    notes.append(f"CYS{res.id}: SG ligado/coordenado ➔ CYX")
            variants.append(v)

        used_modeller = any(v is not None for v in variants)
        if used_modeller:
            modeller = Modeller(fixer.topology, fixer.positions)
            modeller.addHydrogens(pH=ph, variants=variants)
            method = f"Modeller.addHydrogens(pH={ph}, variantes explícitas para sítio metálico)"
        else:
            fixer.addMissingHydrogens(ph)
            modeller = Modeller(fixer.topology, fixer.positions)
            method = f"PDBFixer.addMissingHydrogens({ph})"

        # Limpeza de H em conflito (tioéter His–Cys, esfera de coordenação do metal)
        top2 = modeller.topology
        pos2 = np.array(modeller.positions.value_in_unit(unit.angstrom), dtype=float)
        atoms2 = list(top2.atoms())
        parent = {}
        for b in top2.bonds():
            a1, a2 = b
            if _is_h(a1) and not _is_h(a2):
                parent[a1.index] = a2.index
            elif _is_h(a2) and not _is_h(a1):
                parent[a2.index] = a1.index
        heavy2 = np.array([a.index for a in atoms2 if not _is_h(a)], dtype=int)
        metal2 = np.array([a.index for a in atoms2 if a.residue.name.strip().upper() in CATALYTIC_METALS], dtype=int)
        to_delete = []
        for a in atoms2:
            if not _is_h(a):
                continue
            d = np.linalg.norm(pos2[heavy2] - pos2[a.index], axis=1)
            mask = heavy2 != parent.get(a.index, -1)
            conflict = bool(np.any(d[mask] < 1.6))
            if not conflict and metal2.size > 0:
                conflict = bool(np.min(np.linalg.norm(pos2[metal2] - pos2[a.index], axis=1)) < 1.9)
            if conflict:
                to_delete.append(a)
        if to_delete:
            modeller.delete(to_delete)
            notes.append(f"{len(to_delete)} H removido(s) por conflito estérico/coordenação")

        tmp_pdb = out_pdb.with_suffix(".openmm.pdb")
        with open(tmp_pdb, "w") as fh:
            PDBFile.writeFile(modeller.topology, modeller.positions, fh, keepIds=True)

        # Reorganiza: ATOM (proteína) ➔ TER ➔ HETATM (metais) com numeração contínua
        prot_lines, metal_lines = [], []
        for ln in tmp_pdb.read_text().splitlines():
            if not ln.startswith(("ATOM  ", "HETATM")):
                continue
            rname = ln[17:20].strip().upper()
            if rname in CATALYTIC_METALS:
                metal_lines.append(ln)
            else:
                prot_lines.append(ln)
        out_pdb.write_text("\n".join(renumber_receptor_lines(prot_lines, metal_lines)) + "\n")

        qc = qc_receptor_pdb(out_pdb)
        qc["method"] = method
        qc["notes"] = notes
        qc["n_metal_h_variants"] = sum(1 for v in variants if v is not None)
        qc["n_missing_heavy"] = n_missing_heavy
        qc["memory_guard"] = memory_guard
        # Libera as referências pesadas (C++/OpenMM) antes de devolver o controle
        del modeller, fixer, top, pos, atoms, res_list, metal_atoms, metal_pos, heavy_idx
        del top2, pos2, atoms2, parent, heavy2, metal2, to_delete
        gc.collect()
        return qc

    def extract_receptor_and_native(self, pdb_id, chain, native_resname, keep_metals,
                                    progress=gr.Progress()):
        if not self.pdb_text:
            return None, None, status_box("err", "Nenhum PDB carregado", "Clique em <b>Buscar PDB</b> primeiro.")

        pdb_id = pdb_id.strip().upper()
        sdir = self._get_session_dir(pdb_id, chain, new=True)
        if self.source_kind in ("local", "alphafold") and self.local_source_copy and self.pdb_names.get("pdb_id") == pdb_id:
            try:                              # reprodutibilidade: o PDB enviado/baixado (AFDB) fica na pasta da sessão
                shutil.copy(self.local_source_copy, sdir / f"{pdb_id}_input_{'alphafold' if self.source_kind == 'alphafold' else 'local'}.pdb")
            except Exception:
                pass

        progress(0.20, desc=f"Isolando Cadeia {chain}" +
                 (" e preservando metais catalíticos..." if keep_metals else "..."))
        prot_lines, metal_lines, lig_lines, lig_coords = [], [], [], []
        het_groups = OrderedDict()                       # (resname, resseq) -> linhas HETATM da cadeia
        aa_numbers = set()                               # numeração dos aminoácidos (ATOM) da cadeia
        for line in self.pdb_text.splitlines():
            if line.startswith("ENDMDL"):
                break
            if len(line) < 54 or line[16] not in (" ", "A", "1") or line[21] != chain:
                continue
            clean_line = line[:16] + " " + line[17:]
            if line.startswith("ATOM  "):
                prot_lines.append(clean_line)
                try:
                    aa_numbers.add(int(line[22:26]))
                except ValueError:
                    pass
            elif line.startswith("HETATM"):
                het_groups.setdefault((line[17:20].strip(), line[22:26].strip()), []).append(clean_line)

        # Classifica cada heterogêneo: ligante nativo > íon livre > resíduo modificado > cofator (bypass) > descarte
        cofactor_lines, cofactor_names, modified_names, dropped_het = [], set(), set(), set()
        for (resname, resseq), lns in het_groups.items():
            recs = [_pdb_atom_record(l) for l in lns]
            if native_resname and resname == native_resname:
                for l, r in zip(lns, recs):
                    lig_lines.append(l)
                    lig_coords.append([float(l[30:38]), float(l[38:46]), float(l[46:54])])
            elif resname in CATALYTIC_METALS:
                if keep_metals:
                    metal_lines.extend(lns)
                else:
                    dropped_het.add(resname)
            elif resname not in SOLVENTS_AND_IONS and inchain_modified_residue(
                    resname, [r["name"] for r in recs], chain, resseq, {chain: aa_numbers}):
                prot_lines.extend(lns)                   # MSE, SEP, TPO...: o PDBFixer converte para o resíduo padrão
                modified_names.add(resname)
            elif is_bypass_cofactor(resname, [r["el"] for r in recs]):
                if keep_metals:
                    cofactor_lines.extend(lns)
                    cofactor_names.add(resname)
                else:
                    dropped_het.add(resname)
            elif resname not in SOLVENTS_AND_IONS:
                dropped_het.add(resname)                 # água/solventes comuns não são listados

        if not prot_lines:
            return None, None, status_box("err", f"Cadeia {chain} sem átomos de proteína")
        n_res_chain = pdb_has_amino_acids(prot_lines)
        if n_res_chain == 0:
            return None, None, status_box("err", f"Cadeia {chain} sem aminoácidos",
                                          "Parece ser ácido nucleico (DNA/RNA). Este pipeline prepara cadeias proteicas.")
        if n_res_chain > MAX_CHAIN_RESIDUES:
            return None, None, status_box("err", f"Cadeia {chain} com {n_res_chain} resíduos",
                                          f"Acima do limite de {MAX_CHAIN_RESIDUES}. Selecione um domínio/cadeia menor.")
        size_warn = (f"Cadeia grande ({n_res_chain} resíduos): preparo e docking podem demorar." if n_res_chain > WARN_CHAIN_RESIDUES else "")

        # BYPASS: ao PDBFixer vão a proteína, os íons livres e apenas os "íons substitutos" do metal de cada cofator
        standin_lines, standin_coords = make_metal_standins(cofactor_lines)
        raw_pdb = sdir / f"{pdb_id}_{chain}_protein_metals_raw.pdb"
        raw_pdb.write_text("\n".join(prot_lines + ["TER"] + metal_lines + standin_lines + ["END"]) + "\n")

        progress(0.55, desc="PDBFixer/OpenMM: átomos ausentes e protonação em pH 7.4...")
        rec_final_path = sdir / f"{pdb_id}_{chain}_receptor_h.pdb"
        try:
            qc = self._prepare_receptor_with_pdbfixer(raw_pdb, rec_final_path, ph=7.4)
        except Exception as e:
            self.receptor_qc = None
            return None, None, status_box("err", "Falha no preparo do receptor (PDBFixer/OpenMM)", esc(e))
        # Cola os cofatores orgânicos INTACTOS (sem H, sem parâmetros) e refaz o QC com o metal do cofator
        self.cofactor_names = sorted(cofactor_names)
        if cofactor_lines:
            try:
                merge_cofactors_into_receptor(rec_final_path, cofactor_lines, standin_coords)
                _qc2 = qc_receptor_pdb(rec_final_path)
                for _k in ("n_residues", "n_h", "n_metals", "his_states", "issues", "ok"):
                    qc[_k] = _qc2[_k]
            except Exception as e:
                return None, None, status_box("err", "Falha ao anexar os cofatores ao receptor", esc(e))
        self.receptor_qc = qc
        self.cofactor_lines_cache = list(cofactor_lines)

        # Ligante nativo: sempre extraído quando selecionado (caixa automática e RMSD de redocking)
        self.native_centroid, self.native_box_span, self.native_bbox_center = None, None, None
        native_path, native_txt = None, "Nenhum ligante nativo selecionado (use a caixa manual no Passo 3)."
        if native_resname:
            if lig_lines:
                native_path = sdir / f"{pdb_id}_{chain}_{native_resname}_native.pdb"
                native_path.write_text("\n".join(lig_lines) + "\nEND\n")
                coords_arr = np.array(lig_coords, dtype=float)
                self.native_centroid = coords_arr.mean(axis=0)
                self.native_box_span = np.ptp(coords_arr, axis=0)
                self.native_bbox_center = (coords_arr.min(axis=0) + coords_arr.max(axis=0)) / 2.0
                c = self.native_bbox_center
                native_txt = (f"Ligante nativo <b>{esc(native_resname)}</b> ({len(lig_coords)} átomos). "
                              f"Centro da caixa (bbox): ({c[0]:.3f}, {c[1]:.3f}, {c[2]:.3f}) Å.")
            else:
                native_txt = f"Ligante {esc(native_resname)} não encontrado na cadeia {esc(chain)}."

        metals_found = sorted({f"{m['res']}{m['resseq']}" for m in metal_records(rec_final_path)})
        his_msg = ", ".join(f"{k}: {v}" for k, v in sorted(qc["his_states"].items())) or "sem His"
        kind = "ok" if qc["ok"] else "warn"
        tiles = metric_tiles([
            ("Resíduos", qc["n_residues"], "info"),
            ("H adicionados (pH 7.4)", qc["n_h"], "info"),
            ("Metais retidos", qc["n_metals"], "ok" if qc["n_metals"] else "info"),
            ("Cofatores orgânicos", ", ".join(sorted(cofactor_names)) if cofactor_names else "0",
             "ok" if cofactor_names else "info"),
            ("QC da protonação", "OK" if qc["ok"] else f"{len(qc['issues'])} alerta(s)", kind),
        ])
        details = [f"<b>Metais:</b> {esc(', '.join(metals_found)) if metals_found else 'nenhum no receptor'}",
                   f"<b>Histidinas:</b> {esc(his_msg)}",
                   f"<b>Protonação:</b> {esc(qc['method'])}"]
        if qc["notes"]:
            details.append("<b>Ajustes no sítio metálico:</b> " + esc(" | ".join(qc["notes"])))
        if cofactor_names:
            details.append("<b>Cofatores retidos fora do OpenMM:</b> " + esc(", ".join(sorted(cofactor_names))) +
                           " (sem hidrogênios nem parâmetros de campo de força; o metal do cofator foi usado para definir a His/Cys axial).")
        if modified_names:
            details.append("<b>Resíduos modificados mantidos na cadeia:</b> " + esc(", ".join(sorted(modified_names))) +
                           " (convertidos para o resíduo padrão pelo PDBFixer, quando conhecido).")
        if dropped_het:
            details.append("<b>Heterogêneos descartados:</b> " + esc(", ".join(sorted(dropped_het))) +
                           (" (desmarque/marque a preservação de cofatores se algum for catalítico)." if not keep_metals else "."))
        if size_warn:
            kind = "warn"
            details.append("<b>Tamanho:</b> " + esc(size_warn))
        if qc.get("pdbfixer_peak_mb"):
            details.append(f"<b>Pico de RAM do PDBFixer (processo isolado):</b> {qc['pdbfixer_peak_mb']:.0f} MB")
        if qc.get("memory_guard"):
            kind = "warn"
            details.append("<b>Trava de memória:</b> " + esc(qc["memory_guard"]))
        pn = self.pdb_names if isinstance(self.pdb_names, dict) and self.pdb_names.get("pdb_id") == pdb_id else {"title": "", "chains": {}, "organisms": {}}
        if pn.get("title") or pn["chains"].get(chain):
            ident = f"<b>Proteína:</b> {esc(pn.get('title', ''))}"
            if pn["chains"].get(chain):
                org = pn["organisms"].get(chain, "")
                ident += f" — cadeia {esc(chain)}: {esc(pn['chains'][chain])}{(' (' + esc(org) + ')') if org else ''}"
            details.insert(0, ident)
        if qc["issues"]:
            details.append("<b>Alertas do QC:</b> " + esc(" | ".join(qc["issues"][:6])))
        details.append(native_txt)
        if self.source_kind == "alphafold" and pn.get("pdb_id") == pdb_id:      # pLDDT no resumo do Passo 1
            try:
                site0 = None
                if self.native_bbox_center is not None:                          # AFDB não traz ligante; só se o usuário forneceu um centro
                    site0 = compute_pocket_plddt(self.pdb_text, self.native_bbox_center, chain=chain)
                details.append(f"<b>Modelo AlphaFold DB:</b> UniProt {esc(pn.get('uniprot_id', ''))} (model_{esc(pn.get('af_version', ''))}).<br>"
                               + plddt_html(site0, global_plddt(self.pdb_text, chain)))
            except Exception as e_pl:
                print(f"⚠️ pLDDT do Passo 1 não calculado: {type(e_pl).__name__}: {e_pl}")
        progress(1.0, desc="Receptor preparado!")
        html_out = (status_box(kind, f"Receptor da cadeia {chain} pronto",
                               "Prossiga para o Passo 2." if qc["ok"] else "Revise os alertas abaixo antes de prosseguir.")
                    + tiles + '<div class="dd-detail">' + "<br>".join(details) + "</div>")
        return str(rec_final_path), (str(native_path) if native_path else None), html_out

    def use_native_center(self):
        """Preenche os campos manuais com o centro (bbox) do ligante nativo."""
        if self.native_bbox_center is None:
            return gr.update(), gr.update(), gr.update(), status_box("warn", "Sem ligante nativo",
                                                                    "Fixe o sítio ativo com um ligante nativo no Passo 1.")
        c = self.native_bbox_center
        return (round(float(c[0]), 3), round(float(c[1]), 3), round(float(c[2]), 3),
                status_box("ok", "Centro do ligante nativo aplicado",
                           f"({c[0]:.3f}, {c[1]:.3f}, {c[2]:.3f}) Å — ajuste o tamanho se necessário."))


    @staticmethod
    def _minimize_converged(mol, conf_id=-1, max_cycles=6):
        """Minimiza (MMFF94; UFF como contingência) e devolve (método, energia, rc, |grad|máx kcal/mol/Å)."""
        if AllChem.MMFFHasAllMoleculeParams(mol):
            props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94")
            ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=conf_id)
            method = "MMFF94"
        else:
            ff = AllChem.UFFGetMoleculeForceField(mol, confId=conf_id)
            method = "UFF (contingência: MMFF94 sem parâmetros)"
        ff.Initialize()
        rc = 1
        for _ in range(max_cycles):
            rc = ff.Minimize(maxIts=5000, forceTol=1e-5, energyTol=1e-6)
            if rc == 0:
                break
        grad = np.array(ff.CalcGrad(), dtype=float)
        gmax = float(np.max(np.abs(grad))) if grad.size else 0.0
        return method, float(ff.CalcEnergy()), rc, gmax

    def _prepare_ligand_for_docking(self, user_file_path: str, prep_mode: str, sdir: Path) -> dict:
        raw_sdf = sdir / "step0_raw_input.sdf"
        run_obabel(["obabel", user_file_path, "-O", str(raw_sdf)], "conversão do arquivo do ligante")
        suppl = [m for m in Chem.SDMolSupplier(str(raw_sdf), removeHs=False) if m is not None]
        if not suppl:
            raise ValueError("Falha ao processar arquivo molecular fornecido.")
        multi_warning = f" (Aviso: {len(suppl)} moléculas detectadas; processando a primeira)" if len(suppl) > 1 else ""
        mol0 = suppl[0]

        has_3d_input = False
        if mol0.GetNumConformers() > 0:
            conf0 = mol0.GetConformer()
            z = [conf0.GetAtomPosition(i).z for i in range(mol0.GetNumAtoms())]
            if (max(z) - min(z)) > 0.05:
                has_3d_input = True

        is_redock = prep_mode.startswith("2.")
        applied_rules = []

        # ---- 1) isolamento do fragmento + protonação inicial (OpenBabel) ----
        if is_redock:
            mol_base = mol0
        else:
            mol_desalted = rdMolStandardize.LargestFragmentChooser().choose(mol0)
            desalted_sdf = sdir / "step1_desalted_3d.sdf"
            with Chem.SDWriter(str(desalted_sdf)) as w:
                w.write(mol_desalted)
            ph74_sdf = sdir / "step2_protonated_ph74.sdf"
            run_obabel(["obabel", str(desalted_sdf), "-O", str(ph74_sdf), "-p", "7.4"], "protonação em pH 7.4")
            suppl_ph = [m for m in Chem.SDMolSupplier(str(ph74_sdf), removeHs=False) if m is not None]
            mol_base = suppl_ph[0] if suppl_ph else mol_desalted

        # ---- 2) ajuste iônico por subestrutura (grafo sem H) + reconstrução de H ----
        try:
            Chem.SanitizeMol(mol_base)
        except Exception:
            pass
        mol_heavy = Chem.RemoveHs(mol_base)
        mol_heavy, applied_rules = apply_ph74_ionization(mol_heavy)
        mol_3d = Chem.AddHs(mol_heavy, addCoords=True)

        # ---- 3) geometria 3D e minimização ----
        mmff_status_str = ""
        if is_redock:
            if not has_3d_input:
                params = AllChem.ETKDGv3()
                params.randomSeed = RANDOM_SEED
                AllChem.EmbedMolecule(mol_3d, params)
            mmff_status_str = "Geometria cristalográfica: átomos pesados preservados (sem minimização)"
            final_mol = mol_3d

        else:
            params = AllChem.ETKDGv3()
            params.randomSeed = RANDOM_SEED
            n_confs = LIG_N_CONFORMERS
            cids = list(AllChem.EmbedMultipleConfs(mol_3d, numConfs=n_confs, params=params))
            if not cids:
                params.useRandomCoords = True
                cids = list(AllChem.EmbedMultipleConfs(mol_3d, numConfs=n_confs, params=params))
            if not cids:
                raise RuntimeError("Falha ao gerar geometria 3D (ETKDGv3).")
            if AllChem.MMFFHasAllMoleculeParams(mol_3d):
                res = AllChem.MMFFOptimizeMoleculeConfs(mol_3d, maxIters=5000, mmffVariant="MMFF94")
            else:
                res = AllChem.UFFOptimizeMoleculeConfs(mol_3d, maxIters=5000)
            energies = [(e, cid) for cid, (flag, e) in zip(cids, res)]
            best_cid = min(energies)[1]
            final_mol = Chem.Mol(mol_3d)
            final_mol.RemoveAllConformers()
            final_mol.AddConformer(Chem.Conformer(mol_3d.GetConformer(best_cid)), assignId=True)
            method, e_val, rc, gmax = self._minimize_converged(final_mol)
            conv_txt = "convergiu" if rc == 0 else "NÃO convergiu"
            mmff_status_str = (f"ETKDGv3 (seed {RANDOM_SEED}, {len(cids)} confôrmeros) + {method} {conv_txt} "
                               f"(E = {e_val:.2f} kcal/mol; |∇E|máx = {gmax:.2e} kcal/mol/Å)")

        # ---- 4) cargas de Gasteiger, exportação e torções ----
        AllChem.ComputeGasteigerCharges(final_mol)
        qs = []
        for atom in final_mol.GetAtoms():
            try:
                q = float(atom.GetProp("_GasteigerCharge"))
                if np.isfinite(q):
                    qs.append(q)
            except Exception:
                pass
        gasteiger_range_str = f"[{min(qs):+.3f}, {max(qs):+.3f}] e" if qs else "n/d"

        prep_sdf = sdir / "ligand_prepared_ph74_3d.sdf"
        prep_mol2 = sdir / "ligand_prepared_gasteiger.mol2"
        prep_pdbqt = sdir / "ligand_prepared_docking.pdbqt"
        with Chem.SDWriter(str(prep_sdf)) as w:
            w.write(final_mol)
        for out_f in (prep_mol2, prep_pdbqt):
            run_obabel(["obabel", str(prep_sdf), "-O", str(out_f), "--partialcharge", "gasteiger"],
                       f"exportação {Path(out_f).suffix} com cargas de Gasteiger")

        rot_rdkit = rdMolDescriptors.CalcNumRotatableBonds(Chem.RemoveHs(final_mol))
        torsdof = rot_rdkit
        if prep_pdbqt.exists():
            for line in prep_pdbqt.read_text().splitlines():
                if line.startswith("TORSDOF"):
                    parts = line.split()
                    if len(parts) >= 2 and parts[1].isdigit():
                        torsdof = int(parts[1])
                        break

        hac = final_mol.GetNumHeavyAtoms()
        formal_charge = Chem.GetFormalCharge(final_mol)
        smiles = Chem.MolToSmiles(Chem.RemoveHs(final_mol), isomericSmiles=True)
        ion_txt = "; ".join(applied_rules) if applied_rules else "nenhuma regra aplicável"

        chem_info = (f"HAC: {hac} | Carga (pH 7.4): {formal_charge:+d} | Ionização: {ion_txt} | "
                     f"Gasteiger: {gasteiger_range_str} | TORSDOF (PDBQT): {torsdof} | {mmff_status_str}{multi_warning}")
        return {
            "sdf": str(prep_sdf), "mol2": str(prep_mol2), "pdbqt": str(prep_pdbqt), "mol": final_mol,
            "hac": hac, "formal_charge": formal_charge, "smiles": smiles, "mmff_status": mmff_status_str,
            "gasteiger_range": gasteiger_range_str, "torsdof": torsdof, "rot_rdkit": rot_rdkit,
            "ionization": ion_txt, "chem_info": chem_info,
        }

    # --------------------------------------------------------------------------
    # RMSD simétrico (spyrmsd) -- sem superposição prévia
    # --------------------------------------------------------------------------
    def _compute_symmetry_rmsd_if_applicable(self, native_pdb, docked_mols):
        n = len(docked_mols)
        if not native_pdb or not Path(native_pdb).exists() or not docked_mols:
            return ["N/A (Triagem)"] * n, "Sem referência cristalográfica (triagem prospectiva)"
        ref_raw = Chem.MolFromPDBFile(str(native_pdb), removeHs=True, sanitize=False)
        if ref_raw is None:
            return ["Erro Leitura Ref"] * n, "Falha ao ler o ligante nativo"
        pred_template = Chem.RemoveHs(docked_mols[0])
        if ref_raw.GetNumHeavyAtoms() != pred_template.GetNumHeavyAtoms():
            return ["N/A (Triagem)"] * n, "Número de átomos pesados difere do ligante nativo"
        ref_elems = Counter(a.GetAtomicNum() for a in ref_raw.GetAtoms() if a.GetAtomicNum() > 1)
        pred_elems = Counter(a.GetAtomicNum() for a in pred_template.GetAtoms())
        if ref_elems != pred_elems:
            return ["N/A (Triagem)"] * n, "Composição elementar difere do ligante nativo (RMSD não aplicável)"

        status = "Ordens de ligação do nativo atribuídas por template"
        try:
            Chem.SanitizeMol(ref_raw, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE)
            ref_assigned = AllChem.AssignBondOrdersFromTemplate(pred_template, ref_raw)
        except Exception:
            ref_assigned = ref_raw
            status = "Template de ordens de ligação falhou; usada conectividade por proximidade (grafo idêntico)"
        try:
            s_ref = spy_mol.Molecule.from_rdkit(ref_assigned)
            vals = []
            for m in docked_mols:
                s_pred = spy_mol.Molecule.from_rdkit(Chem.RemoveHs(m))
                v = spy_rmsd.symmrmsd(
                    s_ref.coordinates, s_pred.coordinates,
                    s_ref.atomicnums, s_pred.atomicnums,
                    s_ref.adjacency_matrix, s_pred.adjacency_matrix,
                    center=False, minimize=False,
                )
                vals.append(round(float(v), 3))
            return vals, status
        except Exception as e:
            return ["N/A (Grafo Distinto)"] * n, f"spyrmsd falhou: {e}"


    def _write_complex_pdb_with_conect(self, rec_path: Path, best_mol, chain: str, out_complex_path: Path):
        rec_lines, max_serial = [], 0
        for ln in rec_path.read_text().splitlines():
            if ln.startswith(("ATOM  ", "HETATM", "TER")):
                rec_lines.append(ln)
                if ln.startswith(("ATOM  ", "HETATM")):
                    try:
                        max_serial = max(max_serial, int(ln[6:11]))
                    except ValueError:
                        pass
        offset = max_serial

        lig_block = Chem.MolToPDBBlock(best_mol)
        lig_atom_lines = [l for l in lig_block.splitlines() if l.startswith(("ATOM  ", "HETATM"))]
        if len(lig_atom_lines) != best_mol.GetNumAtoms():
            raise ValueError("Inconsistência entre átomos do ligante e bloco PDB.")
        lig_lines = []
        for i, ln in enumerate(lig_atom_lines):
            lig_lines.append(f"HETATM{offset + i + 1:5d}{ln[11:17]}LIG {chain} 999{ln[26:]}")

        conect = []
        for atom in best_mol.GetAtoms():
            nbrs = sorted(n.GetIdx() for n in atom.GetNeighbors())
            for k in range(0, len(nbrs), 4):
                chunk = nbrs[k:k + 4]
                conect.append("CONECT" + f"{offset + atom.GetIdx() + 1:5d}" +
                              "".join(f"{offset + j + 1:5d}" for j in chunk))

        out_complex_path.write_text("\n".join(rec_lines + ["TER"] + lig_lines + conect + ["END"]) + "\n")

    def _run_gnina(self, cmd, tag, sdir, progress, p_start, stages, timeout=900):
        out_f, err_f = sdir / f"gnina_{tag}_stdout.log", sdir / f"gnina_{tag}_stderr.log"
        used_cmd = list(cmd)

        def _once(c):
            with open(out_f, "w") as fo, open(err_f, "w") as fe:
                proc = subprocess.Popen(c, stdout=fo, stderr=fe, env=GNINA_ENV)
                idx, cur, t0 = 0, p_start, time.time()
                while proc.poll() is None:
                    if time.time() - t0 > timeout:
                        proc.kill()
                        return -9
                    if idx < len(stages):
                        cur, msg = stages[idx]
                        progress(cur, desc=msg)
                        idx += 1
                    else:
                        cur = cur + 0.005
                        progress(min(cur, 0.7), desc="Processando na GPU...")
                    time.sleep(0.7)
                return proc.returncode

        rc = _once(used_cmd)
        if rc not in (0, -9):
            err_txt = err_f.read_text(errors="ignore") if err_f.exists() else ""
            if "pose_sort_order" in err_txt and "--pose_sort_order" in used_cmd:
                k = used_cmd.index("--pose_sort_order")
                used_cmd = used_cmd[:k] + used_cmd[k + 2:]
                rc = _once(used_cmd)
        return rc, used_cmd, out_f, err_f

    # --------------------------------------------------------------------------
    # POSEBUSTERS (veredito = testes oficiais do modo dock)
    # --------------------------------------------------------------------------
    def _run_posebusters(self, poses_sdf: Path, rec_path: Path, lig_sdf: str):
        buster = PoseBusters(config="dock")

        def _bust(full: bool):
            try:
                return buster.bust(mol_pred=str(poses_sdf), mol_true=None, mol_cond=str(rec_path), full_report=full)
            except Exception:
                return buster.bust(mol_pred=str(poses_sdf), mol_true=lig_sdf, mol_cond=str(rec_path), full_report=full)

        df_full = _bust(True).reset_index(drop=True)
        try:
            short_cols = list(_bust(False).columns)
        except Exception:
            short_cols = None
        official, diagnostic, source = pb_select_official_columns(df_full, short_cols)
        status, failed, na = pb_evaluate(df_full, official)

        rest = [c for c in df_full.columns if c not in official and c not in diagnostic]
        export = df_full[official + diagnostic + rest].copy()
        export.insert(0, "Pose_Rank", [f"Pose_{i + 1}" for i in range(len(export))])
        export.insert(1, "PB_Valid_Status", status)
        export.insert(2, "PB_Testes_Oficiais_Avaliados", [len(official)] * len(export))
        export.insert(3, "PB_Testes_Oficiais_Falhos", failed)
        export.insert(4, "PB_Testes_NaN_nao_aplicaveis", na)
        return export, status, failed, official, diagnostic, source

    # --------------------------------------------------------------------------
    # PROLIF (receptor com metais em ramos robustos e blindagem de falhas)
    # --------------------------------------------------------------------------
    def _select_pocket(self, u, poses_sdf=None, center=None):
        """Resíduos da proteína dentro de POCKET_RADIUS_A (12 Å) do ligante (todas as poses); se vier vazio, esfera de 16 Å em torno do
        centro da grid box. NUNCA devolve a proteína inteira (a conversão de ~4.000+ átomos com force=True gerava AtomPDBResidueInfo
        corrompido e SIGSEGV em SplitMolByPDBResidues). Retorna (AtomGroup, descrição)."""
        prot = u.select_atoms("protein")
        if poses_sdf is not None:
            try:
                xyz = [p["xyz"] for p in read_pose_geometries(poses_sdf) if p is not None]
                idx = pocket_resindices(prot.positions, prot.resindices, xyz, POCKET_RADIUS_A)
                if len(idx):
                    ag = prot.residues[np.isin(prot.residues.resindices, idx)].atoms
                    return ag, f"bolsão de {POCKET_RADIUS_A:.0f} Å em torno do ligante ({len(idx)} resíduos)"
            except Exception as e:
                print(f"⚠️ Bolsão por ligante falhou ({type(e).__name__}: {e}); usando esfera da grid box.")
        if center is not None:
            cx, cy, cz = (float(v) for v in center)
            ag = u.select_atoms(f"same residue as (protein and point {cx:.3f} {cy:.3f} {cz:.3f} {POCKET_FALLBACK_RADIUS_A})")
            if len(ag):
                return ag, f"esfera de {POCKET_FALLBACK_RADIUS_A:.0f} Å na grid box ({len(ag.residues)} resíduos)"
        raise RuntimeError("bolsão de ligação vazio (sem poses legíveis nem centro de caixa)")

    def _build_prolif_receptor(self, rec_path: Path, allow_metals: bool = True, poses_sdf=None, center=None):
        """Retorna (molécula ProLIF do BOLSÃO, descrição do ramo, nº de metais no objeto, lista de erros)."""
        u = mda.Universe(str(rec_path))
        metal_sel = "resname " + " ".join(sorted(CATALYTIC_METALS))
        n_metal_atoms = len(metal_records(rec_path)) if allow_metals else 0   # inclui o FE do HEM
        errors = []
        try:
            pocket, pdesc = self._select_pocket(u, poses_sdf, center)
        except Exception as e:
            errors.append(f"P: {type(e).__name__}: {e}")
            return None, "FALHA NA SELEÇÃO DO BOLSÃO", 0, errors

        # RAMO A (preferencial): bolsão via MDAnalysis + metais como átomos isolados via RDKit
        try:
            prot_plf = plf.Molecule.from_mda(pocket, implicit_hydrogens=False, force=True)
            if n_metal_atoms:
                combined = Chem.CombineMols(Chem.Mol(prot_plf), build_metal_rdmol(rec_path))
                prot_plf = plf.Molecule(combined)
                n_in = count_metal_atoms(prot_plf)
                if n_in != n_metal_atoms:
                    raise RuntimeError(f"metais no objeto ProLIF ({n_in}) != metais no receptor ({n_metal_atoms})")
                return prot_plf, f"RAMO A: {pdesc} + metais injetados como átomos isolados (RDKit)", n_in, errors
            return prot_plf, f"RAMO A: {pdesc} (receptor sem metais)", 0, errors
        except Exception as e:
            errors.append(f"A: {type(e).__name__}: {e}")

        # RAMO B: bolsão + íons metálicos por seleção literal no MDAnalysis
        if n_metal_atoms:
            try:
                ag = pocket | u.select_atoms(metal_sel)
                prot_plf = plf.Molecule.from_mda(ag, implicit_hydrogens=False, force=True)
                n_in = count_metal_atoms(prot_plf)
                if n_in:
                    return prot_plf, f"RAMO B: {pdesc} + seleção MDAnalysis 'resname <metais>'", n_in, errors
                errors.append("B: metais ausentes no objeto convertido")
            except Exception as e:
                errors.append(f"B: {type(e).__name__}: {e}")

        # RAMO C: somente o bolsão (metais excluídos do ProLIF)
        try:
            prot_plf = plf.Molecule.from_mda(pocket, implicit_hydrogens=False, force=True)
            return prot_plf, f"FALLBACK C: {pdesc}, sem metais (ver contatos geométricos)", 0, errors
        except Exception as e:
            errors.append(f"C: {type(e).__name__}: {e}")

        # RAMO D: heavy atoms do bolsão com H implícitos
        try:
            prot_plf = plf.Molecule.from_mda(pocket.select_atoms("not name H*"), force=True)
            return prot_plf, f"FALLBACK D: {pdesc}, heavy atoms com H implícitos (protonação NÃO preservada)", 0, errors
        except Exception as e:
            errors.append(f"D: {type(e).__name__}: {e}")

        return None, "FALHA TOTAL DE CONVERSÃO NO PROLIF", 0, errors

    def _run_prolif(self, rec_path: Path, poses_sdf: Path, n_poses: int, center=None):
        base = ["HBDonor", "HBAcceptor", "PiStacking", "PiCation", "CationPi", "Hydrophobic", "Cationic", "Anionic"]
        prot_plf, branch, n_metals, errors = self._build_prolif_receptor(rec_path, True, poses_sdf, center)

        def _fp(prot, with_metal):
            if prot is None:
                raise RuntimeError("Objeto receptor nulo.")
            inter = base + (["MetalAcceptor"] if with_metal else [])
            poses = list(plf.sdf_supplier(str(poses_sdf)))
            fp = plf.Fingerprint(inter)
            fp.run_from_iterable(poses, prot, progress=False)
            return fp.to_dataframe(), inter

        try:
            df_raw, interactions = _fp(prot_plf, n_metals > 0)
        except Exception as e:
            errors.append(f"FP: {type(e).__name__}: {e}")
            prot_plf, branch, n_metals, e2 = self._build_prolif_receptor(rec_path, False, poses_sdf, center)
            errors += e2
            branch = "FALLBACK pós-erro no fingerprint: " + branch
            try:
                # O segundo try garante que o app não quebre se o ligante em si for problemático
                df_raw, interactions = _fp(prot_plf, False)
            except Exception as e3:
                errors.append(f"FP_Fallback: {type(e3).__name__}: {e3}")
                df_raw = pd.DataFrame()
                interactions = base

        if not df_raw.empty and len(df_raw) == n_poses:
            df = df_raw.copy()
            df.columns = [f"{c[1]}_{c[2]}" for c in df_raw.columns]
            df.insert(0, "Pose_Rank", [f"Pose_{i + 1}" for i in range(len(df))])
        else:                                    # ProLIF sem resultado utilizável: contatos geométricos (nunca deixa 'n/d')
            try:
                df = geometric_interactions_df(rec_path, poses_sdf, n_poses)
                branch = GEO_BRANCH + (f" ({'; '.join(errors)[:300]})" if errors else "")
                interactions, n_metals = ["HB", "PiPi", "Hf", "Metal"], 0
            except Exception as e_geo:
                errors.append(f"GEO: {type(e_geo).__name__}: {e_geo}")
                df = pd.DataFrame({
                    "Pose_Rank": [f"Pose_{i + 1}" for i in range(n_poses)],
                    "Status_Contatos": ["Nenhum contato detectado (ou falha interna no mapeamento do ProLIF)"] * n_poses})
        prot_plf, df_raw = None, None
        _free_memory()
        return df, branch, interactions, n_metals, errors


    # --------------------------------------------------------------------------
    # PIPELINE COMPLETO
    # --------------------------------------------------------------------------
    def _fail(self, title, detail="", tb=None):
        body = esc(detail)
        if tb:
            body += f'<pre class="dd-pre">{esc(tb[-1500:])}</pre>'
        return None, "", status_box("err", title, body), None, gr.update()

    def run_docking(self, pdb_id, chain, box_mode, native_pdb, cx, cy, cz, sx, sy, sz,
                    user_ligand_file, pubchem_query, lig_info, prep_mode, exhaustiveness,
                    auto_native=True, target_domain="", domain_function="", progress=gr.Progress()):
        ram0 = _rss_mb()
        try:                                                    # anotação funcional lida no clique (nunca fica desatualizada)
            self.target_annotation = {"target_domain": str(target_domain or "").strip(),
                                      "domain_function": str(domain_function or "").strip()}
        except Exception:
            self.target_annotation = {"target_domain": "", "domain_function": ""}
        try:
            return self._run_batch(pdb_id, chain, box_mode, native_pdb, cx, cy, cz, sx, sy, sz,
                                   user_ligand_file, pubchem_query, lig_info, prep_mode,
                                   exhaustiveness, progress, auto_native)
        except Exception as e:
            import traceback
            return self._fail("Erro inesperado no pipeline", f"{type(e).__name__}: {e}", traceback.format_exc())
        finally:
            # === COLETA DE LIXO FORÇADA PÓS-DOCKING (LIBERA A RAM E GPU), COM OU SEM ERRO ===
            ram1 = _rss_mb()
            _free_memory()
            print(f"[memória] RAM do kernel: {ram0:.0f} MB (início) → {ram1:.0f} MB (fim da corrida) → {_rss_mb():.0f} MB (após limpeza)")

    def _run_batch(self, pdb_id, chain, box_mode, native_pdb, cx, cy, cz, sx, sy, sz,
                   user_ligand_file, pubchem_query, lig_info, prep_mode, exhaustiveness, progress, auto_native=False):
        """Fila sequencial: receptor e caixa fixos; cada ligante passa inteiro por _run_docking_core (preparo, GNINA,
        refino, PoseBusters, ProLIF isolado) e _free_memory() roda depois de CADA um, com sucesso ou erro."""
        import traceback
        t0 = time.time()
        items, qnotes, qerr = self._build_queue(user_ligand_file, pubchem_query, lig_info, progress, native_pdb, auto_native, prep_mode)
        if qerr:
            return self._fail(*qerr)
        n, results = len(items), []
        if any(it["is_native"] for it in items):
            self.native_benchmark = None                         # o controle desta corrida substitui qualquer anterior
        self._lock_control = False
        for i, it in enumerate(items):
            def sub(frac=0.0, desc="", _i=i, _l=it["label"]):
                progress((_i + max(0.0, min(1.0, float(frac)))) / n, desc=(f"[{_i + 1}/{n}] {_l}: {desc}" if n > 1 else desc))
            self._last_result, ui, err = None, None, None
            try:
                ui = self._run_docking_core(pdb_id, chain, box_mode, native_pdb, cx, cy, cz, sx, sy, sz,
                                            it["path"], "", it["info"], it["prep"] or prep_mode, exhaustiveness, sub)
            except Exception as e:
                err = f"{type(e).__name__}: {e}"
                print(f"⚠️ Ligante {it['label']} falhou: {err}\n{traceback.format_exc()[-1200:]}")
            finally:
                _free_memory()                                   # mandatório após cada ligante
            r, self._last_result = self._last_result, None
            if r is not None and err is None:
                r["ok"], r["ui"], r["is_native"], r["prep_item"] = True, ui, it["is_native"], it["prep"]
                if r.get("bm_rec") is None:                      # benchmark indisponível: registro mínimo só para a tabela
                    r["bm_rec"] = bm_make_record(None, r["sel"], r["cnn_var"], r["hac"], r["lig_name"], r["selected_idx"] + 1, r["rmsd_sel"])
                r["rec"] = r["bm_rec"]
            else:
                reason = err or (_strip_tags(ui[2]) if ui and len(ui) > 2 and ui[2] else "falha sem detalhe")
                r = {"ok": False, "name": it["label"], "reason": reason, "ui": ui, "is_native": it["is_native"]}
            results.append(r)
            if it["is_native"]:
                if not r["ok"]:
                    qnotes.append(f"Controle nativo NÃO processado ({str(r['reason'])[:160]}): os Δ dos testes ficam n/d.")
                elif r["bm"].get("mode") == "control_saved":
                    self._lock_control = True                    # um teste idêntico ao nativo não substitui o controle modo 2
                else:
                    qnotes.append("Controle nativo processado mas NÃO validado: " + str(r.get("rmsd_status", "RMSD simétrico indisponível")) + ". Os Δ dos testes ficam n/d.")
        self._lock_control = False
        ctrl = self.native_benchmark
        if n == 1 and not results[0]["ok"]:                      # 1 molécula com falha: devolve a tela de erro do próprio ligante
            if results[0].get("ui") is not None:
                return results[0]["ui"]
            return self._fail("Erro inesperado no pipeline", str(results[0]["reason"]))
        sdir = self._get_session_dir(pdb_id.strip().upper(), chain)
        n_ok = sum(1 for r in results if r["ok"])
        batch_meta = {"Versao_Pipeline": PIPELINE_VERSION, "Ligantes_na_fila": n, "Ligantes_concluidos": n_ok,
                      "Ligantes_com_falha": n - n_ok, "Fonte_Estrutura_Alvo": SOURCE_LABELS.get(getattr(self, "source_kind", "rcsb"), "n/d"), "Tempo_total_s": round(time.time() - t0, 1),
                      "Controle_nativo": (f"{ctrl['ctx']['resname']} (Vina {ctrl['vina']:.2f}; pK {ctrl['pk']:.2f})" if ctrl else BM_NO_CONTROL),
                      "Modo_preparo_controle": PREP_MODES[1] if any(it["is_native"] for it in items) else "— (sem controle automático)",
                      "Modo_preparo_testes": str(prep_mode), "Semente": RANDOM_SEED, "Exaustividade": int(exhaustiveness),
                      "GPU_PyTorch": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"),
                      "Dispositivo_GNINA": next((r["gnina_dev"] for r in results if r["ok"]), "n/d"),
                      "Ordenacao_Leaderboard": "PB_Valid=PASS primeiro; depois Vina (mais negativo) e CNNaffinity",
                      "Avisos_da_fila": " | ".join(qnotes) or "—"}
        clean_pdb = re.sub(r"[^A-Za-z0-9]", "", str(pdb_id)) or "PDB"
        clean_chain = re.sub(r"[^A-Za-z0-9]", "", str(chain)) or "X"
        timestamp_str = time.strftime("%Y%m%d_%H%M%S")
        xlsx_filename = f"{clean_pdb}_{clean_chain}_LEADERBOARD_{VERSION_TAG}_{timestamp_str}.xlsx"
        zip_filename = f"{clean_pdb}_{clean_chain}_Pacote_Completo_{VERSION_TAG}_{timestamp_str}.zip"
        xlsx = sdir / xlsx_filename
        xlsx_err = ""
        try:
            if n_ok:
                write_leaderboard_xlsx(xlsx, results, ctrl, batch_meta)
            else:
                xlsx = None
        except Exception as e:
            xlsx, xlsx_err = None, f"{type(e).__name__}: {e}"
            print(f"⚠️ {xlsx_filename} não gerado: {xlsx_err}")
        figs = []
        if n_ok:
            try:
                figs = gerar_graficos_publicacao(results, ctrl, sdir, f"{clean_pdb}_{clean_chain}_{timestamp_str}")
            except Exception as e_fig:
                print(f"⚠️ Gráficos de publicação não gerados: {type(e_fig).__name__}: {e_fig}")
        rep_out = {"files": [], "html": "", "error": ""}                  # relatório acadêmico: Metodologia + Resultados/Discussão
        if n_ok:
            rep_out = gerar_relatorio_academico(self, results, ctrl, sdir, f"{clean_pdb}_{clean_chain}_{timestamp_str}",
                                                pdb_id, chain, exhaustiveness, qnotes)
            if rep_out["error"]:
                qnotes.append("Relatório acadêmico não gerado: " + rep_out["error"][:160])
            for w_ in rep_out.get("warnings", []):                          # .docx/.pdf que falharam (o resto do relatório existe)
                qnotes.append("Relatório acadêmico: " + w_)
        md_root, md_leader = None, ""
        if n_ok:                                                          # MD turnkey da pose líder (1º teste do ranking; PASS primeiro)
            try:
                _rk = [it for it in _ranked(results, ctrl) if it[0].get("md")]
                _lead = next((it[0] for it in _rk if not is_ctrl(it[1])), None) or (_rk[0][0] if _rk else None)
                if _lead:
                    md_root = stage_md_folder(sdir / MD_DIRNAME, _lead["md"]["complex"], ligand_sdf_src=_lead["md"]["ligand_sdf"] or None)
                    md_leader = _lead["lig_name"]
            except Exception as e_md:
                print(f"⚠️ Pasta {MD_DIRNAME}/ do lote não gerada: {type(e_md).__name__}: {e_md}")
        md_html = md_notice_html(md_root, md_leader)
        md_files = list(md_root["files"]) if md_root else []
        if n == 1:
            ui = results[0]["ui"]
            _ar_append_to_zip((ui[3] or [None])[0], rep_out["files"])            # o .zip do núcleo (1 ligante) também leva 07_relatorio_academico/
            files = list(ui[3] or []) + ([str(xlsx)] if xlsx else []) + [str(f) for f in figs] + rep_out["files"] + md_files[:1]
            return (ui[0], ui[1] + parecer_html(results[0], ctrl) + rep_out["html"], ui[2], files, ui[4])
        files = []
        if n_ok:
            zip_path = sdir / zip_filename
            try:
                build_batch_zip(zip_path, results, xlsx, figs, rep_out["files"], md_files)
                files.append(str(zip_path))
            except Exception as e:
                print(f"⚠️ {zip_filename} não gerado: {type(e).__name__}: {e}")
        files += rep_out["files"] + md_files[:1]
        if xlsx:
            files.insert(0, str(xlsx))
        gpu_ok = all(str(r.get("gnina_dev", "")).startswith("GPU") for r in results if r["ok"])
        kind = "ok" if (n_ok == n and gpu_ok and not xlsx_err and not rep_out["error"] and not rep_out.get("warnings")) else ("err" if n_ok == 0 else "warn")
        status = status_box(kind, "Lote finalizado",
                            f"{n_ok} de {n} ligantes concluídos em {(time.time() - t0) / 60:.1f} min. "
                            + ("" if n_ok == n else f"<b>{n - n_ok} falharam</b> (ver Passo 4). ")
                            + ("" if not xlsx_err else f"<b>Planilha consolidada não gerada:</b> {esc(xlsx_err)}. ")
                            + ("" if not qnotes else esc(" | ".join(qnotes)) + ". ")
                            + f"RAM do kernel: {_rss_mb():.0f} MB.")
        lb = build_leaderboard(results, ctrl) if n_ok else pd.DataFrame(
            [{"Composto": r["name"], "Papel": "Falha", "Classe vs. nativo": "FALHA: " + str(r["reason"])[:90]} for r in results])
        return (lb, leaderboard_html(results, ctrl) + md_html + rep_out["html"], status, files or None, gr.update(selected="t4"))

    def _run_docking_core(self, pdb_id, chain, box_mode, native_pdb, cx, cy, cz, sx, sy, sz,
                          user_ligand_file, pubchem_query, lig_info, prep_mode, exhaustiveness, progress):
        direct_lig_info = None
        lig_path_str = None
        if user_ligand_file is not None:
            lig_path_str = user_ligand_file.name if hasattr(user_ligand_file, "name") else str(user_ligand_file)
        elif pubchem_query and pubchem_query.strip():
            fetched_path, fetch_msg, direct_lig_info = self._pubchem_download(pubchem_query, progress)
            if not fetched_path:
                return self._fail("Ligante não obtido", fetch_msg)
            lig_path_str = fetched_path
        else:
            return self._fail("Nenhum ligante informado",
                              "Envie um arquivo (.sdf, .mol2, .pdb, .pdbqt) ou baixe um composto do PubChem no Passo 2.")

        # Identificação amigável do ligante (PubChem > arquivo) e da proteína (cabeçalho do PDB)
        info = direct_lig_info
        if info is None and isinstance(lig_info, dict) and lig_info.get("file") \
                and Path(str(lig_info["file"])).name == Path(lig_path_str).name:
            info = lig_info
        if info:
            lig_name = info["name"]
            lig_sub = " · ".join(x for x in [(f"CID {info['cid']}" if str(info['cid']).isdigit() else str(info['cid'])), info["formula"],
                                             f"IUPAC: {info['iupac'][:80]}" if info["iupac"] not in ("", "N/A") and info["iupac"] != info["name"] else ""] if x)
            lig_source = str(info.get("source", "PubChem"))
        else:
            lig_name, lig_sub, lig_source = Path(lig_path_str).stem, "arquivo local (nome do arquivo)", "nome do arquivo"

        pdb_id = pdb_id.strip().upper()
        p_names = {"title": "", "chains": {}, "organisms": {}}
        if hasattr(self, "pdb_names") and isinstance(self.pdb_names, dict):
            if self.pdb_names.get("pdb_id") == pdb_id:
                p_names["title"] = str(self.pdb_names.get("title", ""))
                p_names["chains"] = dict(self.pdb_names.get("chains", {})) if isinstance(self.pdb_names.get("chains"), dict) else {}
                p_names["organisms"] = dict(self.pdb_names.get("organisms", {})) if isinstance(self.pdb_names.get("organisms"), dict) else {}
        sdir = self._get_session_dir(pdb_id, chain)
        if Path(lig_path_str).suffix.lower() not in [".sdf", ".mol2", ".pdb", ".pdbqt"]:
            return self._fail("Formato não suportado", "Utilize SDF, MOL2, PDB ou PDBQT.")

        rec_path = sdir / f"{pdb_id}_{chain}_receptor_h.pdb"
        if not rec_path.exists():
            return self._fail("Receptor não preparado", "Volte ao Passo 1 e clique em 'Fixar sítio ativo'.")
        rec_qc = qc_receptor_pdb(rec_path)
        metals = metal_records(rec_path)

        # --- 1. LIGANTE ---
        progress(0.08, desc="Preparando ligante (pH 7.4, ionização, ETKDGv3, MMFF94, Gasteiger, torções)...")
        try:
            lig = self._prepare_ligand_for_docking(lig_path_str, prep_mode, sdir)
        except Exception as e:
            return self._fail("Erro no preparo químico do ligante", str(e))
        prep_mol, hac = lig["mol"], lig["hac"]
        try:
            perfil = classificar_perfil_ligante(prep_mol)
        except Exception as e_perfil:
            print(f"⚠️ Perfil químico indisponível: {type(e_perfil).__name__}: {e_perfil}")
            perfil = None
        _blk, lig_safety = ligand_safety_report(prep_mol, hac, lig.get("torsdof"))
        if _blk:
            return self._fail("Ligante recusado pela checagem de sanidade", _blk)

        conf = prep_mol.GetConformer()
        pts = np.array([list(conf.GetAtomPosition(i)) for i in range(prep_mol.GetNumAtoms())], dtype=float)
        max_lig_diameter = float(np.max(np.linalg.norm(pts[:, None, :] - pts[None, :, :], axis=-1)))

        # --- 2. CAIXA EXPLÍCITA ---
        if box_mode == BOX_AUTO:
            if not native_pdb:
                return self._fail("Ligante nativo ausente",
                                  "A caixa automática exige um ligante nativo. Fixe o sítio no Passo 1 ou use a caixa manual.")
            center, size, centroid, span = compute_autobox(native_pdb, max_lig_diameter, AUTOBOX_ADD)
            box_meta_str = (f"Explícita a partir do ligante nativo (bbox + {AUTOBOX_ADD} Å por lado; arestas ≥ diâmetro "
                            f"do ligante {max_lig_diameter:.2f} Å) | Centro: ({center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f}) Å | "
                            f"Tamanho: ({size[0]:.2f}, {size[1]:.2f}, {size[2]:.2f}) Å")
        else:
            try:
                center = np.array([cx, cy, cz], dtype=float)
                size = np.array([sx, sy, sz], dtype=float)
            except Exception:
                return self._fail("Coordenadas manuais inválidas", "Preencha centro e tamanho da caixa.")
            if np.any(size <= 0):
                return self._fail("Tamanho de caixa inválido", "Os tamanhos devem ser positivos (Å).")
            if np.any(size > MAX_MANUAL_BOX_EDGE):
                return self._fail("Caixa grande demais",
                                  f"Nenhuma aresta deve passar de {MAX_MANUAL_BOX_EDGE:.0f} Å (informado: {size.tolist()}). "
                                  "Caixas grandes diluem a amostragem e podem estourar o tempo limite.")
            box_meta_str = f"Manual | Centro: ({cx}, {cy}, {cz}) Å | Tamanho: ({sx}, {sy}, {sz}) Å"
        box_args = [
            "--center_x", f"{center[0]:.3f}", "--center_y", f"{center[1]:.3f}", "--center_z", f"{center[2]:.3f}",
            "--size_x", f"{size[0]:.3f}", "--size_y", f"{size[1]:.3f}", "--size_z", f"{size[2]:.3f}",
        ]
        gpu_args = ["--no_gpu"] if self.no_gpu else []

        # --- pLDDT do sítio (somente modelos AlphaFold): resíduos a < 10 Å do centro EFETIVO da caixa, sobre o PDB original
        plddt_site, af_rows = None, []
        if self.source_kind == "alphafold" and isinstance(self.pdb_names, dict) and self.pdb_names.get("pdb_id") == pdb_id:
            try:
                plddt_site = compute_pocket_plddt(self.pdb_text, center, chain=chain, radius=PLDDT_POCKET_RADIUS)
            except Exception as e_pl:
                print(f"⚠️ pLDDT do sítio não calculado: {type(e_pl).__name__}: {e_pl}")
            pq = plddt_site or {}
            af_rows = [("AF_UniProt", self.pdb_names.get("uniprot_id", "n/d")), ("AF_Versao_Modelo", self.pdb_names.get("af_version", "n/d")),
                       ("AF_URL_Modelo", self.pdb_names.get("af_url", "n/d")),
                       ("AF_pLDDT_Medio_Bolsao", pq.get("mean", "n/d")), ("AF_pLDDT_Minimo_Bolsao", pq.get("min", "n/d")),
                       ("AF_pLDDT_Maximo_Bolsao", pq.get("max", "n/d")), ("AF_N_Residuos_Bolsao", pq.get("n_residues", "n/d")),
                       ("AF_N_Atomos_Pesados_Bolsao", pq.get("n_atoms", "n/d")), ("AF_Raio_Bolsao (Å)", PLDDT_POCKET_RADIUS),
                       ("AF_Classe_Confiabilidade_Bolsao", pq.get("classe", "n/d")),
                       ("AF_pLDDT_Fonte", "coluna do fator B do PDB original do AFDB (átomos pesados dos resíduos a < raio do centro da caixa)")]

        # --- 3. DOCKING (GNINA) ---
        _mem_log("antes do docking GNINA")
        out_raw_sdf = sdir / "gnina_raw_poses.sdf"
        out_refined_sdf = sdir / "gnina_refined_poses.sdf"
        out_sdf = sdir / "docking_poses.sdf"
        for f in (out_raw_sdf, out_refined_sdf, out_sdf):
            if f.exists():
                f.unlink()

        rec_gnina = rec_path
        cof_in_rec = sorted({r["res"] for r in _read_pdb_records(rec_path)
                             if r["rec"] == "HETATM" and r["res"].upper() not in CATALYTIC_METALS})
        gnina_cofactor_fallback, gnina_first_err = "", ""
        cmd_dock = [
            GNINA_BIN, "--receptor", str(rec_gnina), "--ligand", lig["sdf"],
            "--exhaustiveness", str(int(exhaustiveness)), "--num_modes", str(GNINA_DOCK_PARAMS["num_modes"]),
            "--min_rmsd_filter", str(GNINA_DOCK_PARAMS["min_rmsd_filter"]), "--num_mc_saved", str(GNINA_DOCK_PARAMS["num_mc_saved"]),
            "--cnn_scoring", GNINA_DOCK_PARAMS["cnn_scoring"], "--pose_sort_order", GNINA_DOCK_PARAMS["pose_sort_order"],
            "--seed", str(RANDOM_SEED),
        ] + box_args + gpu_args + ["--out", str(out_raw_sdf)]

        progress(0.22, desc="GNINA: amostragem Monte Carlo + rescoring 3D-CNN...")
        rc, used_dock, dock_out, dock_err = self._run_gnina(
            cmd_dock, "docking", sdir, progress, 0.26,
            [(0.34, "Amostrando poses via Monte Carlo..."),
             (0.44, "Otimizando complexos ligante-receptor..."),
             (0.52, "Processando tensores 3D no ensemble 3D-CNN...")], timeout=_gnina_dock_timeout(exhaustiveness))
        if rc not in (0, -9) and cof_in_rec:
            # O cofator orgânico (sem H, tipos inferidos pelo OpenBabel) pode ser recusado pelo GNINA: 2ª tentativa sem ele.
            gnina_first_err = dock_err.read_text(errors="ignore")[:400] if dock_err.exists() else ""
            rec_gnina = sdir / f"{pdb_id}_{chain}_receptor_sem_cofatores.pdb"
            strip_cofactors_from_pdb(rec_path, rec_gnina, cof_in_rec)
            cmd_dock = [str(rec_gnina) if x == str(rec_path) else x for x in cmd_dock]
            gnina_cofactor_fallback = (f"GNINA falhou com {', '.join(cof_in_rec)} no receptor; repetido SEM o cofator "
                                       "(as poses foram geradas sem ele e podem sobrepô-lo)")
            print("⚠️ " + gnina_cofactor_fallback)
            progress(0.30, desc="GNINA recusou o cofator; repetindo sem o cofator orgânico...")
            rc, used_dock, dock_out, dock_err = self._run_gnina(
                cmd_dock, "docking", sdir, progress, 0.32,
                [(0.40, "Amostrando poses via Monte Carlo (sem cofator)..."),
                 (0.52, "Processando tensores 3D no ensemble 3D-CNN...")], timeout=_gnina_dock_timeout(exhaustiveness))
        if rc == -9:
            return self._fail("Tempo limite excedido", f"O GNINA ultrapassou {_gnina_dock_timeout(exhaustiveness)} s na etapa de docking.")
        if rc != 0 or not out_raw_sdf.exists():
            return self._fail("Erro no GNINA (docking)", "", dock_err.read_text(errors="ignore") if dock_err.exists() else "")

        # --- 4. REFINO NO CAMPO DO RECEPTOR ---
        cmd_ref = [
            GNINA_BIN, "--receptor", str(rec_gnina), "--ligand", str(out_raw_sdf),
            "--minimize", "--cnn_scoring", "rescore", "--seed", str(RANDOM_SEED),
        ] + box_args + gpu_args + ["--out", str(out_refined_sdf)]
        progress(0.56, desc="GNINA: minimização das poses no sítio ativo (receptor + metais)...")
        rc2, used_ref, ref_out, ref_err = self._run_gnina(
            cmd_ref, "refino", sdir, progress, 0.58,
            [(0.60, "Minimizando poses no campo do receptor..."),
             (0.64, "Reavaliando CNNscore/CNNaffinity das poses finais...")], timeout=600)
        refine_ok = (rc2 == 0 and out_refined_sdf.exists())
        if refine_ok:
            pose_source, refine_note = out_refined_sdf, "Poses refinadas com gnina --minimize no campo do receptor"
        else:
            pose_source = out_raw_sdf
            refine_note = ("Refino --minimize falhou; usadas poses brutas do docking "
                           f"({(ref_err.read_text(errors='ignore')[:200] if ref_err.exists() else '')})")

        # --- 5. LEITURA, COMPLETAR H (sem minimização) E REORDENAÇÃO ---
        _mem_log("após o docking e refino")
        raw_suppl = [m for m in Chem.SDMolSupplier(str(pose_source), removeHs=False) if m is not None]
        loaded = []
        for i, m in enumerate(raw_suppl):
            m_h = Chem.AddHs(m, addCoords=True)
            for pn in m.GetPropNames():
                m_h.SetProp(pn, m.GetProp(pn))
            m_h.SetIntProp("Rank_Amostragem", i + 1)
            loaded.append(m_h)
        if not loaded:
            return self._fail("Nenhuma conformação válida", "O GNINA não gerou poses legíveis.")
        order = sorted(range(len(loaded)), key=lambda k: -float(loaded[k].GetPropsAsDict().get("CNNscore", 0.0)))
        valid_mols = [loaded[k] for k in order]
        with Chem.SDWriter(str(out_sdf)) as w:
            for m in valid_mols:
                w.write(m)

        # --- 6. RMSD SIMÉTRICO (redocking, quando há nativo compatível) ---
        progress(0.74, desc="RMSD simétrico sem superposição prévia (spyrmsd)...")
        rmsd_list, rmsd_status = self._compute_symmetry_rmsd_if_applicable(native_pdb, valid_mols)

        # --- 7. POSEBUSTERS (testes oficiais) ---
        progress(0.80, desc="Auditoria físico-química (PoseBusters, modo dock)...")
        _mem_log("antes do PoseBusters")
        pb_receptor_note, pb_cof_fallback = "receptor completo (com cofatores)" if cof_in_rec else "receptor completo", False
        try:
            _pb_res, _pb_peak = run_isolated(self._run_posebusters, (out_sdf, rec_path, lig["sdf"]),
                                             timeout_s=900, label="PoseBusters")
        except Exception as e_pb:
            if not cof_in_rec:
                raise
            rec_pb = sdir / f"{pdb_id}_{chain}_receptor_sem_cofatores.pdb"
            if not rec_pb.exists():
                strip_cofactors_from_pdb(rec_path, rec_pb, cof_in_rec)
            print(f"⚠️ PoseBusters falhou com o cofator no receptor ({e_pb}); repetindo sem ele.")
            _pb_res, _pb_peak = run_isolated(self._run_posebusters, (out_sdf, rec_pb, lig["sdf"]),
                                             timeout_s=900, label="PoseBusters (sem cofator)")
            pb_receptor_note = (f"PoseBusters falhou com {', '.join(cof_in_rec)} no receptor ({str(e_pb)[:160]}); "
                                "repetido sem o cofator: choques com ele NÃO foram avaliados")
            pb_cof_fallback = True
        pb_df_export, pb_valid_list, pb_failed, pb_official, pb_diag, pb_source = _pb_res

        # --- 8. GEOMETRIA DE COORDENAÇÃO METÁLICA (independente do ProLIF) ---
        metal_dmin, metal_txt = [], []
        for m in valid_mols:
            xyz = m.GetConformer().GetPositions()
            syms = [a.GetSymbol() for a in m.GetAtoms()]
            d, t = metal_contacts(xyz, syms, metals)
            metal_dmin.append(d)
            metal_txt.append(t)
        native_dmin = None
        if native_pdb and Path(native_pdb).exists() and metals:
            nat = [r for r in _read_pdb_records(native_pdb) if r["el"] != "H"]
            if nat:
                native_dmin, _ = metal_contacts(np.array([r["xyz"] for r in nat]), [r["el"] for r in nat], metals)

        # --- 9. PROLIF ---
        progress(0.88, desc="Mapeando interações no ProLIF...")
        _mem_log("antes do ProLIF")
        try:
            (df_prolif, prolif_branch, interactions, n_plf_metals, plf_errors), _ = run_isolated(
                self._run_prolif, (rec_path, out_sdf, len(valid_mols), tuple(float(v) for v in center)), timeout_s=900, label="ProLIF")
        except Exception as e_plf:      # o docking e a auditoria já estão prontos: segue sem o mapa de interações
            msg_plf = f"{type(e_plf).__name__}: {e_plf}"
            print("⚠️ " + msg_plf)
            try:                                                # SIGSEGV/timeout/memória no filho: contatos geométricos no processo principal
                df_prolif = geometric_interactions_df(rec_path, out_sdf, len(valid_mols))
                prolif_branch, interactions, n_plf_metals, plf_errors = GEO_BRANCH + f" ({msg_plf[:160]})", ["HB", "PiPi", "Hf", "Metal"], 0, [msg_plf]
            except Exception as e_geo:
                df_prolif = pd.DataFrame({"Pose_Rank": [f"Pose_{i + 1}" for i in range(len(valid_mols))],
                                          "Status_Contatos": ["ProLIF indisponível nesta corrida: " + msg_plf[:200]] * len(valid_mols)})
                prolif_branch, interactions, n_plf_metals, plf_errors = "FALHA (etapa isolada encerrada)", [], 0, [msg_plf, f"GEO: {type(e_geo).__name__}: {e_geo}"]
        df_prolif["Dist_Min_Metal_Ligante_A"] = [round(d, 2) if d is not None else None for d in metal_dmin]
        df_prolif["Contatos_Metal_Geometricos (<=2.8 A)"] = [t or "—" for t in metal_txt]
        n_plf_metal_cols = sum(1 for c in df_prolif.columns if str(c).endswith("_MetalAcceptor"))

        # --- CNNvariance por pose (SDF ou log de refino; casamento por valores) ---
        cnn_var_list, cnn_var_src = collect_cnn_variance(valid_mols, [ref_out] if refine_ok else [])
        # --- 10. CONSOLIDAÇÃO ---
        _mem_log("antes da consolidação")
        progress(0.94, desc="Consolidando métricas e gerando planilhas...")
        rows, selected_idx = [], None
        for i, m in enumerate(valid_mols):
            p = m.GetPropsAsDict()
            vina = float(p.get("minimizedAffinity", 0.0))
            cnn_s, cnn_a = float(p.get("CNNscore", 0.0)), float(p.get("CNNaffinity", 0.0))
            pb_status = pb_valid_list[i] if i < len(pb_valid_list) else "ERRO_ALINHAMENTO"
            if selected_idx is None and pb_status == "PASS":
                selected_idx = i
            rows.append({
                "Rank da Pose": i + 1,
                "Rank Amostragem (pré-refino)": int(p.get("Rank_Amostragem", i + 1)),
                "Vina score (kcal/mol)": round(vina, 2),
                "CNNscore (0-1)": round(cnn_s, 3),
                "CNNaffinity (pK)": round(cnn_a, 2),
                "CNNvariance (incerteza)": round(cnn_var_list[i], 3) if cnn_var_list[i] is not None else "n/d",
                "CNN_VS (score×pK)": round(cnn_s * cnn_a, 2),
                "LE Vina (-Vina/HAC)": round((-vina) / hac, 3) if hac > 0 else 0.0,
                "LE CNN (1.364·pK/HAC)": round(1.364 * cnn_a / hac, 3) if hac > 0 else 0.0,
                "RMSD Simétrico (Å)": rmsd_list[i],
                "Dist. mín. metal–ligante (Å)": round(metal_dmin[i], 2) if metal_dmin[i] is not None else "—",
                "PB_Valid (PoseBusters)": pb_status,
                "PB_Testes_Falhos": pb_failed[i] if i < len(pb_failed) else "",
            })
        df_exec = pd.DataFrame(rows)
        if metals:
            _pos = df_exec.columns.get_loc("Dist. mín. metal–ligante (Å)") + 1
            if native_dmin is not None:
                df_exec.insert(_pos, "Δd metal–L vs nativo (Å)",
                               [round(d - native_dmin, 2) if d is not None else "—" for d in metal_dmin])
                _pos += 1
            df_exec.insert(_pos, "Faixa metal–ligante", [metal_tier(d) for d in metal_dmin])

        if selected_idx is None:
            selected_idx = 0
            export_pose_note = "Nenhuma pose obteve PB_Valid=PASS; exportada a Pose 1 com alerta."
            export_kind = "warn"
        else:
            export_pose_note = f"Exportada a Pose {selected_idx + 1} (melhor CNNscore entre as aprovadas no PoseBusters)."
            export_kind = "ok"
        best_valid_mol = valid_mols[selected_idx]

        import torch
        dev_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        try:
            _dock_txt = dock_out.read_text(errors="ignore") + dock_err.read_text(errors="ignore")
        except Exception:
            _dock_txt = ""
        gnina_gpu_ok = ("No GPU detected" not in _dock_txt) and not self.no_gpu
        gnina_dev = ("GPU (CUDA)" if gnina_gpu_ok else
                     ("CPU (--no_gpu)" if self.no_gpu else "CPU: o GNINA não detectou a GPU (log: 'No GPU detected')"))
        if not gnina_gpu_ok:
            print("⚠️ GNINA rodou em CPU. Verifique `!ldconfig -p | grep libcuda` e o LD_LIBRARY_PATH do GNINA.")
        box_log = None
        try:
            box_log = parse_box_echo(dock_out.read_text(errors="ignore") + "\n" + dock_err.read_text(errors="ignore"))
        except Exception:
            pass
        qc_txt = ("OK" if rec_qc["ok"] else f"{len(rec_qc['issues'])} problema(s): " + " | ".join(rec_qc["issues"][:10]))
        prep_qc = self.receptor_qc or {}

        # ---- BENCHMARK vs. LIGANTE NATIVO (somente leitura; uma falha aqui NUNCA afeta as saídas já geradas) ----
        try:
            if native_pdb and Path(native_pdb).exists():
                _bm_ctx = bm_make_ctx(pdb_id, chain, native_pdb, rec_path, center, size, exhaustiveness, RANDOM_SEED,
                                      prep_mode, gnina_dev)
                bm = bm_process(self, _bm_ctx, native_pdb, df_exec.iloc[selected_idx].to_dict(),
                                cnn_var_list[selected_idx], hac, lig_name, selected_idx + 1, rmsd_list[selected_idx],
                                any(isinstance(v, (int, float)) for v in rmsd_list))
            else:
                bm = bm_no_native()
        except Exception as e_bm:
            print(f"⚠️ Benchmark nativo indisponível: {type(e_bm).__name__}: {e_bm}")
            bm = bm_failsafe(native_pdb, e_bm)

        meta = [
            ("Versao_Pipeline", PIPELINE_VERSION),
            ("Plataforma", PLATFORM_NAME),
            ("Fonte_Estrutura_Alvo", SOURCE_LABELS.get(self.source_kind, str(self.source_kind))),
            ("Perfil_Quimico", (f"{perfil['classe_estrutural']}; {perfil['perfil_medchem']}" if perfil else "n/d")),
            ("Proteina_Titulo_PDB", p_names.get("title") or "n/d"),
            ("Proteina_Molecula_Cadeia", p_names.get("chains", {}).get(chain, "n/d")),
            ("Proteina_Organismo", p_names.get("organisms", {}).get(chain, "n/d")),
            ("Ligante_Nome", lig_name), ("Ligante_Fonte_Nome", lig_source),
            ("Ligante_CID_PubChem", info["cid"] if info else "n/d"),
            ("Ligante_IUPAC", info["iupac"] if info else "n/d"),
            ("PDBFixer_Atomos_Pesados_Ausentes", (self.receptor_qc or {}).get("n_missing_heavy", "n/d")),
            ("PDBFixer_Pico_RAM_Filho (MB)", (self.receptor_qc or {}).get("pdbfixer_peak_mb", "n/d")),
            ("PDBFixer_Trava_Memoria", (self.receptor_qc or {}).get("memory_guard") or "não acionada"),
            ("RAM_Kernel_ao_Consolidar (MB)", round(_rss_mb(), 0)),
            ("PDB_ID", pdb_id), ("Cadeia", chain), ("Dispositivo_Torch (informativo)", dev_name), ("GNINA_Dispositivo_Efetivo", gnina_dev),
            ("Versao_Python", sys.version.split()[0]),
            ("GNINA_Versao", GNINA_VERSION), ("GNINA_SHA256", GNINA_SHA256),
            ("OpenBabel_Versao", _obabel_version()),
            ("RDKit_Versao", rdkit.__version__),
            ("OpenMM_Versao", _pkg_ver("openmm")), ("PDBFixer_Versao", _pkg_ver("pdbfixer")),
            ("PoseBusters_Versao", getattr(posebusters, "__version__", _pkg_ver("posebusters"))),
            ("ProLIF_Versao", plf.__version__), ("spyRMSD_Versao", spyrmsd.__version__),
            ("MDAnalysis_Versao", _pkg_ver("MDAnalysis")), ("Gradio_Versao", _pkg_ver("gradio")),
            ("Semente_Estocastica (GNINA/ETKDG/PDBFixer)", RANDOM_SEED),
            ("Comando_GNINA_Docking", " ".join(used_dock)),
            ("Comando_GNINA_Refino", " ".join(used_ref)),
            ("Refino_Status", refine_note),
            ("Grid_Box_Efetiva (passada ao GNINA via CLI)", box_meta_str),
            ("Grid_Box_Ecoada_no_Log_GNINA", json.dumps(box_log) if box_log else "não ecoada no log; a caixa acima foi fornecida explicitamente"),
            ("Diametro_Maximo_Ligante (Å)", round(max_lig_diameter, 2)),
            ("Exhaustiveness", int(exhaustiveness)),
            ("Receptor_Metodo_Protonacao", prep_qc.get("method", "n/d")),
            ("Receptor_Notas_Preparo", " | ".join(prep_qc.get("notes", [])) or "—"),
            ("Receptor_Estados_His", json.dumps(rec_qc["his_states"])),
            ("Receptor_N_Hidrogenios", rec_qc["n_h"]), ("Receptor_N_Metais", rec_qc["n_metals"]),
            ("Receptor_QC_Protonacao", qc_txt),
            ("Metais_no_Receptor (inclui metal de cofator)", ", ".join(f"{m['res']}{m['resseq']}:{m['name']}" for m in metals) or "nenhum"),
            ("Cofatores_Organicos_no_Receptor", ", ".join(cof_in_rec) or "nenhum"),
            ("Cofatores_Politica", "Retidos fora do OpenMM (sem H nem parâmetros); metal do cofator entra no PDBFixer como íon substituto, removido depois"),
            ("GNINA_Fallback_Sem_Cofator", gnina_cofactor_fallback or "não acionado"),
            ("GNINA_Erro_1a_Tentativa", gnina_first_err or "—"),
            ("PoseBusters_Receptor_Usado", pb_receptor_note),
            ("Ligante_Avisos_Seguranca", " | ".join(lig_safety) or "nenhum"),
            ("Modo_Preparo_Ligante", prep_mode), ("Modo_Ionizacao", "Regras RDKit de subestrutura em pH 7.4 (padrão, incondicional)"),
            ("Ionizacao_Aplicada", lig["ionization"]),
            ("SMILES_Protonado_pH7.4", lig["smiles"]), ("Heavy_Atom_Count (HAC)", hac),
            ("Carga_Formal_pH7.4", lig["formal_charge"]),
            ("Faixa_Cargas_Gasteiger (RDKit)", lig["gasteiger_range"]),
            ("TORSDOF_PDBQT_OpenBabel", lig["torsdof"]),
            ("Ligacoes_Rotacionaveis_RDKit (estritas)", lig["rot_rdkit"]),
            ("Status_Minimizacao_Ligante", lig["mmff_status"]),
            ("Nota_Arquivos_PDBQT_MOL2", "Artefatos de exportação; o GNINA recebe o SDF preparado."),
            ("CNNvariance_Fonte", f"SDF: {cnn_var_src.count('SDF')} | log de refino: {cnn_var_src.count('log')} | ambíguo: {cnn_var_src.count('ambíguo')} | n/d: {cnn_var_src.count('n/d')}"),
            ("CNNvariance_Nota", "Variância reportada pelo GNINA entre os modelos do ensemble 3D-CNN; maior = mais desacordo. Sem limiar universal: compare entre poses da mesma corrida."),
            ("PoseBusters_Criterio_PB_Valid", "PASS = todos os testes OFICIAIS do modo dock True; NaN (não aplicável) não reprova"),
            ("PoseBusters_Fonte_Lista_Testes", pb_source),
            ("PoseBusters_N_Testes_Oficiais", len(pb_official)),
            ("PoseBusters_Colunas_Auxiliares_Fora_do_Veredito", ", ".join(pb_diag) or "—"),
            ("PoseBusters_Nota_Auxiliares", "most_extreme_clash_*: False = sem choque; not_too_far_away_*: False quando o cofator/água não existe. Não são testes."),
            ("RMSD_Referencia_Status", rmsd_status),
            ("Dist_Min_Metal_Ligante_Nativo (Å)", round(native_dmin, 2) if native_dmin is not None else "n/d"),
            ("ProLIF_Interacoes", ", ".join(interactions)), ("ProLIF_Ramo_Proteina", prolif_branch),
            ("ProLIF_N_Metais_no_Objeto", n_plf_metals),
            ("ProLIF_N_Colunas_MetalAcceptor", n_plf_metal_cols),
            ("ProLIF_Erros_Ramos", " || ".join(plf_errors) or "—"),
            ("Pose_Exportada_PDB", export_pose_note),
        ]
        meta = meta + list(bm["meta"]) + af_rows
        df_meta = pd.DataFrame([{"Parâmetro": k, "Valor": v} for k, v in meta])

        stem = Path(lig_path_str).stem
        complex_path = sdir / f"{pdb_id}_{chain}_{stem}_Complexo.pdb"
        poses_sdf_path = sdir / f"{pdb_id}_{chain}_{stem}_Poses.sdf"
        raw_sdf_export = sdir / f"{pdb_id}_{chain}_{stem}_Poses_Brutas_Amostragem.sdf"
        lig_mol2_export = sdir / f"{stem}_Preparado_Gasteiger.mol2"
        lig_pdbqt_export = sdir / f"{stem}_Preparado_Docking.pdbqt"
        excel_path = sdir / f"{pdb_id}_{chain}_{stem}_Suplementar_{VERSION_TAG}.xlsx"
        zip_path = sdir / f"{pdb_id}_{chain}_{stem}_Pacote_Completo_{VERSION_TAG}.zip"

        self._write_complex_pdb_with_conect(rec_path, best_valid_mol, chain, complex_path)
        md_info = None
        try:                                  # script turnkey de MD (OpenMM) da pose exportada, em 08_dinamica_molecular/<ligante>/
            md_info = stage_md_folder(sdir / MD_DIRNAME / re.sub(r"[^A-Za-z0-9_.-]", "_", str(stem))[:50], complex_path, best_valid_mol)
        except Exception as e_md:
            print(f"⚠️ Script de MD não gerado: {type(e_md).__name__}: {e_md}")
        shutil.copy(str(out_sdf), str(poses_sdf_path))
        shutil.copy(str(out_raw_sdf), str(raw_sdf_export))
        shutil.copy(lig["mol2"], str(lig_mol2_export))
        shutil.copy(lig["pdbqt"], str(lig_pdbqt_export))

        with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
            df_exec.to_excel(writer, sheet_name="Resumo_Executivo", index=False)
            pb_df_export.to_excel(writer, sheet_name="Auditoria_PoseBusters", index=False)
            df_prolif.to_excel(writer, sheet_name="Interacoes_ProLIF", index=False)
            df_meta.to_excel(writer, sheet_name="Metadados_Sessao", index=False)
            try:
                bm["df"].to_excel(writer, sheet_name="Benchmark_Nativo", index=False)
            except Exception as e_bm_x:
                print(f"⚠️ Aba Benchmark_Nativo não gravada: {e_bm_x}")

        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for folder, f in [("00_receptor", rec_path), ("01_complexo", complex_path),
                              ("02_poses", poses_sdf_path), ("02_poses", raw_sdf_export),
                              ("03_ligante", lig_mol2_export), ("03_ligante", lig_pdbqt_export),
                              ("04_planilha", excel_path),
                              ("05_logs", dock_out), ("05_logs", dock_err), ("05_logs", ref_out), ("05_logs", ref_err)]:
                if Path(f).exists():
                    zf.write(str(f), f"{folder}/{Path(f).name}")
            for f in (md_info or {}).get("files", []):
                if Path(f).exists():
                    zf.write(str(f), f"{MD_DIRNAME}/{Path(f).name}")

        # --- 11. RESUMO PARA A INTERFACE ---
        sel = df_exec.iloc[selected_idx]
        pb_sel = sel["PB_Valid (PoseBusters)"]
        tiles = metric_tiles([
            ("Pose exportada", f"#{selected_idx + 1}", export_kind),
            ("Vina (kcal/mol)", f"{sel['Vina score (kcal/mol)']:.2f}", "info"),
            ("CNNscore", f"{sel['CNNscore (0-1)']:.3f}", "info"),
            ("CNNaffinity (pK)", f"{sel['CNNaffinity (pK)']:.2f}", "info"),
            ("CNNvariance", (f"{sel['CNNvariance (incerteza)']:.2f}" if cnn_var_list[selected_idx] is not None else "n/d"), "info"),
            ("LE (Vina)", f"{sel['LE Vina (-Vina/HAC)']:.3f}", "info"),
            ("PoseBusters", pb_sel, "ok" if pb_sel == "PASS" else "err"),
        ])
        notes = [esc(export_pose_note)]
        _vv = sorted(v for v in cnn_var_list if v is not None)
        if cnn_var_list[selected_idx] is not None and _vv:
            _med = _vv[len(_vv) // 2] if len(_vv) % 2 else (_vv[len(_vv) // 2 - 1] + _vv[len(_vv) // 2]) / 2
            notes.append(f"<b>Incerteza da rede:</b> CNNvariance da pose exportada = {cnn_var_list[selected_idx]:.3f} "
                         f"(mediana das {len(_vv)} poses = {_med:.3f}; menor = mais consenso entre os modelos do ensemble).")
        numeric_rmsd = [v for v in rmsd_list if isinstance(v, (int, float))]
        if numeric_rmsd:
            best = min(numeric_rmsd)
            top1 = rmsd_list[0]
            verdict = "sucesso (< 2,0 Å)" if isinstance(top1, (int, float)) and top1 < 2.0 else "falha (≥ 2,0 Å)"
            notes.append(f"<b>Redocking:</b> RMSD simétrico da pose 1 = {top1} Å → {verdict}; melhor entre as poses = {best:.3f} Å.")
        else:
            notes.append(f"<b>RMSD:</b> {esc(rmsd_status)}.")
        if metals:
            d1 = metal_dmin[selected_idx]
            txt = (f"<b>Metal:</b> distância mínima O/N/S–metal na pose exportada = {d1:.2f} Å ({esc(metal_tier(d1))})"
                   if d1 is not None else "<b>Metal:</b> sem átomos O/N/S no ligante")
            if native_dmin is not None and d1 is not None:
                txt += f"; ligante nativo: {native_dmin:.2f} Å (Δ = {d1 - native_dmin:+.2f} Å)"
            if d1 is not None and d1 > METAL_COORD_CUTOFF:
                txt += ". O RMSD sozinho não garante a quelação do metal; o GNINA não tem termo explícito de coordenação."
            notes.append(txt)
        notes.append(f"<b>ProLIF:</b> {esc(prolif_branch)}; metais no objeto: {n_plf_metals}.")
        if self.source_kind == "alphafold":
            notes.append(plddt_html(plddt_site) if plddt_site else "<b>pLDDT do sítio:</b> indisponível (bolsão vazio ou fator B ausente).")
        for _w in lig_safety:
            notes.append("<b>Atenção — ligante:</b> " + esc(_w))
        if cof_in_rec:
            notes.append(f"<b>Cofatores no receptor:</b> {esc(', '.join(cof_in_rec))} (sem hidrogênios; tipos atômicos inferidos pelo GNINA). "
                         "O ProLIF mapeia apenas o metal do cofator, não as interações com o anel orgânico.")
        if gnina_cofactor_fallback:
            notes.append("<b>Atenção — GNINA:</b> " + esc(gnina_cofactor_fallback) + ".")
        if pb_cof_fallback:
            notes.append("<b>Atenção — PoseBusters:</b> " + esc(pb_receptor_note) + ".")
        plf_ok = not str(prolif_branch).startswith("FALHA")
        if str(prolif_branch).startswith("FALLBACK GEOMÉTRICO"):
            notes.append("<b>Interações:</b> o ProLIF não concluiu; as interações da aba Interacoes_ProLIF e da tabela do artigo foram "
                         "calculadas por proximidade geométrica (H, π, hidrofóbico, metal). Motivo na aba Metadados_Sessao.")
        if not plf_ok:
            notes.append("<b>Atenção — ProLIF:</b> o processo isolado falhou; o docking e o PoseBusters são válidos, mas a aba "
                         "Interacoes_ProLIF não traz interações (apenas contatos geométricos com metais). Motivo na aba Metadados_Sessao.")
        if not gnina_gpu_ok:
            notes.append(f"<b>Atenção — dispositivo do GNINA:</b> {esc(gnina_dev)}. O CNN foi avaliado em CPU (mais lento); "
                         "os escores não mudam, mas o tempo e a metodologia 'acelerada por GPU' não se aplicam a esta corrida.")
        notes.append(f"<b>Refino:</b> {esc(refine_note[:90])}")
        ident_html = identity_card(pdb_id, p_names.get("title", ""), p_names.get("chains", {}).get(chain, ""),
                                   p_names.get("organisms", {}).get(chain, ""), lig_name, lig_sub)
        summary_html = (ident_html + status_box("ok", "Docking concluído", esc(lig['chem_info']))
                        + tiles + '<div class="dd-detail">' + "<br>".join(notes) + "</div>" + bm["html"]
                        + md_notice_html(md_info, lig_name))

        df_ui = df_exec.copy()
        df_ui.insert(1, "★", ["★" if i == selected_idx else "" for i in range(len(df_ui))])
        df_ui["PB_Valid (PoseBusters)"] = df_ui["PB_Valid (PoseBusters)"].map(
            lambda v: "✅ PASS" if v == "PASS" else "❌ FAIL")

        progress(1.0, desc="Docking finalizado!")
        files = [str(zip_path), str(complex_path), str(poses_sdf_path), str(raw_sdf_export),
                 str(lig_mol2_export), str(lig_pdbqt_export), str(excel_path)]
        _ui_out = (df_ui, summary_html,
                status_box("ok" if (gnina_gpu_ok and plf_ok and not gnina_cofactor_fallback and not pb_cof_fallback and not lig_safety) else "warn", "Pipeline finalizado",
                           "Veja o Passo 4 para resultados, glossário e downloads. " + bm["short"]
                           + ("" if plf_ok else "<b>ProLIF indisponível nesta corrida: sem mapa de interações.</b> ")
                           + f"GNINA: {esc(gnina_dev)}. RAM do kernel: {_rss_mb():.0f} MB."),
                files, gr.update(selected="t4"))
        try:                                  # resultado estruturado para o lote/leaderboard (somente leitura)
            self._last_result = {
                "lig_name": str(lig_name), "lig_sub": lig_sub, "stem": stem, "hac": hac, "pdb_id": pdb_id, "chain": chain,
                "native_pdb": str(native_pdb) if native_pdb else "", "selected_idx": selected_idx,
                "sel": df_exec.iloc[selected_idx].to_dict(), "cnn_var": cnn_var_list[selected_idx],
                "rmsd_sel": rmsd_list[selected_idx], "rmsd_status": rmsd_status, "df_exec": df_exec, "pb_df": pb_df_export, "df_prolif": df_prolif, "prolif_branch": str(prolif_branch),
                "df_meta": df_meta, "summary_html": summary_html, "bm": bm, "bm_rec": bm.get("rec"),
                "key_inter": key_interactions_summary(df_prolif, selected_idx), "zip_path": str(zip_path),
                "plddt": plddt_site, "md": md_info, "source_kind": self.source_kind,
                "excel_path": str(excel_path), "gnina_dev": gnina_dev, "exh": int(exhaustiveness), "prep_mode": prep_mode, "perfil": perfil,
            }
        except Exception as e_lr:
            print(f"⚠️ Resultado estruturado não guardado: {type(e_lr).__name__}: {e_lr}")
            self._last_result = None
        return _ui_out



# Autoteste químico (não interrompe a execução; apenas avisa)
_selftest_warnings = run_ionization_self_test()
if _selftest_warnings:
    print("⚠️ Autoteste das regras de ionização apresentou divergências:\n  - " + "\n  - ".join(_selftest_warnings))
else:
    print("✅ Autoteste das regras de ionização: OK (quercetina, tropolona, ácidos carboxílicos, controles negativos).")

try:
    gr.close_all(verbose=False)      # re-execução da célula: fecha servidores/túneis Gradio anteriores
except Exception:
    pass
engine = DockingInterfaceEngine()

# ==============================================================================
# 6. INTERFACE GRADIO: tema, CSS e conteúdo didático
# ==============================================================================
CUSTOM_CSS = """
:root{
  --dd-indigo:#4338ca; --dd-royal:#1d4ed8; --dd-emerald:#059669; --dd-amber:#b45309; --dd-red:#b91c1c;
  --dd-ink:#0f172a; --dd-slate:#475569; --dd-card:#ffffff; --dd-soft:#f1f5f9; --dd-border:#e2e8f0;
  --dd-shadow:0 1px 2px rgba(15,23,42,.06),0 10px 28px -14px rgba(15,23,42,.25);
}
.dark{ --dd-ink:#e2e8f0; --dd-slate:#94a3b8; --dd-card:#1e293b; --dd-soft:#0f172a; --dd-border:#334155;
  --dd-shadow:0 1px 2px rgba(0,0,0,.4),0 10px 28px -14px rgba(0,0,0,.6); }
.gradio-container{ max-width:1180px !important; margin:auto; }
.dd-hero{ background:linear-gradient(120deg,#1e3a8a 0%,#4338ca 60%,#4f46e5 100%); color:#fff;
  border-radius:18px; padding:26px 30px; box-shadow:var(--dd-shadow); }
.dd-hero h1{ margin:0 0 6px 0; font-size:1.75rem; font-weight:700; letter-spacing:-.01em; color:#fff; }
.dd-hero p{ margin:0 0 14px 0; color:#e0e7ff; font-size:.98rem; max-width:78ch; line-height:1.5; }
.dd-hero .dd-chip{ display:inline-block; margin:0 6px 6px 0; padding:3px 11px; border-radius:999px;
  background:rgba(255,255,255,.14); border:1px solid rgba(255,255,255,.28); font-size:.8rem; }
.dd-card{ background:var(--dd-card); border:1px solid var(--dd-border); border-radius:14px;
  box-shadow:var(--dd-shadow); padding:16px 18px !important; }
.dd-explain{ border-left:4px solid var(--dd-indigo); background:var(--dd-soft); border-radius:10px;
  padding:14px 18px; color:var(--dd-ink); line-height:1.55; margin-bottom:6px; }
.dd-explain h3{ margin:0 0 6px 0; font-size:1.05rem; color:var(--dd-ink); }
.dd-explain ol,.dd-explain ul{ margin:6px 0 0 0; padding-left:20px; }
.dd-explain li{ margin-bottom:4px; }
.dd-explain.dd-green{ border-left-color:var(--dd-emerald); }
.dd-explain.dd-blue{ border-left-color:var(--dd-royal); }
.dd-table{ width:100%; border-collapse:collapse; font-size:.9rem; color:var(--dd-ink); }
.dd-table th,.dd-table td{ border:1px solid var(--dd-border); padding:8px 10px; text-align:left; vertical-align:top; }
.dd-table th{ background:var(--dd-soft); }
.dd-pill{ display:inline-block; padding:2px 10px; border-radius:999px; font-size:.78rem; font-weight:600; }
.dd-pill-ok{ background:#d1fae5; color:#065f46; } .dd-pill-info{ background:#dbeafe; color:#1e40af; }
.dd-pill-warn{ background:#fef3c7; color:#92400e; } .dd-pill-err{ background:#fee2e2; color:#991b1b; }
.dd-status{ display:flex; gap:12px; align-items:flex-start; border-radius:12px; padding:12px 14px;
  border:1px solid var(--dd-border); background:var(--dd-card); margin-bottom:8px; color:var(--dd-ink); }
.dd-status-icon{ flex:0 0 26px; height:26px; border-radius:50%; display:flex; align-items:center;
  justify-content:center; font-weight:700; color:#fff; }
.dd-status-ok{ border-color:#34d399; } .dd-status-ok .dd-status-icon{ background:var(--dd-emerald); }
.dd-status-info{ border-color:#93c5fd; } .dd-status-info .dd-status-icon{ background:var(--dd-royal); }
.dd-status-warn{ border-color:#fbbf24; } .dd-status-warn .dd-status-icon{ background:#d97706; }
.dd-status-err{ border-color:#f87171; } .dd-status-err .dd-status-icon{ background:var(--dd-red); }
.dd-status-title{ font-weight:650; } .dd-status-text{ color:var(--dd-slate); font-size:.9rem; margin-top:2px; line-height:1.45; }
.dd-tiles{ display:grid; grid-template-columns:repeat(auto-fit,minmax(130px,1fr)); gap:10px; margin:8px 0; }
.dd-tile{ border:1px solid var(--dd-border); border-radius:12px; padding:10px 12px; background:var(--dd-card); }
.dd-tile-value{ font-size:1.25rem; font-weight:700; color:var(--dd-ink); }
.dd-tile-label{ font-size:.78rem; color:var(--dd-slate); margin-top:2px; }
.dd-tile-ok{ border-top:3px solid var(--dd-emerald); } .dd-tile-info{ border-top:3px solid var(--dd-royal); }
.dd-tile-warn{ border-top:3px solid #d97706; } .dd-tile-err{ border-top:3px solid var(--dd-red); }
.dd-detail{ font-size:.88rem; color:var(--dd-slate); line-height:1.6; padding:4px 2px; }
.dd-pre{ background:var(--dd-soft); padding:8px; border-radius:8px; overflow:auto; max-height:220px; font-size:.75rem; }
.dd-gloss dt{ font-weight:650; color:var(--dd-ink); margin-top:10px; }
.dd-gloss dd{ margin:2px 0 0 0; color:var(--dd-slate); line-height:1.5; }
.dd-gloss{ background:var(--dd-card); border:1px solid var(--dd-border); border-radius:12px; padding:6px 18px 14px; }
.dd-ident{ display:grid; grid-template-columns:repeat(auto-fit,minmax(260px,1fr)); gap:12px; margin-bottom:10px; }
.dd-ident-col{ background:linear-gradient(135deg,rgba(67,56,202,.10),rgba(5,150,105,.10)); border:1px solid var(--dd-border);
  border-radius:14px; padding:14px 16px; }
.dd-ident-tag{ font-size:.78rem; color:var(--dd-slate); font-weight:600; }
.dd-ident-name{ font-size:1.15rem; font-weight:700; color:var(--dd-ink); margin-top:2px; line-height:1.3; }
.dd-ident-sub{ font-size:.82rem; color:var(--dd-slate); margin-top:4px; line-height:1.4; }
.dd-foot{ font-size:.8rem; color:var(--dd-slate); text-align:center; padding:12px 0 4px; }
button.primary, .dd-run{ background:linear-gradient(135deg,#4338ca,#1d4ed8) !important; color:#fff !important;
  border:none !important; font-weight:650 !important; box-shadow:0 6px 16px -6px rgba(67,56,202,.7) !important; }
button.primary:hover{ filter:brightness(1.08); }
.dd-go{ background:linear-gradient(135deg,#059669,#10b981) !important; color:#fff !important; border:none !important; font-weight:650 !important; }
button[role=tab][aria-selected=true]{ color:var(--dd-indigo) !important; font-weight:650; }
"""

HERO_HTML = f"""
<div class="dd-hero">
  <h1>{PLATFORM_NAME}</h1>
  <p>Docking molecular com rede convolucional 3D (GNINA), preparo reprodutível do alvo e do ligante,
     e auditoria físico-química das poses. Siga os quatro passos nas abas abaixo.</p>
  <span class="dd-chip">GNINA 1.3.2</span><span class="dd-chip">GPU NVIDIA A100</span>
  <span class="dd-chip">pH 7.4</span><span class="dd-chip">semente 42</span>
  <span class="dd-chip">PoseBusters + ProLIF + spyRMSD</span><span class="dd-chip">v{PIPELINE_VERSION}</span>
</div>
"""

STEP1_HTML = """
<div class="dd-explain"><h3>O que acontece neste passo</h3>
<ol>
<li><b>Isolamento da cadeia:</b> baixamos o PDB do RCSB (ou lemos o arquivo .pdb que você enviar, como um modelo do SWISS-MODEL) e mantemos só a cadeia escolhida, sem águas, íons de cristalização (SO4, GOL…) nem co-solventes.</li>
<li><b>Cofatores catalíticos preservados:</b> metais como o Cu²⁺ da tirosinase continuam no receptor, porque definem o sítio ativo.</li>
<li><b>Protonação em pH 7.4 (PDBFixer/OpenMM):</b> completa átomos ausentes e adiciona hidrogênios. Histidinas que coordenam o metal recebem o tautômero explícito (HID/HIE).</li>
<li><b>Controle de qualidade:</b> verificamos hidrogênios inválidos em anéis aromáticos, histidinas sem N–H e nitrogênios coordenados ao metal mas protonados.</li>
</ol></div>
"""

STEP2_HTML = """
<div class="dd-explain dd-green"><h3>Como obter o ligante</h3>
<ul><li><b>PubChem:</b> busque por nome ou CID; para vários, separe por vírgula+espaço ou quebra de linha. O 3D oficial é usado quando existe; caso contrário a estrutura 2D é convertida.</li>
<li><b>Arquivo local:</b> envie SDF, MOL2, PDB ou PDBQT, um ou vários (SDF multi-molécula é separado em uma molécula por ligante).</li>
<li><b>Controle nativo automático:</b> com a caixa marcada e um ligante nativo fixado no Passo 1, ele entra como item <b>#0</b> (Modo 2, coordenadas de cristal) e roda antes dos testes; os Δ de cada teste saem contra esse controle.</li>
<li><b>Fila:</b> até 50 ligantes de teste por corrida, em sequência, com o mesmo receptor e a mesma caixa.</li></ul></div>
"""

MODES_HTML = """
<table class="dd-table"><tr><th>Modo</th><th>Quando usar</th><th>O que faz com a geometria</th></tr>
<tr><td><b>1. Triagem prospectiva</b></td><td>Compostos novos, sem pose experimental.</td>
<td>Remove sais, ajusta a ionização em pH 7.4, gera 20 confôrmeros ETKDGv3 (semente 42) e minimiza o melhor com MMFF94.</td></tr>
<tr><td><b>2. Redocking cristalográfico</b></td><td>Validação do método com o ligante do cristal.</td>
<td>Mantém a geometria exata do cristal; sem minimização. O RMSD simétrico é calculado contra o nativo.</td></tr></table>
"""

TERMS_HTML = """
<dl class="dd-gloss">
<dt>Cargas de Gasteiger</dt><dd>Cargas parciais atômicas calculadas por equalização de eletronegatividade. Entram no arquivo PDBQT/MOL2 exportado; o GNINA recebe o SDF preparado.</dd>
<dt>Torções ativas (TORSDOF)</dt><dd>Número de ligações rotacionáveis que o docking pode girar. Mais torções ampliam o espaço de busca e pedem maior exaustividade.</dd>
<dt>Ionização em pH 7.4</dt><dd>Grupos ácidos são desprotonados quando o pKa fica abaixo de ~7.4: carboxilatos (pKa ~4–5), tropolonas (~7) e o 7-OH de flavonoides (~7.0–7.2). Os dois últimos estão perto do pH, então a forma carregada é uma hipótese de modelagem; compare com a forma neutra se o resultado depender dela.</dd>
<dt>Hidrogênios polares</dt><dd>São ajustados no campo do receptor pela minimização do GNINA (Passo 3), não no vácuo.</dd>
</dl>
"""

STEP3_HTML = """
<div class="dd-explain dd-blue"><h3>Caixa de busca (grid box)</h3>
<ul><li><b>Automática:</b> usa o contorno do ligante nativo mais 4 Å por lado; cada aresta nunca fica menor que o diâmetro do ligante que será docado.</li>
<li><b>Manual:</b> informe centro e tamanho em Å. O botão abaixo copia o centro do ligante nativo para os campos.</li></ul>
<p style="margin:8px 0 0 0">A caixa efetiva é registrada na planilha (Metadados_Sessao), para reprodutibilidade.</p></div>
<div class="dd-explain dd-blue" style="margin-top:8px"><h3>Como as poses são avaliadas</h3>
<ol><li>O GNINA amostra poses por Monte Carlo (exaustividade controla o esforço) e pontua com a função empírica tipo Vina.</li>
<li>Um <b>ensemble de redes convolucionais 3D</b> reclassifica cada pose: CNNscore (qualidade da pose) e CNNaffinity (afinidade em pK).</li>
<li>Em seguida, <code>--minimize</code> relaxa a pose no campo do receptor, removendo choques estéricos sem sair do bolsão. Os hidrogênios polares (como -OH) se ajustam ao ambiente real, não ao vácuo.</li></ol></div>
"""

FIXED_PARAMS_HTML = """
<table class="dd-table"><tr><th>Parâmetro</th><th>Valor</th><th>Papel</th></tr>
<tr><td>num_modes</td><td>10</td><td>Poses retornadas</td></tr>
<tr><td>num_mc_saved</td><td>50</td><td>Poses estocásticas retidas por corrida de Monte Carlo</td></tr>
<tr><td>min_rmsd_filter</td><td>0.5 Å</td><td>Filtro de diversidade entre poses</td></tr>
<tr><td>cnn_scoring</td><td>rescore</td><td>CNN reavalia as poses finais</td></tr>
<tr><td>pose_sort_order</td><td>CNNscore</td><td>Ordenação das poses</td></tr>
<tr><td>seed</td><td>42</td><td>Reprodutibilidade</td></tr></table>
"""

GLOSSARY_HTML = """
<dl class="dd-gloss">
<dt>Vina score (kcal/mol)</dt><dd>Energia de ligação estimada pela função empírica após a minimização. Mais negativo é melhor. Compare apenas ligantes no mesmo alvo; o erro típico é de alguns kcal/mol.</dd>
<dt>CNNscore (0–1)</dt><dd>Probabilidade, estimada pela rede 3D, de a pose ser de boa qualidade. É o critério de ordenação das poses. Quanto mais perto de 1, mais confiável.</dd>
<dt>CNNaffinity (pK)</dt><dd>Afinidade prevista em −log10(Kd/Ki). pK 6 ≈ 1 µM; pK 9 ≈ 1 nM. Equivale a ΔG ≈ −1,364·pK kcal/mol a 298 K.</dd>
<dt>CNNvariance (incerteza)</dt><dd>Variância reportada pelo GNINA entre os modelos do ensemble 3D-CNN para a pose. Quanto maior, mais os modelos discordam, ou seja, maior a incerteza da rede. Não há limiar universal: compare entre as poses da mesma corrida. Vem da tag do SDF ou do log de refino; "n/d" quando indisponível.</dd>
<dt>CNN_VS</dt><dd>CNNscore × CNNaffinity. Combina plausibilidade da pose e afinidade para ranquear compostos em triagem virtual.</dd>
<dt>LE (eficiência de ligante)</dt><dd>Afinidade por átomo pesado: −Vina/HAC (e 1,364·pK/HAC para a versão CNN). Valores a partir de ~0,3 kcal/mol por átomo pesado são o ponto de partida usual para seleção de líderes.</dd>
<dt>ΔVina, ΔpK, potência relativa, ΔLE</dt><dd>Diferenças entre o composto e o controle nativo re-dockado: ΔVina = Vina(teste) − Vina(nativo) (negativo = melhor); ΔpK = pK(teste) − pK(nativo); potência relativa = 10^ΔpK (&gt; 1 = teste previsto mais potente); ΔLE = LE(teste) − LE(nativo). As classes Superior/Equivalente/Inferior usam a banda de ±0,5 kcal/mol no ΔVina (e o equivalente em ΔG para a CNN). É um ranking computacional, não afinidade experimental.</dd>
<dt>Veredito do leaderboard</dt><dd>Decisão em cinco filtros, aplicados em ordem. <b>Inviável (PoseBusters FAIL)</b>: a pose não passou nos testes físico-químicos. <b>Inconclusivo (Discordância Vina/CNN)</b>: o Vina e a CNN apontam classes opostas (Superior × Inferior). <b>Superior (Consenso Robusto)</b>: Vina Superior, com a CNN Superior ou Equivalente. <b>Superior (Ressalva: Baixa LE / Efeito Volume)</b>: o mesmo, mas LE Vina &lt; 0,28 e ΔLE Vina &lt; −0,05, ou seja, o ganho de energia vem do tamanho da molécula. <b>Promissor (LE/CNN Superior ao Nativo)</b>: Vina Equivalente, CNN Superior e ΔLE Vina &gt; 0,02. <b>Equivalente (Alerta: CNN Desfavorável)</b>: Vina Equivalente, CNN Inferior. <b>Equivalente ao Controle</b>: nenhum sinal adicional. <b>Bioativo Eficiente (Potencial Lead-like)</b>: ΔVina ≥ +0,5 kcal/mol, mas LE Vina ≥ 0,35 (molécula pequena e eficiente, ponto de partida para otimização). <b>Inferior ao Nativo</b>: ΔVina ≥ +0,5 kcal/mol com LE Vina &lt; 0,35. <b>Inconclusivo (Apoio Fraco da CNN)</b>: caso residual sem apoio da CNN. É uma triagem computacional, não afinidade experimental.</dd>
<dt>RMSD simétrico (Å)</dt><dd>Desvio entre a pose predita e a pose cristalográfica, corrigido pela simetria do ligante e sem superposição prévia. Abaixo de 2,0 Å conta como redocking bem-sucedido. Só existe quando há ligante nativo compatível.</dd>
<dt>Dist. mín. metal–ligante (Å)</dt><dd>Menor distância entre um O, N ou S do ligante e um metal do sítio, inclusive o metal preso a um cofator (ex.: FE do heme). Distâncias de coordenação cristalográficas ficam em torno de 2 Å; a faixa "coordenação provável" usa ≤ 2,5 Å, "contato próximo" até 2,8 Å (raio do ProLIF). No redocking, "Δd vs nativo" é a diferença para o ligante cristalográfico. Um RMSD baixo não garante a quelação do metal.</dd>
<dt>PoseBusters (PASS/FAIL)</dt><dd>PASS quando a pose passa em todos os testes físico-químicos oficiais do modo dock: geometria de ligações e ângulos, planaridade, choques internos e com o receptor/cofatores, energia interna e distâncias.</dd>
<dt>ProLIF</dt><dd>Impressão digital das interações por resíduo: ligações de hidrogênio, empilhamento π, cátion–π, contatos hidrofóbicos, interações iônicas e aceitação de metal. Está na aba Interacoes_ProLIF da planilha.</dd>
</dl>
"""

PB_HELP_HTML = """
<div class="dd-explain"><b>Por que algumas colunas booleanas ficam <code>False</code> sem reprovar a pose?</b>
<p style="margin:6px 0 0 0">Na planilha, colunas como <code>most_extreme_clash_protein</code> indicam se o par mais próximo é um choque (False = sem choque).
As colunas <code>not_too_far_away_organic_cofactors</code> e <code>…_waters</code> ficam False quando o sistema não tem cofator orgânico ou água.
Elas são detalhes dos módulos, não testes. O veredito PASS/FAIL usa apenas os testes oficiais, listados primeiro na aba Auditoria_PoseBusters.</p></div>
"""

FOOT_HTML = ('<div class="dd-foot">GNINA 1.3 (McNutt et al., 2025) · AutoDock Vina (Trott &amp; Olson, 2010) · '
             'PoseBusters (Buttenschoen et al., 2024) · ProLIF (Bouysset &amp; Fiorucci, 2021) · '
             'spyrmsd (Meli &amp; Biggin, 2020) · MMFF94 (Halgren, 1996)</div>')

_theme_kwargs = {}
try:
    THEME = gr.themes.Soft(
        primary_hue=gr.themes.colors.indigo, secondary_hue=gr.themes.colors.emerald,
        neutral_hue=gr.themes.colors.slate,
        font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
    )
    try:
        THEME = THEME.set(button_primary_background_fill="linear-gradient(135deg,#4338ca,#1d4ed8)",
                          button_primary_background_fill_hover="linear-gradient(135deg,#3730a3,#1e40af)",
                          button_primary_text_color="white", block_radius="14px")
    except Exception:
        pass
except Exception:
    THEME = None

LAUNCH_KW = {}
try:
    app = gr.Blocks(theme=THEME, css=CUSTOM_CSS, title=PLATFORM_NAME)
except TypeError:   # Gradio 6: tema/CSS migram para launch()
    app = gr.Blocks(title=PLATFORM_NAME)
    LAUNCH_KW = dict(theme=THEME, css=CUSTOM_CSS)

with app:
    gr.HTML(HERO_HTML)
    state_native_path = gr.State()
    state_rec_path = gr.State()
    state_lig_info = gr.State()      # {"file", "name", "cid", "iupac", ...} do ligante obtido no PubChem

    with gr.Tabs(selected="t1") as tabs:
        # ---------------------------------------------------------------- PASSO 1
        with gr.TabItem("1 · Alvo e sítio ativo", id="t1"):
            gr.HTML(STEP1_HTML)
            with gr.Row():
                with gr.Column(scale=5, elem_classes="dd-card"):
                    input_mode = gr.Radio(choices=[INPUT_RCSB, INPUT_LOCAL, INPUT_AFDB], value=INPUT_RCSB, label="Origem do alvo",
                                          info="RCSB: estrutura experimental. Upload: modelos de homologia (ex.: SWISS-MODEL) ou estruturas próprias. "
                                               "AlphaFold DB: modelo predito por código UniProt (a confiabilidade do sítio é auditada pelo pLDDT).")
                    pdb_input = gr.Textbox(label="PDB ID", value="2Y9X", placeholder="2Y9X",
                                           info="Código de 4 caracteres do RCSB PDB.")
                    uniprot_input = gr.Textbox(label="UniProt ID (AlphaFold DB)", value="P00533", placeholder="P00533", visible=False,
                                               info="Código de acesso do UniProt (ex.: P00533, P04150). Baixa AF-<ID>-F1-model_v4.pdb (404 -> v3).")
                    pdb_upload = gr.File(label="Envie seu arquivo .pdb", file_types=[".pdb"], visible=False)
                    btn_fetch = gr.Button("Buscar PDB", variant="primary")
                    status_pdb = gr.HTML()
                    chain_drop = gr.Dropdown(label="Cadeia proteica", choices=[], interactive=True,
                                             info="Apenas esta cadeia segue para o docking.")
                    lig_drop = gr.Dropdown(label="Ligante nativo (define a caixa e a referência de redocking)",
                                           choices=[], interactive=True,
                                           info="Opcional se você usar a caixa manual.")
                    chk_metals = gr.Checkbox(value=False, label="Preservar cofatores catalíticos (íons metálicos e cofatores como HEM, FAD, NAD)",
                                             info="Ajustado automaticamente ao buscar o PDB: marcado se a cadeia tem metais ou cofatores (Cu, Zn, Mg, Fe, HEM, FAD, NAD…).")
                    with gr.Accordion("Anotação Funcional do Alvo (Opcional)", open=False):
                        gr.Markdown("Descreve o sítio onde o docking ocorre; o texto entra na **Metodologia** e na **classificação de "
                                    "compartimentos** do relatório acadêmico. Os campos são **preenchidos automaticamente** a partir do "
                                    "cabeçalho do PDB (classe, código EC, nome da molécula e metais); edite para refinar. Se ficarem "
                                    "vazios, o relatório faz a mesma inferência ao ser gerado.")
                        target_domain = gr.Textbox(label="Domínio-alvo (target_domain)", lines=1, max_lines=2,
                                                   placeholder="ex.: domínio catalítico tipo-3 cobre   |   domínio quinase (KD)",
                                                   info="Inferido do PDB (HEADER/COMPND/EC); pode ser editado.")
                        domain_function = gr.Textbox(label="Função do domínio (domain_function)", lines=1, max_lines=2,
                                                     placeholder="ex.: oxidação sequencial de monofenóis e difenóis   |   fosforilação de substratos",
                                                     info="Inferida do PDB; pode ser editada.")
                    btn_extract = gr.Button("Fixar sítio ativo e preparar receptor", variant="primary")
                with gr.Column(scale=6, elem_classes="dd-card"):
                    gr.Markdown("**Resumo do receptor**")
                    status_rec = gr.HTML(status_box("info", "Aguardando", "Busque um PDB e fixe o sítio ativo."))
            btn_next1 = gr.Button("Continuar para o Passo 2", elem_classes="dd-go")

        # ---------------------------------------------------------------- PASSO 2
        with gr.TabItem("2 · Ligante", id="t2") as tab2:
            gr.HTML(STEP2_HTML)
            with gr.Row():
                with gr.Column(scale=6, elem_classes="dd-card"):
                    with gr.Tabs():
                        with gr.TabItem("Pesquisar no PubChem"):
                            gr.Markdown("Busque por **CID** ou **nome**. Para uma **triagem em lote**, informe vários separados por vírgula+espaço ou por "
                                        "quebra de linha (ex.: `5280343, 158637, 5280805` ou `Quercetin, Rutin, Oleuropein`).")
                            with gr.Row():
                                pubchem_input = gr.Textbox(label="CIDs ou nomes dos compostos (um ou vários)", lines=3, max_lines=8,
                                                           placeholder="5280343, 158637, 5280805", scale=3)
                                btn_pubchem = gr.Button("Baixar do PubChem", variant="secondary", scale=1)
                            status_pubchem = gr.HTML()
                        with gr.TabItem("Enviar arquivo local"):
                            gr.Markdown("Envie um ou **vários** arquivos (ex.: `0TR.mol2`, ou um `.sdf` com várias moléculas: cada uma vira um ligante da fila).")
                    user_lig_file = gr.File(label="Ligante(s) selecionado(s) (.sdf, .mol2, .pdb, .pdbqt)", file_count="multiple")
                    queue_html = gr.HTML()
                with gr.Column(scale=6, elem_classes="dd-card"):
                    chk_auto_native = gr.Checkbox(value=True, elem_id="dd-auto-native",
                                                  label="Incluir automaticamente o ligante nativo como Controle (#0 - Redocking cristalográfico)",
                                                  info="Executa o controle nativo no Modo 2 preservando as coordenadas de cristal para validação e calibração de benchmark antes da triagem dos testes.")
                    prep_mode_radio = gr.Radio(choices=PREP_MODES, value=PREP_MODES[0],
                                               label="Modo de preparo dos ligantes de TESTE",
                                               info="Modo 1 (padrão): triagem de compostos novos, com ionização por regras em pH 7.4 e geometria ETKDGv3 + MMFF94. O Modo 2 (redocking cristalográfico) é usado automaticamente no controle nativo.")
            with gr.Accordion("Guia dos modos de preparo", open=False):
                gr.HTML(MODES_HTML)
            with gr.Accordion("Entenda os termos: Gasteiger, torções (TORSDOF) e ionização", open=False):
                gr.HTML(TERMS_HTML)
            btn_next2 = gr.Button("Continuar para o Passo 3", elem_classes="dd-go")

        # ---------------------------------------------------------------- PASSO 3
        with gr.TabItem("3 · Docking GNINA 3D-CNN", id="t3"):
            gr.HTML(STEP3_HTML)
            with gr.Row():
                with gr.Column(scale=6, elem_classes="dd-card"):
                    box_mode = gr.Radio([BOX_AUTO, BOX_MANUAL], value=BOX_AUTO, label="Definição da grid box")
                    with gr.Column(visible=False) as manual_box_col:
                        with gr.Row():
                            cx = gr.Number(label="Centro X (Å)", value=-10.021)
                            cy = gr.Number(label="Centro Y (Å)", value=-28.823)
                            cz = gr.Number(label="Centro Z (Å)", value=-43.596)
                        with gr.Row():
                            sx = gr.Number(label="Tamanho X (Å)", value=20.0)
                            sy = gr.Number(label="Tamanho Y (Å)", value=20.0)
                            sz = gr.Number(label="Tamanho Z (Å)", value=20.0)
                        btn_center = gr.Button("Usar o centro do ligante nativo", variant="secondary")
                        status_center = gr.HTML()
                with gr.Column(scale=5, elem_classes="dd-card"):
                    slider_exh = gr.Slider(minimum=8, maximum=128, step=8, value=32,
                                           label="Exaustividade (esforço da amostragem)",
                                           info="Valores maiores exploram mais o espaço conformacional e demoram mais. 32 é o padrão do pipeline.")
                    with gr.Accordion("Parâmetros fixos (reprodutibilidade)", open=False):
                        gr.HTML(FIXED_PARAMS_HTML)
            btn_dock = gr.Button("Preparar ligante(s) e rodar o docking (GPU A100)", variant="primary", size="lg",
                                 elem_classes="dd-run")
            status_dock = gr.HTML()

        # ---------------------------------------------------------------- PASSO 4
        with gr.TabItem("4 · Resultados e validação", id="t4"):
            summary_html = gr.HTML(status_box("info", "Nenhum resultado ainda", "Execute o docking no Passo 3."))
            gr.Markdown("**Resultados** — um ligante: poses ordenadas por CNNscore (★ marca a pose exportada). Vários ligantes: *leaderboard* frente ao controle nativo, com um bloco recolhível por composto.")
            table_output = gr.Dataframe(interactive=False, wrap=True)
            with gr.Accordion("Glossário: o que cada coluna significa", open=True):
                gr.HTML(GLOSSARY_HTML)
            with gr.Accordion("Como ler o PoseBusters", open=False):
                gr.HTML(PB_HELP_HTML)
            gr.Markdown(f"**Downloads** — o ZIP reúne receptor, complexo, poses, ligante, planilha e logs. Com controle ou vários ligantes: <b>&lt;PDB&gt;_&lt;cadeia&gt;_LEADERBOARD_{VERSION_TAG}_&lt;data&gt;.xlsx</b> (7 abas, a 1ª pronta para o artigo; inclui <b>Parecer_Cientifico</b>) e <b>&lt;PDB&gt;_&lt;cadeia&gt;_Pacote_Completo_{VERSION_TAG}_&lt;data&gt;.zip</b> com uma subpasta por ligante e as pastas <b>06_graficos_artigo</b> (Fig. 1 fingerprint ProLIF e Fig. 2 paisagem de LE, em português e inglês: PNG 600 DPI, SVG e PDF) e <b>07_relatorio_academico</b> (Metodologia e Resultados/Discussão gerados dos dados da corrida: .docx, .pdf, .md, .tex, .txt e referencias.bib; nomes <b>&lt;PDB&gt;_&lt;cadeia&gt;_&lt;data&gt;_Relatorio_Academico_{VERSION_TAG}</b>). Os nomes levam PDB, cadeia e data/hora, então planilhas de alvos diferentes abrem juntas no Excel.")
            file_download = gr.File(label="Arquivos gerados", file_count="multiple")

    gr.HTML(FOOT_HTML)

    # ------------------------------------------------------------------ EVENTOS
    def toggle_box_mode(mode):
        return gr.update(visible=(mode == BOX_MANUAL))

    box_mode.change(fn=toggle_box_mode, inputs=[box_mode], outputs=[manual_box_col])
    def toggle_input_mode(mode, current_id):
        rcsb, local, afdb = (mode == INPUT_RCSB), (mode == INPUT_LOCAL), (mode == INPUT_AFDB)
        id_update = gr.update(visible=rcsb)
        if rcsb and str(current_id or "").startswith((LOCAL_ID_PREFIX, AF_ID_PREFIX)):
            id_update = gr.update(visible=True, value="")      # volta ao RCSB sem deixar o ID interno do upload / do AFDB
        label = "Carregar arquivo local" if local else ("Baixar do AlphaFold DB" if afdb else "Buscar PDB")
        return id_update, gr.update(visible=local), gr.update(value=label), gr.update(visible=afdb)

    input_mode.change(fn=toggle_input_mode, inputs=[input_mode, pdb_input], outputs=[pdb_input, pdb_upload, btn_fetch, uniprot_input])
    btn_fetch.click(fn=engine.load_receptor_source, concurrency_limit=1, concurrency_id="dd_engine",
                    inputs=[input_mode, pdb_input, pdb_upload, uniprot_input],
                    outputs=[chain_drop, lig_drop, status_pdb, chk_metals, pdb_input, target_domain, domain_function])
    chain_drop.change(fn=engine.update_ligands_dropdown, concurrency_limit=1, concurrency_id="dd_engine", inputs=[chain_drop, target_domain, domain_function],
                      outputs=[lig_drop, status_pdb, chk_metals, target_domain, domain_function])
    btn_extract.click(fn=engine.extract_receptor_and_native, concurrency_limit=1, concurrency_id="dd_engine",
                      inputs=[pdb_input, chain_drop, lig_drop, chk_metals],
                      outputs=[state_rec_path, state_native_path, status_rec])
    btn_pubchem.click(fn=engine.fetch_ligand_from_pubchem, concurrency_limit=1, concurrency_id="dd_engine", inputs=[pubchem_input],
                      outputs=[user_lig_file, status_pubchem, state_lig_info])
    _pq_in = [user_lig_file, state_lig_info, pubchem_input, chk_auto_native, state_native_path, prep_mode_radio]
    for _ev in (user_lig_file.change, chk_auto_native.change, prep_mode_radio.change, pubchem_input.blur, tab2.select):
        _ev(fn=preview_queue, inputs=_pq_in, outputs=[queue_html])
    btn_center.click(fn=engine.use_native_center, concurrency_limit=1, concurrency_id="dd_engine", inputs=None, outputs=[cx, cy, cz, status_center])
    btn_next1.click(fn=lambda: gr.update(selected="t2"), inputs=None, outputs=[tabs])
    btn_next2.click(fn=lambda: gr.update(selected="t3"), inputs=None, outputs=[tabs])
    btn_dock.click(
        fn=engine.run_docking, concurrency_limit=1, concurrency_id="dd_engine",
        inputs=[pdb_input, chain_drop, box_mode, state_native_path, cx, cy, cz, sx, sy, sz,
                user_lig_file, pubchem_input, state_lig_info, prep_mode_radio, slider_exh, chk_auto_native,
                target_domain, domain_function],
        outputs=[table_output, summary_html, status_dock, file_download, tabs],
    )

if __name__ == "__main__":
    # Todos os eventos do motor compartilham concurrency_id='dd_engine' (limite 1): o motor guarda estado
    # global (PDB, receptor, nativo), então só uma operação por vez. Uso recomendado: UMA cópia do notebook por aluno.
    app.queue(default_concurrency_limit=1)
    app.launch(share=True, debug=False, quiet=True, inline=False, **LAUNCH_KW)
